<a href="https://colab.research.google.com/github/tiagoeletro-bot/Ar-Condicionado-HVAC---notebookLM-/blob/main/Projeto_Alarme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
# MÓDULO 1 - IMPORTAÇÃO, CONSOLIDAÇÃO E QUALIDADE DOS DADOS
# ============================================================

# ============================================================
# 1. BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ============================================================
# 2. UPLOAD DO ARQUIVO
# ============================================================

from google.colab import files

print("=" * 80)
print("UPLOAD DA BASE DE ALARMES METASYS")
print("=" * 80)

uploaded = files.upload()

ARQUIVO = "02 - Alarmes Metasys 2026 R0.xlsx"

if not Path(ARQUIVO).exists():
    raise FileNotFoundError(
        f"\nO arquivo esperado '{ARQUIVO}' não foi localizado.\n"
        "Verifique se o nome do arquivo enviado ao Colab é exatamente esse."
    )

print(f"\nArquivo localizado com sucesso: {ARQUIVO}")


# ============================================================
# 3. DICIONÁRIOS DE REFERÊNCIA
# ============================================================

# ATENÇÃO:
# Estes dicionários serão utilizados apenas para AUDITORIA.
# Eles NÃO irão sobrescrever as classificações existentes
# nas colunas Categoria e Evento.

MAPA_CATEGORIA_INFORMADO = {
    0: "HVAC",
    1: "SDAI",
    2: "Segurança",
    3: "Serviço",
    4: "Administrativo",
    5: "Geral",
    6: "Iluminação",
    7: "Refrigeração",
    8: "Ambiente Crítico",
    9: "Qualidade do Ar",
    10: "Potência",
    11: "Energia",
    12: "Sistema",
    100: "Controle de Demanda",
    101: "Hidráulica"
}


MAPA_STATUS_INFORMADO = {
    2: "Normal",
    28: "Alarme Alto",
    18: "Alarme Baixo",
    14: "Pré Alarme Alto",
    3: "Pré Alarme Baixo",
    120: "Unreliable",
    106: "Offline",
    4: "Sistema",
    5: "Sistema",
    554: "Sistema",
    872: "SDAI",
    3267: "Sistema",
    22: "Alarme"
}


MAPA_MESES = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}


MAPA_DIA_SEMANA = {
    1: "Segunda",
    2: "Terça",
    3: "Quarta",
    4: "Quinta",
    5: "Sexta",
    6: "Sábado",
    7: "Domingo"
}


# ============================================================
# 4. INSPEÇÃO DO ARQUIVO
# ============================================================

print("\n" + "=" * 80)
print("INSPEÇÃO DAS ABAS")
print("=" * 80)

excel = pd.ExcelFile(ARQUIVO)

abas = excel.sheet_names

print(f"\nQuantidade de abas encontradas: {len(abas)}")

for aba in abas:
    print(f" - {aba}")


# ============================================================
# 5. COLUNAS ESPERADAS
# ============================================================

COLUNAS_ESPERADAS = [
    "utcCreationDateTime",
    "priority",
    "currentStatusEnumInfoId",
    "itemName",
    "itemDescription",
    "itemCategoryEnumInfoId",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",
    "previousStatusEnumInfoId"
]


# ============================================================
# 6. LEITURA E CONSOLIDAÇÃO DAS ABAS
# ============================================================

print("\n" + "=" * 80)
print("IMPORTAÇÃO E CONSOLIDAÇÃO")
print("=" * 80)

lista_dfs = []
resumo_importacao = []

for aba in abas:

    print(f"Lendo aba: {aba}")

    df_temp = pd.read_excel(
        ARQUIVO,
        sheet_name=aba
    )

    # Padronização dos nomes das colunas
    df_temp.columns = df_temp.columns.str.strip()

    colunas_faltantes = [
        col
        for col in COLUNAS_ESPERADAS
        if col not in df_temp.columns
    ]

    colunas_extras = [
        col
        for col in df_temp.columns
        if col not in COLUNAS_ESPERADAS
    ]

    resumo_importacao.append({
        "Aba": aba,
        "Registros": len(df_temp),
        "Colunas": len(df_temp.columns),
        "Colunas_Faltantes": ", ".join(colunas_faltantes),
        "Colunas_Extras": ", ".join(colunas_extras)
    })

    # Identificação da aba de origem
    df_temp["Aba_Origem"] = aba

    lista_dfs.append(df_temp)


# Consolidação
df = pd.concat(
    lista_dfs,
    ignore_index=True
)

resumo_importacao = pd.DataFrame(resumo_importacao)

print("\nImportação concluída.")
print(f"Total de registros consolidados: {len(df):,}")
print(f"Total de colunas: {df.shape[1]}")


# ============================================================
# 7. RESUMO DA IMPORTAÇÃO
# ============================================================

print("\n" + "=" * 80)
print("RESUMO DA IMPORTAÇÃO")
print("=" * 80)

display(resumo_importacao)


# ============================================================
# 8. PADRONIZAÇÃO DOS TIPOS DE DADOS
# ============================================================

print("\nPadronizando tipos de dados...")


# Data/Hora
df["utcCreationDateTime"] = pd.to_datetime(
    df["utcCreationDateTime"],
    errors="coerce"
)


# Colunas numéricas
colunas_numericas = [
    "priority",
    "currentStatusEnumInfoId",
    "itemCategoryEnumInfoId",
    "previousStatusEnumInfoId"
]

for coluna in colunas_numericas:

    df[coluna] = pd.to_numeric(
        df[coluna],
        errors="coerce"
    )


# Colunas textuais
colunas_texto = [
    "itemName",
    "itemDescription",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]

for coluna in colunas_texto:

    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .replace(
            ["", "nan", "None", "<NA>"],
            pd.NA
        )
    )

print("Tipos de dados padronizados.")


# ============================================================
# 9. CRIAÇÃO DE DATA/HORA UTC E LOCAL
# ============================================================

print("\nConvertendo horário UTC para America/Sao_Paulo...")


df["DataHoraUTC"] = df["utcCreationDateTime"]


# Converte somente se a coluna estiver sem timezone
if df["DataHoraUTC"].dt.tz is None:

    df["DataHoraLocal"] = (
        df["DataHoraUTC"]
        .dt.tz_localize(
            "UTC",
            ambiguous="NaT",
            nonexistent="NaT"
        )
        .dt.tz_convert(
            "America/Sao_Paulo"
        )
    )

else:

    df["DataHoraLocal"] = (
        df["DataHoraUTC"]
        .dt.tz_convert(
            "America/Sao_Paulo"
        )
    )


print("Conversão concluída.")


# ============================================================
# 10. CRIAÇÃO DAS VARIÁVEIS TEMPORAIS
# ============================================================

df["Data"] = (
    df["DataHoraLocal"]
    .dt.date
)

df["Ano"] = (
    df["DataHoraLocal"]
    .dt.year
)

df["Mes_Numero"] = (
    df["DataHoraLocal"]
    .dt.month
)

df["Mes"] = (
    df["Mes_Numero"]
    .map(MAPA_MESES)
)

df["Dia"] = (
    df["DataHoraLocal"]
    .dt.day
)

df["Hora"] = (
    df["DataHoraLocal"]
    .dt.hour
)

df["Minuto"] = (
    df["DataHoraLocal"]
    .dt.minute
)

df["Dia_Semana_Numero"] = (
    df["DataHoraLocal"]
    .dt.dayofweek + 1
)

df["Dia_Semana"] = (
    df["Dia_Semana_Numero"]
    .map(MAPA_DIA_SEMANA)
)


# ============================================================
# 11. CLASSIFICAÇÃO DO PERÍODO DO DIA
# ============================================================

def classificar_periodo(hora):

    if pd.isna(hora):
        return pd.NA

    if 0 <= hora < 6:
        return "Madrugada"

    elif 6 <= hora < 12:
        return "Manhã"

    elif 12 <= hora < 18:
        return "Tarde"

    else:
        return "Noite"


df["Periodo_Dia"] = (
    df["Hora"]
    .apply(classificar_periodo)
)


# ============================================================
# 12. FLAGS DE QUALIDADE
# ============================================================

df["Flag_ItemName_Vazio"] = (
    df["itemName"]
    .isna()
)

df["Flag_Descricao_Vazia"] = (
    df["itemDescription"]
    .isna()
)

df["Flag_Categoria_Vazia"] = (
    df["Categoria"]
    .isna()
)

df["Flag_Evento_Vazio"] = (
    df["Evento"]
    .isna()
)

df["Flag_Mantenedor_Vazio"] = (
    df["Mantenedor"]
    .isna()
)

df["Flag_Tipo_Vazio"] = (
    df["Tipo"]
    .isna()
)

df["Flag_Torre_Vazia"] = (
    df["Torre"]
    .isna()
)


# ============================================================
# 13. IDENTIFICAÇÃO DE POSSÍVEIS DUPLICIDADES
# ============================================================

COLUNAS_CHAVE_DUPLICIDADE = [
    "utcCreationDateTime",
    "priority",
    "currentStatusEnumInfoId",
    "itemName",
    "previousStatusEnumInfoId"
]


df["Flag_Duplicado"] = (
    df
    .duplicated(
        subset=COLUNAS_CHAVE_DUPLICIDADE,
        keep=False
    )
)


qtd_duplicados = (
    df["Flag_Duplicado"]
    .sum()
)

perc_duplicados = (
    qtd_duplicados
    / len(df)
    * 100
)

print("\n" + "=" * 80)
print("DUPLICIDADES")
print("=" * 80)

print(f"Possíveis registros duplicados: {qtd_duplicados:,}")
print(f"Percentual de possíveis duplicados: {perc_duplicados:.4f}%")


# ============================================================
# 14. AUDITORIA itemCategoryEnumInfoId × Categoria
# ============================================================

auditoria_categoria = (
    df
    .groupby(
        [
            "itemCategoryEnumInfoId",
            "Categoria"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


auditoria_categoria["Categoria_Dicionario_Informado"] = (
    auditoria_categoria[
        "itemCategoryEnumInfoId"
    ]
    .map(MAPA_CATEGORIA_INFORMADO)
)


# Começa classificando tudo como divergente
auditoria_categoria["Validacao_Dicionario"] = "Divergente"

# Código sem correspondência no dicionário informado
mascara_sem_codigo = (
    auditoria_categoria[
        "Categoria_Dicionario_Informado"
    ].isna()
)

auditoria_categoria.loc[
    mascara_sem_codigo,
    "Validacao_Dicionario"
] = "Código não existente no dicionário informado"


# Código e descrição coincidentes
mascara_coincidente = (
    auditoria_categoria["Categoria"].fillna("")
    ==
    auditoria_categoria[
        "Categoria_Dicionario_Informado"
    ].fillna("")
)

auditoria_categoria.loc[
    mascara_coincidente,
    "Validacao_Dicionario"
] = "Coincidente"


print("\n" + "=" * 80)
print("AUDITORIA DE CATEGORIAS")
print("=" * 80)

display(
    auditoria_categoria.head(100)
)


# ============================================================
# 15. AUDITORIA currentStatusEnumInfoId × Evento
# ============================================================

auditoria_status = (
    df
    .groupby(
        [
            "currentStatusEnumInfoId",
            "Evento"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


auditoria_status["Evento_Dicionario_Informado"] = (
    auditoria_status[
        "currentStatusEnumInfoId"
    ]
    .map(MAPA_STATUS_INFORMADO)
)


# Começa classificando tudo como divergente
auditoria_status["Validacao_Dicionario"] = "Divergente"

# Código sem correspondência no dicionário informado
mascara_sem_codigo = (
    auditoria_status[
        "Evento_Dicionario_Informado"
    ].isna()
)

auditoria_status.loc[
    mascara_sem_codigo,
    "Validacao_Dicionario"
] = "Código não existente no dicionário informado"


# Código e descrição coincidentes
mascara_coincidente = (
    auditoria_status["Evento"].fillna("")
    ==
    auditoria_status[
        "Evento_Dicionario_Informado"
    ].fillna("")
)

auditoria_status.loc[
    mascara_coincidente,
    "Validacao_Dicionario"
] = "Coincidente"


print("\n" + "=" * 80)
print("AUDITORIA DE STATUS / EVENTOS")
print("=" * 80)

display(
    auditoria_status.head(100)
)


# ============================================================
# 16. CONSISTÊNCIA DOS CÓDIGOS DE CATEGORIA
# ============================================================

consistencia_categoria = (
    df
    .groupby(
        "itemCategoryEnumInfoId",
        observed=True
    )["Categoria"]
    .nunique()
    .reset_index(
        name="Qtd_Categorias_Diferentes"
    )
    .sort_values(
        "Qtd_Categorias_Diferentes",
        ascending=False
    )
)


# ============================================================
# 17. CONSISTÊNCIA DOS CÓDIGOS DE EVENTO
# ============================================================

consistencia_evento = (
    df
    .groupby(
        "currentStatusEnumInfoId",
        observed=True
    )["Evento"]
    .nunique()
    .reset_index(
        name="Qtd_Eventos_Diferentes"
    )
    .sort_values(
        "Qtd_Eventos_Diferentes",
        ascending=False
    )
)


# ============================================================
# 18. DIAGNÓSTICO DAS DIMENSÕES
# ============================================================

DIMENSOES = [
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]


diagnosticos_dimensoes = {}


print("\n" + "=" * 80)
print("DIAGNÓSTICO DAS DIMENSÕES")
print("=" * 80)


for coluna in DIMENSOES:

    resultado = (
        df[coluna]
        .value_counts(
            dropna=False
        )
        .reset_index()
    )

    resultado.columns = [
        coluna,
        "Quantidade"
    ]

    resultado["Percentual"] = (
        resultado["Quantidade"]
        / len(df)
        * 100
    )

    diagnosticos_dimensoes[coluna] = resultado

    print("\n" + "-" * 80)
    print(f"COLUNA: {coluna}")
    print("-" * 80)

    display(
        resultado.head(50)
    )


# ============================================================
# 19. RELATÓRIO DE QUALIDADE GERAL
# ============================================================

qualidade = []


for coluna in df.columns:

    quantidade_nulos = (
        df[coluna]
        .isna()
        .sum()
    )

    percentual_nulos = (
        quantidade_nulos
        / len(df)
        * 100
    )

    quantidade_unicos = (
        df[coluna]
        .nunique(
            dropna=True
        )
    )

    qualidade.append({
        "Coluna": coluna,
        "Tipo_Dado": str(df[coluna].dtype),
        "Registros": len(df),
        "Nulos": quantidade_nulos,
        "Percentual_Nulos": percentual_nulos,
        "Valores_Unicos": quantidade_unicos
    })


df_qualidade = (
    pd.DataFrame(
        qualidade
    )
    .sort_values(
        "Percentual_Nulos",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("QUALIDADE GERAL DA BASE")
print("=" * 80)

display(df_qualidade)


# ============================================================
# 20. AUDITORIA DAS PRIORIDADES
# ============================================================

prioridades = (
    df["priority"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .reset_index()
)

prioridades.columns = [
    "Priority",
    "Quantidade"
]

prioridades["Percentual"] = (
    prioridades["Quantidade"]
    / len(df)
    * 100
)


print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DAS PRIORIDADES")
print("=" * 80)

display(prioridades)


# ============================================================
# 21. RELAÇÃO TORRE × TTL TORRE
# ============================================================

torre_ttl = (
    df
    .groupby(
        [
            "Torre",
            "TTL Torre"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


consistencia_torre = (
    df
    .groupby(
        "Torre",
        observed=True
    )["TTL Torre"]
    .nunique()
    .reset_index(
        name="Qtd_TTL_Torre"
    )
    .sort_values(
        "Qtd_TTL_Torre",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("RELAÇÃO TORRE × TTL TORRE")
print("=" * 80)

display(
    torre_ttl.head(100)
)


# ============================================================
# 22. RESUMO EXECUTIVO DA BASE
# ============================================================

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO DA BASE")
print("=" * 80)

print(f"Registros totais.............: {len(df):,}")

print(
    f"Período inicial..............: "
    f"{df['DataHoraLocal'].min()}"
)

print(
    f"Período final................: "
    f"{df['DataHoraLocal'].max()}"
)

print(
    f"Equipamentos únicos..........: "
    f"{df['itemName'].nunique():,}"
)

print(
    f"Descrições únicas............: "
    f"{df['itemDescription'].nunique():,}"
)

print(
    f"Categorias...................: "
    f"{df['Categoria'].nunique():,}"
)

print(
    f"Eventos......................: "
    f"{df['Evento'].nunique():,}"
)

print(
    f"Mantenedores.................: "
    f"{df['Mantenedor'].nunique():,}"
)

print(
    f"Tipos........................: "
    f"{df['Tipo'].nunique():,}"
)

print(
    f"Torres.......................: "
    f"{df['Torre'].nunique():,}"
)

print(
    f"Prioridades diferentes.......: "
    f"{df['priority'].nunique():,}"
)

print(
    f"Possíveis duplicidades.......: "
    f"{qtd_duplicados:,}"
)

print("=" * 80)


# ============================================================
# 23. ORDENAÇÃO CRONOLÓGICA
# ============================================================

print("\nOrdenando a base cronologicamente...")


df = (
    df
    .sort_values(
        [
            "itemName",
            "DataHoraUTC"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("Ordenação concluída.")


# ============================================================
# 24. OTIMIZAÇÃO DE MEMÓRIA
# ============================================================

memoria_antes = (
    df
    .memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


colunas_categoria = [
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",
    "Aba_Origem",
    "Dia_Semana",
    "Periodo_Dia",
    "Mes"
]


for coluna in colunas_categoria:

    if coluna in df.columns:

        df[coluna] = (
            df[coluna]
            .astype("category")
        )


memoria_depois = (
    df
    .memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


print("\n" + "=" * 80)
print("OTIMIZAÇÃO DE MEMÓRIA")
print("=" * 80)

print(
    f"Memória antes da otimização.: "
    f"{memoria_antes:,.2f} MB"
)

print(
    f"Memória após otimização......: "
    f"{memoria_depois:,.2f} MB"
)

print(
    f"Redução......................: "
    f"{((memoria_antes - memoria_depois) / memoria_antes) * 100:.2f}%"
)


# ============================================================
# 25. SALVAMENTO DA BASE TRATADA EM PARQUET
# ============================================================

ARQUIVO_PARQUET = (
    "Base_Alarmes_Metasys_2026_Tratada.parquet"
)


print("\nSalvando base tratada em formato Parquet...")


df.to_parquet(
    ARQUIVO_PARQUET,
    index=False
)


print(
    f"Base tratada salva com sucesso:\n"
    f"{ARQUIVO_PARQUET}"
)


# ============================================================
# 26. EXPORTAÇÃO DO RELATÓRIO DE QUALIDADE
# ============================================================

ARQUIVO_QUALIDADE = (
    "Relatorio_Qualidade_Alarmes_Metasys_2026.xlsx"
)


print("\nGerando relatório de qualidade em Excel...")


with pd.ExcelWriter(
    ARQUIVO_QUALIDADE,
    engine="openpyxl"
) as writer:

    resumo_importacao.to_excel(
        writer,
        sheet_name="Importacao",
        index=False
    )

    df_qualidade.to_excel(
        writer,
        sheet_name="Qualidade",
        index=False
    )

    auditoria_categoria.to_excel(
        writer,
        sheet_name="Auditoria Categoria",
        index=False
    )

    auditoria_status.to_excel(
        writer,
        sheet_name="Auditoria Status",
        index=False
    )

    consistencia_categoria.to_excel(
        writer,
        sheet_name="Consist Categoria",
        index=False
    )

    consistencia_evento.to_excel(
        writer,
        sheet_name="Consist Evento",
        index=False
    )

    prioridades.to_excel(
        writer,
        sheet_name="Prioridades",
        index=False
    )

    torre_ttl.to_excel(
        writer,
        sheet_name="Torre x TTL",
        index=False
    )

    consistencia_torre.to_excel(
        writer,
        sheet_name="Consist Torre",
        index=False
    )

    for nome, tabela in diagnosticos_dimensoes.items():

        nome_aba = (
            f"Diag {nome}"
            [:31]
        )

        tabela.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )


print(
    f"Relatório de qualidade criado:\n"
    f"{ARQUIVO_QUALIDADE}"
)


# ============================================================
# 27. AMOSTRA FINAL DA BASE TRATADA
# ============================================================

print("\n" + "=" * 80)
print("AMOSTRA DA BASE FINAL TRATADA")
print("=" * 80)

colunas_amostra = [
    "DataHoraUTC",
    "DataHoraLocal",
    "Mes",
    "Dia_Semana",
    "Hora",
    "Periodo_Dia",
    "priority",
    "currentStatusEnumInfoId",
    "previousStatusEnumInfoId",
    "itemName",
    "itemDescription",
    "itemCategoryEnumInfoId",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]


display(
    df[
        colunas_amostra
    ].head(20)
)


# ============================================================
# 28. DOWNLOAD DOS ARQUIVOS GERADOS
# ============================================================

print("\n" + "=" * 80)
print("PROCESSAMENTO CONCLUÍDO COM SUCESSO")
print("=" * 80)

print("\nArquivos gerados:")

print(
    f"1. {ARQUIVO_PARQUET}"
)

print(
    f"2. {ARQUIVO_QUALIDADE}"
)


print(
    "\nIniciando download do relatório de qualidade..."
)

files.download(
    ARQUIVO_QUALIDADE
)


print(
    "\nOBSERVAÇÃO:"
    "\nA base Parquet pode ser baixada executando:"
    "\nfiles.download(ARQUIVO_PARQUET)"
)


UPLOAD DA BASE DE ALARMES METASYS


Saving 02 - Alarmes Metasys 2026 R0.xlsx to 02 - Alarmes Metasys 2026 R0 (1).xlsx

Arquivo localizado com sucesso: 02 - Alarmes Metasys 2026 R0.xlsx

INSPEÇÃO DAS ABAS

Quantidade de abas encontradas: 7
 - Janeiro
 - Fevereiro
 - Março
 - Abril
 - Maio
 - Junho
 - Julho

IMPORTAÇÃO E CONSOLIDAÇÃO
Lendo aba: Janeiro
Lendo aba: Fevereiro
Lendo aba: Março
Lendo aba: Abril
Lendo aba: Maio
Lendo aba: Junho
Lendo aba: Julho

Importação concluída.
Total de registros consolidados: 1,413,882
Total de colunas: 14

RESUMO DA IMPORTAÇÃO


,Aba,Registros,Colunas,Colunas_Faltantes,Colunas_Extras
0,Janeiro,294183,13,,
1,Fevereiro,218226,13,,
2,Março,207241,13,,
3,Abril,180463,13,,
4,Maio,157589,13,,
5,Junho,174446,13,,
6,Julho,181734,13,,



Padronizando tipos de dados...
Tipos de dados padronizados.

Convertendo horário UTC para America/Sao_Paulo...
Conversão concluída.

DUPLICIDADES
Possíveis registros duplicados: 12,118
Percentual de possíveis duplicados: 0.8571%

AUDITORIA DE CATEGORIAS


,itemCategoryEnumInfoId,Categoria,Quantidade,Categoria_Dicionario_Informado,Validacao_Dicionario
2,16,HVAC,1067619,NaN,Código não existente no dicionário informado
1,12,Sistema,193061,Sistema,Coincidente
0,8,Geral,89192,Ambiente Crítico,Divergente
6,285,Hidráulica,37499,NaN,Código não existente no dicionário informado
7,387,Iluminação,24617,NaN,Código não existente no dicionário informado
5,153,Energia,1773,NaN,Código não existente no dicionário informado
4,129,SDAI,66,NaN,Código não existente no dicionário informado
3,84,Potência,40,NaN,Código não existente no dicionário informado
8,2184,Administrativo,15,NaN,Código não existente no dicionário informado


...

,DataHoraUTC,DataHoraLocal,Mes,Dia_Semana,Hora,Periodo_Dia,priority,currentStatusEnumInfoId,previousStatusEnumInfoId,itemName,itemDescription,itemCategoryEnumInfoId,Categoria,Evento,Mantenedor,Tipo,Torre,TTL Torre
0,2026-05-28 21:52:33,2026-05-28 18:52:33-03:00,Maio,Quinta,18,Noite,106,22,2,89-CGM09090 Default State,<NA>,8,Geral,Off-line,Others,Off-line,TAE,CEITOB
1,2026-01-01 09:41:46,2026-01-01 06:41:46-03:00,Janeiro,Quinta,6,Manhã,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
2,2026-01-01 10:31:48,2026-01-01 07:31:48-03:00,Janeiro,Quinta,7,Manhã,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
3,2026-01-03 08:38:33,2026-01-03 05:38:33-03:00,Janeiro,Sábado,5,Madrugada,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
4,2026-01-03 09:28:33,2026-01-03 06:28:33-03:00,Janeiro,Sábado,6,Manhã,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
5,2026-01-16 17:59:12,2026-01-16 14:59:12-03:00,Janeiro,Sexta,14,Tarde,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
6,2026-01-16 18:49:08,2026-01-16 15:49:08-03:00,Janeiro,Sexta,15,Tarde,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
7,2026-01-18 01:40:44,2026-01-17 22:40:44-03:00,Janeiro,Sábado,22,Noite,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
8,2026-01-18 02:30:40,2026-01-17 23:30:40-03:00,Janeiro,Sábado,23,Noite,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
9,2026-01-18 03:58:33,2026-01-18 00:58:33-03:00,Janeiro,Domingo,0,Madrugada,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6



PROCESSAMENTO CONCLUÍDO COM SUCESSO

Arquivos gerados:
1. Base_Alarmes_Metasys_2026_Tratada.parquet
2. Relatorio_Qualidade_Alarmes_Metasys_2026.xlsx

Iniciando download do relatório de qualidade...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


OBSERVAÇÃO:
A base Parquet pode ser baixada executando:
files.download(ARQUIVO_PARQUET)


In [6]:
# ============================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
# MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES DE ESTADO
# ============================================================

# ============================================================
# 1. BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ============================================================
# 2. ARQUIVOS
# ============================================================

ARQUIVO_ENTRADA = "Base_Alarmes_Metasys_2026_Tratada.parquet"

ARQUIVO_SAIDA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo2_Classificacao_Alarmes_Metasys.xlsx"
)


# ============================================================
# 3. VERIFICAÇÃO DO ARQUIVO
# ============================================================

print("=" * 90)
print("MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES")
print("=" * 90)

if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' não foi localizado.\n"
        "Faça o upload do arquivo Parquet gerado no Módulo 1."
    )


# ============================================================
# 4. LEITURA DA BASE
# ============================================================

print("\nCarregando base tratada do Módulo 1...")

df = pd.read_parquet(
    ARQUIVO_ENTRADA
)

print("Base carregada com sucesso.")
print(f"Registros: {len(df):,}")
print(f"Colunas iniciais: {df.shape[1]}")


# ============================================================
# 5. VERIFICAÇÃO DAS COLUNAS NECESSÁRIAS
# ============================================================

COLUNAS_NECESSARIAS = [
    "DataHoraUTC",
    "DataHoraLocal",
    "priority",
    "currentStatusEnumInfoId",
    "previousStatusEnumInfoId",
    "itemName",
    "Evento",
    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre"
]

faltantes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS
    if coluna not in df.columns
]

if faltantes:

    raise ValueError(
        "As seguintes colunas necessárias não foram encontradas:\n"
        + "\n".join(faltantes)
    )


# ============================================================
# 6. FUNÇÃO DE NORMALIZAÇÃO DE TEXTO
# ============================================================

def normalizar_texto(valor):

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto


# ============================================================
# 7. DICIONÁRIO EMPÍRICO:
#    currentStatusEnumInfoId → Evento mais frequente
# ============================================================

print("\nConstruindo dicionário empírico de códigos de estado...")


mapa_status_empirico_df = (
    df[
        [
            "currentStatusEnumInfoId",
            "Evento"
        ]
    ]
    .dropna(
        subset=[
            "currentStatusEnumInfoId",
            "Evento"
        ]
    )
    .groupby(
        [
            "currentStatusEnumInfoId",
            "Evento"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
)


# Evento predominante para cada código
mapa_status_principal = (
    mapa_status_empirico_df
    .sort_values(
        [
            "currentStatusEnumInfoId",
            "Quantidade"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop_duplicates(
        subset="currentStatusEnumInfoId",
        keep="first"
    )
)


MAPA_STATUS_EMPIRICO = dict(
    zip(
        mapa_status_principal[
            "currentStatusEnumInfoId"
        ],
        mapa_status_principal[
            "Evento"
        ].astype(str)
    )
)


print(
    f"Códigos mapeados empiricamente: "
    f"{len(MAPA_STATUS_EMPIRICO):,}"
)


# ============================================================
# 8. EVENTO ATUAL E EVENTO ANTERIOR
# ============================================================

# Evento atual da própria base
df["Evento_Atual"] = (
    df["Evento"]
    .astype("string")
)


# Evento anterior reconstruído através do código anterior
df["Evento_Anterior"] = (
    df[
        "previousStatusEnumInfoId"
    ]
    .map(
        MAPA_STATUS_EMPIRICO
    )
    .astype("string")
)


# Flag para códigos anteriores sem tradução possível
df["Flag_Evento_Anterior_Desconhecido"] = (
    df["previousStatusEnumInfoId"].notna()
    &
    df["Evento_Anterior"].isna()
)


# ============================================================
# 9. CLASSIFICAÇÃO DA FAMÍLIA DO EVENTO
# ============================================================

def classificar_familia_evento(evento):

    texto = normalizar_texto(evento)

    if texto == "":
        return "Desconhecido"

    # --------------------------------------------------------
    # NORMAL
    # --------------------------------------------------------

    if texto == "normal":
        return "Normal"

    # --------------------------------------------------------
    # OFFLINE
    # --------------------------------------------------------

    if (
        "offline" in texto
        or "off-line" in texto
    ):
        return "Offline"

    # --------------------------------------------------------
    # UNRELIABLE
    # --------------------------------------------------------

    if (
        "unreliable" in texto
        or "nao confiavel" in texto
    ):
        return "Unreliable"

    # --------------------------------------------------------
    # PRÉ-ALARME
    # Tem que vir antes de "alarme"
    # --------------------------------------------------------

    if (
        "pre alarme" in texto
        or "pre-alarme" in texto
    ):

        if (
            "alta" in texto
            or "alto" in texto
        ):
            return "Pré-Alarme Alto"

        if (
            "baixa" in texto
            or "baixo" in texto
        ):
            return "Pré-Alarme Baixo"

        return "Pré-Alarme"

    # --------------------------------------------------------
    # ALARME
    # --------------------------------------------------------

    if "alarme" in texto:

        if (
            "alta" in texto
            or "alto" in texto
        ):
            return "Alarme Alto"

        if (
            "baixa" in texto
            or "baixo" in texto
        ):
            return "Alarme Baixo"

        return "Alarme"

    # --------------------------------------------------------
    # SDAI
    # --------------------------------------------------------

    if "sdai" in texto:
        return "SDAI"

    # --------------------------------------------------------
    # SISTEMA
    # --------------------------------------------------------

    if "sistema" in texto:
        return "Sistema"

    # --------------------------------------------------------
    # DEMAIS SITUAÇÕES
    # --------------------------------------------------------

    return "Outros"


df["Familia_Evento_Atual"] = (
    df["Evento_Atual"]
    .apply(
        classificar_familia_evento
    )
)


df["Familia_Evento_Anterior"] = (
    df["Evento_Anterior"]
    .apply(
        classificar_familia_evento
    )
)


# ============================================================
# 10. GRUPOS MACRO DOS EVENTOS
# ============================================================

def classificar_grupo_macro(familia):

    if familia == "Normal":
        return "Normal"

    if familia in [
        "Alarme",
        "Alarme Alto",
        "Alarme Baixo"
    ]:
        return "Alarme"

    if familia in [
        "Pré-Alarme",
        "Pré-Alarme Alto",
        "Pré-Alarme Baixo"
    ]:
        return "Pré-Alarme"

    if familia in [
        "Offline",
        "Unreliable"
    ]:
        return "Comunicação / Confiabilidade"

    if familia == "Sistema":
        return "Sistema"

    if familia == "SDAI":
        return "SDAI"

    if familia == "Desconhecido":
        return "Desconhecido"

    return "Outros"


df["Grupo_Evento_Atual"] = (
    df["Familia_Evento_Atual"]
    .apply(
        classificar_grupo_macro
    )
)


df["Grupo_Evento_Anterior"] = (
    df["Familia_Evento_Anterior"]
    .apply(
        classificar_grupo_macro
    )
)


# ============================================================
# 11. CLASSIFICAÇÃO DA TRANSIÇÃO DE ESTADO
# ============================================================

def classificar_transicao(row):

    anterior = row["Grupo_Evento_Anterior"]
    atual = row["Grupo_Evento_Atual"]

    familia_anterior = row[
        "Familia_Evento_Anterior"
    ]

    familia_atual = row[
        "Familia_Evento_Atual"
    ]

    # --------------------------------------------------------
    # INFORMAÇÃO ANTERIOR AUSENTE
    # --------------------------------------------------------

    if anterior == "Desconhecido":

        if atual == "Alarme":
            return "Alarme sem estado anterior conhecido"

        if atual == "Pré-Alarme":
            return "Pré-Alarme sem estado anterior conhecido"

        if atual == "Comunicação / Confiabilidade":
            return "Falha comunicação sem estado anterior conhecido"

        return "Estado anterior desconhecido"

    # --------------------------------------------------------
    # ESTADO SEM ALTERAÇÃO
    # --------------------------------------------------------

    if familia_anterior == familia_atual:
        return "Estado mantido"

    # --------------------------------------------------------
    # NORMAL → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Pré-Alarme"
    ):
        return "Entrada em Pré-Alarme"

    # --------------------------------------------------------
    # NORMAL → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Alarme"
    ):
        return "Entrada em Alarme"

    # --------------------------------------------------------
    # PRÉ-ALARME → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Alarme"
    ):
        return "Escalada Pré-Alarme → Alarme"

    # --------------------------------------------------------
    # ALARME → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Pré-Alarme"
    ):
        return "Redução Alarme → Pré-Alarme"

    # --------------------------------------------------------
    # ALARME → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Normal"
    ):
        return "Normalização de Alarme"

    # --------------------------------------------------------
    # PRÉ-ALARME → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Normal"
    ):
        return "Normalização de Pré-Alarme"

    # --------------------------------------------------------
    # NORMAL → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Entrada em Falha de Comunicação"

    # --------------------------------------------------------
    # ALARME → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Alarme → Falha de Comunicação"

    # --------------------------------------------------------
    # PRÉ-ALARME → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Pré-Alarme → Falha de Comunicação"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Normal"
    ):
        return "Restabelecimento de Comunicação"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Alarme"
    ):
        return "Comunicação Restabelecida → Alarme"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Pré-Alarme"
    ):
        return "Comunicação Restabelecida → Pré-Alarme"

    # --------------------------------------------------------
    # TROCA ENTRE TIPOS DE ALARME
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Alarme"
    ):
        return "Mudança entre Alarmes"

    # --------------------------------------------------------
    # TROCA ENTRE PRÉ-ALARMES
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Pré-Alarme"
    ):
        return "Mudança entre Pré-Alarmes"

    # --------------------------------------------------------
    # EVENTOS DE SISTEMA
    # --------------------------------------------------------

    if atual == "Sistema":
        return "Evento de Sistema"

    if atual == "SDAI":
        return "Evento SDAI"

    # --------------------------------------------------------
    # OUTRAS TRANSIÇÕES
    # --------------------------------------------------------

    return (
        f"{anterior} → {atual}"
    )


print("\nClassificando transições de estado...")

df["Tipo_Transicao"] = (
    df.apply(
        classificar_transicao,
        axis=1
    )
)

print("Transições classificadas.")


# ============================================================
# 12. FLAGS OPERACIONAIS PRINCIPAIS
# ============================================================

df["Flag_Entrada_Alarme"] = (
    df["Tipo_Transicao"]
    .isin(
        [
            "Entrada em Alarme",
            "Escalada Pré-Alarme → Alarme",
            "Comunicação Restabelecida → Alarme",
            "Alarme sem estado anterior conhecido"
        ]
    )
)


df["Flag_Entrada_PreAlarme"] = (
    df["Tipo_Transicao"]
    .isin(
        [
            "Entrada em Pré-Alarme",
            "Comunicação Restabelecida → Pré-Alarme",
            "Pré-Alarme sem estado anterior conhecido"
        ]
    )
)


df["Flag_Normalizacao_Alarme"] = (
    df["Tipo_Transicao"]
    ==
    "Normalização de Alarme"
)


df["Flag_Normalizacao_PreAlarme"] = (
    df["Tipo_Transicao"]
    ==
    "Normalização de Pré-Alarme"
)


df["Flag_Falha_Comunicacao"] = (
    df["Grupo_Evento_Atual"]
    ==
    "Comunicação / Confiabilidade"
)


df["Flag_Restabelecimento_Comunicacao"] = (
    df["Tipo_Transicao"]
    ==
    "Restabelecimento de Comunicação"
)


df["Flag_Evento_Sistema"] = (
    df["Grupo_Evento_Atual"]
    ==
    "Sistema"
)


df["Flag_Evento_SDAI"] = (
    df["Grupo_Evento_Atual"]
    ==
    "SDAI"
)


# ============================================================
# 13. NATUREZA DO REGISTRO
# ============================================================

def classificar_natureza(row):

    if row["Flag_Entrada_Alarme"]:
        return "Geração de Alarme"

    if row["Flag_Entrada_PreAlarme"]:
        return "Geração de Pré-Alarme"

    if row["Flag_Normalizacao_Alarme"]:
        return "Normalização de Alarme"

    if row["Flag_Normalizacao_PreAlarme"]:
        return "Normalização de Pré-Alarme"

    if row["Flag_Restabelecimento_Comunicacao"]:
        return "Restabelecimento de Comunicação"

    if row["Flag_Falha_Comunicacao"]:
        return "Falha de Comunicação / Confiabilidade"

    if row["Flag_Evento_Sistema"]:
        return "Evento de Sistema"

    if row["Flag_Evento_SDAI"]:
        return "Evento SDAI"

    if row["Grupo_Evento_Atual"] == "Normal":
        return "Evento Normal"

    if row["Grupo_Evento_Atual"] == "Alarme":
        return "Alarme - outra transição"

    if row["Grupo_Evento_Atual"] == "Pré-Alarme":
        return "Pré-Alarme - outra transição"

    if row["Grupo_Evento_Atual"] == "Desconhecido":
        return "Não Classificado"

    return "Outro Evento"


df["Natureza_Evento"] = (
    df.apply(
        classificar_natureza,
        axis=1
    )
)


# ============================================================
# 14. FLAG DE EVENTO OPERACIONAL
# ============================================================

df["Flag_Evento_Operacional"] = (
    df["Natureza_Evento"]
    .isin(
        [
            "Geração de Alarme",
            "Geração de Pré-Alarme",
            "Normalização de Alarme",
            "Normalização de Pré-Alarme",
            "Falha de Comunicação / Confiabilidade",
            "Restabelecimento de Comunicação",
            "Alarme - outra transição",
            "Pré-Alarme - outra transição"
        ]
    )
)


# ============================================================
# 15. FLAG DE GERAÇÃO EFETIVA
# ============================================================

df["Flag_Geracao_Efetiva"] = (
    df["Natureza_Evento"]
    .isin(
        [
            "Geração de Alarme",
            "Geração de Pré-Alarme",
            "Falha de Comunicação / Confiabilidade"
        ]
    )
)


# ============================================================
# 16. DIREÇÃO DA TRANSIÇÃO
# ============================================================

def classificar_direcao(row):

    tipo = row["Tipo_Transicao"]

    if tipo in [
        "Entrada em Alarme",
        "Entrada em Pré-Alarme",
        "Escalada Pré-Alarme → Alarme",
        "Entrada em Falha de Comunicação"
    ]:
        return "Piora"

    if tipo in [
        "Normalização de Alarme",
        "Normalização de Pré-Alarme",
        "Restabelecimento de Comunicação",
        "Redução Alarme → Pré-Alarme"
    ]:
        return "Melhora"

    if tipo in [
        "Estado mantido",
        "Mudança entre Alarmes",
        "Mudança entre Pré-Alarmes"
    ]:
        return "Estável / Lateral"

    return "Indeterminado"


df["Direcao_Transicao"] = (
    df.apply(
        classificar_direcao,
        axis=1
    )
)


# ============================================================
# 17. SEVERIDADE LÓGICA DO ESTADO
# ============================================================

MAPA_SEVERIDADE = {
    "Normal": 0,
    "Pré-Alarme": 1,
    "Pré-Alarme Alto": 1,
    "Pré-Alarme Baixo": 1,
    "Alarme": 2,
    "Alarme Alto": 2,
    "Alarme Baixo": 2,
    "Offline": 3,
    "Unreliable": 3,
    "Sistema": np.nan,
    "SDAI": np.nan,
    "Outros": np.nan,
    "Desconhecido": np.nan
}


df["Nivel_Severidade_Anterior"] = (
    df["Familia_Evento_Anterior"]
    .map(
        MAPA_SEVERIDADE
    )
)


df["Nivel_Severidade_Atual"] = (
    df["Familia_Evento_Atual"]
    .map(
        MAPA_SEVERIDADE
    )
)


df["Variacao_Severidade"] = (
    df["Nivel_Severidade_Atual"]
    -
    df["Nivel_Severidade_Anterior"]
)


# ============================================================
# 18. IDENTIFICAÇÃO DE MUDANÇA REAL DE ESTADO
# ============================================================

df["Flag_Mudanca_Estado"] = (
    df["currentStatusEnumInfoId"]
    !=
    df["previousStatusEnumInfoId"]
)


# Cuida de NaNs
df["Flag_Mudanca_Estado"] = (
    df["Flag_Mudanca_Estado"]
    .fillna(False)
)


# ============================================================
# 19. RESUMO DAS FAMÍLIAS DE EVENTOS
# ============================================================

resumo_familias = (
    df["Familia_Evento_Atual"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_familias.columns = [
    "Familia_Evento",
    "Quantidade"
]

resumo_familias["Percentual"] = (
    resumo_familias["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 20. RESUMO DOS GRUPOS MACRO
# ============================================================

resumo_grupos = (
    df["Grupo_Evento_Atual"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_grupos.columns = [
    "Grupo_Evento",
    "Quantidade"
]

resumo_grupos["Percentual"] = (
    resumo_grupos["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 21. RESUMO DAS TRANSIÇÕES
# ============================================================

resumo_transicoes = (
    df["Tipo_Transicao"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_transicoes.columns = [
    "Tipo_Transicao",
    "Quantidade"
]

resumo_transicoes["Percentual"] = (
    resumo_transicoes["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 22. RESUMO DA NATUREZA DOS EVENTOS
# ============================================================

resumo_natureza = (
    df["Natureza_Evento"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_natureza.columns = [
    "Natureza_Evento",
    "Quantidade"
]

resumo_natureza["Percentual"] = (
    resumo_natureza["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 23. MATRIZ ESTADO ANTERIOR × ESTADO ATUAL
# ============================================================

matriz_transicao = pd.crosstab(
    df["Familia_Evento_Anterior"],
    df["Familia_Evento_Atual"],
    margins=True,
    margins_name="Total"
)


# ============================================================
# 24. MATRIZ GRUPO ANTERIOR × GRUPO ATUAL
# ============================================================

matriz_grupo_transicao = pd.crosstab(
    df["Grupo_Evento_Anterior"],
    df["Grupo_Evento_Atual"],
    margins=True,
    margins_name="Total"
)


# ============================================================
# 25. CÓDIGOS SEM TRADUÇÃO DO ESTADO ANTERIOR
# ============================================================

codigos_anteriores_desconhecidos = (
    df.loc[
        df["Flag_Evento_Anterior_Desconhecido"],
        "previousStatusEnumInfoId"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

codigos_anteriores_desconhecidos.columns = [
    "previousStatusEnumInfoId",
    "Quantidade"
]


# ============================================================
# 26. MAPA COMPLETO DOS CÓDIGOS EMPÍRICOS
# ============================================================

mapa_status_exportacao = (
    mapa_status_empirico_df
    .sort_values(
        [
            "currentStatusEnumInfoId",
            "Quantidade"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)


# Percentual interno por código
mapa_status_exportacao[
    "Total_Codigo"
] = (
    mapa_status_exportacao
    .groupby(
        "currentStatusEnumInfoId"
    )["Quantidade"]
    .transform("sum")
)


mapa_status_exportacao[
    "Percentual_no_Codigo"
] = (
    mapa_status_exportacao[
        "Quantidade"
    ]
    /
    mapa_status_exportacao[
        "Total_Codigo"
    ]
    *
    100
)


# ============================================================
# 27. RESUMO MENSAL
# ============================================================

resumo_mensal = (
    df
    .groupby(
        [
            "Ano",
            "Mes_Numero",
            "Mes"
        ],
        observed=True
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Normalizacoes_PreAlarme=(
            "Flag_Normalizacao_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Restabelecimentos_Comunicacao=(
            "Flag_Restabelecimento_Comunicacao",
            "sum"
        ),

        Eventos_Sistema=(
            "Flag_Evento_Sistema",
            "sum"
        ),

        Eventos_Operacionais=(
            "Flag_Evento_Operacional",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
)


# ============================================================
# 28. RESUMO POR MANTENEDOR
# ============================================================

resumo_mantenedor = (
    df
    .groupby(
        "Mantenedor",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Eventos_Sistema=(
            "Flag_Evento_Sistema",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 29. RESUMO POR CATEGORIA
# ============================================================

resumo_categoria = (
    df
    .groupby(
        "Categoria",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 30. RESUMO POR TIPO
# ============================================================

resumo_tipo = (
    df
    .groupby(
        "Tipo",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 31. RESUMO POR TORRE
# ============================================================

resumo_torre = (
    df
    .groupby(
        "Torre",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 32. RESUMO POR EQUIPAMENTO
# ============================================================

resumo_equipamento = (
    df
    .groupby(
        [
            "itemName",
            "itemDescription",
            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre"
        ],
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 33. KPIs DO MÓDULO 2
# ============================================================

total_eventos = len(df)

total_entrada_alarme = int(
    df["Flag_Entrada_Alarme"].sum()
)

total_entrada_prealarme = int(
    df["Flag_Entrada_PreAlarme"].sum()
)

total_normalizacao_alarme = int(
    df["Flag_Normalizacao_Alarme"].sum()
)

total_normalizacao_prealarme = int(
    df["Flag_Normalizacao_PreAlarme"].sum()
)

total_falha_comunicacao = int(
    df["Flag_Falha_Comunicacao"].sum()
)

total_restabelecimento = int(
    df[
        "Flag_Restabelecimento_Comunicacao"
    ].sum()
)

total_sistema = int(
    df["Flag_Evento_Sistema"].sum()
)

total_operacionais = int(
    df["Flag_Evento_Operacional"].sum()
)

total_geracoes_efetivas = int(
    df["Flag_Geracao_Efetiva"].sum()
)

eventos_anteriores_desconhecidos = int(
    df[
        "Flag_Evento_Anterior_Desconhecido"
    ].sum()
)


kpis_modulo2 = pd.DataFrame(
    {
        "Indicador": [
            "Registros totais",
            "Gerações de alarme",
            "Gerações de pré-alarme",
            "Normalizações de alarme",
            "Normalizações de pré-alarme",
            "Eventos em falha de comunicação",
            "Restabelecimentos de comunicação",
            "Eventos de sistema",
            "Eventos operacionais",
            "Gerações efetivas",
            "Estados anteriores sem tradução"
        ],

        "Quantidade": [
            total_eventos,
            total_entrada_alarme,
            total_entrada_prealarme,
            total_normalizacao_alarme,
            total_normalizacao_prealarme,
            total_falha_comunicacao,
            total_restabelecimento,
            total_sistema,
            total_operacionais,
            total_geracoes_efetivas,
            eventos_anteriores_desconhecidos
        ]
    }
)


kpis_modulo2["Percentual_Base"] = (
    kpis_modulo2["Quantidade"]
    /
    total_eventos
    *
    100
)


# ============================================================
# 34. EXIBIÇÃO DOS PRINCIPAIS RESULTADOS
# ============================================================

print("\n" + "=" * 90)
print("KPIs DO MÓDULO 2")
print("=" * 90)

display(
    kpis_modulo2
)


print("\n" + "=" * 90)
print("FAMÍLIAS DE EVENTOS")
print("=" * 90)

display(
    resumo_familias
)


print("\n" + "=" * 90)
print("TRANSIÇÕES DE ESTADO")
print("=" * 90)

display(
    resumo_transicoes.head(50)
)


print("\n" + "=" * 90)
print("NATUREZA DOS EVENTOS")
print("=" * 90)

display(
    resumo_natureza
)


print("\n" + "=" * 90)
print("RESUMO MENSAL")
print("=" * 90)

display(
    resumo_mensal
)


print("\n" + "=" * 90)
print("TOP 20 EQUIPAMENTOS POR GERAÇÕES EFETIVAS")
print("=" * 90)

display(
    resumo_equipamento.head(20)
)


# ============================================================
# 35. VALIDAÇÕES IMPORTANTES
# ============================================================

print("\n" + "=" * 90)
print("VALIDAÇÕES")
print("=" * 90)

print(
    f"Códigos de estado atuais identificados: "
    f"{df['currentStatusEnumInfoId'].nunique(dropna=True):,}"
)

print(
    f"Códigos anteriores identificados: "
    f"{df['previousStatusEnumInfoId'].nunique(dropna=True):,}"
)

print(
    f"Estados anteriores sem tradução: "
    f"{eventos_anteriores_desconhecidos:,}"
)

print(
    f"Percentual sem tradução: "
    f"{eventos_anteriores_desconhecidos / len(df) * 100:.4f}%"
)

print(
    f"Registros com mudança de estado: "
    f"{df['Flag_Mudanca_Estado'].sum():,}"
)

print(
    f"Registros sem mudança de estado: "
    f"{(~df['Flag_Mudanca_Estado']).sum():,}"
)


# ============================================================
# 36. OTIMIZAÇÃO DE MEMÓRIA
# ============================================================

colunas_category = [
    "Evento_Atual",
    "Evento_Anterior",
    "Familia_Evento_Atual",
    "Familia_Evento_Anterior",
    "Grupo_Evento_Atual",
    "Grupo_Evento_Anterior",
    "Tipo_Transicao",
    "Natureza_Evento",
    "Direcao_Transicao"
]


for coluna in colunas_category:

    if coluna in df.columns:

        df[coluna] = (
            df[coluna]
            .astype("category")
        )


# ============================================================
# 37. SALVAMENTO DA BASE CLASSIFICADA
# ============================================================

print("\nSalvando base classificada do Módulo 2...")


df.to_parquet(
    ARQUIVO_SAIDA,
    index=False
)


print(
    f"Base classificada criada:\n"
    f"{ARQUIVO_SAIDA}"
)


# ============================================================
# 38. EXPORTAÇÃO DO RELATÓRIO DO MÓDULO 2
# ============================================================

print("\nGerando relatório Excel do Módulo 2...")


with pd.ExcelWriter(
    ARQUIVO_RELATORIO,
    engine="openpyxl"
) as writer:

    kpis_modulo2.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    mapa_status_exportacao.to_excel(
        writer,
        sheet_name="Mapa Codigos Status",
        index=False
    )

    codigos_anteriores_desconhecidos.to_excel(
        writer,
        sheet_name="Codigos Sem Traducao",
        index=False
    )

    resumo_familias.to_excel(
        writer,
        sheet_name="Familias Eventos",
        index=False
    )

    resumo_grupos.to_excel(
        writer,
        sheet_name="Grupos Eventos",
        index=False
    )

    resumo_transicoes.to_excel(
        writer,
        sheet_name="Transicoes",
        index=False
    )

    resumo_natureza.to_excel(
        writer,
        sheet_name="Natureza Eventos",
        index=False
    )

    matriz_transicao.to_excel(
        writer,
        sheet_name="Matriz Estados"
    )

    matriz_grupo_transicao.to_excel(
        writer,
        sheet_name="Matriz Grupos"
    )

    resumo_mensal.to_excel(
        writer,
        sheet_name="Resumo Mensal",
        index=False
    )

    resumo_mantenedor.to_excel(
        writer,
        sheet_name="Mantenedor",
        index=False
    )

    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )

    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )

    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )

    resumo_equipamento.to_excel(
        writer,
        sheet_name="Equipamentos",
        index=False
    )


print(
    f"Relatório criado:\n"
    f"{ARQUIVO_RELATORIO}"
)


# ============================================================
# 39. AMOSTRA DA BASE CLASSIFICADA
# ============================================================

COLUNAS_AMOSTRA = [
    "DataHoraLocal",
    "priority",

    "previousStatusEnumInfoId",
    "currentStatusEnumInfoId",

    "Evento_Anterior",
    "Evento_Atual",

    "Familia_Evento_Anterior",
    "Familia_Evento_Atual",

    "Grupo_Evento_Anterior",
    "Grupo_Evento_Atual",

    "Tipo_Transicao",
    "Direcao_Transicao",
    "Natureza_Evento",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao",

    "itemName",
    "itemDescription",
    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre"
]


print("\n" + "=" * 90)
print("AMOSTRA DA BASE CLASSIFICADA")
print("=" * 90)

display(
    df[
        COLUNAS_AMOSTRA
    ].head(30)
)


# ============================================================
# 40. FINALIZAÇÃO
# ============================================================

print("\n" + "=" * 90)
print("MÓDULO 2 CONCLUÍDO COM SUCESSO")
print("=" * 90)

print("\nArquivos gerados:")

print(
    f"1. {ARQUIVO_SAIDA}"
)

print(
    f"2. {ARQUIVO_RELATORIO}"
)


# ============================================================
# 41. DOWNLOAD DO RELATÓRIO
# ============================================================

from google.colab import files

print(
    "\nIniciando download do relatório Excel..."
)

files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar também a base classificada em Parquet,"
    "\nexecute posteriormente:"
)

print(
    f'files.download("{ARQUIVO_SAIDA}")'
)

MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES

Carregando base tratada do Módulo 1...
Base carregada com sucesso.
Registros: 1,413,882
Colunas iniciais: 34

Construindo dicionário empírico de códigos de estado...
Códigos mapeados empiricamente: 12

Classificando transições de estado...
Transições classificadas.

KPIs DO MÓDULO 2


,Indicador,Quantidade,Percentual_Base
0,Registros totais,1413882,100.00
1,Gerações de alarme,240277,16.99
2,Gerações de pré-alarme,296725,20.99
3,Normalizações de alarme,266494,18.85
4,Normalizações de pré-alarme,263966,18.67
5,Eventos em falha de comunicação,100564,7.11
6,Restabelecimentos de comunicação,8074,0.57
7,Eventos de sistema,7171,0.51
8,Eventos operacionais,1320425,93.39
9,Gerações efetivas,637566,45.09



FAMÍLIAS DE EVENTOS


,Familia_Evento,Quantidade,Percentual
0,Normal,624812,44.19
1,Pré-Alarme Baixo,196477,13.90
2,Pré-Alarme Alto,137370,9.72
3,Alarme,136278,9.64
4,Alarme Alto,116421,8.23
5,Alarme Baixo,94781,6.70
6,Offline,85864,6.07
7,Unreliable,14700,1.04
8,Sistema,7171,0.51
9,SDAI,8,0.00



TRANSIÇÕES DE ESTADO


...

,DataHoraLocal,priority,previousStatusEnumInfoId,currentStatusEnumInfoId,Evento_Anterior,Evento_Atual,Familia_Evento_Anterior,Familia_Evento_Atual,Grupo_Evento_Anterior,Grupo_Evento_Atual,Tipo_Transicao,Direcao_Transicao,Natureza_Evento,Flag_Entrada_Alarme,Flag_Entrada_PreAlarme,Flag_Normalizacao_Alarme,Flag_Normalizacao_PreAlarme,Flag_Falha_Comunicacao,Flag_Restabelecimento_Comunicacao,itemName,itemDescription,Categoria,Mantenedor,Tipo,Torre
0,2026-05-28 18:52:33-03:00,106,2,22,Normal,Off-line,Normal,Offline,Normal,Comunicação / Confiabilidade,Entrada em Falha de Comunicação,Piora,Falha de Comunicação / Confiabilidade,False,False,False,False,True,False,89-CGM09090 Default State,<NA>,Geral,Others,Off-line,TAE
1,2026-01-01 06:41:46-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
2,2026-01-01 07:31:48-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
3,2026-01-03 05:38:33-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
4,2026-01-03 06:28:33-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
5,2026-01-16 14:59:12-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
6,2026-01-16 15:49:08-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
7,2026-01-17 22:40:44-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
8,2026-01-17 23:30:40-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
9,2026-01-18 00:58:33-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6



MÓDULO 2 CONCLUÍDO COM SUCESSO

Arquivos gerados:
1. Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
2. Relatorio_Modulo2_Classificacao_Alarmes_Metasys.xlsx

Iniciando download do relatório Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também a base classificada em Parquet,
execute posteriormente:
files.download("Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet")


In [8]:
# ================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 3
# ANÁLISE DE VOLUMETRIA DOS EVENTOS
#
# Base de entrada:
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Objetivos:
# - Análise mensal
# - Análise diária
# - Análise horária
# - Dia da semana
# - Período do dia
# - Prioridade
# - Categoria
# - Mantenedor
# - Tipo
# - Torre
# - TTL Torre
# - Matrizes cruzadas
# - Rankings preliminares
# - Identificação de dias e horas de maior volumetria
# - Comparação Eventos Totais × Eventos Operacionais ×
#   Gerações Efetivas
# ================================================================


# ================================================================
# 1. BIBLIOTECAS
# ================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    200
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    250
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ================================================================
# 2. CONFIGURAÇÃO DOS ARQUIVOS
# ================================================================

ARQUIVO_ENTRADA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo3_Volumetria_Alarmes_Metasys.xlsx"
)


print("=" * 100)
print("MÓDULO 3 - ANÁLISE DE VOLUMETRIA DOS ALARMES METASYS")
print("=" * 100)


# ================================================================
# 3. VERIFICAÇÃO / UPLOAD DO ARQUIVO
# ================================================================

if not Path(ARQUIVO_ENTRADA).exists():

    print(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "não foi localizado no ambiente atual."
    )

    print(
        "\nSelecione o arquivo Parquet gerado no Módulo 2."
    )

    from google.colab import files

    uploaded = files.upload()


if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "continua não disponível."
    )


# ================================================================
# 4. CARREGAMENTO DA BASE
# ================================================================

print("\nCarregando base do Módulo 2...")


df = pd.read_parquet(
    ARQUIVO_ENTRADA
)


print("Base carregada com sucesso.")

print(
    f"Registros: {len(df):,}"
)

print(
    f"Colunas: {df.shape[1]}"
)


# ================================================================
# 5. VERIFICAÇÃO DAS COLUNAS NECESSÁRIAS
# ================================================================

COLUNAS_NECESSARIAS = [

    "DataHoraLocal",

    "Ano",
    "Mes_Numero",
    "Mes",

    "Data",
    "Hora",

    "Dia_Semana_Numero",
    "Dia_Semana",
    "Periodo_Dia",

    "priority",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Familia_Evento_Atual",
    "Grupo_Evento_Atual",
    "Natureza_Evento",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao",

    "Flag_Evento_Sistema",

    "Flag_Evento_Operacional",
    "Flag_Geracao_Efetiva"
]


colunas_faltantes = [

    coluna

    for coluna
    in COLUNAS_NECESSARIAS

    if coluna not in df.columns
]


if colunas_faltantes:

    raise ValueError(

        "\nAs seguintes colunas necessárias "
        "não foram encontradas:\n\n"

        + "\n".join(
            colunas_faltantes
        )
    )


# ================================================================
# 6. AJUSTES DE TIPOS
# ================================================================

df["DataHoraLocal"] = pd.to_datetime(
    df["DataHoraLocal"],
    errors="coerce"
)


# Cria uma data pandas sem timezone para agrupamento e Excel
df["Data_Analise"] = (
    df["DataHoraLocal"]
    .dt.tz_localize(None)
    .dt.normalize()
)


df["AnoMes"] = (
    df["DataHoraLocal"]
    .dt.strftime("%Y-%m")
)


# Hora cheia
df["DataHora_Hora"] = (
    df["DataHoraLocal"]
    .dt.tz_localize(None)
    .dt.floor("h")
)


# ================================================================
# 7. ORDEM DOS MESES / DIAS
# ================================================================

ORDEM_MESES = [

    "Janeiro",
    "Fevereiro",
    "Março",
    "Abril",
    "Maio",
    "Junho",
    "Julho",
    "Agosto",
    "Setembro",
    "Outubro",
    "Novembro",
    "Dezembro"
]


ORDEM_DIAS = [

    "Segunda",
    "Terça",
    "Quarta",
    "Quinta",
    "Sexta",
    "Sábado",
    "Domingo"
]


ORDEM_PERIODOS = [

    "Madrugada",
    "Manhã",
    "Tarde",
    "Noite"
]


# ================================================================
# 8. FUNÇÃO BASE DE AGREGAÇÃO
# ================================================================

def gerar_resumo(df_base, dimensoes):

    """
    Gera os principais indicadores de volumetria
    para uma ou mais dimensões.
    """

    resultado = (

        df_base
        .groupby(
            dimensoes,
            observed=True,
            dropna=False
        )
        .agg(

            Eventos_Totais=(
                "itemName",
                "size"
            ),

            Eventos_Operacionais=(
                "Flag_Evento_Operacional",
                "sum"
            ),

            Geracoes_Efetivas=(
                "Flag_Geracao_Efetiva",
                "sum"
            ),

            Geracoes_Alarme=(
                "Flag_Entrada_Alarme",
                "sum"
            ),

            Geracoes_PreAlarme=(
                "Flag_Entrada_PreAlarme",
                "sum"
            ),

            Falhas_Comunicacao=(
                "Flag_Falha_Comunicacao",
                "sum"
            ),

            Normalizacoes_Alarme=(
                "Flag_Normalizacao_Alarme",
                "sum"
            ),

            Normalizacoes_PreAlarme=(
                "Flag_Normalizacao_PreAlarme",
                "sum"
            ),

            Restabelecimentos_Comunicacao=(
                "Flag_Restabelecimento_Comunicacao",
                "sum"
            ),

            Eventos_Sistema=(
                "Flag_Evento_Sistema",
                "sum"
            ),

            Equipamentos_Unicos=(
                "itemName",
                "nunique"
            )
        )

        .reset_index()
    )


    # ------------------------------------------------------------
    # Percentual de gerações efetivas sobre eventos totais
    # ------------------------------------------------------------

    resultado[
        "Perc_Geracoes_Efetivas"
    ] = np.where(

        resultado[
            "Eventos_Totais"
        ] > 0,

        (
            resultado[
                "Geracoes_Efetivas"
            ]
            /
            resultado[
                "Eventos_Totais"
            ]
            *
            100
        ),

        0
    )


    # ------------------------------------------------------------
    # Percentual operacional
    # ------------------------------------------------------------

    resultado[
        "Perc_Eventos_Operacionais"
    ] = np.where(

        resultado[
            "Eventos_Totais"
        ] > 0,

        (
            resultado[
                "Eventos_Operacionais"
            ]
            /
            resultado[
                "Eventos_Totais"
            ]
            *
            100
        ),

        0
    )


    return resultado


# ================================================================
# 9. KPIs GERAIS
# ================================================================

TOTAL_EVENTOS = len(df)

TOTAL_OPERACIONAIS = int(
    df[
        "Flag_Evento_Operacional"
    ].sum()
)

TOTAL_GERACOES = int(
    df[
        "Flag_Geracao_Efetiva"
    ].sum()
)

TOTAL_ALARMES = int(
    df[
        "Flag_Entrada_Alarme"
    ].sum()
)

TOTAL_PREALARMES = int(
    df[
        "Flag_Entrada_PreAlarme"
    ].sum()
)

TOTAL_COMUNICACAO = int(
    df[
        "Flag_Falha_Comunicacao"
    ].sum()
)

TOTAL_NORMALIZACOES_ALARME = int(
    df[
        "Flag_Normalizacao_Alarme"
    ].sum()
)

TOTAL_NORMALIZACOES_PREALARME = int(
    df[
        "Flag_Normalizacao_PreAlarme"
    ].sum()
)

TOTAL_EQUIPAMENTOS = (
    df[
        "itemName"
    ].nunique()
)

TOTAL_DIAS = (
    df[
        "Data_Analise"
    ].nunique()
)


kpis_gerais = pd.DataFrame({

    "Indicador": [

        "Eventos totais",
        "Eventos operacionais",
        "Gerações efetivas",

        "Gerações de alarme",
        "Gerações de pré-alarme",
        "Falhas de comunicação",

        "Normalizações de alarme",
        "Normalizações de pré-alarme",

        "Equipamentos únicos",
        "Dias analisados"
    ],

    "Quantidade": [

        TOTAL_EVENTOS,
        TOTAL_OPERACIONAIS,
        TOTAL_GERACOES,

        TOTAL_ALARMES,
        TOTAL_PREALARMES,
        TOTAL_COMUNICACAO,

        TOTAL_NORMALIZACOES_ALARME,
        TOTAL_NORMALIZACOES_PREALARME,

        TOTAL_EQUIPAMENTOS,
        TOTAL_DIAS
    ]

})


# Percentual somente para indicadores que representam registros
kpis_gerais["Percentual_Base"] = np.nan


mascara_percentual = (

    kpis_gerais[
        "Indicador"
    ].isin([

        "Eventos totais",
        "Eventos operacionais",
        "Gerações efetivas",

        "Gerações de alarme",
        "Gerações de pré-alarme",
        "Falhas de comunicação",

        "Normalizações de alarme",
        "Normalizações de pré-alarme"
    ])
)


kpis_gerais.loc[
    mascara_percentual,
    "Percentual_Base"
] = (

    kpis_gerais.loc[
        mascara_percentual,
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 10. MÉDIAS GERAIS
# ================================================================

media_eventos_dia = (
    TOTAL_EVENTOS
    /
    TOTAL_DIAS
)

media_geracoes_dia = (
    TOTAL_GERACOES
    /
    TOTAL_DIAS
)

media_alarm_dia = (
    TOTAL_ALARMES
    /
    TOTAL_DIAS
)

media_prealarme_dia = (
    TOTAL_PREALARMES
    /
    TOTAL_DIAS
)


medias_gerais = pd.DataFrame({

    "Indicador": [

        "Média de eventos totais por dia",
        "Média de gerações efetivas por dia",
        "Média de gerações de alarme por dia",
        "Média de gerações de pré-alarme por dia"
    ],

    "Valor": [

        media_eventos_dia,
        media_geracoes_dia,
        media_alarm_dia,
        media_prealarme_dia
    ]

})


# ================================================================
# 11. VOLUMETRIA MENSAL
# ================================================================

resumo_mensal = gerar_resumo(

    df,

    [
        "Ano",
        "Mes_Numero",
        "Mes"
    ]
)


resumo_mensal = (

    resumo_mensal
    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
    .reset_index(
        drop=True
    )
)


# Número de dias existentes em cada mês
dias_mes = (

    df
    .groupby(
        [
            "Ano",
            "Mes_Numero"
        ],
        observed=True
    )[
        "Data_Analise"
    ]
    .nunique()
    .reset_index(
        name="Dias_Analisados"
    )
)


resumo_mensal = resumo_mensal.merge(

    dias_mes,

    on=[
        "Ano",
        "Mes_Numero"
    ],

    how="left"
)


# Médias por dia
resumo_mensal[
    "Media_Eventos_Dia"
] = (

    resumo_mensal[
        "Eventos_Totais"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_Geracoes_Dia"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_Alarmes_Dia"
] = (

    resumo_mensal[
        "Geracoes_Alarme"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_PreAlarmes_Dia"
] = (

    resumo_mensal[
        "Geracoes_PreAlarme"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


# ================================================================
# 12. VARIAÇÃO MÊS CONTRA MÊS
# ================================================================

resumo_mensal[
    "Var_MoM_Eventos_Perc"
] = (

    resumo_mensal[
        "Eventos_Totais"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_Geracoes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_Alarmes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Alarme"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_PreAlarmes_Perc"
] = (

    resumo_mensal[
        "Geracoes_PreAlarme"
    ]
    .pct_change()
    *
    100
)


# ================================================================
# 13. VARIAÇÃO CONTRA O PRIMEIRO MÊS
# ================================================================

if len(resumo_mensal) > 0:

    base_eventos = (
        resumo_mensal.iloc[0][
            "Eventos_Totais"
        ]
    )

    base_geracoes = (
        resumo_mensal.iloc[0][
            "Geracoes_Efetivas"
        ]
    )

    base_alarm = (
        resumo_mensal.iloc[0][
            "Geracoes_Alarme"
        ]
    )

    base_prealarme = (
        resumo_mensal.iloc[0][
            "Geracoes_PreAlarme"
        ]
    )


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Eventos_Perc"
    ] = (

        (
            resumo_mensal[
                "Eventos_Totais"
            ]
            /
            base_eventos
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Geracoes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_Efetivas"
            ]
            /
            base_geracoes
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Alarmes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_Alarme"
            ]
            /
            base_alarm
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_PreAlarmes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_PreAlarme"
            ]
            /
            base_prealarme
        )
        -
        1
    ) * 100


# ================================================================
# 14. PARTICIPAÇÃO MENSAL
# ================================================================

resumo_mensal[
    "Participacao_Geracoes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 15. VOLUMETRIA DIÁRIA
# ================================================================

resumo_diario = gerar_resumo(

    df,

    [
        "Data_Analise"
    ]
)


resumo_diario = (

    resumo_diario
    .sort_values(
        "Data_Analise"
    )
    .reset_index(
        drop=True
    )
)


# Acrescenta informações temporais
resumo_diario[
    "Ano"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.year
)


resumo_diario[
    "Mes_Numero"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.month
)


resumo_diario[
    "Mes"
] = (
    resumo_diario[
        "Mes_Numero"
    ]
    .map({

        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    })
)


resumo_diario[
    "Dia_Semana_Numero"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.dayofweek
    +
    1
)


resumo_diario[
    "Dia_Semana"
] = (
    resumo_diario[
        "Dia_Semana_Numero"
    ]
    .map({

        1: "Segunda",
        2: "Terça",
        3: "Quarta",
        4: "Quinta",
        5: "Sexta",
        6: "Sábado",
        7: "Domingo"
    })
)


# ================================================================
# 16. TOP DIAS POR GERAÇÕES EFETIVAS
# ================================================================

top_dias = (

    resumo_diario
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .head(50)
    .reset_index(
        drop=True
    )
)


# ================================================================
# 17. VOLUMETRIA HORÁRIA - HORA DO DIA
# ================================================================

resumo_hora = gerar_resumo(

    df,

    [
        "Hora"
    ]
)


resumo_hora = (

    resumo_hora
    .sort_values(
        "Hora"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 18. VOLUMETRIA POR HORA CRONOLÓGICA
# ================================================================

resumo_datahora = gerar_resumo(

    df,

    [
        "DataHora_Hora"
    ]
)


resumo_datahora = (

    resumo_datahora
    .sort_values(
        "DataHora_Hora"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 19. TOP HORAS DE MAIOR VOLUMETRIA
# ================================================================

top_horas = (

    resumo_datahora
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .head(100)
    .reset_index(
        drop=True
    )
)


# ================================================================
# 20. DIA DA SEMANA
# ================================================================

resumo_dia_semana = gerar_resumo(

    df,

    [
        "Dia_Semana_Numero",
        "Dia_Semana"
    ]
)


resumo_dia_semana = (

    resumo_dia_semana
    .sort_values(
        "Dia_Semana_Numero"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 21. MÉDIA POR OCORRÊNCIA DO DIA DA SEMANA
# ================================================================

dias_semana_contagem = (

    df[
        [
            "Data_Analise",
            "Dia_Semana_Numero"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "Dia_Semana_Numero"
    )
    .size()
    .reset_index(
        name="Qtd_Dias"
    )
)


resumo_dia_semana = resumo_dia_semana.merge(

    dias_semana_contagem,

    on="Dia_Semana_Numero",

    how="left"
)


resumo_dia_semana[
    "Media_Geracoes_por_Dia"
] = (

    resumo_dia_semana[
        "Geracoes_Efetivas"
    ]
    /
    resumo_dia_semana[
        "Qtd_Dias"
    ]
)


resumo_dia_semana[
    "Media_Alarmes_por_Dia"
] = (

    resumo_dia_semana[
        "Geracoes_Alarme"
    ]
    /
    resumo_dia_semana[
        "Qtd_Dias"
    ]
)


# ================================================================
# 22. PERÍODO DO DIA
# ================================================================

resumo_periodo = gerar_resumo(

    df,

    [
        "Periodo_Dia"
    ]
)


ordem_periodo_map = {

    "Madrugada": 1,
    "Manhã": 2,
    "Tarde": 3,
    "Noite": 4
}


resumo_periodo[
    "Ordem"
] = (

    resumo_periodo[
        "Periodo_Dia"
    ]
    .astype(str)
    .map(
        ordem_periodo_map
    )
)


resumo_periodo = (

    resumo_periodo
    .sort_values(
        "Ordem"
    )
    .drop(
        columns="Ordem"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 23. PRIORIDADE
# ================================================================

resumo_prioridade = gerar_resumo(

    df,

    [
        "priority"
    ]
)


resumo_prioridade = (

    resumo_prioridade
    .sort_values(
        "priority",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


# Participação das gerações efetivas
resumo_prioridade[
    "Participacao_Geracoes_Perc"
] = (

    resumo_prioridade[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 24. CATEGORIA
# ================================================================

resumo_categoria = gerar_resumo(

    df,

    [
        "Categoria"
    ]
)


resumo_categoria = (

    resumo_categoria
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_categoria[
    "Participacao_Geracoes_Perc"
] = (

    resumo_categoria[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_categoria[
    "Participacao_Acumulada_Perc"
] = (

    resumo_categoria[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# ================================================================
# 25. MANTENEDOR
# ================================================================

resumo_mantenedor = gerar_resumo(

    df,

    [
        "Mantenedor"
    ]
)


resumo_mantenedor = (

    resumo_mantenedor
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_mantenedor[
    "Participacao_Geracoes_Perc"
] = (

    resumo_mantenedor[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 26. TIPO
# ================================================================

resumo_tipo = gerar_resumo(

    df,

    [
        "Tipo"
    ]
)


resumo_tipo = (

    resumo_tipo
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_tipo[
    "Participacao_Geracoes_Perc"
] = (

    resumo_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_tipo[
    "Participacao_Acumulada_Perc"
] = (

    resumo_tipo[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# ================================================================
# 27. TORRE
# ================================================================

resumo_torre = gerar_resumo(

    df,

    [
        "Torre"
    ]
)


resumo_torre = (

    resumo_torre
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_torre[
    "Participacao_Geracoes_Perc"
] = (

    resumo_torre[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 28. TTL TORRE
# ================================================================

resumo_ttl_torre = gerar_resumo(

    df,

    [
        "TTL Torre"
    ]
)


resumo_ttl_torre = (

    resumo_ttl_torre
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 29. MATRIZ MANTENEDOR × TIPO
# ================================================================

matriz_mantenedor_tipo = pd.pivot_table(

    df,

    index="Mantenedor",

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 30. MATRIZ CATEGORIA × TIPO
# ================================================================

matriz_categoria_tipo = pd.pivot_table(

    df,

    index="Categoria",

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 31. MATRIZ TORRE × CATEGORIA
# ================================================================

matriz_torre_categoria = pd.pivot_table(

    df,

    index="Torre",

    columns="Categoria",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 32. MATRIZ MÊS × MANTENEDOR
# ================================================================

matriz_mes_mantenedor = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Mantenedor",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 33. MATRIZ MÊS × CATEGORIA
# ================================================================

matriz_mes_categoria = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Categoria",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 34. MATRIZ MÊS × TIPO
# ================================================================

matriz_mes_tipo = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 35. MATRIZ DIA DA SEMANA × HORA
# ================================================================

matriz_dia_hora = pd.pivot_table(

    df,

    index=[
        "Dia_Semana_Numero",
        "Dia_Semana"
    ],

    columns="Hora",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 36. MATRIZ MÊS × HORA
# ================================================================

matriz_mes_hora = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Hora",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 37. RANKING PRELIMINAR DOS EQUIPAMENTOS
# ================================================================

resumo_equipamentos = (

    df
    .groupby(

        [
            "itemName",
            "itemDescription",
            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre",
            "TTL Torre"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Eventos_Operacionais=(
            "Flag_Evento_Operacional",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Normalizacoes_PreAlarme=(
            "Flag_Normalizacao_PreAlarme",
            "sum"
        ),

        Prioridade_Minima=(
            "priority",
            "min"
        ),

        Prioridade_Mediana=(
            "priority",
            "median"
        ),

        Prioridade_Media=(
            "priority",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


resumo_equipamentos[
    "Participacao_Geracoes_Perc"
] = (

    resumo_equipamentos[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_equipamentos[
    "Participacao_Acumulada_Perc"
] = (

    resumo_equipamentos[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# Top 100 preliminar
top100_equipamentos = (

    resumo_equipamentos
    .head(100)
    .copy()
)


# ================================================================
# 38. RANKING POR MÊS E EQUIPAMENTO
# ================================================================

ranking_mensal_equip = (

    df
    .groupby(

        [
            "Ano",
            "Mes_Numero",
            "Mes",

            "itemName",
            "itemDescription",

            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()
)


ranking_mensal_equip = (

    ranking_mensal_equip[

        ranking_mensal_equip[
            "Geracoes_Efetivas"
        ] > 0

    ]

    .sort_values(

        [
            "Ano",
            "Mes_Numero",
            "Geracoes_Efetivas"
        ],

        ascending=[
            True,
            True,
            False
        ]
    )
)


ranking_mensal_equip[
    "Ranking_Mes"
] = (

    ranking_mensal_equip
    .groupby(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
    .cumcount()
    +
    1
)


top20_mensal = (

    ranking_mensal_equip[
        ranking_mensal_equip[
            "Ranking_Mes"
        ] <= 20
    ]
    .copy()
)


# ================================================================
# 39. VOLUMETRIA POR NATUREZA DO EVENTO
# ================================================================

resumo_natureza = (

    df[
        "Natureza_Evento"
    ]

    .value_counts(
        dropna=False
    )

    .reset_index()
)


resumo_natureza.columns = [

    "Natureza_Evento",
    "Quantidade"
]


resumo_natureza[
    "Percentual"
] = (

    resumo_natureza[
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 40. VOLUMETRIA POR FAMÍLIA
# ================================================================

resumo_familia = (

    df[
        "Familia_Evento_Atual"
    ]

    .value_counts(
        dropna=False
    )

    .reset_index()
)


resumo_familia.columns = [

    "Familia_Evento",
    "Quantidade"
]


resumo_familia[
    "Percentual"
] = (

    resumo_familia[
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 41. TOP COMBINAÇÕES CATEGORIA × TIPO
# ================================================================

ranking_categoria_tipo = (

    df
    .groupby(

        [
            "Categoria",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_categoria_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_categoria_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 42. TOP COMBINAÇÕES MANTENEDOR × TIPO
# ================================================================

ranking_mantenedor_tipo = (

    df
    .groupby(

        [
            "Mantenedor",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_mantenedor_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_mantenedor_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 43. TOP COMBINAÇÕES TORRE × TIPO
# ================================================================

ranking_torre_tipo = (

    df
    .groupby(

        [
            "Torre",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_torre_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_torre_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 44. RESUMO DOS PONTOS PRINCIPAIS
# ================================================================

print("\n" + "=" * 100)
print("KPIs GERAIS")
print("=" * 100)

display(
    kpis_gerais
)


print("\n" + "=" * 100)
print("MÉDIAS GERAIS")
print("=" * 100)

display(
    medias_gerais
)


print("\n" + "=" * 100)
print("VOLUMETRIA MENSAL")
print("=" * 100)

display(
    resumo_mensal
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR CATEGORIA")
print("=" * 100)

display(
    resumo_categoria
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR MANTENEDOR")
print("=" * 100)

display(
    resumo_mantenedor
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR TIPO")
print("=" * 100)

display(
    resumo_tipo.head(30)
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR TORRE")
print("=" * 100)

display(
    resumo_torre
)


print("\n" + "=" * 100)
print("TOP 20 EQUIPAMENTOS")
print("=" * 100)

display(
    resumo_equipamentos.head(20)
)


print("\n" + "=" * 100)
print("TOP 20 DIAS")
print("=" * 100)

display(
    top_dias.head(20)
)


print("\n" + "=" * 100)
print("TOP 20 HORAS")
print("=" * 100)

display(
    top_horas.head(20)
)


# ================================================================
# 45. CRIAÇÃO DE RESUMO EXECUTIVO
# ================================================================

mes_maior_geracao = (
    resumo_mensal
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .iloc[0]
)

mes_menor_geracao = (
    resumo_mensal
    .sort_values(
        "Geracoes_Efetivas",
        ascending=True
    )
    .iloc[0]
)

dia_maior_geracao = top_dias.iloc[0]

hora_maior_geracao = top_horas.iloc[0]

categoria_maior = resumo_categoria.iloc[0]

mantenedor_maior = resumo_mantenedor.iloc[0]

tipo_maior = resumo_tipo.iloc[0]

torre_maior = resumo_torre.iloc[0]

equip_maior = resumo_equipamentos.iloc[0]


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [
            "Total de registros",
            "Total de gerações efetivas",
            "% gerações efetivas / registros",
            "Média de gerações por dia",
            "Mês com maior geração",
            "Quantidade no mês de maior geração",
            "Mês com menor geração",
            "Quantidade no mês de menor geração",
            "Dia com maior geração",
            "Quantidade no dia de maior geração",
            "Hora cronológica de maior geração",
            "Quantidade na hora de maior geração",
            "Categoria com maior geração",
            "Mantenedor com maior geração",
            "Tipo com maior geração",
            "Torre com maior geração",
            "Equipamento com maior geração",
            "Gerações do equipamento líder"
        ],

        "Resultado": [
            f"{TOTAL_EVENTOS:,}",
            f"{TOTAL_GERACOES:,}",
            f"{TOTAL_GERACOES / TOTAL_EVENTOS * 100:.2f}%",
            f"{media_geracoes_dia:,.2f}",

            str(
                mes_maior_geracao["Mes"]
            ),

            f"{int(mes_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                mes_menor_geracao["Mes"]
            ),

            f"{int(mes_menor_geracao['Geracoes_Efetivas']):,}",

            str(
                dia_maior_geracao["Data_Analise"].date()
            ),

            f"{int(dia_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                hora_maior_geracao["DataHora_Hora"]
            ),

            f"{int(hora_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                categoria_maior["Categoria"]
            ),

            str(
                mantenedor_maior["Mantenedor"]
            ),

            str(
                tipo_maior["Tipo"]
            ),

            str(
                torre_maior["Torre"]
            ),

            str(
                equip_maior["itemName"]
            ),

            f"{int(equip_maior['Geracoes_Efetivas']):,}"
        ]
    }
)


# ================================================================
# 46. EXPORTAÇÃO PARA EXCEL
# ================================================================

print("\nGerando relatório Excel do Módulo 3...")


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(

        writer,

        sheet_name="Resumo Executivo",

        index=False
    )


    kpis_gerais.to_excel(

        writer,

        sheet_name="KPIs Gerais",

        index=False
    )


    medias_gerais.to_excel(

        writer,

        sheet_name="Medias Gerais",

        index=False
    )


    resumo_mensal.to_excel(

        writer,

        sheet_name="Mensal",

        index=False
    )


    resumo_diario.to_excel(

        writer,

        sheet_name="Diario",

        index=False
    )


    top_dias.to_excel(

        writer,

        sheet_name="Top Dias",

        index=False
    )


    resumo_hora.to_excel(

        writer,

        sheet_name="Hora Dia",

        index=False
    )


    resumo_datahora.to_excel(

        writer,

        sheet_name="Hora Cronologica",

        index=False
    )


    top_horas.to_excel(

        writer,

        sheet_name="Top Horas",

        index=False
    )


    resumo_dia_semana.to_excel(

        writer,

        sheet_name="Dia Semana",

        index=False
    )


    resumo_periodo.to_excel(

        writer,

        sheet_name="Periodo Dia",

        index=False
    )


    resumo_prioridade.to_excel(

        writer,

        sheet_name="Prioridades",

        index=False
    )


    resumo_categoria.to_excel(

        writer,

        sheet_name="Categorias",

        index=False
    )


    resumo_mantenedor.to_excel(

        writer,

        sheet_name="Mantenedores",

        index=False
    )


    resumo_tipo.to_excel(

        writer,

        sheet_name="Tipos",

        index=False
    )


    resumo_torre.to_excel(

        writer,

        sheet_name="Torres",

        index=False
    )


    resumo_ttl_torre.to_excel(

        writer,

        sheet_name="TTL Torre",

        index=False
    )


    resumo_natureza.to_excel(

        writer,

        sheet_name="Natureza Eventos",

        index=False
    )


    resumo_familia.to_excel(

        writer,

        sheet_name="Familia Eventos",

        index=False
    )


    top100_equipamentos.to_excel(

        writer,

        sheet_name="Top100 Equipamentos",

        index=False
    )


    top20_mensal.to_excel(

        writer,

        sheet_name="Top20 Mensal",

        index=False
    )


    ranking_categoria_tipo.to_excel(

        writer,

        sheet_name="Categoria x Tipo",

        index=False
    )


    ranking_mantenedor_tipo.to_excel(

        writer,

        sheet_name="Mantenedor x Tipo",

        index=False
    )


    ranking_torre_tipo.to_excel(

        writer,

        sheet_name="Torre x Tipo",

        index=False
    )


    matriz_mantenedor_tipo.to_excel(

        writer,

        sheet_name="Matriz Mant x Tipo"
    )


    matriz_categoria_tipo.to_excel(

        writer,

        sheet_name="Matriz Cat x Tipo"
    )


    matriz_torre_categoria.to_excel(

        writer,

        sheet_name="Matriz Torre x Cat"
    )


    matriz_mes_mantenedor.to_excel(

        writer,

        sheet_name="Mes x Mantenedor"
    )


    matriz_mes_categoria.to_excel(

        writer,

        sheet_name="Mes x Categoria"
    )


    matriz_mes_tipo.to_excel(

        writer,

        sheet_name="Mes x Tipo"
    )


    matriz_dia_hora.to_excel(

        writer,

        sheet_name="Dia Semana x Hora"
    )


    matriz_mes_hora.to_excel(

        writer,

        sheet_name="Mes x Hora"
    )


# ================================================================
# 47. FORMATAÇÃO BÁSICA DO EXCEL
# ================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)

from openpyxl.utils import get_column_letter


wb = load_workbook(
    ARQUIVO_RELATORIO
)


thin_border = Border(

    bottom=Side(
        style="thin"
    )
)


for ws in wb.worksheets:

    # Congela cabeçalho
    ws.freeze_panes = "A2"

    # Filtro automático
    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # Cabeçalho
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

        cell.border = thin_border


    # Ajuste de largura
    for column_cells in ws.columns:

        max_length = 0

        coluna = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                if len(valor) > max_length:
                    max_length = len(valor)

            except:
                pass


        largura = min(
            max(
                max_length + 2,
                12
            ),
            45
        )

        ws.column_dimensions[
            coluna
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ================================================================
# 48. VALIDAÇÃO FINAL
# ================================================================

print("\n" + "=" * 100)
print("VALIDAÇÃO DOS TOTAIS")
print("=" * 100)


print(
    f"Eventos totais na base................: "
    f"{TOTAL_EVENTOS:,}"
)


print(
    f"Gerações efetivas na base.............: "
    f"{TOTAL_GERACOES:,}"
)


print(
    f"Soma das gerações no resumo mensal....: "
    f"{int(resumo_mensal['Geracoes_Efetivas'].sum()):,}"
)


print(
    f"Soma das gerações no resumo diário....: "
    f"{int(resumo_diario['Geracoes_Efetivas'].sum()):,}"
)


print(
    f"Soma das gerações por hora do dia.....: "
    f"{int(resumo_hora['Geracoes_Efetivas'].sum()):,}"
)


# Verificação
validacao_mensal = (
    int(
        resumo_mensal[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


validacao_diaria = (
    int(
        resumo_diario[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


validacao_hora = (
    int(
        resumo_hora[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


print("\nChecagens:")

print(
    f"Resumo mensal consistente..............: "
    f"{validacao_mensal}"
)

print(
    f"Resumo diário consistente..............: "
    f"{validacao_diaria}"
)

print(
    f"Resumo horário consistente.............: "
    f"{validacao_hora}"
)


# ================================================================
# 49. FINALIZAÇÃO
# ================================================================

print("\n" + "=" * 100)
print("MÓDULO 3 CONCLUÍDO COM SUCESSO")
print("=" * 100)


print(
    f"\nRelatório criado:\n"
    f"{ARQUIVO_RELATORIO}"
)


print(
    "\nPrincipais análises disponíveis:"
)

print(
    """
1. Volumetria mensal
2. Volumetria diária
3. Volumetria horária
4. Dias da semana
5. Períodos do dia
6. Prioridades
7. Categorias
8. Mantenedores
9. Tipos
10. Torres
11. TTL Torre
12. Top dias
13. Top horas
14. Top equipamentos
15. Ranking mensal de equipamentos
16. Categoria × Tipo
17. Mantenedor × Tipo
18. Torre × Tipo
19. Dia da Semana × Hora
20. Mês × Hora
"""
)


# ================================================================
# 50. DOWNLOAD DO RELATÓRIO
# ================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)

MÓDULO 3 - ANÁLISE DE VOLUMETRIA DOS ALARMES METASYS

Carregando base do Módulo 2...
Base carregada com sucesso.
Registros: 1,413,882
Colunas: 58

KPIs GERAIS


,Indicador,Quantidade,Percentual_Base
0,Eventos totais,1413882,100.00
1,Eventos operacionais,1320425,93.39
2,Gerações efetivas,637566,45.09
3,Gerações de alarme,240277,16.99
4,Gerações de pré-alarme,296725,20.99
5,Falhas de comunicação,100564,7.11
6,Normalizações de alarme,266494,18.85
7,Normalizações de pré-alarme,263966,18.67
8,Equipamentos únicos,8254,NaN
9,Dias analisados,211,NaN



MÉDIAS GERAIS


,Indicador,Valor
0,Média de eventos totais por dia,"6,700.86"
1,Média de gerações efetivas por dia,"3,021.64"
2,Média de gerações de alarme por dia,"1,138.75"
3,Média de gerações de pré-alarme por dia,"1,406.28"



VOLUMETRIA MENSAL


...


TOP 20 HORAS


,DataHora_Hora,Eventos_Totais,Eventos_Operacionais,Geracoes_Efetivas,Geracoes_Alarme,Geracoes_PreAlarme,Falhas_Comunicacao,Normalizacoes_Alarme,Normalizacoes_PreAlarme,Restabelecimentos_Comunicacao,Eventos_Sistema,Equipamentos_Unicos,Perc_Geracoes_Efetivas,Perc_Eventos_Operacionais
0,2026-01-17 22:00:00,5499,5430,2736,143,10,2583,2577,17,11,13,966,49.75,98.75
1,2026-06-29 06:00:00,4118,3784,1770,222,156,1392,1433,108,73,15,1591,42.98,91.89
2,2026-02-17 17:00:00,3491,3395,1630,220,72,1338,1510,101,35,3,1038,46.69,97.25
3,2026-01-18 01:00:00,2956,2931,1455,91,27,1337,1401,21,4,14,888,49.22,99.15
4,2026-01-22 17:00:00,3005,2869,1417,283,193,941,1083,181,32,15,1455,47.15,95.47
5,2026-01-24 20:00:00,2798,2746,1363,47,11,1305,1302,21,1,10,753,48.71,98.14
6,2026-01-27 15:00:00,3635,3115,1318,261,216,841,970,185,46,13,1871,36.26,85.69
7,2026-01-19 17:00:00,3349,2898,1242,215,162,865,976,139,33,14,1668,37.09,86.53
8,2026-02-16 17:00:00,2962,2676,1223,203,111,909,1019,86,23,16,1373,41.29,90.34
9,2026-06-29 07:00:00,3008,2627,1129,182,119,828,934,114,29,15,1530,37.53,87.33



Gerando relatório Excel do Módulo 3...

VALIDAÇÃO DOS TOTAIS
Eventos totais na base................: 1,413,882
Gerações efetivas na base.............: 637,566
Soma das gerações no resumo mensal....: 637,566
Soma das gerações no resumo diário....: 637,566
Soma das gerações por hora do dia.....: 637,566

Checagens:
Resumo mensal consistente..............: True
Resumo diário consistente..............: True
Resumo horário consistente.............: True

MÓDULO 3 CONCLUÍDO COM SUCESSO

Relatório criado:
Relatorio_Modulo3_Volumetria_Alarmes_Metasys.xlsx

Principais análises disponíveis:

1. Volumetria mensal
2. Volumetria diária
3. Volumetria horária
4. Dias da semana
5. Períodos do dia
6. Prioridades
7. Categorias
8. Mantenedores
9. Tipos
10. Torres
11. TTL Torre
12. Top dias
13. Top horas
14. Top equipamentos
15. Ranking mensal de equipamentos
16. Categoria × Tipo
17. Mantenedor × Tipo
18. Torre × Tipo
19. Dia da Semana × Hora
20. Mês × Hora


Iniciando download do relatório Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# =====================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 4
# PARETO E ÍNDICE ANALÍTICO DE CRITICIDADE
#
# Base:
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# OBJETIVOS
# ---------------------------------------------------------------------
# 1. Pareto por equipamento
# 2. Pareto por Categoria
# 3. Pareto por Mantenedor
# 4. Pareto por Tipo
# 5. Pareto por Torre
# 6. Concentração Top N
# 7. Identificação do grupo responsável por 80%, 90% e 95%
# 8. Construção de Índice Analítico de Criticidade
#
# COMPONENTES DO ÍNDICE
# ---------------------------------------------------------------------
# 40% Frequência
# 25% Prioridade
# 20% Severidade da composição
# 15% Persistência mensal
#
# IMPORTANTE:
# Quanto MENOR o valor de priority, MAIOR a prioridade.
# =====================================================================


# =====================================================================
# 1. BIBLIOTECAS
# =====================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    200
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    250
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# =====================================================================
# 2. ARQUIVOS
# =====================================================================

ARQUIVO_ENTRADA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo4_Pareto_Criticidade_Metasys.xlsx"
)

ARQUIVO_RANKING = (
    "Ranking_Criticidade_Alarmes_Metasys_2026.parquet"
)


print("=" * 105)
print("MÓDULO 4 - PARETO E ÍNDICE ANALÍTICO DE CRITICIDADE")
print("=" * 105)


# =====================================================================
# 3. VERIFICAÇÃO / UPLOAD
# =====================================================================

if not Path(ARQUIVO_ENTRADA).exists():

    print(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "não foi encontrado."
    )

    print(
        "\nSelecione o arquivo Parquet "
        "gerado no Módulo 2."
    )

    from google.colab import files

    uploaded = files.upload()


if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "continua indisponível."
    )


# =====================================================================
# 4. CARREGAMENTO
# =====================================================================

print("\nCarregando base classificada...")


df = pd.read_parquet(
    ARQUIVO_ENTRADA
)


print("Base carregada.")

print(
    f"Registros totais: {len(df):,}"
)

print(
    f"Colunas: {df.shape[1]}"
)


# =====================================================================
# 5. COLUNAS NECESSÁRIAS
# =====================================================================

COLUNAS_NECESSARIAS = [

    "DataHoraLocal",
    "Mes_Numero",

    "priority",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Flag_Geracao_Efetiva",
    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",
    "Flag_Falha_Comunicacao"
]


faltantes = [

    coluna

    for coluna
    in COLUNAS_NECESSARIAS

    if coluna not in df.columns
]


if faltantes:

    raise ValueError(

        "Colunas necessárias ausentes:\n\n"

        + "\n".join(
            faltantes
        )
    )


# =====================================================================
# 6. BASE APENAS DE GERAÇÕES EFETIVAS
# =====================================================================

df_geracoes = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()
)


TOTAL_GERACOES = len(
    df_geracoes
)


TOTAL_EQUIPAMENTOS_GERADORES = (

    df_geracoes[
        "itemName"
    ]
    .nunique()
)


TOTAL_MESES = (

    df[
        "Mes_Numero"
    ]
    .nunique()
)


print("\n" + "=" * 105)
print("BASE DE GERAÇÕES EFETIVAS")
print("=" * 105)

print(
    f"Gerações efetivas..............: "
    f"{TOTAL_GERACOES:,}"
)

print(
    f"Equipamentos com geração.......: "
    f"{TOTAL_EQUIPAMENTOS_GERADORES:,}"
)

print(
    f"Meses analisados...............: "
    f"{TOTAL_MESES}"
)


# =====================================================================
# 7. FUNÇÃO DE PARETO
# =====================================================================

def gerar_pareto(
    df_base,
    coluna,
    nome_coluna=None
):

    if nome_coluna is None:
        nome_coluna = coluna

    resultado = (

        df_base
        .groupby(
            coluna,
            observed=True,
            dropna=False
        )
        .size()
        .reset_index(
            name="Geracoes_Efetivas"
        )
        .sort_values(
            "Geracoes_Efetivas",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    resultado[
        "Participacao_Perc"
    ] = (

        resultado[
            "Geracoes_Efetivas"
        ]
        /
        resultado[
            "Geracoes_Efetivas"
        ].sum()
        *
        100
    )


    resultado[
        "Participacao_Acumulada_Perc"
    ] = (

        resultado[
            "Participacao_Perc"
        ]
        .cumsum()
    )


    resultado[
        "Ranking"
    ] = (

        np.arange(
            1,
            len(resultado) + 1
        )
    )


    # -------------------------------------------------------------
    # Classe de Pareto
    #
    # A = até aproximadamente 80%
    # B = 80% até 95%
    # C = acima de 95%
    # -------------------------------------------------------------

    resultado[
        "Classe_Pareto"
    ] = np.select(

        [
            resultado[
                "Participacao_Acumulada_Perc"
            ] <= 80,

            resultado[
                "Participacao_Acumulada_Perc"
            ] <= 95
        ],

        [
            "A",
            "B"
        ],

        default="C"
    )


    return resultado


# =====================================================================
# 8. PARETOS DAS PRINCIPAIS DIMENSÕES
# =====================================================================

pareto_categoria = gerar_pareto(
    df_geracoes,
    "Categoria"
)

pareto_mantenedor = gerar_pareto(
    df_geracoes,
    "Mantenedor"
)

pareto_tipo = gerar_pareto(
    df_geracoes,
    "Tipo"
)

pareto_torre = gerar_pareto(
    df_geracoes,
    "Torre"
)

pareto_ttl_torre = gerar_pareto(
    df_geracoes,
    "TTL Torre"
)


# =====================================================================
# 9. RANKING DETALHADO POR EQUIPAMENTO
# =====================================================================

ranking_equipamentos = (

    df_geracoes
    .groupby(

        [
            "itemName",
            "itemDescription",
            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre",
            "TTL Torre"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Prioridade_Minima=(
            "priority",
            "min"
        ),

        Prioridade_Mediana=(
            "priority",
            "median"
        ),

        Prioridade_Media=(
            "priority",
            "mean"
        ),

        Meses_Com_Geracao=(
            "Mes_Numero",
            "nunique"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


# =====================================================================
# 10. PARTICIPAÇÃO E PARETO POR EQUIPAMENTO
# =====================================================================

ranking_equipamentos[
    "Participacao_Geracoes_Perc"
] = (

    ranking_equipamentos[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


ranking_equipamentos[
    "Participacao_Acumulada_Perc"
] = (

    ranking_equipamentos[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


ranking_equipamentos[
    "Ranking_Frequencia"
] = (

    np.arange(
        1,
        len(ranking_equipamentos) + 1
    )
)


ranking_equipamentos[
    "Classe_Pareto"
] = np.select(

    [
        ranking_equipamentos[
            "Participacao_Acumulada_Perc"
        ] <= 80,

        ranking_equipamentos[
            "Participacao_Acumulada_Perc"
        ] <= 95
    ],

    [
        "A",
        "B"
    ],

    default="C"
)


# =====================================================================
# 11. COMPONENTE 1:
#     SCORE DE FREQUÊNCIA
#
# Maior quantidade de gerações = maior score
# Percentil empírico 0-100
# =====================================================================

ranking_equipamentos[
    "Score_Frequencia"
] = (

    ranking_equipamentos[
        "Geracoes_Efetivas"
    ]
    .rank(
        pct=True,
        method="average"
    )
    *
    100
)


# =====================================================================
# 12. COMPONENTE 2:
#     SCORE DE PRIORIDADE
#
# Menor número = maior prioridade
#
# Usaremos a prioridade mediana das GERAÇÕES EFETIVAS,
# não dos eventos de normalização.
# =====================================================================

# Caso existam prioridades ausentes
mediana_global_prioridade = (

    ranking_equipamentos[
        "Prioridade_Mediana"
    ]
    .median()
)


ranking_equipamentos[
    "Prioridade_Mediana_Ajustada"
] = (

    ranking_equipamentos[
        "Prioridade_Mediana"
    ]
    .fillna(
        mediana_global_prioridade
    )
)


# ascending=False:
# valores menores recebem percentil maior.
ranking_equipamentos[
    "Score_Prioridade"
] = (

    ranking_equipamentos[
        "Prioridade_Mediana_Ajustada"
    ]
    .rank(
        pct=True,
        ascending=False,
        method="average"
    )
    *
    100
)


# =====================================================================
# 13. COMPONENTE 3:
#     SCORE DE SEVERIDADE DA COMPOSIÇÃO
#
# Pesos analíticos:
#
# Alarme                 = 100
# Falha de Comunicação   = 80
# Pré-Alarme             = 60
#
# Não representa classificação oficial do Metasys.
# É um índice analítico criado para este projeto.
# =====================================================================

ranking_equipamentos[
    "Score_Severidade"
] = (

    (
        ranking_equipamentos[
            "Geracoes_Alarme"
        ]
        * 100
    )

    +

    (
        ranking_equipamentos[
            "Falhas_Comunicacao"
        ]
        * 80
    )

    +

    (
        ranking_equipamentos[
            "Geracoes_PreAlarme"
        ]
        * 60
    )

) / (

    ranking_equipamentos[
        "Geracoes_Efetivas"
    ]
)


# Segurança
ranking_equipamentos[
    "Score_Severidade"
] = (

    ranking_equipamentos[
        "Score_Severidade"
    ]
    .fillna(0)
    .clip(
        lower=0,
        upper=100
    )
)


# =====================================================================
# 14. COMPONENTE 4:
#     SCORE DE PERSISTÊNCIA
#
# Quantos meses o ponto apresentou gerações efetivas.
# =====================================================================

ranking_equipamentos[
    "Score_Persistencia"
] = (

    ranking_equipamentos[
        "Meses_Com_Geracao"
    ]
    /
    TOTAL_MESES
    *
    100
)


# =====================================================================
# 15. ÍNDICE ANALÍTICO DE CRITICIDADE V1
#
# 40% Frequência
# 25% Prioridade
# 20% Severidade
# 15% Persistência
# =====================================================================

PESO_FREQUENCIA = 0.40

PESO_PRIORIDADE = 0.25

PESO_SEVERIDADE = 0.20

PESO_PERSISTENCIA = 0.15


ranking_equipamentos[
    "Indice_Criticidade_V1"
] = (

    ranking_equipamentos[
        "Score_Frequencia"
    ]
    *
    PESO_FREQUENCIA

    +

    ranking_equipamentos[
        "Score_Prioridade"
    ]
    *
    PESO_PRIORIDADE

    +

    ranking_equipamentos[
        "Score_Severidade"
    ]
    *
    PESO_SEVERIDADE

    +

    ranking_equipamentos[
        "Score_Persistencia"
    ]
    *
    PESO_PERSISTENCIA
)


# =====================================================================
# 16. CLASSIFICAÇÃO DO ÍNDICE
# =====================================================================

ranking_equipamentos[
    "Classe_Criticidade"
] = np.select(

    [
        ranking_equipamentos[
            "Indice_Criticidade_V1"
        ] >= 80,

        ranking_equipamentos[
            "Indice_Criticidade_V1"
        ] >= 60,

        ranking_equipamentos[
            "Indice_Criticidade_V1"
        ] >= 40
    ],

    [
        "Crítica",
        "Alta",
        "Moderada"
    ],

    default="Baixa"
)


# =====================================================================
# 17. RANKING FINAL POR CRITICIDADE
# =====================================================================

ranking_criticidade = (

    ranking_equipamentos
    .sort_values(

        [
            "Indice_Criticidade_V1",
            "Geracoes_Efetivas"
        ],

        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


ranking_criticidade[
    "Ranking_Criticidade"
] = (

    np.arange(
        1,
        len(ranking_criticidade) + 1
    )
)


# =====================================================================
# 18. REORDENAÇÃO DAS COLUNAS
# =====================================================================

COLUNAS_RANKING = [

    "Ranking_Criticidade",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Geracoes_Efetivas",
    "Geracoes_Alarme",
    "Geracoes_PreAlarme",
    "Falhas_Comunicacao",

    "Participacao_Geracoes_Perc",
    "Participacao_Acumulada_Perc",

    "Classe_Pareto",

    "Prioridade_Minima",
    "Prioridade_Mediana",
    "Prioridade_Media",

    "Meses_Com_Geracao",

    "Score_Frequencia",
    "Score_Prioridade",
    "Score_Severidade",
    "Score_Persistencia",

    "Indice_Criticidade_V1",
    "Classe_Criticidade"
]


ranking_criticidade = (

    ranking_criticidade[
        COLUNAS_RANKING
    ]
)


# =====================================================================
# 19. CONCENTRAÇÃO TOP N
# =====================================================================

top_ns = [

    1,
    5,
    10,
    20,
    50,
    100,
    250,
    500,
    1000
]


concentracao_topn = []


for n in top_ns:

    n_real = min(
        n,
        len(ranking_equipamentos)
    )

    qtd = (

        ranking_equipamentos
        .head(
            n_real
        )[
            "Geracoes_Efetivas"
        ]
        .sum()
    )

    percentual = (

        qtd
        /
        TOTAL_GERACOES
        *
        100
    )


    concentracao_topn.append({

        "Top_N": n_real,

        "Geracoes_Efetivas": int(
            qtd
        ),

        "Percentual_Total": percentual
    })


concentracao_topn = pd.DataFrame(
    concentracao_topn
)


# =====================================================================
# 20. QUANTIDADE DE EQUIPAMENTOS NECESSÁRIA PARA
#     EXPLICAR 50%, 80%, 90% E 95%
# =====================================================================

limites_pareto = [

    50,
    80,
    90,
    95
]


resultado_limites = []


for limite in limites_pareto:

    mascara = (

        ranking_equipamentos[
            "Participacao_Acumulada_Perc"
        ]
        >= limite
    )


    if mascara.any():

        indice = (
            mascara.idxmax()
        )

        quantidade_equipamentos = (
            indice + 1
        )

        percentual_real = (

            ranking_equipamentos
            .loc[
                indice,
                "Participacao_Acumulada_Perc"
            ]
        )

        geracoes = (

            ranking_equipamentos
            .iloc[
                :quantidade_equipamentos
            ][
                "Geracoes_Efetivas"
            ]
            .sum()
        )


        resultado_limites.append({

            "Limite_Pareto_Perc": limite,

            "Quantidade_Equipamentos": (
                quantidade_equipamentos
            ),

            "Percentual_Equipamentos": (
                quantidade_equipamentos
                /
                len(ranking_equipamentos)
                *
                100
            ),

            "Geracoes_Efetivas": int(
                geracoes
            ),

            "Participacao_Real_Perc": (
                percentual_real
            )
        })


resumo_limites_pareto = pd.DataFrame(
    resultado_limites
)


# =====================================================================
# 21. QUANTIDADE POR CLASSE DE PARETO
# =====================================================================

resumo_classes_pareto = (

    ranking_equipamentos
    .groupby(
        "Classe_Pareto",
        observed=True
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        )
    )
    .reset_index()
)


resumo_classes_pareto[
    "Percentual_Equipamentos"
] = (

    resumo_classes_pareto[
        "Equipamentos"
    ]
    /
    len(ranking_equipamentos)
    *
    100
)


resumo_classes_pareto[
    "Percentual_Geracoes"
] = (

    resumo_classes_pareto[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# =====================================================================
# 22. DISTRIBUIÇÃO DAS CLASSES DE CRITICIDADE
# =====================================================================

resumo_criticidade = (

    ranking_criticidade
    .groupby(
        "Classe_Criticidade",
        observed=True
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Media_Indice=(
            "Indice_Criticidade_V1",
            "mean"
        ),

        Mediana_Indice=(
            "Indice_Criticidade_V1",
            "median"
        )
    )
    .reset_index()
)


resumo_criticidade[
    "Participacao_Geracoes_Perc"
] = (

    resumo_criticidade[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# =====================================================================
# 23. TOP 100 POR CRITICIDADE
# =====================================================================

top100_criticidade = (

    ranking_criticidade
    .head(100)
    .copy()
)


# =====================================================================
# 24. TOP 100 POR FREQUÊNCIA
# =====================================================================

top100_frequencia = (

    ranking_equipamentos
    .head(100)
    .copy()
)


# =====================================================================
# 25. COMPARAÇÃO:
#     RANKING DE FREQUÊNCIA × RANKING DE CRITICIDADE
# =====================================================================

ranking_freq_aux = (

    ranking_equipamentos[
        [
            "itemName",
            "Ranking_Frequencia"
        ]
    ]
    .copy()
)


comparativo_rankings = (

    ranking_criticidade
    .merge(

        ranking_freq_aux,

        on="itemName",

        how="left"
    )
)


comparativo_rankings[
    "Variacao_Posicao"
] = (

    comparativo_rankings[
        "Ranking_Frequencia"
    ]
    -
    comparativo_rankings[
        "Ranking_Criticidade"
    ]
)


comparativo_rankings = (

    comparativo_rankings
    .sort_values(
        "Ranking_Criticidade"
    )
)


# =====================================================================
# 26. PARETO MENSAL POR EQUIPAMENTO
# =====================================================================

pareto_mensal_equip = (

    df_geracoes
    .groupby(

        [
            "Mes_Numero",
            "Mes",

            "itemName",
            "itemDescription",

            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre"
        ],

        observed=True,

        dropna=False
    )

    .size()

    .reset_index(
        name="Geracoes_Efetivas"
    )
)


# Total mensal
pareto_mensal_equip[
    "Total_Mes"
] = (

    pareto_mensal_equip
    .groupby(
        "Mes_Numero"
    )[
        "Geracoes_Efetivas"
    ]
    .transform("sum")
)


pareto_mensal_equip[
    "Participacao_Mes_Perc"
] = (

    pareto_mensal_equip[
        "Geracoes_Efetivas"
    ]
    /
    pareto_mensal_equip[
        "Total_Mes"
    ]
    *
    100
)


pareto_mensal_equip = (

    pareto_mensal_equip
    .sort_values(

        [
            "Mes_Numero",
            "Geracoes_Efetivas"
        ],

        ascending=[
            True,
            False
        ]
    )
)


pareto_mensal_equip[
    "Participacao_Acumulada_Mes_Perc"
] = (

    pareto_mensal_equip
    .groupby(
        "Mes_Numero"
    )[
        "Participacao_Mes_Perc"
    ]
    .cumsum()
)


pareto_mensal_equip[
    "Ranking_Mes"
] = (

    pareto_mensal_equip
    .groupby(
        "Mes_Numero"
    )
    .cumcount()
    +
    1
)


pareto_mensal_equip[
    "Classe_Pareto_Mensal"
] = np.select(

    [
        pareto_mensal_equip[
            "Participacao_Acumulada_Mes_Perc"
        ] <= 80,

        pareto_mensal_equip[
            "Participacao_Acumulada_Mes_Perc"
        ] <= 95
    ],

    [
        "A",
        "B"
    ],

    default="C"
)


# =====================================================================
# 27. TOP 20 DE CADA MÊS
# =====================================================================

top20_mensal = (

    pareto_mensal_equip.loc[

        pareto_mensal_equip[
            "Ranking_Mes"
        ] <= 20

    ]

    .copy()
)


# =====================================================================
# 28. CRITICIDADE POR CATEGORIA
# =====================================================================

criticidade_categoria = (

    ranking_criticidade
    .groupby(
        "Categoria",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Criticidade_V1",
            "mean"
        ),

        Indice_Maximo=(
            "Indice_Criticidade_V1",
            "max"
        ),

        Equipamentos_Criticos=(
            "Classe_Criticidade",
            lambda x: (
                x == "Crítica"
            ).sum()
        ),

        Equipamentos_Alta=(
            "Classe_Criticidade",
            lambda x: (
                x == "Alta"
            ).sum()
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# =====================================================================
# 29. CRITICIDADE POR MANTENEDOR
# =====================================================================

criticidade_mantenedor = (

    ranking_criticidade
    .groupby(
        "Mantenedor",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Criticidade_V1",
            "mean"
        ),

        Indice_Maximo=(
            "Indice_Criticidade_V1",
            "max"
        ),

        Equipamentos_Criticos=(
            "Classe_Criticidade",
            lambda x: (
                x == "Crítica"
            ).sum()
        ),

        Equipamentos_Alta=(
            "Classe_Criticidade",
            lambda x: (
                x == "Alta"
            ).sum()
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# =====================================================================
# 30. CRITICIDADE POR TIPO
# =====================================================================

criticidade_tipo = (

    ranking_criticidade
    .groupby(
        "Tipo",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Criticidade_V1",
            "mean"
        ),

        Indice_Maximo=(
            "Indice_Criticidade_V1",
            "max"
        ),

        Equipamentos_Criticos=(
            "Classe_Criticidade",
            lambda x: (
                x == "Crítica"
            ).sum()
        ),

        Equipamentos_Alta=(
            "Classe_Criticidade",
            lambda x: (
                x == "Alta"
            ).sum()
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# =====================================================================
# 31. CRITICIDADE POR TORRE
# =====================================================================

criticidade_torre = (

    ranking_criticidade
    .groupby(
        "Torre",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Criticidade_V1",
            "mean"
        ),

        Indice_Maximo=(
            "Indice_Criticidade_V1",
            "max"
        ),

        Equipamentos_Criticos=(
            "Classe_Criticidade",
            lambda x: (
                x == "Crítica"
            ).sum()
        ),

        Equipamentos_Alta=(
            "Classe_Criticidade",
            lambda x: (
                x == "Alta"
            ).sum()
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# =====================================================================
# 32. RESUMO EXECUTIVO
# =====================================================================

equip_freq = (

    ranking_equipamentos
    .iloc[0]
)


equip_crit = (

    ranking_criticidade
    .iloc[0]
)


linha80 = (

    resumo_limites_pareto.loc[
        resumo_limites_pareto[
            "Limite_Pareto_Perc"
        ] == 80
    ]
    .iloc[0]
)


linha90 = (

    resumo_limites_pareto.loc[
        resumo_limites_pareto[
            "Limite_Pareto_Perc"
        ] == 90
    ]
    .iloc[0]
)


linha95 = (

    resumo_limites_pareto.loc[
        resumo_limites_pareto[
            "Limite_Pareto_Perc"
        ] == 95
    ]
    .iloc[0]
)


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Gerações efetivas",

            "Equipamentos geradores",

            "Equipamento líder por frequência",

            "Gerações do líder",

            "Participação do líder",

            "Equipamento líder por criticidade",

            "Índice máximo de criticidade",

            "Equipamentos para atingir 80%",

            "% dos equipamentos para atingir 80%",

            "Equipamentos para atingir 90%",

            "% dos equipamentos para atingir 90%",

            "Equipamentos para atingir 95%",

            "% dos equipamentos para atingir 95%",

            "Categoria líder",

            "Mantenedor líder",

            "Tipo líder",

            "Torre líder"
        ],

        "Resultado": [

            f"{TOTAL_GERACOES:,}",

            f"{TOTAL_EQUIPAMENTOS_GERADORES:,}",

            str(
                equip_freq[
                    "itemName"
                ]
            ),

            f"{int(equip_freq['Geracoes_Efetivas']):,}",

            f"{equip_freq['Participacao_Geracoes_Perc']:.2f}%",

            str(
                equip_crit[
                    "itemName"
                ]
            ),

            f"{equip_crit['Indice_Criticidade_V1']:.2f}",

            f"{int(linha80['Quantidade_Equipamentos']):,}",

            f"{linha80['Percentual_Equipamentos']:.2f}%",

            f"{int(linha90['Quantidade_Equipamentos']):,}",

            f"{linha90['Percentual_Equipamentos']:.2f}%",

            f"{int(linha95['Quantidade_Equipamentos']):,}",

            f"{linha95['Percentual_Equipamentos']:.2f}%",

            str(
                pareto_categoria.iloc[0][
                    "Categoria"
                ]
            ),

            str(
                pareto_mantenedor.iloc[0][
                    "Mantenedor"
                ]
            ),

            str(
                pareto_tipo.iloc[0][
                    "Tipo"
                ]
            ),

            str(
                pareto_torre.iloc[0][
                    "Torre"
                ]
            )
        ]
    }
)


# =====================================================================
# 33. EXIBIÇÃO DOS RESULTADOS
# =====================================================================

print("\n" + "=" * 105)
print("RESUMO EXECUTIVO")
print("=" * 105)

display(
    resumo_executivo
)


print("\n" + "=" * 105)
print("CONCENTRAÇÃO TOP N")
print("=" * 105)

display(
    concentracao_topn
)


print("\n" + "=" * 105)
print("QUANTIDADE DE EQUIPAMENTOS PARA EXPLICAR O VOLUME")
print("=" * 105)

display(
    resumo_limites_pareto
)


print("\n" + "=" * 105)
print("CLASSES DE PARETO")
print("=" * 105)

display(
    resumo_classes_pareto
)


print("\n" + "=" * 105)
print("DISTRIBUIÇÃO DA CRITICIDADE")
print("=" * 105)

display(
    resumo_criticidade
)


print("\n" + "=" * 105)
print("TOP 30 POR ÍNDICE DE CRITICIDADE")
print("=" * 105)

display(
    ranking_criticidade.head(30)
)


print("\n" + "=" * 105)
print("TOP 30 POR FREQUÊNCIA")
print("=" * 105)

display(
    ranking_equipamentos.head(30)
)


print("\n" + "=" * 105)
print("PARETO POR CATEGORIA")
print("=" * 105)

display(
    pareto_categoria
)


print("\n" + "=" * 105)
print("PARETO POR MANTENEDOR")
print("=" * 105)

display(
    pareto_mantenedor
)


print("\n" + "=" * 105)
print("PARETO POR TIPO")
print("=" * 105)

display(
    pareto_tipo
)


print("\n" + "=" * 105)
print("PARETO POR TORRE")
print("=" * 105)

display(
    pareto_torre
)


# =====================================================================
# 34. SALVAMENTO DO RANKING EM PARQUET
# =====================================================================

print(
    "\nSalvando ranking de criticidade..."
)


ranking_criticidade.to_parquet(
    ARQUIVO_RANKING,
    index=False
)


print(
    f"Arquivo criado:\n"
    f"{ARQUIVO_RANKING}"
)


# =====================================================================
# 35. EXPORTAÇÃO PARA EXCEL
# =====================================================================

print(
    "\nGerando relatório Excel..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    concentracao_topn.to_excel(
        writer,
        sheet_name="Concentracao TopN",
        index=False
    )


    resumo_limites_pareto.to_excel(
        writer,
        sheet_name="Limites Pareto",
        index=False
    )


    resumo_classes_pareto.to_excel(
        writer,
        sheet_name="Classes Pareto",
        index=False
    )


    resumo_criticidade.to_excel(
        writer,
        sheet_name="Classes Criticidade",
        index=False
    )


    ranking_criticidade.to_excel(
        writer,
        sheet_name="Ranking Criticidade",
        index=False
    )


    top100_criticidade.to_excel(
        writer,
        sheet_name="Top100 Criticidade",
        index=False
    )


    top100_frequencia.to_excel(
        writer,
        sheet_name="Top100 Frequencia",
        index=False
    )


    comparativo_rankings.to_excel(
        writer,
        sheet_name="Freq x Criticidade",
        index=False
    )


    pareto_categoria.to_excel(
        writer,
        sheet_name="Pareto Categoria",
        index=False
    )


    pareto_mantenedor.to_excel(
        writer,
        sheet_name="Pareto Mantenedor",
        index=False
    )


    pareto_tipo.to_excel(
        writer,
        sheet_name="Pareto Tipo",
        index=False
    )


    pareto_torre.to_excel(
        writer,
        sheet_name="Pareto Torre",
        index=False
    )


    pareto_ttl_torre.to_excel(
        writer,
        sheet_name="Pareto TTL Torre",
        index=False
    )


    top20_mensal.to_excel(
        writer,
        sheet_name="Top20 Mensal",
        index=False
    )


    criticidade_categoria.to_excel(
        writer,
        sheet_name="Criticidade Categoria",
        index=False
    )


    criticidade_mantenedor.to_excel(
        writer,
        sheet_name="Criticidade Mantenedor",
        index=False
    )


    criticidade_tipo.to_excel(
        writer,
        sheet_name="Criticidade Tipo",
        index=False
    )


    criticidade_torre.to_excel(
        writer,
        sheet_name="Criticidade Torre",
        index=False
    )


# =====================================================================
# 36. FORMATAÇÃO BÁSICA DO EXCEL
# =====================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # Cabeçalho
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    # Ajuste de largura
    for coluna_cells in ws.columns:

        max_length = 0

        coluna_letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                max_length = max(
                    max_length,
                    len(valor)
                )

            except:
                pass


        largura = min(
            max(
                max_length + 2,
                12
            ),
            45
        )


        ws.column_dimensions[
            coluna_letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# =====================================================================
# 37. VALIDAÇÕES
# =====================================================================

print("\n" + "=" * 105)
print("VALIDAÇÕES")
print("=" * 105)


soma_ranking = int(

    ranking_equipamentos[
        "Geracoes_Efetivas"
    ]
    .sum()
)


soma_categoria = int(

    pareto_categoria[
        "Geracoes_Efetivas"
    ]
    .sum()
)


soma_mantenedor = int(

    pareto_mantenedor[
        "Geracoes_Efetivas"
    ]
    .sum()
)


soma_tipo = int(

    pareto_tipo[
        "Geracoes_Efetivas"
    ]
    .sum()
)


soma_torre = int(

    pareto_torre[
        "Geracoes_Efetivas"
    ]
    .sum()
)


print(
    f"Total base gerações efetivas.........: "
    f"{TOTAL_GERACOES:,}"
)

print(
    f"Soma ranking equipamentos............: "
    f"{soma_ranking:,}"
)

print(
    f"Soma Pareto Categoria................: "
    f"{soma_categoria:,}"
)

print(
    f"Soma Pareto Mantenedor...............: "
    f"{soma_mantenedor:,}"
)

print(
    f"Soma Pareto Tipo.....................: "
    f"{soma_tipo:,}"
)

print(
    f"Soma Pareto Torre....................: "
    f"{soma_torre:,}"
)


print("\nChecagens:")

print(
    "Equipamentos consistente.............:",
    soma_ranking == TOTAL_GERACOES
)

print(
    "Categoria consistente................:",
    soma_categoria == TOTAL_GERACOES
)

print(
    "Mantenedor consistente...............:",
    soma_mantenedor == TOTAL_GERACOES
)

print(
    "Tipo consistente.....................:",
    soma_tipo == TOTAL_GERACOES
)

print(
    "Torre consistente....................:",
    soma_torre == TOTAL_GERACOES
)


# =====================================================================
# 38. DOCUMENTAÇÃO DO ÍNDICE
# =====================================================================

metodologia_indice = pd.DataFrame(
    {
        "Componente": [

            "Frequência",
            "Prioridade",
            "Severidade",
            "Persistência"
        ],

        "Peso": [

            40,
            25,
            20,
            15
        ],

        "Descrição": [

            (
                "Percentil da quantidade de gerações "
                "efetivas do equipamento."
            ),

            (
                "Percentil inverso da prioridade mediana. "
                "Quanto menor o número priority, "
                "maior o score."
            ),

            (
                "Composição ponderada: "
                "Alarme=100, Comunicação=80, Pré-Alarme=60."
            ),

            (
                "Percentual de meses nos quais o equipamento "
                "apresentou geração efetiva."
            )
        ]
    }
)


# Acrescenta metodologia ao Excel
with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl",

    mode="a",

    if_sheet_exists="replace"

) as writer:


    metodologia_indice.to_excel(
        writer,
        sheet_name="Metodologia Indice",
        index=False
    )


# =====================================================================
# 39. FINALIZAÇÃO
# =====================================================================

print("\n" + "=" * 105)
print("MÓDULO 4 CONCLUÍDO COM SUCESSO")
print("=" * 105)


print(
    "\nArquivos gerados:"
)

print(
    f"\n1. {ARQUIVO_RELATORIO}"
)

print(
    f"2. {ARQUIVO_RANKING}"
)


print(
    "\nO relatório possui:"
)

print(
    """
1. Resumo executivo
2. Concentração Top N
3. Limites de Pareto 50/80/90/95%
4. Classes A/B/C
5. Ranking de criticidade
6. Ranking de frequência
7. Comparação frequência × criticidade
8. Pareto por Categoria
9. Pareto por Mantenedor
10. Pareto por Tipo
11. Pareto por Torre
12. Pareto por TTL Torre
13. Top 20 mensal
14. Criticidade por Categoria
15. Criticidade por Mantenedor
16. Criticidade por Tipo
17. Criticidade por Torre
18. Metodologia do índice
"""
)


# =====================================================================
# 40. DOWNLOAD
# =====================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar também o ranking em Parquet:"
)

print(
    f'files.download("{ARQUIVO_RANKING}")'
)

MÓDULO 4 - PARETO E ÍNDICE ANALÍTICO DE CRITICIDADE

Carregando base classificada...
Base carregada.
Registros totais: 1,413,882
Colunas: 58

BASE DE GERAÇÕES EFETIVAS
Gerações efetivas..............: 637,566
Equipamentos com geração.......: 5,826
Meses analisados...............: 7

RESUMO EXECUTIVO


,Indicador,Resultado
0,Gerações efetivas,"637,566"
1,Equipamentos geradores,"5,826"
2,Equipamento líder por frequência,KRON73 (Elev Panoramico)
3,Gerações do líder,"6,442"
4,Participação do líder,1.01%
5,Equipamento líder por criticidade,TE2-PG-NAE-55114-MC [...
6,Índice máximo de criticidade,98.48
7,Equipamentos para atingir 80%,"1,425"
8,% dos equipamentos para atingir 80%,22.75%
9,Equipamentos para atingir 90%,"2,308"



CONCENTRAÇÃO TOP N


,Top_N,Geracoes_Efetivas,Percentual_Total
0,1,6442,1.01
1,5,21815,3.42
2,10,36676,5.75
3,20,57893,9.08
4,50,103947,16.30
5,100,153212,24.03
6,250,242201,37.99
7,500,333992,52.39
8,1000,450142,70.60



QUANTIDADE DE EQUIPAMENTOS PARA EXPLICAR O VOLUME


...

,Torre,Geracoes_Efetivas,Participacao_Perc,Participacao_Acumulada_Perc,Ranking,Classe_Pareto
0,CEA,288710,45.28,45.28,1,A
1,TEV,96726,15.17,60.45,2,A
2,TOS,91640,14.37,74.83,3,A
3,TCO,53540,8.40,83.23,4,B
4,TAE,47351,7.43,90.65,5,B
5,TWMS,44164,6.93,97.58,6,C
6,Bloco E6,14770,2.32,99.90,7,C
7,X,665,0.10,100.00,8,C



Salvando ranking de criticidade...
Arquivo criado:
Ranking_Criticidade_Alarmes_Metasys_2026.parquet

Gerando relatório Excel...

VALIDAÇÕES
Total base gerações efetivas.........: 637,566
Soma ranking equipamentos............: 637,566
Soma Pareto Categoria................: 637,566
Soma Pareto Mantenedor...............: 637,566
Soma Pareto Tipo.....................: 637,566
Soma Pareto Torre....................: 637,566

Checagens:
Equipamentos consistente.............: True
Categoria consistente................: True
Mantenedor consistente...............: True
Tipo consistente.....................: True
Torre consistente....................: True

MÓDULO 4 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo4_Pareto_Criticidade_Metasys.xlsx
2. Ranking_Criticidade_Alarmes_Metasys_2026.parquet

O relatório possui:

1. Resumo executivo
2. Concentração Top N
3. Limites de Pareto 50/80/90/95%
4. Classes A/B/C
5. Ranking de criticidade
6. Ranking de frequência
7. Comparação frequênc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também o ranking em Parquet:
files.download("Ranking_Criticidade_Alarmes_Metasys_2026.parquet")


In [14]:
# =====================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 5
# REINCIDÊNCIA, CHATTERING E COMPORTAMENTO REPETITIVO
#
# Base principal:
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Base opcional:
# Ranking_Criticidade_Alarmes_Metasys_2026.parquet
#
# OBJETIVOS
# ---------------------------------------------------------------------
# 1. Medir tempo entre gerações do mesmo ponto
# 2. Identificar reincidência
# 3. Identificar retorno rápido à condição normal
# 4. Identificar reativação rápida
# 5. Detectar alternância Entrada ↔ Retorno
# 6. Detectar episódios de chattering
# 7. Avaliar diferentes janelas:
#       1, 5, 10, 15 e 30 minutos
# 8. Ranking por equipamento
# 9. Análise por Categoria, Mantenedor, Tipo e Torre
# 10. Análise mensal
#
# DEFINIÇÃO OPERACIONAL DO PROJETO
# ---------------------------------------------------------------------
# Entrada:
#     Geração efetiva de Alarme, Pré-Alarme ou Falha de Comunicação
#
# Retorno:
#     Normalização de Alarme,
#     Normalização de Pré-Alarme ou
#     Restabelecimento de Comunicação
#
# CHATTERING:
#     Alternância rápida Entrada ↔ Retorno.
#
# Episódio de chattering (referência de 10 minutos):
#     sequência contendo no mínimo:
#       - 4 transições
#       - 2 entradas
#       - 2 retornos
#
# IMPORTANTE:
# Os limites utilizados são critérios analíticos deste projeto,
# não limites oficiais do Metasys / Johnson Controls.
# =====================================================================


# =====================================================================
# 1. BIBLIOTECAS
# =====================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    250
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    280
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# =====================================================================
# 2. ARQUIVOS
# =====================================================================

ARQUIVO_ENTRADA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_CRITICIDADE = (
    "Ranking_Criticidade_Alarmes_Metasys_2026.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo5_Reincidencia_Chattering_Metasys.xlsx"
)

ARQUIVO_RANKING = (
    "Ranking_Modulo5_Reincidencia_Chattering.parquet"
)

ARQUIVO_EPISODIOS = (
    "Episodios_Chattering_10min_Metasys.parquet"
)


# =====================================================================
# 3. PARÂMETROS
# =====================================================================

# Janelas para análise de sensibilidade
JANELAS_CHATTERING = [
    1,
    5,
    10,
    15,
    30
]


# Janela principal adotada como referência operacional do projeto
JANELA_REFERENCIA_MIN = 10


# Reincidência entre duas gerações efetivas
JANELAS_REINCIDENCIA = [
    5,
    10,
    30,
    60,
    1440       # 24 horas
]


print("=" * 110)
print("MÓDULO 5 - REINCIDÊNCIA, CHATTERING E COMPORTAMENTO REPETITIVO")
print("=" * 110)


# =====================================================================
# 4. VERIFICAÇÃO / UPLOAD DA BASE
# =====================================================================

if not Path(ARQUIVO_ENTRADA).exists():

    print(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' não foi encontrado."
    )

    print(
        "\nSelecione o arquivo Parquet gerado no Módulo 2."
    )

    from google.colab import files

    uploaded = files.upload()


if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"O arquivo '{ARQUIVO_ENTRADA}' não foi localizado."
    )


# =====================================================================
# 5. CARREGAMENTO
# =====================================================================

print("\nCarregando base...")

df = pd.read_parquet(
    ARQUIVO_ENTRADA
)


print("Base carregada com sucesso.")

print(
    f"Registros: {len(df):,}"
)

print(
    f"Colunas: {df.shape[1]}"
)


# =====================================================================
# 6. COLUNAS NECESSÁRIAS
# =====================================================================

COLUNAS_NECESSARIAS = [

    "DataHoraUTC",
    "DataHoraLocal",

    "Ano",
    "Mes_Numero",
    "Mes",

    "priority",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Flag_Geracao_Efetiva",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",
    "Flag_Falha_Comunicacao",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",
    "Flag_Restabelecimento_Comunicacao"
]


faltantes = [

    coluna

    for coluna in COLUNAS_NECESSARIAS

    if coluna not in df.columns
]


if faltantes:

    raise ValueError(
        "Colunas necessárias ausentes:\n\n"
        + "\n".join(faltantes)
    )


# =====================================================================
# 7. AJUSTE DE DATAS
# =====================================================================

df["DataHoraUTC"] = pd.to_datetime(
    df["DataHoraUTC"],
    errors="coerce"
)

df["DataHoraLocal"] = pd.to_datetime(
    df["DataHoraLocal"],
    errors="coerce"
)


# =====================================================================
# 8. DEFINIÇÃO DE ENTRADA E RETORNO
# =====================================================================

df["Flag_Retorno_Condicao"] = (

    df[
        "Flag_Normalizacao_Alarme"
    ]

    |

    df[
        "Flag_Normalizacao_PreAlarme"
    ]

    |

    df[
        "Flag_Restabelecimento_Comunicacao"
    ]
)


df["Classe_Transicao_Chattering"] = np.select(

    [
        df[
            "Flag_Geracao_Efetiva"
        ] == True,

        df[
            "Flag_Retorno_Condicao"
        ] == True
    ],

    [
        "Entrada",
        "Retorno"
    ],

    default="Outro"
)


# =====================================================================
# 9. BASE DE TRANSIÇÕES RELEVANTES
# =====================================================================

df_transicoes = (

    df.loc[
        df[
            "Classe_Transicao_Chattering"
        ].isin(
            [
                "Entrada",
                "Retorno"
            ]
        )
    ]

    .copy()
)


df_transicoes = (

    df_transicoes
    .sort_values(
        [
            "itemName",
            "DataHoraUTC"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 110)
print("BASE DE TRANSIÇÕES RELEVANTES")
print("=" * 110)

print(
    f"Transições Entrada/Retorno: "
    f"{len(df_transicoes):,}"
)


# =====================================================================
# 10. TRANSIÇÃO ANTERIOR DO MESMO PONTO
# =====================================================================

df_transicoes[
    "DataHora_Transicao_Anterior"
] = (

    df_transicoes
    .groupby(
        "itemName",
        observed=True
    )[
        "DataHoraUTC"
    ]
    .shift(1)
)


df_transicoes[
    "Classe_Transicao_Anterior"
] = (

    df_transicoes
    .groupby(
        "itemName",
        observed=True
    )[
        "Classe_Transicao_Chattering"
    ]
    .shift(1)
)


# =====================================================================
# 11. INTERVALO ENTRE TRANSIÇÕES
# =====================================================================

df_transicoes[
    "Intervalo_Transicao_Min"
] = (

    (
        df_transicoes[
            "DataHoraUTC"
        ]

        -

        df_transicoes[
            "DataHora_Transicao_Anterior"
        ]
    )

    .dt.total_seconds()

    /

    60
)


# =====================================================================
# 12. ALTERNÂNCIA DE ESTADO
# =====================================================================

df_transicoes[
    "Flag_Alternancia"
] = (

    df_transicoes[
        "Classe_Transicao_Chattering"
    ]

    !=

    df_transicoes[
        "Classe_Transicao_Anterior"
    ]
)


# Primeira ocorrência do equipamento não pode ser alternância
df_transicoes.loc[
    df_transicoes[
        "Classe_Transicao_Anterior"
    ].isna(),
    "Flag_Alternancia"
] = False


# =====================================================================
# 13. FLAGS DE CHATTERING PARA MÚLTIPLAS JANELAS
# =====================================================================

for janela in JANELAS_CHATTERING:

    nome_flag = (
        f"Flag_Chattering_{janela}min"
    )

    df_transicoes[
        nome_flag
    ] = (

        df_transicoes[
            "Flag_Alternancia"
        ]

        &

        (
            df_transicoes[
                "Intervalo_Transicao_Min"
            ]
            <= janela
        )

        &

        (
            df_transicoes[
                "Intervalo_Transicao_Min"
            ]
            >= 0
        )
    )


# =====================================================================
# 14. RETORNO RÁPIDO E REATIVAÇÃO RÁPIDA
# =====================================================================

df_transicoes[
    "Flag_Retorno_Rapido_10min"
] = (

    (
        df_transicoes[
            "Classe_Transicao_Chattering"
        ]
        ==
        "Retorno"
    )

    &

    (
        df_transicoes[
            "Classe_Transicao_Anterior"
        ]
        ==
        "Entrada"
    )

    &

    (
        df_transicoes[
            "Intervalo_Transicao_Min"
        ]
        <= JANELA_REFERENCIA_MIN
    )

    &

    (
        df_transicoes[
            "Intervalo_Transicao_Min"
        ]
        >= 0
    )
)


df_transicoes[
    "Flag_Reativacao_Rapida_10min"
] = (

    (
        df_transicoes[
            "Classe_Transicao_Chattering"
        ]
        ==
        "Entrada"
    )

    &

    (
        df_transicoes[
            "Classe_Transicao_Anterior"
        ]
        ==
        "Retorno"
    )

    &

    (
        df_transicoes[
            "Intervalo_Transicao_Min"
        ]
        <= JANELA_REFERENCIA_MIN
    )

    &

    (
        df_transicoes[
            "Intervalo_Transicao_Min"
        ]
        >= 0
    )
)


# =====================================================================
# 15. ANÁLISE DE REINCIDÊNCIA ENTRE GERAÇÕES
# =====================================================================

df_entradas = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()

    .sort_values(
        [
            "itemName",
            "DataHoraUTC"
        ]
    )

    .reset_index(
        drop=True
    )
)


df_entradas[
    "DataHora_Geracao_Anterior"
] = (

    df_entradas
    .groupby(
        "itemName",
        observed=True
    )[
        "DataHoraUTC"
    ]
    .shift(1)
)


df_entradas[
    "Intervalo_Entre_Geracoes_Min"
] = (

    (
        df_entradas[
            "DataHoraUTC"
        ]

        -

        df_entradas[
            "DataHora_Geracao_Anterior"
        ]
    )

    .dt.total_seconds()

    /

    60
)


for janela in JANELAS_REINCIDENCIA:

    nome_flag = (
        f"Flag_Reincidencia_{janela}min"
    )

    df_entradas[
        nome_flag
    ] = (

        (
            df_entradas[
                "Intervalo_Entre_Geracoes_Min"
            ]
            <= janela
        )

        &

        (
            df_entradas[
                "Intervalo_Entre_Geracoes_Min"
            ]
            >= 0
        )
    )


# =====================================================================
# 16. IDENTIFICAÇÃO DE EPISÓDIOS DE CHATTERING
#     REFERÊNCIA = 10 MINUTOS
# =====================================================================

FLAG_CHAT_REF = (
    f"Flag_Chattering_{JANELA_REFERENCIA_MIN}min"
)


# Uma nova sequência começa quando:
# - não há alternância rápida com o registro anterior
df_transicoes[
    "Flag_Novo_Episodio"
] = (

    ~df_transicoes[
        FLAG_CHAT_REF
    ]
)


# Primeira linha de cada equipamento inicia novo episódio
primeira_linha_item = (

    df_transicoes
    .groupby(
        "itemName",
        observed=True
    )
    .cumcount()
    ==
    0
)


df_transicoes.loc[
    primeira_linha_item,
    "Flag_Novo_Episodio"
] = True


# Numeração do episódio dentro de cada equipamento
df_transicoes[
    "Numero_Episodio"
] = (

    df_transicoes
    .groupby(
        "itemName",
        observed=True
    )[
        "Flag_Novo_Episodio"
    ]
    .cumsum()
)


# =====================================================================
# 17. RESUMO DOS EPISÓDIOS
# =====================================================================

episodios = (

    df_transicoes
    .groupby(

        [
            "itemName",
            "Numero_Episodio"
        ],

        observed=True
    )

    .agg(

        Inicio_Episodio=(
            "DataHoraUTC",
            "min"
        ),

        Fim_Episodio=(
            "DataHoraUTC",
            "max"
        ),

        Transicoes=(
            "Classe_Transicao_Chattering",
            "size"
        ),

        Entradas=(
            "Classe_Transicao_Chattering",
            lambda x: (
                x == "Entrada"
            ).sum()
        ),

        Retornos=(
            "Classe_Transicao_Chattering",
            lambda x: (
                x == "Retorno"
            ).sum()
        ),

        Alternancias_10min=(
            FLAG_CHAT_REF,
            "sum"
        ),

        Intervalo_Medio_Min=(
            "Intervalo_Transicao_Min",
            "mean"
        ),

        Intervalo_Minimo_Min=(
            "Intervalo_Transicao_Min",
            "min"
        ),

        Intervalo_Maximo_Min=(
            "Intervalo_Transicao_Min",
            "max"
        )
    )

    .reset_index()
)


episodios[
    "Duracao_Episodio_Min"
] = (

    (
        episodios[
            "Fim_Episodio"
        ]

        -

        episodios[
            "Inicio_Episodio"
        ]
    )

    .dt.total_seconds()

    /

    60
)


# =====================================================================
# 18. CLASSIFICAÇÃO DE EPISÓDIO DE CHATTERING
# =====================================================================

episodios[
    "Flag_Episodio_Chattering"
] = (

    (
        episodios[
            "Transicoes"
        ] >= 4
    )

    &

    (
        episodios[
            "Entradas"
        ] >= 2
    )

    &

    (
        episodios[
            "Retornos"
        ] >= 2
    )

    &

    (
        episodios[
            "Alternancias_10min"
        ] >= 3
    )
)


episodios_chattering = (

    episodios.loc[
        episodios[
            "Flag_Episodio_Chattering"
        ] == True
    ]

    .copy()
)


# =====================================================================
# 19. CADASTRO PRINCIPAL DOS EQUIPAMENTOS
# =====================================================================

def moda_segura(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return pd.NA

    moda = serie.mode()

    if len(moda) > 0:
        return moda.iloc[0]

    return serie.iloc[0]


cadastro_equipamentos = (

    df
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        itemDescription=(
            "itemDescription",
            moda_segura
        ),

        Categoria=(
            "Categoria",
            moda_segura
        ),

        Mantenedor=(
            "Mantenedor",
            moda_segura
        ),

        Tipo=(
            "Tipo",
            moda_segura
        ),

        Torre=(
            "Torre",
            moda_segura
        ),

        TTL_Torre=(
            "TTL Torre",
            moda_segura
        )
    )

    .reset_index()
)


# ======================================================================
# 20. RESUMO INICIAL DOS EPISÓDIOS
# ======================================================================

if len(
    floods
) > 0:

    episodios = (

        floods
        .groupby(
            "ID_Episodio",
            observed=True
        )
        .agg(

            Inicio_Episodio=(
                "DataHoraAnalise",
                "min"
            ),

            Fim_Episodio=(
                "DataHoraAnalise",
                "max"
            ),

            Janelas_Flood=(
                "DataHoraAnalise",
                "size"
            ),

            Eventos_Janelas=(
                "Eventos",
                "sum"
            ),

            Pico_Eventos_10min=(
                "Eventos",
                "max"
            ),

            Pico_Equipamentos_10min=(
                "Equipamentos_Unicos",
                "max"
            ),

            Max_Torres=(
                "Torres",
                "max"
            ),

            Max_Categorias=(
                "Categorias",
                "max"
            ),

            # ----------------------------------------------------------
            # IMPORTANTE:
            # Esses valores vêm da soma das janelas de 10 minutos.
            #
            # Usamos nomes diferentes para evitar colisão posteriormente
            # com Eventos_P1 / Eventos_P2 calculados sobre os eventos
            # reais de cada episódio.
            # ----------------------------------------------------------

            Eventos_P1_Janelas=(
                "Eventos_P1",
                "sum"
            ),

            Eventos_P2_Janelas=(
                "Eventos_P2",
                "sum"
            ),

            Eventos_P1_P2_Janelas=(
                "Eventos_P1_P2",
                "sum"
            )
        )

        .reset_index()
    )


    # --------------------------------------------------------------
    # Cada janela representa 10 minutos.
    # Portanto, o término real do episódio é o início da
    # última janela + duração da janela de referência.
    # --------------------------------------------------------------

    episodios[
        "Fim_Episodio"
    ] = (

        episodios[
            "Fim_Episodio"
        ]

        +

        pd.Timedelta(
            minutes=JANELA_REFERENCIA_MIN
        )
    )


    # --------------------------------------------------------------
    # Duração total do episódio
    # --------------------------------------------------------------

    episodios[
        "Duracao_Episodio_Min"
    ] = (

        (
            episodios[
                "Fim_Episodio"
            ]

            -

            episodios[
                "Inicio_Episodio"
            ]
        )

        .dt.total_seconds()

        /

        60
    )


else:

    episodios = pd.DataFrame()


# =====================================================================
# 21. MÉTRICAS DE CHATTERING POR EQUIPAMENTO
# =====================================================================

agregacoes_chattering = {

    "Transicoes_Relevantes": (
        "Classe_Transicao_Chattering",
        "size"
    ),

    "Retornos_Rapidos_10min": (
        "Flag_Retorno_Rapido_10min",
        "sum"
    ),

    "Reativacoes_Rapidas_10min": (
        "Flag_Reativacao_Rapida_10min",
        "sum"
    )
}


for janela in JANELAS_CHATTERING:

    agregacoes_chattering[
        f"Transicoes_Chattering_{janela}min"
    ] = (

        f"Flag_Chattering_{janela}min",
        "sum"
    )


metricas_chattering = (

    df_transicoes
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(
        **agregacoes_chattering
    )

    .reset_index()
)


# =====================================================================
# 22. EPISÓDIOS POR EQUIPAMENTO
# =====================================================================

metricas_episodios = (

    episodios_chattering
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios_Chattering_10min=(
            "Numero_Episodio",
            "size"
        ),

        Total_Transicoes_em_Episodios=(
            "Transicoes",
            "sum"
        ),

        Total_Entradas_em_Episodios=(
            "Entradas",
            "sum"
        ),

        Total_Retornos_em_Episodios=(
            "Retornos",
            "sum"
        ),

        Duracao_Media_Episodio_Min=(
            "Duracao_Episodio_Min",
            "mean"
        ),

        Duracao_Maxima_Episodio_Min=(
            "Duracao_Episodio_Min",
            "max"
        ),

        Maior_Episodio_Transicoes=(
            "Transicoes",
            "max"
        )
    )

    .reset_index()
)


# =====================================================================
# 23. RANKING CONSOLIDADO POR EQUIPAMENTO
# =====================================================================

ranking = (

    cadastro_equipamentos
    .merge(
        metricas_reincidencia,
        on="itemName",
        how="left"
    )
    .merge(
        metricas_chattering,
        on="itemName",
        how="left"
    )
    .merge(
        metricas_episodios,
        on="itemName",
        how="left"
    )
)


# =====================================================================
# 24. PREENCHIMENTO DOS CONTADORES NULOS
# =====================================================================

colunas_contagem = [

    "Geracoes_Efetivas",

    "Reincidencias_5min",
    "Reincidencias_10min",
    "Reincidencias_30min",
    "Reincidencias_60min",
    "Reincidencias_24h",

    "Transicoes_Relevantes",

    "Retornos_Rapidos_10min",
    "Reativacoes_Rapidas_10min",

    "Episodios_Chattering_10min",

    "Total_Transicoes_em_Episodios",
    "Total_Entradas_em_Episodios",
    "Total_Retornos_em_Episodios",

    "Maior_Episodio_Transicoes"
]


for janela in JANELAS_CHATTERING:

    colunas_contagem.append(
        f"Transicoes_Chattering_{janela}min"
    )


for coluna in colunas_contagem:

    if coluna in ranking.columns:

        ranking[coluna] = (
            ranking[coluna]
            .fillna(0)
        )


# =====================================================================
# 25. TAXAS POR EQUIPAMENTO
# =====================================================================

ranking[
    "Taxa_Reincidencia_10min_Perc"
] = np.where(

    ranking[
        "Geracoes_Efetivas"
    ] > 1,

    ranking[
        "Reincidencias_10min"
    ]

    /

    (
        ranking[
            "Geracoes_Efetivas"
        ] - 1
    )

    * 100,

    0
)


ranking[
    "Taxa_Reincidencia_60min_Perc"
] = np.where(

    ranking[
        "Geracoes_Efetivas"
    ] > 1,

    ranking[
        "Reincidencias_60min"
    ]

    /

    (
        ranking[
            "Geracoes_Efetivas"
        ] - 1
    )

    * 100,

    0
)


ranking[
    "Taxa_Chattering_10min_Perc"
] = np.where(

    ranking[
        "Transicoes_Relevantes"
    ] > 1,

    ranking[
        "Transicoes_Chattering_10min"
    ]

    /

    (
        ranking[
            "Transicoes_Relevantes"
        ] - 1
    )

    * 100,

    0
)


# =====================================================================
# 26. CLASSIFICAÇÃO DO COMPORTAMENTO
# =====================================================================

ranking[
    "Classe_Comportamento"
] = np.select(

    [
        ranking[
            "Episodios_Chattering_10min"
        ] >= 10,

        ranking[
            "Episodios_Chattering_10min"
        ] >= 3,

        ranking[
            "Episodios_Chattering_10min"
        ] >= 1,

        ranking[
            "Reincidencias_10min"
        ] >= 10,

        ranking[
            "Reincidencias_10min"
        ] >= 1
    ],

    [
        "Chattering Muito Alto",
        "Chattering Alto",
        "Chattering Detectado",
        "Reincidência Alta",
        "Reincidência Detectada"
    ],

    default="Sem comportamento rápido relevante"
)


# =====================================================================
# 27. MERGE OPCIONAL COM CRITICIDADE DO MÓDULO 4
# =====================================================================

if Path(
    ARQUIVO_CRITICIDADE
).exists():

    print(
        "\nRanking de criticidade do Módulo 4 encontrado."
    )

    df_criticidade = pd.read_parquet(
        ARQUIVO_CRITICIDADE
    )


    colunas_criticidade = [

        "itemName",

        "Ranking_Criticidade",
        "Indice_Criticidade_V1",
        "Classe_Criticidade",

        "Classe_Pareto",

        "Score_Frequencia",
        "Score_Prioridade",
        "Score_Severidade",
        "Score_Persistencia"
    ]


    colunas_criticidade = [

        c

        for c in colunas_criticidade

        if c in df_criticidade.columns
    ]


    ranking = ranking.merge(

        df_criticidade[
            colunas_criticidade
        ],

        on="itemName",

        how="left"
    )


else:

    print(
        "\nRanking do Módulo 4 não encontrado."
    )

    print(
        "O Módulo 5 continuará normalmente, "
        "apenas sem as colunas de criticidade."
    )


# =====================================================================
# 28. PRIORIZAÇÃO MANUTENÇÃO:
#     CRITICIDADE + CHATTERING
# =====================================================================

if (
    "Indice_Criticidade_V1"
    in ranking.columns
):

    ranking[
        "Score_Chattering"
    ] = (

        ranking[
            "Episodios_Chattering_10min"
        ]
        .rank(
            pct=True,
            method="average"
        )
        *
        100
    )


    ranking[
        "Score_Reincidencia"
    ] = (

        ranking[
            "Reincidencias_10min"
        ]
        .rank(
            pct=True,
            method="average"
        )
        *
        100
    )


    ranking[
        "Indice_Prioridade_Investigacao"
    ] = (

        ranking[
            "Indice_Criticidade_V1"
        ]
        .fillna(0)
        *
        0.60

        +

        ranking[
            "Score_Chattering"
        ]
        *
        0.25

        +

        ranking[
            "Score_Reincidencia"
        ]
        *
        0.15
    )


    ranking[
        "Classe_Prioridade_Investigacao"
    ] = np.select(

        [
            ranking[
                "Indice_Prioridade_Investigacao"
            ] >= 80,

            ranking[
                "Indice_Prioridade_Investigacao"
            ] >= 60,

            ranking[
                "Indice_Prioridade_Investigacao"
            ] >= 40
        ],

        [
            "Prioridade 1",
            "Prioridade 2",
            "Prioridade 3"
        ],

        default="Prioridade 4"
    )


# =====================================================================
# 29. ORDENAÇÃO DO RANKING
# =====================================================================

if (
    "Indice_Prioridade_Investigacao"
    in ranking.columns
):

    ranking = (

        ranking
        .sort_values(

            [
                "Indice_Prioridade_Investigacao",
                "Episodios_Chattering_10min",
                "Geracoes_Efetivas"
            ],

            ascending=[
                False,
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


else:

    ranking = (

        ranking
        .sort_values(

            [
                "Episodios_Chattering_10min",
                "Reativacoes_Rapidas_10min",
                "Geracoes_Efetivas"
            ],

            ascending=[
                False,
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


ranking[
    "Ranking_Modulo5"
] = (

    np.arange(
        1,
        len(ranking) + 1
    )
)


# =====================================================================
# 30. SENSIBILIDADE DAS JANELAS DE CHATTERING
# =====================================================================

sensibilidade = []


for janela in JANELAS_CHATTERING:

    coluna = (
        f"Flag_Chattering_{janela}min"
    )


    quantidade = int(
        df_transicoes[
            coluna
        ].sum()
    )


    equipamentos = (

        df_transicoes.loc[
            df_transicoes[
                coluna
            ],
            "itemName"
        ]
        .nunique()
    )


    percentual = (

        quantidade
        /
        max(
            len(df_transicoes) - 1,
            1
        )
        *
        100
    )


    sensibilidade.append({

        "Janela_Minutos": janela,

        "Transicoes_Rapidas": quantidade,

        "Equipamentos_Envolvidos": equipamentos,

        "Percentual_Transicoes": percentual
    })


sensibilidade_chattering = pd.DataFrame(
    sensibilidade
)


# =====================================================================
# 31. KPIs GERAIS
# =====================================================================

TOTAL_GERACOES = len(
    df_entradas
)

TOTAL_TRANSICOES = len(
    df_transicoes
)

TOTAL_EPISODIOS_CHATTERING = len(
    episodios_chattering
)

EQUIPAMENTOS_CHATTERING = (

    episodios_chattering[
        "itemName"
    ]
    .nunique()
)

REINCIDENCIAS_10MIN = int(

    df_entradas[
        "Flag_Reincidencia_10min"
    ]
    .sum()
)

REINCIDENCIAS_60MIN = int(

    df_entradas[
        "Flag_Reincidencia_60min"
    ]
    .sum()
)

REATIVACOES_10MIN = int(

    df_transicoes[
        "Flag_Reativacao_Rapida_10min"
    ]
    .sum()
)

RETORNOS_RAPIDOS_10MIN = int(

    df_transicoes[
        "Flag_Retorno_Rapido_10min"
    ]
    .sum()
)


kpis = pd.DataFrame(
    {
        "Indicador": [

            "Gerações efetivas",

            "Transições Entrada/Retorno",

            "Reincidências até 10 min",

            "Reincidências até 60 min",

            "Reativações após retorno até 10 min",

            "Retornos após entrada até 10 min",

            "Episódios de chattering - 10 min",

            "Equipamentos com chattering - 10 min"
        ],

        "Quantidade": [

            TOTAL_GERACOES,

            TOTAL_TRANSICOES,

            REINCIDENCIAS_10MIN,

            REINCIDENCIAS_60MIN,

            REATIVACOES_10MIN,

            RETORNOS_RAPIDOS_10MIN,

            TOTAL_EPISODIOS_CHATTERING,

            EQUIPAMENTOS_CHATTERING
        ]
    }
)


# =====================================================================
# 32. RESUMO MENSAL DE REINCIDÊNCIA
# =====================================================================

resumo_mensal_reincidencia = (

    df_entradas
    .groupby(

        [
            "Ano",
            "Mes_Numero",
            "Mes"
        ],

        observed=True
    )
    .agg(

        Geracoes_Efetivas=(
            "itemName",
            "size"
        ),

        Reincidencias_5min=(
            "Flag_Reincidencia_5min",
            "sum"
        ),

        Reincidencias_10min=(
            "Flag_Reincidencia_10min",
            "sum"
        ),

        Reincidencias_30min=(
            "Flag_Reincidencia_30min",
            "sum"
        ),

        Reincidencias_60min=(
            "Flag_Reincidencia_60min",
            "sum"
        ),

        Reincidencias_24h=(
            "Flag_Reincidencia_1440min",
            "sum"
        ),

        Equipamentos=(
            "itemName",
            "nunique"
        )
    )

    .reset_index()

    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
)


resumo_mensal_reincidencia[
    "Taxa_Reincidencia_10min_Perc"
] = (

    resumo_mensal_reincidencia[
        "Reincidencias_10min"
    ]

    /

    resumo_mensal_reincidencia[
        "Geracoes_Efetivas"
    ]

    * 100
)


# =====================================================================
# 33. MÊS DOS EPISÓDIOS DE CHATTERING
# =====================================================================

if len(
    episodios_chattering
) > 0:

    episodios_chattering[
        "Mes_Numero"
    ] = (

        episodios_chattering[
            "Inicio_Episodio"
        ]
        .dt.month
    )


    MAPA_MESES = {

        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }


    episodios_chattering[
        "Mes"
    ] = (

        episodios_chattering[
            "Mes_Numero"
        ]
        .map(
            MAPA_MESES
        )
    )


    resumo_mensal_chattering = (

        episodios_chattering
        .groupby(

            [
                "Mes_Numero",
                "Mes"
            ],

            observed=True
        )
        .agg(

            Episodios_Chattering=(
                "Numero_Episodio",
                "size"
            ),

            Equipamentos_Envolvidos=(
                "itemName",
                "nunique"
            ),

            Transicoes=(
                "Transicoes",
                "sum"
            ),

            Duracao_Media_Min=(
                "Duracao_Episodio_Min",
                "mean"
            )
        )

        .reset_index()

        .sort_values(
            "Mes_Numero"
        )
    )


else:

    resumo_mensal_chattering = pd.DataFrame(
        columns=[
            "Mes_Numero",
            "Mes",
            "Episodios_Chattering",
            "Equipamentos_Envolvidos",
            "Transicoes",
            "Duracao_Media_Min"
        ]
    )


# =====================================================================
# 34. RESUMO POR CATEGORIA
# =====================================================================

resumo_categoria = (

    ranking
    .groupby(
        "Categoria",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Reincidencias_10min=(
            "Reincidencias_10min",
            "sum"
        ),

        Reincidencias_60min=(
            "Reincidencias_60min",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Reativacoes_10min=(
            "Reativacoes_Rapidas_10min",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios_Chattering",
        ascending=False
    )
)


# =====================================================================
# 35. RESUMO POR MANTENEDOR
# =====================================================================

resumo_mantenedor = (

    ranking
    .groupby(
        "Mantenedor",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Reincidencias_10min=(
            "Reincidencias_10min",
            "sum"
        ),

        Reincidencias_60min=(
            "Reincidencias_60min",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Reativacoes_10min=(
            "Reativacoes_Rapidas_10min",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios_Chattering",
        ascending=False
    )
)


# =====================================================================
# 36. RESUMO POR TIPO
# =====================================================================

resumo_tipo = (

    ranking
    .groupby(
        "Tipo",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Reincidencias_10min=(
            "Reincidencias_10min",
            "sum"
        ),

        Reincidencias_60min=(
            "Reincidencias_60min",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Reativacoes_10min=(
            "Reativacoes_Rapidas_10min",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios_Chattering",
        ascending=False
    )
)


# =====================================================================
# 37. RESUMO POR TORRE
# =====================================================================

resumo_torre = (

    ranking
    .groupby(
        "Torre",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Reincidencias_10min=(
            "Reincidencias_10min",
            "sum"
        ),

        Reincidencias_60min=(
            "Reincidencias_60min",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Reativacoes_10min=(
            "Reativacoes_Rapidas_10min",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios_Chattering",
        ascending=False
    )
)


# =====================================================================
# 38. TOP EPISÓDIOS DE CHATTERING
# =====================================================================

top_episodios = (

    episodios_chattering
    .sort_values(

        [
            "Transicoes",
            "Duracao_Episodio_Min"
        ],

        ascending=[
            False,
            False
        ]
    )

    .head(200)

    .copy()
)


# Acrescenta cadastro aos episódios
top_episodios = (

    top_episodios
    .merge(

        cadastro_equipamentos,

        on="itemName",

        how="left"
    )
)


# =====================================================================
# 39. TOP REINCIDÊNCIA
# =====================================================================

top_reincidencia = (

    ranking
    .sort_values(

        [
            "Reincidencias_10min",
            "Geracoes_Efetivas"
        ],

        ascending=[
            False,
            False
        ]
    )

    .head(200)

    .copy()
)


# =====================================================================
# 40. TOP CHATTERING
# =====================================================================

top_chattering = (

    ranking
    .sort_values(

        [
            "Episodios_Chattering_10min",
            "Transicoes_Chattering_10min",
            "Reativacoes_Rapidas_10min"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )

    .head(200)

    .copy()
)


# =====================================================================
# 41. RESUMO DE COMPORTAMENTO
# =====================================================================

resumo_comportamento = (

    ranking[
        "Classe_Comportamento"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)


resumo_comportamento.columns = [

    "Classe_Comportamento",
    "Equipamentos"
]


resumo_comportamento[
    "Percentual_Equipamentos"
] = (

    resumo_comportamento[
        "Equipamentos"
    ]

    /

    len(
        ranking
    )

    * 100
)


# =====================================================================
# 42. RESUMO EXECUTIVO
# =====================================================================

if len(
    top_chattering
) > 0:

    equipamento_chatter = (
        top_chattering.iloc[0]
    )

else:

    equipamento_chatter = None


if len(
    top_reincidencia
) > 0:

    equipamento_reincidente = (
        top_reincidencia.iloc[0]
    )

else:

    equipamento_reincidente = None


resumo_executivo_dados = {

    "Indicador": [

        "Gerações efetivas analisadas",

        "Transições Entrada/Retorno",

        "Reincidências em até 10 min",

        "Reincidências em até 60 min",

        "Reativações em até 10 min",

        "Retornos rápidos em até 10 min",

        "Episódios de chattering - 10 min",

        "Equipamentos com chattering",

        "Equipamento líder em chattering",

        "Equipamento líder em reincidência"
    ],

    "Resultado": [

        f"{TOTAL_GERACOES:,}",

        f"{TOTAL_TRANSICOES:,}",

        f"{REINCIDENCIAS_10MIN:,}",

        f"{REINCIDENCIAS_60MIN:,}",

        f"{REATIVACOES_10MIN:,}",

        f"{RETORNOS_RAPIDOS_10MIN:,}",

        f"{TOTAL_EPISODIOS_CHATTERING:,}",

        f"{EQUIPAMENTOS_CHATTERING:,}",

        (
            str(
                equipamento_chatter[
                    "itemName"
                ]
            )
            if equipamento_chatter is not None
            else "Nenhum"
        ),

        (
            str(
                equipamento_reincidente[
                    "itemName"
                ]
            )
            if equipamento_reincidente is not None
            else "Nenhum"
        )
    ]
}


resumo_executivo = pd.DataFrame(
    resumo_executivo_dados
)


# =====================================================================
# 43. EXIBIÇÃO DOS RESULTADOS
# =====================================================================

print("\n" + "=" * 110)
print("RESUMO EXECUTIVO")
print("=" * 110)

display(
    resumo_executivo
)


print("\n" + "=" * 110)
print("SENSIBILIDADE DA JANELA DE CHATTERING")
print("=" * 110)

display(
    sensibilidade_chattering
)


print("\n" + "=" * 110)
print("RESUMO DE COMPORTAMENTO")
print("=" * 110)

display(
    resumo_comportamento
)


print("\n" + "=" * 110)
print("TOP 30 CHATTERING")
print("=" * 110)

display(
    top_chattering.head(30)
)


print("\n" + "=" * 110)
print("TOP 30 REINCIDÊNCIA")
print("=" * 110)

display(
    top_reincidencia.head(30)
)


print("\n" + "=" * 110)
print("TOP 30 EPISÓDIOS")
print("=" * 110)

display(
    top_episodios.head(30)
)


print("\n" + "=" * 110)
print("RESUMO MENSAL - REINCIDÊNCIA")
print("=" * 110)

display(
    resumo_mensal_reincidencia
)


print("\n" + "=" * 110)
print("RESUMO MENSAL - CHATTERING")
print("=" * 110)

display(
    resumo_mensal_chattering
)


# =====================================================================
# 44. SALVAMENTO DOS PARQUETS
# =====================================================================

print(
    "\nSalvando ranking do Módulo 5..."
)


ranking.to_parquet(
    ARQUIVO_RANKING,
    index=False
)


episodios_chattering.to_parquet(
    ARQUIVO_EPISODIOS,
    index=False
)


print(
    f"\nArquivos Parquet criados:\n"
    f"1. {ARQUIVO_RANKING}\n"
    f"2. {ARQUIVO_EPISODIOS}"
)


# =====================================================================
# 45. EXPORTAÇÃO PARA EXCEL
# =====================================================================

print(
    "\nGerando relatório Excel..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    kpis.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )


    sensibilidade_chattering.to_excel(
        writer,
        sheet_name="Sensibilidade",
        index=False
    )


    resumo_comportamento.to_excel(
        writer,
        sheet_name="Comportamento",
        index=False
    )


    ranking.to_excel(
        writer,
        sheet_name="Ranking Equipamentos",
        index=False
    )


    top_chattering.to_excel(
        writer,
        sheet_name="Top Chattering",
        index=False
    )


    top_reincidencia.to_excel(
        writer,
        sheet_name="Top Reincidencia",
        index=False
    )


    top_episodios.to_excel(
        writer,
        sheet_name="Top Episodios",
        index=False
    )


    resumo_mensal_reincidencia.to_excel(
        writer,
        sheet_name="Mensal Reincidencia",
        index=False
    )


    resumo_mensal_chattering.to_excel(
        writer,
        sheet_name="Mensal Chattering",
        index=False
    )


    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )


    resumo_mantenedor.to_excel(
        writer,
        sheet_name="Mantenedor",
        index=False
    )


    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )


    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )


# =====================================================================
# 46. METODOLOGIA
# =====================================================================

metodologia = pd.DataFrame(
    {
        "Parametro": [

            "Geração efetiva",

            "Retorno",

            "Janela principal",

            "Chattering",

            "Episódio de chattering",

            "Reincidência",

            "Observação"
        ],

        "Definicao": [

            (
                "Entrada em Alarme, Pré-Alarme "
                "ou Falha de Comunicação."
            ),

            (
                "Normalização de Alarme, "
                "Normalização de Pré-Alarme "
                "ou Restabelecimento de Comunicação."
            ),

            (
                "10 minutos como referência "
                "analítica inicial."
            ),

            (
                "Alternância Entrada ↔ Retorno "
                "dentro da janela definida."
            ),

            (
                "Sequência com pelo menos 4 transições, "
                "2 entradas, 2 retornos e "
                "3 alternâncias rápidas."
            ),

            (
                "Nova geração do mesmo itemName "
                "dentro de determinado intervalo."
            ),

            (
                "Os limites são critérios analíticos "
                "do projeto e não parâmetros oficiais "
                "Johnson Controls / Metasys."
            )
        ]
    }
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl",

    mode="a",

    if_sheet_exists="replace"

) as writer:


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


# =====================================================================
# 47. FORMATAÇÃO DO EXCEL
# =====================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0

        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                max_length = max(
                    max_length,
                    len(valor)
                )

            except:
                pass


        largura = min(
            max(
                max_length + 2,
                12
            ),
            45
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# =====================================================================
# 48. VALIDAÇÕES
# =====================================================================

print("\n" + "=" * 110)
print("VALIDAÇÕES")
print("=" * 110)


print(
    f"Gerações efetivas base original........: "
    f"{int(df['Flag_Geracao_Efetiva'].sum()):,}"
)

print(
    f"Gerações efetivas base reincidência....: "
    f"{len(df_entradas):,}"
)

print(
    f"Transições relevantes..................: "
    f"{len(df_transicoes):,}"
)

print(
    f"Episódios classificados como chattering: "
    f"{len(episodios_chattering):,}"
)


validacao_geracoes = (

    int(
        df[
            "Flag_Geracao_Efetiva"
        ].sum()
    )

    ==

    len(
        df_entradas
    )
)


print(
    "\nGerações consistentes..................:",
    validacao_geracoes
)


# Validação da hierarquia das janelas
valores_sensibilidade = (

    sensibilidade_chattering[
        "Transicoes_Rapidas"
    ]
    .tolist()
)


validacao_janelas = all(

    valores_sensibilidade[i]
    <=
    valores_sensibilidade[i + 1]

    for i in range(
        len(valores_sensibilidade) - 1
    )
)


print(
    "Janelas de chattering consistentes.....:",
    validacao_janelas
)


# =====================================================================
# 49. FINALIZAÇÃO
# =====================================================================

print("\n" + "=" * 110)
print("MÓDULO 5 CONCLUÍDO COM SUCESSO")
print("=" * 110)


print(
    "\nArquivos gerados:"
)

print(
    f"\n1. {ARQUIVO_RELATORIO}"
)

print(
    f"2. {ARQUIVO_RANKING}"
)

print(
    f"3. {ARQUIVO_EPISODIOS}"
)


print(
    """
Principais resultados disponíveis:

- Reincidência em 5, 10, 30, 60 minutos e 24 horas
- Chattering em 1, 5, 10, 15 e 30 minutos
- Retorno rápido
- Reativação rápida
- Episódios de chattering
- Ranking de equipamentos
- Ranking integrado com criticidade do Módulo 4
- Categoria
- Mantenedor
- Tipo
- Torre
- Evolução mensal
- Sensibilidade da janela temporal
"""
)


# =====================================================================
# 50. DOWNLOAD
# =====================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar os arquivos Parquet:"
)

print(
    f'files.download("{ARQUIVO_RANKING}")'
)

print(
    f'files.download("{ARQUIVO_EPISODIOS}")'
)

MÓDULO 5 - REINCIDÊNCIA, CHATTERING E COMPORTAMENTO REPETITIVO

Carregando base...
Base carregada com sucesso.
Registros: 1,413,882
Colunas: 58

BASE DE TRANSIÇÕES RELEVANTES
Transições Entrada/Retorno: 1,176,100


KeyboardInterrupt: 

In [11]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 6
# RECONSTRUÇÃO DAS OCORRÊNCIAS E ANÁLISE DE DURAÇÃO
#
# Base principal:
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Bases opcionais:
# Ranking_Criticidade_Alarmes_Metasys_2026.parquet
# Ranking_Modulo5_Reincidencia_Chattering.parquet
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Reconstruir ocorrências completas:
#       Entrada -> Retorno
#
# 2. Separar:
#       - Alarme
#       - Pré-Alarme
#       - Comunicação / Confiabilidade
#
# 3. Calcular:
#       - duração
#       - ocorrências abertas
#       - retornos sem entrada correspondente
#       - reentradas enquanto a ocorrência estava ativa
#
# 4. Produzir análises:
#       - mensal
#       - Categoria
#       - Mantenedor
#       - Tipo
#       - Torre
#       - equipamento
#       - faixas de duração
#
# 5. Integrar:
#       - criticidade do Módulo 4
#       - chattering / reincidência do Módulo 5
#
# IMPORTANTE
# ----------------------------------------------------------------------
# Eventos ainda abertos ao final da base NÃO terão duração presumida.
# Eles serão classificados como "Aberta ao final da base".
#
# Para esses casos será calculado apenas:
# "Tempo_Minimo_Aberto_Ate_Fim_Base_Min".
#
# As faixas de duração utilizadas são critérios analíticos do projeto,
# não parâmetros oficiais Johnson Controls / Metasys.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    250
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    280
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_ENTRADA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_CRITICIDADE = (
    "Ranking_Criticidade_Alarmes_Metasys_2026.parquet"
)

ARQUIVO_MODULO5 = (
    "Ranking_Modulo5_Reincidencia_Chattering.parquet"
)

ARQUIVO_OCORRENCIAS = (
    "Base_Ocorrencias_Reconstruidas_Modulo6.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo6_Duracao_Ocorrencias_Metasys.xlsx"
)


print("=" * 110)
print("MÓDULO 6 - RECONSTRUÇÃO DAS OCORRÊNCIAS E ANÁLISE DE DURAÇÃO")
print("=" * 110)


# ======================================================================
# 3. VERIFICAÇÃO / UPLOAD
# ======================================================================

if not Path(ARQUIVO_ENTRADA).exists():

    print(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' não foi localizado."
    )

    print(
        "\nSelecione o arquivo Parquet gerado no Módulo 2."
    )

    from google.colab import files

    uploaded = files.upload()


if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"O arquivo '{ARQUIVO_ENTRADA}' não foi localizado."
    )


# ======================================================================
# 4. CARREGAMENTO DA BASE
# ======================================================================

print("\nCarregando base classificada...")


df = pd.read_parquet(
    ARQUIVO_ENTRADA
)


print("Base carregada com sucesso.")

print(
    f"Registros: {len(df):,}"
)

print(
    f"Colunas: {df.shape[1]}"
)


# ======================================================================
# 5. COLUNAS NECESSÁRIAS
# ======================================================================

COLUNAS_NECESSARIAS = [

    "DataHoraUTC",
    "DataHoraLocal",

    "priority",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao"
]


faltantes = [

    coluna

    for coluna in COLUNAS_NECESSARIAS

    if coluna not in df.columns
]


if faltantes:

    raise ValueError(

        "Colunas necessárias ausentes:\n\n"

        + "\n".join(
            faltantes
        )
    )


# ======================================================================
# 6. PADRONIZAÇÃO TEMPORAL
# ======================================================================

df["DataHoraUTC"] = pd.to_datetime(
    df["DataHoraUTC"],
    errors="coerce"
)

df["DataHoraLocal"] = pd.to_datetime(
    df["DataHoraLocal"],
    errors="coerce"
)


# Final real da base no horário local
FIM_BASE_LOCAL = (
    df["DataHoraLocal"]
    .max()
)

INICIO_BASE_LOCAL = (
    df["DataHoraLocal"]
    .min()
)


print("\nPeríodo da base:")

print(
    f"Início: {INICIO_BASE_LOCAL}"
)

print(
    f"Fim...: {FIM_BASE_LOCAL}"
)


# ======================================================================
# 7. FUNÇÃO PARA MODA SEGURA
# ======================================================================

def moda_segura(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return pd.NA

    moda = serie.mode()

    if len(moda) > 0:
        return moda.iloc[0]

    return serie.iloc[0]


# ======================================================================
# 8. CADASTRO CONSOLIDADO POR itemName
# ======================================================================

cadastro = (

    df
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        itemDescription=(
            "itemDescription",
            moda_segura
        ),

        Categoria=(
            "Categoria",
            moda_segura
        ),

        Mantenedor=(
            "Mantenedor",
            moda_segura
        ),

        Tipo=(
            "Tipo",
            moda_segura
        ),

        Torre=(
            "Torre",
            moda_segura
        ),

        TTL_Torre=(
            "TTL Torre",
            moda_segura
        )
    )
    .reset_index()
)


# ======================================================================
# 9. FUNÇÃO PARA CRIAR BASE DE TRANSIÇÕES DE CADA FAMÍLIA
# ======================================================================

def preparar_transicoes(
    df_base,
    flag_entrada,
    flag_retorno,
    tipo_ocorrencia
):

    temp = (

        df_base.loc[
            (
                df_base[
                    flag_entrada
                ] == True
            )

            |

            (
                df_base[
                    flag_retorno
                ] == True
            ),

            [
                "DataHoraUTC",
                "DataHoraLocal",

                "priority",

                "itemName",
                "itemDescription",

                "Categoria",
                "Mantenedor",
                "Tipo",
                "Torre",
                "TTL Torre",

                flag_entrada,
                flag_retorno
            ]
        ]

        .copy()
    )


    temp[
        "Tipo_Ocorrencia"
    ] = tipo_ocorrencia


    temp[
        "Classe_Movimento"
    ] = np.where(

        temp[
            flag_entrada
        ] == True,

        "Entrada",

        "Retorno"
    )


    temp = (

        temp
        .sort_values(
            [
                "itemName",
                "DataHoraLocal"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    return temp


# ======================================================================
# 10. PREPARAÇÃO DAS TRÊS FAMÍLIAS
# ======================================================================

print("\nPreparando transições por família...")


trans_alarm = preparar_transicoes(

    df,

    "Flag_Entrada_Alarme",
    "Flag_Normalizacao_Alarme",

    "Alarme"
)


trans_pre = preparar_transicoes(

    df,

    "Flag_Entrada_PreAlarme",
    "Flag_Normalizacao_PreAlarme",

    "Pré-Alarme"
)


trans_com = preparar_transicoes(

    df,

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao",

    "Comunicação"
)


df_transicoes = pd.concat(

    [
        trans_alarm,
        trans_pre,
        trans_com
    ],

    ignore_index=True
)


df_transicoes = (

    df_transicoes
    .sort_values(
        [
            "Tipo_Ocorrencia",
            "itemName",
            "DataHoraLocal"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Transições utilizadas na reconstrução: "
    f"{len(df_transicoes):,}"
)


# ======================================================================
# 11. RECONSTRUÇÃO DAS OCORRÊNCIAS
#
# Regra:
#
# Entrada sem ocorrência aberta:
#     abre nova ocorrência.
#
# Entrada enquanto ocorrência já aberta:
#     contabiliza Reentrada_Enquanto_Aberta.
#
# Retorno com ocorrência aberta:
#     fecha a ocorrência.
#
# Retorno sem ocorrência aberta:
#     classificado como retorno órfão.
# ======================================================================

print("\nReconstruindo ocorrências...")


ocorrencias = []

retornos_orfaos = []


for (
    tipo_ocorrencia,
    item_name
), grupo in df_transicoes.groupby(

    [
        "Tipo_Ocorrencia",
        "itemName"
    ],

    observed=True,
    sort=False
):

    grupo = (
        grupo
        .sort_values(
            "DataHoraLocal"
        )
    )


    ocorrencia_aberta = None

    reentradas = 0


    for row in grupo.itertuples(
        index=False
    ):

        movimento = (
            row.Classe_Movimento
        )


        # ==============================================================
        # ENTRADA
        # ==============================================================

        if movimento == "Entrada":

            if ocorrencia_aberta is None:

                ocorrencia_aberta = {

                    "Tipo_Ocorrencia": (
                        tipo_ocorrencia
                    ),

                    "itemName": (
                        item_name
                    ),

                    "Inicio_Ocorrencia": (
                        row.DataHoraLocal
                    ),

                    "Inicio_Ocorrencia_UTC": (
                        row.DataHoraUTC
                    ),

                    "Prioridade_Entrada": (
                        row.priority
                    ),

                    "itemDescription": (
                        row.itemDescription
                    ),

                    "Categoria": (
                        row.Categoria
                    ),

                    "Mantenedor": (
                        row.Mantenedor
                    ),

                    "Tipo": (
                        row.Tipo
                    ),

                    "Torre": (
                        row.Torre
                    ),

                    "TTL_Torre": (
                        getattr(
                            row,
                            "_10",
                            pd.NA
                        )
                    )
                }


                # Como nomes com espaço podem gerar campos
                # alterados no itertuples, usamos cadastro
                # posteriormente para TTL_Torre.

                reentradas = 0


            else:

                reentradas += 1


        # ==============================================================
        # RETORNO
        # ==============================================================

        elif movimento == "Retorno":

            if ocorrencia_aberta is not None:

                fim = (
                    row.DataHoraLocal
                )

                duracao_min = (

                    (
                        fim

                        -

                        ocorrencia_aberta[
                            "Inicio_Ocorrencia"
                        ]
                    )

                    .total_seconds()

                    /

                    60
                )


                ocorrencia_aberta[
                    "Fim_Ocorrencia"
                ] = fim


                ocorrencia_aberta[
                    "Fim_Ocorrencia_UTC"
                ] = (
                    row.DataHoraUTC
                )


                ocorrencia_aberta[
                    "Duracao_Min"
                ] = max(
                    duracao_min,
                    0
                )


                ocorrencia_aberta[
                    "Duracao_Horas"
                ] = (

                    max(
                        duracao_min,
                        0
                    )

                    /

                    60
                )


                ocorrencia_aberta[
                    "Reentradas_Enquanto_Aberta"
                ] = (
                    reentradas
                )


                ocorrencia_aberta[
                    "Status_Fechamento"
                ] = (
                    "Fechada"
                )


                ocorrencia_aberta[
                    "Tempo_Minimo_Aberto_Ate_Fim_Base_Min"
                ] = np.nan


                ocorrencias.append(
                    ocorrencia_aberta
                )


                ocorrencia_aberta = None

                reentradas = 0


            else:

                retornos_orfaos.append({

                    "Tipo_Ocorrencia": (
                        tipo_ocorrencia
                    ),

                    "itemName": (
                        item_name
                    ),

                    "DataHora_Retorno": (
                        row.DataHoraLocal
                    )
                })


    # ==================================================================
    # OCORRÊNCIA AINDA ABERTA NO FINAL DA BASE
    # ==================================================================

    if ocorrencia_aberta is not None:

        tempo_minimo = (

            (
                FIM_BASE_LOCAL

                -

                ocorrencia_aberta[
                    "Inicio_Ocorrencia"
                ]
            )

            .total_seconds()

            /

            60
        )


        ocorrencia_aberta[
            "Fim_Ocorrencia"
        ] = pd.NaT


        ocorrencia_aberta[
            "Fim_Ocorrencia_UTC"
        ] = pd.NaT


        ocorrencia_aberta[
            "Duracao_Min"
        ] = np.nan


        ocorrencia_aberta[
            "Duracao_Horas"
        ] = np.nan


        ocorrencia_aberta[
            "Reentradas_Enquanto_Aberta"
        ] = (
            reentradas
        )


        ocorrencia_aberta[
            "Status_Fechamento"
        ] = (
            "Aberta ao final da base"
        )


        ocorrencia_aberta[
            "Tempo_Minimo_Aberto_Ate_Fim_Base_Min"
        ] = max(
            tempo_minimo,
            0
        )


        ocorrencias.append(
            ocorrencia_aberta
        )


# ======================================================================
# 12. CONVERSÃO PARA DATAFRAME
# ======================================================================

df_ocorrencias = pd.DataFrame(
    ocorrencias
)


df_retornos_orfaos = pd.DataFrame(
    retornos_orfaos
)


print(
    f"Ocorrências reconstruídas: "
    f"{len(df_ocorrencias):,}"
)

print(
    f"Retornos sem entrada correspondente: "
    f"{len(df_retornos_orfaos):,}"
)


# ======================================================================
# 13. ATUALIZAÇÃO DO CADASTRO
#
# Utilizamos cadastro consolidado para evitar inconsistências
# e também evitar problemas de nomes de colunas com espaços.
# ======================================================================

colunas_metadata = [

    "itemDescription",
    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL_Torre"
]


df_ocorrencias = (

    df_ocorrencias
    .drop(
        columns=[
            c
            for c in colunas_metadata
            if c in df_ocorrencias.columns
        ],
        errors="ignore"
    )

    .merge(
        cadastro,
        on="itemName",
        how="left"
    )
)


# ======================================================================
# 14. DATA LOCAL PARA COMPETÊNCIA
#
# IMPORTANTE:
# Usamos DataHoraLocal, não UTC.
# Isso corrige o efeito observado no Módulo 5,
# onde surgiram registros classificados em agosto.
# ======================================================================

df_ocorrencias[
    "Ano_Inicio"
] = (

    df_ocorrencias[
        "Inicio_Ocorrencia"
    ]
    .dt.year
)


df_ocorrencias[
    "Mes_Numero_Inicio"
] = (

    df_ocorrencias[
        "Inicio_Ocorrencia"
    ]
    .dt.month
)


MAPA_MESES = {

    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}


df_ocorrencias[
    "Mes_Inicio"
] = (

    df_ocorrencias[
        "Mes_Numero_Inicio"
    ]
    .map(
        MAPA_MESES
    )
)


df_ocorrencias[
    "Data_Inicio"
] = (

    df_ocorrencias[
        "Inicio_Ocorrencia"
    ]
    .dt.date
)


df_ocorrencias[
    "Hora_Inicio"
] = (

    df_ocorrencias[
        "Inicio_Ocorrencia"
    ]
    .dt.hour
)


# ======================================================================
# 15. FAIXAS DE DURAÇÃO
# ======================================================================

def classificar_duracao(valor):

    if pd.isna(valor):
        return "Ocorrência Aberta"

    if valor <= 1:
        return "Até 1 min"

    elif valor <= 5:
        return "1 a 5 min"

    elif valor <= 15:
        return "5 a 15 min"

    elif valor <= 30:
        return "15 a 30 min"

    elif valor <= 60:
        return "30 a 60 min"

    elif valor <= 240:
        return "1 a 4 horas"

    elif valor <= 720:
        return "4 a 12 horas"

    elif valor <= 1440:
        return "12 a 24 horas"

    else:
        return "Acima de 24 horas"


df_ocorrencias[
    "Faixa_Duracao"
] = (

    df_ocorrencias[
        "Duracao_Min"
    ]
    .apply(
        classificar_duracao
    )
)


ORDEM_FAIXAS = [

    "Até 1 min",
    "1 a 5 min",
    "5 a 15 min",
    "15 a 30 min",
    "30 a 60 min",
    "1 a 4 horas",
    "4 a 12 horas",
    "12 a 24 horas",
    "Acima de 24 horas",
    "Ocorrência Aberta"
]


# ======================================================================
# 16. FLAGS DE DURAÇÃO
# ======================================================================

df_ocorrencias[
    "Flag_Fechada"
] = (

    df_ocorrencias[
        "Status_Fechamento"
    ]
    ==
    "Fechada"
)


df_ocorrencias[
    "Flag_Aberta"
] = (

    df_ocorrencias[
        "Status_Fechamento"
    ]
    ==
    "Aberta ao final da base"
)


df_ocorrencias[
    "Flag_Maior_15min"
] = (

    df_ocorrencias[
        "Duracao_Min"
    ]
    >
    15
)


df_ocorrencias[
    "Flag_Maior_60min"
] = (

    df_ocorrencias[
        "Duracao_Min"
    ]
    >
    60
)


df_ocorrencias[
    "Flag_Maior_4h"
] = (

    df_ocorrencias[
        "Duracao_Min"
    ]
    >
    240
)


df_ocorrencias[
    "Flag_Maior_24h"
] = (

    df_ocorrencias[
        "Duracao_Min"
    ]
    >
    1440
)


# ======================================================================
# 17. BASE APENAS DAS OCORRÊNCIAS FECHADAS
# ======================================================================

df_fechadas = (

    df_ocorrencias.loc[
        df_ocorrencias[
            "Flag_Fechada"
        ] == True
    ]

    .copy()
)


# ======================================================================
# 18. KPIs GERAIS
# ======================================================================

TOTAL_OCORRENCIAS = len(
    df_ocorrencias
)

TOTAL_FECHADAS = int(
    df_ocorrencias[
        "Flag_Fechada"
    ].sum()
)

TOTAL_ABERTAS = int(
    df_ocorrencias[
        "Flag_Aberta"
    ].sum()
)

TOTAL_REENTRADAS = int(
    df_ocorrencias[
        "Reentradas_Enquanto_Aberta"
    ].sum()
)


DURACAO_MEDIA_MIN = (
    df_fechadas[
        "Duracao_Min"
    ]
    .mean()
)

DURACAO_MEDIANA_MIN = (
    df_fechadas[
        "Duracao_Min"
    ]
    .median()
)

DURACAO_P90_MIN = (
    df_fechadas[
        "Duracao_Min"
    ]
    .quantile(
        0.90
    )
)

DURACAO_P95_MIN = (
    df_fechadas[
        "Duracao_Min"
    ]
    .quantile(
        0.95
    )
)

DURACAO_P99_MIN = (
    df_fechadas[
        "Duracao_Min"
    ]
    .quantile(
        0.99
    )
)


HORAS_TOTAIS_CONDICAO = (

    df_fechadas[
        "Duracao_Horas"
    ]
    .sum()
)


kpis = pd.DataFrame(
    {
        "Indicador": [

            "Ocorrências reconstruídas",

            "Ocorrências fechadas",

            "Ocorrências abertas ao final da base",

            "Retornos órfãos",

            "Reentradas enquanto ocorrência aberta",

            "Duração média - minutos",

            "Duração mediana - minutos",

            "Percentil 90 - minutos",

            "Percentil 95 - minutos",

            "Percentil 99 - minutos",

            "Horas acumuladas de condição anormal"
        ],

        "Valor": [

            TOTAL_OCORRENCIAS,

            TOTAL_FECHADAS,

            TOTAL_ABERTAS,

            len(
                df_retornos_orfaos
            ),

            TOTAL_REENTRADAS,

            DURACAO_MEDIA_MIN,

            DURACAO_MEDIANA_MIN,

            DURACAO_P90_MIN,

            DURACAO_P95_MIN,

            DURACAO_P99_MIN,

            HORAS_TOTAIS_CONDICAO
        ]
    }
)


# ======================================================================
# 19. RESUMO POR TIPO DE OCORRÊNCIA
# ======================================================================

resumo_tipo_ocorrencia = (

    df_ocorrencias
    .groupby(
        "Tipo_Ocorrencia",
        observed=True
    )
    .agg(

        Ocorrencias=(
            "itemName",
            "size"
        ),

        Fechadas=(
            "Flag_Fechada",
            "sum"
        ),

        Abertas=(
            "Flag_Aberta",
            "sum"
        ),

        Reentradas=(
            "Reentradas_Enquanto_Aberta",
            "sum"
        ),

        Duracao_Media_Min=(
            "Duracao_Min",
            "mean"
        ),

        Duracao_Mediana_Min=(
            "Duracao_Min",
            "median"
        ),

        Duracao_Maxima_Min=(
            "Duracao_Min",
            "max"
        ),

        Horas_Acumuladas=(
            "Duracao_Horas",
            "sum"
        ),

        Equipamentos=(
            "itemName",
            "nunique"
        )
    )

    .reset_index()

    .sort_values(
        "Ocorrencias",
        ascending=False
    )
)


# ======================================================================
# 20. RESUMO POR FAIXA DE DURAÇÃO
# ======================================================================

resumo_faixas = (

    df_ocorrencias[
        "Faixa_Duracao"
    ]
    .value_counts(
        dropna=False
    )
    .reindex(
        ORDEM_FAIXAS,
        fill_value=0
    )
    .reset_index()
)


resumo_faixas.columns = [

    "Faixa_Duracao",
    "Ocorrencias"
]


resumo_faixas[
    "Percentual"
] = (

    resumo_faixas[
        "Ocorrencias"
    ]

    /

    TOTAL_OCORRENCIAS

    * 100
)


# ======================================================================
# 21. FUNÇÃO DE RESUMO POR DIMENSÃO
# ======================================================================

def gerar_resumo_dimensao(
    df_base,
    coluna
):

    resultado = (

        df_base
        .groupby(
            coluna,
            observed=True,
            dropna=False
        )
        .agg(

            Ocorrencias=(
                "itemName",
                "size"
            ),

            Fechadas=(
                "Flag_Fechada",
                "sum"
            ),

            Abertas=(
                "Flag_Aberta",
                "sum"
            ),

            Reentradas=(
                "Reentradas_Enquanto_Aberta",
                "sum"
            ),

            Duracao_Media_Min=(
                "Duracao_Min",
                "mean"
            ),

            Duracao_Mediana_Min=(
                "Duracao_Min",
                "median"
            ),

            Duracao_Maxima_Min=(
                "Duracao_Min",
                "max"
            ),

            Ocorrencias_Maior_60min=(
                "Flag_Maior_60min",
                "sum"
            ),

            Ocorrencias_Maior_4h=(
                "Flag_Maior_4h",
                "sum"
            ),

            Ocorrencias_Maior_24h=(
                "Flag_Maior_24h",
                "sum"
            ),

            Horas_Acumuladas=(
                "Duracao_Horas",
                "sum"
            ),

            Equipamentos=(
                "itemName",
                "nunique"
            )
        )

        .reset_index()
    )


    resultado[
        "Perc_Abertas"
    ] = np.where(

        resultado[
            "Ocorrencias"
        ] > 0,

        resultado[
            "Abertas"
        ]

        /

        resultado[
            "Ocorrencias"
        ]

        * 100,

        0
    )


    resultado[
        "Perc_Maior_60min"
    ] = np.where(

        resultado[
            "Fechadas"
        ] > 0,

        resultado[
            "Ocorrencias_Maior_60min"
        ]

        /

        resultado[
            "Fechadas"
        ]

        * 100,

        0
    )


    return (

        resultado
        .sort_values(
            "Horas_Acumuladas",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


# ======================================================================
# 22. RESUMOS POR DIMENSÃO
# ======================================================================

resumo_categoria = gerar_resumo_dimensao(
    df_ocorrencias,
    "Categoria"
)

resumo_mantenedor = gerar_resumo_dimensao(
    df_ocorrencias,
    "Mantenedor"
)

resumo_tipo = gerar_resumo_dimensao(
    df_ocorrencias,
    "Tipo"
)

resumo_torre = gerar_resumo_dimensao(
    df_ocorrencias,
    "Torre"
)


# ======================================================================
# 23. RESUMO MENSAL
# ======================================================================

resumo_mensal = (

    df_ocorrencias
    .groupby(

        [
            "Ano_Inicio",
            "Mes_Numero_Inicio",
            "Mes_Inicio"
        ],

        observed=True
    )
    .agg(

        Ocorrencias=(
            "itemName",
            "size"
        ),

        Fechadas=(
            "Flag_Fechada",
            "sum"
        ),

        Abertas=(
            "Flag_Aberta",
            "sum"
        ),

        Reentradas=(
            "Reentradas_Enquanto_Aberta",
            "sum"
        ),

        Duracao_Media_Min=(
            "Duracao_Min",
            "mean"
        ),

        Duracao_Mediana_Min=(
            "Duracao_Min",
            "median"
        ),

        Duracao_Maxima_Min=(
            "Duracao_Min",
            "max"
        ),

        Horas_Acumuladas=(
            "Duracao_Horas",
            "sum"
        ),

        Equipamentos=(
            "itemName",
            "nunique"
        )
    )

    .reset_index()

    .sort_values(
        [
            "Ano_Inicio",
            "Mes_Numero_Inicio"
        ]
    )
)


# ======================================================================
# 24. RANKING POR EQUIPAMENTO
# ======================================================================

ranking_equipamentos = (

    df_ocorrencias
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        Ocorrencias=(
            "itemName",
            "size"
        ),

        Fechadas=(
            "Flag_Fechada",
            "sum"
        ),

        Abertas=(
            "Flag_Aberta",
            "sum"
        ),

        Reentradas=(
            "Reentradas_Enquanto_Aberta",
            "sum"
        ),

        Duracao_Media_Min=(
            "Duracao_Min",
            "mean"
        ),

        Duracao_Mediana_Min=(
            "Duracao_Min",
            "median"
        ),

        Duracao_Maxima_Min=(
            "Duracao_Min",
            "max"
        ),

        Horas_Acumuladas=(
            "Duracao_Horas",
            "sum"
        ),

        Ocorrencias_Maior_60min=(
            "Flag_Maior_60min",
            "sum"
        ),

        Ocorrencias_Maior_4h=(
            "Flag_Maior_4h",
            "sum"
        ),

        Ocorrencias_Maior_24h=(
            "Flag_Maior_24h",
            "sum"
        )
    )

    .reset_index()

    .merge(
        cadastro,
        on="itemName",
        how="left"
    )
)


ranking_equipamentos[
    "Perc_Abertas"
] = np.where(

    ranking_equipamentos[
        "Ocorrencias"
    ] > 0,

    ranking_equipamentos[
        "Abertas"
    ]

    /

    ranking_equipamentos[
        "Ocorrencias"
    ]

    * 100,

    0
)


# ======================================================================
# 25. INTEGRAÇÃO COM CRITICIDADE - MÓDULO 4
#
# IMPORTANTE:
# Deduplicação explícita por itemName para impedir
# merge muitos-para-muitos observado no Módulo 5.
# ======================================================================

if Path(
    ARQUIVO_CRITICIDADE
).exists():

    print(
        "\nIntegrando criticidade do Módulo 4..."
    )


    crit = pd.read_parquet(
        ARQUIVO_CRITICIDADE
    )


    colunas_crit = [

        "itemName",

        "Indice_Criticidade_V1",
        "Classe_Criticidade",
        "Ranking_Criticidade"
    ]


    colunas_crit = [

        c

        for c in colunas_crit

        if c in crit.columns
    ]


    crit = (

        crit[
            colunas_crit
        ]

        .sort_values(
            "Indice_Criticidade_V1",
            ascending=False
        )

        .drop_duplicates(
            subset="itemName",
            keep="first"
        )
    )


    ranking_equipamentos = (

        ranking_equipamentos
        .merge(
            crit,
            on="itemName",
            how="left"
        )
    )


# ======================================================================
# 26. INTEGRAÇÃO COM CHATTERING - MÓDULO 5
#
# Também deduplicado por itemName.
# ======================================================================

if Path(
    ARQUIVO_MODULO5
).exists():

    print(
        "Integrando comportamento do Módulo 5..."
    )


    mod5 = pd.read_parquet(
        ARQUIVO_MODULO5
    )


    colunas_mod5 = [

        "itemName",

        "Episodios_Chattering_10min",
        "Reincidencias_10min",
        "Taxa_Chattering_10min_Perc",
        "Classe_Comportamento",

        "Indice_Prioridade_Investigacao",
        "Classe_Prioridade_Investigacao"
    ]


    colunas_mod5 = [

        c

        for c in colunas_mod5

        if c in mod5.columns
    ]


    # Mantém a linha de maior prioridade de investigação
    if (
        "Indice_Prioridade_Investigacao"
        in mod5.columns
    ):

        mod5 = (

            mod5[
                colunas_mod5
            ]
            .sort_values(
                "Indice_Prioridade_Investigacao",
                ascending=False
            )
            .drop_duplicates(
                subset="itemName",
                keep="first"
            )
        )


    else:

        mod5 = (

            mod5[
                colunas_mod5
            ]
            .drop_duplicates(
                subset="itemName",
                keep="first"
            )
        )


    ranking_equipamentos = (

        ranking_equipamentos
        .merge(
            mod5,
            on="itemName",
            how="left"
        )
    )


# ======================================================================
# 27. SCORE DE IMPACTO POR DURAÇÃO
# ======================================================================

ranking_equipamentos[
    "Score_Horas_Acumuladas"
] = (

    ranking_equipamentos[
        "Horas_Acumuladas"
    ]
    .rank(
        pct=True,
        method="average"
    )

    * 100
)


ranking_equipamentos[
    "Score_Duracao_Mediana"
] = (

    ranking_equipamentos[
        "Duracao_Mediana_Min"
    ]
    .fillna(0)
    .rank(
        pct=True,
        method="average"
    )

    * 100
)


ranking_equipamentos[
    "Score_Ocorrencias_Abertas"
] = (

    ranking_equipamentos[
        "Abertas"
    ]
    .rank(
        pct=True,
        method="average"
    )

    * 100
)


# ======================================================================
# 28. ÍNDICE DE IMPACTO TEMPORAL
#
# 50% horas acumuladas
# 30% duração mediana
# 20% ocorrências abertas
# ======================================================================

ranking_equipamentos[
    "Indice_Impacto_Temporal"
] = (

    ranking_equipamentos[
        "Score_Horas_Acumuladas"
    ]
    * 0.50

    +

    ranking_equipamentos[
        "Score_Duracao_Mediana"
    ]
    * 0.30

    +

    ranking_equipamentos[
        "Score_Ocorrencias_Abertas"
    ]
    * 0.20
)


ranking_equipamentos[
    "Classe_Impacto_Temporal"
] = np.select(

    [
        ranking_equipamentos[
            "Indice_Impacto_Temporal"
        ] >= 80,

        ranking_equipamentos[
            "Indice_Impacto_Temporal"
        ] >= 60,

        ranking_equipamentos[
            "Indice_Impacto_Temporal"
        ] >= 40
    ],

    [
        "Muito Alto",
        "Alto",
        "Moderado"
    ],

    default="Baixo"
)


# ======================================================================
# 29. PRIORIZAÇÃO INTEGRADA
#
# Se criticidade estiver disponível:
#
# 50% Criticidade
# 30% Impacto temporal
# 20% Chattering (se disponível)
# ======================================================================

if (
    "Indice_Criticidade_V1"
    in ranking_equipamentos.columns
):

    if (
        "Episodios_Chattering_10min"
        in ranking_equipamentos.columns
    ):

        ranking_equipamentos[
            "Score_Chattering_Mod6"
        ] = (

            ranking_equipamentos[
                "Episodios_Chattering_10min"
            ]
            .fillna(0)
            .rank(
                pct=True,
                method="average"
            )

            * 100
        )


        ranking_equipamentos[
            "Indice_Prioridade_Integrada_Mod6"
        ] = (

            ranking_equipamentos[
                "Indice_Criticidade_V1"
            ]
            .fillna(0)
            * 0.50

            +

            ranking_equipamentos[
                "Indice_Impacto_Temporal"
            ]
            * 0.30

            +

            ranking_equipamentos[
                "Score_Chattering_Mod6"
            ]
            * 0.20
        )


    else:

        ranking_equipamentos[
            "Indice_Prioridade_Integrada_Mod6"
        ] = (

            ranking_equipamentos[
                "Indice_Criticidade_V1"
            ]
            .fillna(0)
            * 0.60

            +

            ranking_equipamentos[
                "Indice_Impacto_Temporal"
            ]
            * 0.40
        )


    ranking_equipamentos[
        "Classe_Prioridade_Integrada_Mod6"
    ] = np.select(

        [
            ranking_equipamentos[
                "Indice_Prioridade_Integrada_Mod6"
            ] >= 80,

            ranking_equipamentos[
                "Indice_Prioridade_Integrada_Mod6"
            ] >= 60,

            ranking_equipamentos[
                "Indice_Prioridade_Integrada_Mod6"
            ] >= 40
        ],

        [
            "Prioridade 1",
            "Prioridade 2",
            "Prioridade 3"
        ],

        default="Prioridade 4"
    )


    ranking_equipamentos = (

        ranking_equipamentos
        .sort_values(
            [
                "Indice_Prioridade_Integrada_Mod6",
                "Horas_Acumuladas"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


else:

    ranking_equipamentos = (

        ranking_equipamentos
        .sort_values(
            [
                "Indice_Impacto_Temporal",
                "Horas_Acumuladas"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


ranking_equipamentos[
    "Ranking_Modulo6"
] = (

    np.arange(
        1,
        len(
            ranking_equipamentos
        ) + 1
    )
)


# ======================================================================
# 30. TOP OCORRÊNCIAS MAIS LONGAS
# ======================================================================

top_ocorrencias_longas = (

    df_fechadas
    .sort_values(
        "Duracao_Min",
        ascending=False
    )
    .head(500)
    .copy()
)


# ======================================================================
# 31. OCORRÊNCIAS AINDA ABERTAS
# ======================================================================

ocorrencias_abertas = (

    df_ocorrencias.loc[
        df_ocorrencias[
            "Flag_Aberta"
        ] == True
    ]
    .sort_values(
        "Tempo_Minimo_Aberto_Ate_Fim_Base_Min",
        ascending=False
    )
    .copy()
)


# ======================================================================
# 32. TOP REENTRADAS
# ======================================================================

top_reentradas = (

    df_ocorrencias
    .sort_values(
        "Reentradas_Enquanto_Aberta",
        ascending=False
    )
    .head(500)
    .copy()
)


# ======================================================================
# 33. MATRIZ TIPO DE OCORRÊNCIA × FAIXA
# ======================================================================

matriz_tipo_faixa = pd.crosstab(

    df_ocorrencias[
        "Tipo_Ocorrencia"
    ],

    df_ocorrencias[
        "Faixa_Duracao"
    ]
)


matriz_tipo_faixa = (

    matriz_tipo_faixa
    .reindex(
        columns=[
            c
            for c in ORDEM_FAIXAS
            if c in matriz_tipo_faixa.columns
        ],
        fill_value=0
    )
)


# ======================================================================
# 34. MATRIZ CATEGORIA × TIPO DE OCORRÊNCIA
#     HORAS ACUMULADAS
# ======================================================================

matriz_categoria_ocorrencia = pd.pivot_table(

    df_ocorrencias,

    index="Categoria",

    columns="Tipo_Ocorrencia",

    values="Duracao_Horas",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ======================================================================
# 35. RESUMO EXECUTIVO
# ======================================================================

if len(
    ranking_equipamentos
) > 0:

    lider_temporal = (
        ranking_equipamentos
        .iloc[0]
    )

else:

    lider_temporal = None


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Ocorrências reconstruídas",

            "Ocorrências fechadas",

            "Ocorrências abertas",

            "% ocorrências abertas",

            "Retornos sem entrada correspondente",

            "Reentradas enquanto abertas",

            "Duração média das fechadas - min",

            "Duração mediana das fechadas - min",

            "P90 de duração - min",

            "P95 de duração - min",

            "P99 de duração - min",

            "Horas acumuladas de condição anormal",

            "Equipamento líder em impacto temporal"
        ],

        "Resultado": [

            f"{TOTAL_OCORRENCIAS:,}",

            f"{TOTAL_FECHADAS:,}",

            f"{TOTAL_ABERTAS:,}",

            (
                f"{TOTAL_ABERTAS / TOTAL_OCORRENCIAS * 100:.2f}%"
                if TOTAL_OCORRENCIAS > 0
                else "0.00%"
            ),

            f"{len(df_retornos_orfaos):,}",

            f"{TOTAL_REENTRADAS:,}",

            f"{DURACAO_MEDIA_MIN:,.2f}",

            f"{DURACAO_MEDIANA_MIN:,.2f}",

            f"{DURACAO_P90_MIN:,.2f}",

            f"{DURACAO_P95_MIN:,.2f}",

            f"{DURACAO_P99_MIN:,.2f}",

            f"{HORAS_TOTAIS_CONDICAO:,.2f}",

            (
                str(
                    lider_temporal[
                        "itemName"
                    ]
                )
                if lider_temporal is not None
                else "N/A"
            )
        ]
    }
)


# ======================================================================
# 36. EXIBIÇÃO DOS PRINCIPAIS RESULTADOS
# ======================================================================

print("\n" + "=" * 110)
print("RESUMO EXECUTIVO")
print("=" * 110)

display(
    resumo_executivo
)


print("\n" + "=" * 110)
print("TIPO DE OCORRÊNCIA")
print("=" * 110)

display(
    resumo_tipo_ocorrencia
)


print("\n" + "=" * 110)
print("FAIXAS DE DURAÇÃO")
print("=" * 110)

display(
    resumo_faixas
)


print("\n" + "=" * 110)
print("RESUMO MENSAL")
print("=" * 110)

display(
    resumo_mensal
)


print("\n" + "=" * 110)
print("TOP 30 EQUIPAMENTOS - IMPACTO TEMPORAL / PRIORIDADE")
print("=" * 110)

display(
    ranking_equipamentos.head(30)
)


print("\n" + "=" * 110)
print("TOP 30 OCORRÊNCIAS MAIS LONGAS")
print("=" * 110)

display(
    top_ocorrencias_longas.head(30)
)


print("\n" + "=" * 110)
print("OCORRÊNCIAS ABERTAS MAIS ANTIGAS")
print("=" * 110)

display(
    ocorrencias_abertas.head(30)
)


# ======================================================================
# 37. PREPARAÇÃO PARA EXPORTAÇÃO EXCEL
#
# Excel não aceita datetime timezone-aware.
# Criamos cópias sem timezone SOMENTE para exportação.
# ======================================================================

def remover_timezone_dataframe(df_original):

    temp = (
        df_original
        .copy()
    )

    for coluna in temp.columns:

        if isinstance(
            temp[
                coluna
            ].dtype,
            pd.DatetimeTZDtype
        ):

            temp[
                coluna
            ] = (

                temp[
                    coluna
                ]
                .dt.tz_localize(
                    None
                )
            )

    return temp


top_ocorrencias_excel = remover_timezone_dataframe(
    top_ocorrencias_longas
)

abertas_excel = remover_timezone_dataframe(
    ocorrencias_abertas.head(5000)
)

reentradas_excel = remover_timezone_dataframe(
    top_reentradas
)

retornos_orfaos_excel = remover_timezone_dataframe(
    df_retornos_orfaos.head(10000)
)


# ======================================================================
# 38. SALVAMENTO DA BASE COMPLETA EM PARQUET
# ======================================================================

print(
    "\nSalvando base completa de ocorrências reconstruídas..."
)


df_ocorrencias.to_parquet(
    ARQUIVO_OCORRENCIAS,
    index=False
)


print(
    f"Arquivo criado:\n"
    f"{ARQUIVO_OCORRENCIAS}"
)


# ======================================================================
# 39. EXPORTAÇÃO DO RELATÓRIO EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 6..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    kpis.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )


    resumo_tipo_ocorrencia.to_excel(
        writer,
        sheet_name="Tipo Ocorrencia",
        index=False
    )


    resumo_faixas.to_excel(
        writer,
        sheet_name="Faixas Duracao",
        index=False
    )


    resumo_mensal.to_excel(
        writer,
        sheet_name="Mensal",
        index=False
    )


    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )


    resumo_mantenedor.to_excel(
        writer,
        sheet_name="Mantenedor",
        index=False
    )


    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )


    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )


    ranking_equipamentos.to_excel(
        writer,
        sheet_name="Ranking Equipamentos",
        index=False
    )


    top_ocorrencias_excel.to_excel(
        writer,
        sheet_name="Top Ocorrencias Longas",
        index=False
    )


    abertas_excel.to_excel(
        writer,
        sheet_name="Ocorrencias Abertas",
        index=False
    )


    reentradas_excel.to_excel(
        writer,
        sheet_name="Top Reentradas",
        index=False
    )


    retornos_orfaos_excel.to_excel(
        writer,
        sheet_name="Retornos Orfaos",
        index=False
    )


    matriz_tipo_faixa.to_excel(
        writer,
        sheet_name="Matriz Tipo x Faixa"
    )


    matriz_categoria_ocorrencia.to_excel(
        writer,
        sheet_name="Cat x Tipo Ocorr"
    )


# ======================================================================
# 40. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Item": [

            "Alarme",

            "Pré-Alarme",

            "Comunicação",

            "Reentrada",

            "Ocorrência fechada",

            "Ocorrência aberta",

            "Duração",

            "Tempo mínimo aberto",

            "Competência mensal",

            "Índice de Impacto Temporal"
        ],

        "Definicao": [

            (
                "Entrada de alarme pareada com "
                "Normalização de Alarme."
            ),

            (
                "Entrada de pré-alarme pareada com "
                "Normalização de Pré-Alarme."
            ),

            (
                "Falha de comunicação/confiabilidade pareada "
                "com Restabelecimento de Comunicação."
            ),

            (
                "Nova entrada registrada enquanto uma ocorrência "
                "da mesma família ainda estava aberta."
            ),

            (
                "Entrada com retorno correspondente posterior."
            ),

            (
                "Entrada sem retorno correspondente até o "
                "último registro disponível da base."
            ),

            (
                "Diferença entre retorno e entrada para "
                "ocorrências fechadas."
            ),

            (
                "Tempo conhecido entre a entrada e o final da base "
                "para ocorrências ainda abertas."
            ),

            (
                "Definida pela DataHoraLocal da entrada da ocorrência."
            ),

            (
                "50% horas acumuladas + "
                "30% duração mediana + "
                "20% ocorrências abertas."
            )
        ]
    }
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl",

    mode="a",

    if_sheet_exists="replace"

) as writer:


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


# ======================================================================
# 41. FORMATAÇÃO BÁSICA DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0

        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                max_length = max(
                    max_length,
                    len(valor)
                )

            except:
                pass


        largura = min(
            max(
                max_length + 2,
                12
            ),
            45
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 42. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 110)
print("VALIDAÇÕES")
print("=" * 110)


print(
    f"Ocorrências reconstruídas..............: "
    f"{TOTAL_OCORRENCIAS:,}"
)


print(
    f"Ocorrências fechadas...................: "
    f"{TOTAL_FECHADAS:,}"
)


print(
    f"Ocorrências abertas....................: "
    f"{TOTAL_ABERTAS:,}"
)


print(
    f"Fechadas + abertas.....................: "
    f"{TOTAL_FECHADAS + TOTAL_ABERTAS:,}"
)


print(
    f"Retornos órfãos........................: "
    f"{len(df_retornos_orfaos):,}"
)


validacao_ocorrencias = (

    TOTAL_OCORRENCIAS

    ==

    (
        TOTAL_FECHADAS
        +
        TOTAL_ABERTAS
    )
)


print(
    "\nFechadas + abertas consistente..........:",
    validacao_ocorrencias
)


# ----------------------------------------------------------------------
# Validação de durações negativas
# ----------------------------------------------------------------------

duracoes_negativas = (

    df_fechadas[
        "Duracao_Min"
    ]
    .lt(0)
    .sum()
)


print(
    "Durações negativas......................:",
    int(
        duracoes_negativas
    )
)


# ----------------------------------------------------------------------
# Validação da competência mensal
# ----------------------------------------------------------------------

meses_identificados = sorted(

    df_ocorrencias[
        "Mes_Numero_Inicio"
    ]
    .dropna()
    .unique()
    .tolist()
)


print(
    "Meses identificados pela DataHoraLocal..:",
    meses_identificados
)


# ======================================================================
# 43. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 110)
print("MÓDULO 6 CONCLUÍDO COM SUCESSO")
print("=" * 110)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


print(
    f"2. {ARQUIVO_OCORRENCIAS}"
)


print(
    """
Principais resultados:

- Ocorrências efetivamente reconstruídas
- Duração média e mediana
- P90, P95 e P99
- Ocorrências ainda abertas
- Reentradas enquanto condição estava ativa
- Retornos órfãos
- Horas acumuladas de condição anormal
- Faixas de duração
- Ranking de equipamentos
- Categoria
- Mantenedor
- Tipo
- Torre
- Evolução mensal
- Integração com criticidade
- Integração com chattering
- Índice de Impacto Temporal
- Prioridade Integrada do Módulo 6
"""
)


# ======================================================================
# 44. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar a base completa de ocorrências:"
)


print(
    f'files.download("{ARQUIVO_OCORRENCIAS}")'
)

MÓDULO 6 - RECONSTRUÇÃO DAS OCORRÊNCIAS E ANÁLISE DE DURAÇÃO

Carregando base classificada...
Base carregada com sucesso.
Registros: 1,413,882
Colunas: 58

Período da base:
Início: 2026-01-01 00:00:29-03:00
Fim...: 2026-07-31 23:58:32-03:00

Preparando transições por família...
Transições utilizadas na reconstrução: 1,176,100

Reconstruindo ocorrências...
Ocorrências reconstruídas: 438,482
Retornos sem entrada correspondente: 104,820

Integrando criticidade do Módulo 4...
Integrando comportamento do Módulo 5...

RESUMO EXECUTIVO


,Indicador,Resultado
0,Ocorrências reconstruídas,"438,482"
1,Ocorrências fechadas,"433,714"
2,Ocorrências abertas,"4,768"
3,% ocorrências abertas,1.09%
4,Retornos sem entrada correspondente,"104,820"
5,Reentradas enquanto abertas,"199,084"
6,Duração média das fechadas - min,629.65
7,Duração mediana das fechadas - min,15.77
8,P90 de duração - min,839.85
9,P95 de duração - min,"1,856.45"



TIPO DE OCORRÊNCIA


,Tipo_Ocorrencia,Ocorrencias,Fechadas,Abertas,Reentradas,Duracao_Media_Min,Duracao_Mediana_Min,Duracao_Maxima_Min,Horas_Acumuladas,Equipamentos
2,Pré-Alarme,247582,246934,648,49143,738.11,53.17,"244,097.53","3,037,746.28",1857
0,Alarme,179813,178734,1079,60464,387.04,3.07,"291,397.02","1,152,965.62",2075
1,Comunicação,11087,8046,3041,89477,"2,690.15",0.70,"230,563.45","360,748.95",3497



FAIXAS DE DURAÇÃO


...

,Tipo_Ocorrencia,itemName,Inicio_Ocorrencia,Inicio_Ocorrencia_UTC,Prioridade_Entrada,Fim_Ocorrencia,Fim_Ocorrencia_UTC,Duracao_Min,Duracao_Horas,Reentradas_Enquanto_Aberta,Status_Fechamento,Tempo_Minimo_Aberto_Ate_Fim_Base_Min,itemDescription,Categoria,Mantenedor,Tipo,Torre,TTL_Torre,Ano_Inicio,Mes_Numero_Inicio,Mes_Inicio,Data_Inicio,Hora_Inicio,Faixa_Duracao,Flag_Fechada,Flag_Aberta,Flag_Maior_15min,Flag_Maior_60min,Flag_Maior_4h,Flag_Maior_24h
189699,Comunicação,VAG030 (CEITOC1ICM01FANC06),2026-01-01 00:09:15-03:00,2026-01-01 03:09:15,106,NaT,NaT,NaN,NaN,264,Aberta ao final da base,"305,269.28",Energy Valve - Fancoil 06 01º Interm T.C,Sistema,Automação,Off-line,TCO,CEITOC,2026,1,Janeiro,2026-01-01,0,Ocorrência Aberta,False,True,False,False,False,False
189726,Comunicação,VAG034 (CEITE22SCM01FANC01),2026-01-01 00:19:11-03:00,2026-01-01 03:19:11,106,NaT,NaT,NaN,NaN,563,Aberta ao final da base,"305,259.35",CEITE22SCM01FANC01,Sistema,Automação,Off-line,TOS,CEITE2,2026,1,Janeiro,2026-01-01,0,Ocorrência Aberta,False,True,False,False,False,False
188970,Comunicação,FEC048,2026-01-01 01:07:32-03:00,2026-01-01 04:07:32,106,NaT,NaT,NaN,NaN,12,Aberta ao final da base,"305,211.00",<NA>,Sistema,Automação,Off-line,CEA,CEICEA,2026,1,Janeiro,2026-01-01,1,Ocorrência Aberta,False,True,False,False,False,False
188386,Comunicação,CVM007 [CVAV049],2026-01-01 01:16:46-03:00,2026-01-01 04:16:46,106,NaT,NaT,NaN,NaN,36,Aberta ao final da base,"305,201.77","VAV CTRL/ACT/DP, 3UI, 2CO, 3 BO",Sistema,Automação,Off-line,CEA,CEICEA,2026,1,Janeiro,2026-01-01,1,Ocorrência Aberta,False,True,False,False,False,False
188506,Comunicação,CVM025 [CVAV067],2026-01-01 01:16:46-03:00,2026-01-01 04:16:46,106,NaT,NaT,NaN,NaN,36,Aberta ao final da base,"305,201.77","VAV CTRL/ACT/DP, 3UI, 2CO, 3 BO",Sistema,Automação,Off-line,CEA,CEICEA,2026,1,Janeiro,2026-01-01,1,Ocorrência Aberta,False,True,False,False,False,False
188492,Comunicação,CVM023 [CVAV065],2026-01-01 01:16:47-03:00,2026-01-01 04:16:47,106,NaT,NaT,NaN,NaN,39,Aberta ao final da base,"305,201.75","VAV CTRL/ACT/DP, 3UI, 2CO, 3 BO",Sistema,Automação,Off-line,CEA,CEICEA,2026,1,Janeiro,2026-01-01,1,Ocorrência Aberta,False,True,False,False,False,False
111056,Alarme,CEITE501ST01FCLT01_ZN-TEM,2026-01-01 01:30:39-03:00,2026-01-01 04:30:39,2,NaT,NaT,NaN,NaN,192,Aberta ao final da base,"305,187.88",T. Ambiente 01° (Sala Técnica) 1º Andar T. EV,HVAC,HVAC,Temperatura,TEV,CEITE5,2026,1,Janeiro,2026-01-01,1,Ocorrência Aberta,False,True,False,False,False,False
189756,Comunicação,VAG042 (CEITE23SCM01FANC02),2026-01-01 03:32:06-03:00,2026-01-01 06:32:06,106,NaT,NaT,NaN,NaN,82,Aberta ao final da base,"305,066.43",CEITE23SCM01FANC02,Sistema,Automação,Off-line,TOS,CEITE2,2026,1,Janeiro,2026-01-01,3,Ocorrência Aberta,False,True,False,False,False,False
177292,Alarme,TOC-06-SNE-11061-MC [CASS - 07o a...,2026-01-01 05:00:00-03:00,2026-01-01 08:00:00,70,NaT,NaT,NaN,NaN,149,Aberta ao final da base,"304,978.53",<NA>,Sistema,Automação,Falha de Comando,TCO,CEITOC,2026,1,Janeiro,2026-01-01,5,Ocorrência Aberta,False,True,False,False,False,False
187872,Comunicação,CEITOCPGCM01FANC02_ZN-HUM,2026-01-01 05:00:04-03:00,2026-01-01 08:00:04,2,NaT,NaT,NaN,NaN,149,Aberta ao final da base,"304,978.47",Umidade de Retorno (Sl Obras de Arte) PG T.C,HVAC,HVAC,Umidade,TCO,CEITOC,2026,1,Janeiro,2026-01-01,5,Ocorrência Aberta,False,True,False,False,False,False



Salvando base completa de ocorrências reconstruídas...
Arquivo criado:
Base_Ocorrencias_Reconstruidas_Modulo6.parquet

Gerando relatório Excel do Módulo 6...

VALIDAÇÕES
Ocorrências reconstruídas..............: 438,482
Ocorrências fechadas...................: 433,714
Ocorrências abertas....................: 4,768
Fechadas + abertas.....................: 438,482
Retornos órfãos........................: 104,820

Fechadas + abertas consistente..........: True
Durações negativas......................: 0
Meses identificados pela DataHoraLocal..: [1, 2, 3, 4, 5, 6, 7]

MÓDULO 6 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo6_Duracao_Ocorrencias_Metasys.xlsx
2. Base_Ocorrencias_Reconstruidas_Modulo6.parquet

Principais resultados:

- Ocorrências efetivamente reconstruídas
- Duração média e mediana
- P90, P95 e P99
- Ocorrências ainda abertas
- Reentradas enquanto condição estava ativa
- Retornos órfãos
- Horas acumuladas de condição anormal
- Faixas de duração
- Ranking de equ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar a base completa de ocorrências:
files.download("Base_Ocorrencias_Reconstruidas_Modulo6.parquet")


In [12]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 7
# CRITICIDADE CONSOLIDADA E PRIORIZAÇÃO DE AÇÕES
#
# ENTRADAS
# ----------------------------------------------------------------------
# Ranking_Criticidade_Alarmes_Metasys_2026.parquet
#
# Ranking_Modulo5_Reincidencia_Chattering.parquet
#
# Base_Ocorrencias_Reconstruidas_Modulo6.parquet
#
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Consolidar resultados dos Módulos 4, 5 e 6
#
# 2. Criar índice final de prioridade:
#
#    25% Criticidade técnica
#    20% Chattering
#    15% Reincidência
#    20% Impacto temporal
#    10% Persistência / duração prolongada
#    10% Frequência
#
# 3. Criar classificação:
#
#    P1 - Intervenção Prioritária
#    P2 - Alta Prioridade
#    P3 - Programar Investigação
#    P4 - Monitorar
#
# 4. Criar recomendação automática de investigação
#
# 5. Criar análises:
#
#    - Equipamentos
#    - Categoria
#    - Mantenedor
#    - Tipo
#    - Torre
#    - Prioridade
#    - Pareto
#    - Top equipamentos
#
#
# IMPORTANTE
# ----------------------------------------------------------------------
# O índice criado neste projeto NÃO representa classificação
# oficial da Johnson Controls / Metasys.
#
# Trata-se de uma metodologia analítica para auxiliar
# priorização, manutenção e racionalização de alarmes.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    300
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    300
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_MODULO4 = (
    "Ranking_Criticidade_Alarmes_Metasys_2026.parquet"
)

ARQUIVO_MODULO5 = (
    "Ranking_Modulo5_Reincidencia_Chattering.parquet"
)

ARQUIVO_MODULO6 = (
    "Base_Ocorrencias_Reconstruidas_Modulo6.parquet"
)


ARQUIVO_RANKING_FINAL = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


ARQUIVO_RELATORIO = (
    "Relatorio_Modulo7_Priorizacao_Final_Alarmes_Metasys.xlsx"
)


print("=" * 115)
print("MÓDULO 7 - CRITICIDADE CONSOLIDADA E PRIORIZAÇÃO DE AÇÕES")
print("=" * 115)


# ======================================================================
# 3. VERIFICAÇÃO / UPLOAD DOS ARQUIVOS
# ======================================================================

arquivos_necessarios = [

    ARQUIVO_MODULO4,
    ARQUIVO_MODULO5,
    ARQUIVO_MODULO6
]


arquivos_faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if arquivos_faltantes:

    print(
        "\nAlguns arquivos não foram encontrados:"
    )

    for arquivo in arquivos_faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos necessários."
    )


    from google.colab import files

    uploaded = files.upload()


# Verificação final
arquivos_faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if arquivos_faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            arquivos_faltantes
        )
    )


# ======================================================================
# 4. CARREGAMENTO
# ======================================================================

print(
    "\nCarregando resultados dos módulos anteriores..."
)


m4 = pd.read_parquet(
    ARQUIVO_MODULO4
)


m5 = pd.read_parquet(
    ARQUIVO_MODULO5
)


ocorrencias = pd.read_parquet(
    ARQUIVO_MODULO6
)


print(
    "Arquivos carregados."
)


print(
    f"\nMódulo 4: {len(m4):,} registros"
)

print(
    f"Módulo 5: {len(m5):,} registros"
)

print(
    f"Módulo 6: {len(ocorrencias):,} ocorrências"
)


# ======================================================================
# 5. PADRONIZAÇÃO DO itemName
# ======================================================================

for base in [

    m4,
    m5,
    ocorrencias

]:

    base[
        "itemName"
    ] = (

        base[
            "itemName"
        ]
        .astype("string")
        .str.strip()
    )


# ======================================================================
# 6. DEDUPLICAÇÃO DO MÓDULO 4
#
# O Módulo 5 mostrou que poderiam existir linhas repetidas
# por itemName após alguns merges.
#
# Aqui adotamos explicitamente UMA linha por equipamento.
# ======================================================================

if (
    "Indice_Criticidade_V1"
    in m4.columns
):

    m4 = (

        m4
        .sort_values(
            "Indice_Criticidade_V1",
            ascending=False
        )
        .drop_duplicates(
            subset="itemName",
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )


else:

    m4 = (

        m4
        .drop_duplicates(
            subset="itemName",
            keep="first"
        )
    )


# ======================================================================
# 7. DEDUPLICAÇÃO DO MÓDULO 5
# ======================================================================

if (
    "Indice_Prioridade_Investigacao"
    in m5.columns
):

    m5 = (

        m5
        .sort_values(
            "Indice_Prioridade_Investigacao",
            ascending=False
        )
        .drop_duplicates(
            subset="itemName",
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )


else:

    m5 = (

        m5
        .drop_duplicates(
            subset="itemName",
            keep="first"
        )
    )


# ======================================================================
# 8. CADASTRO CONSOLIDADO DO MÓDULO 6
# ======================================================================

def moda_segura(serie):

    serie = (
        serie
        .dropna()
    )

    if len(
        serie
    ) == 0:

        return pd.NA


    moda = (
        serie
        .mode()
    )


    if len(
        moda
    ) > 0:

        return moda.iloc[0]


    return serie.iloc[0]


cadastro = (

    ocorrencias
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        itemDescription=(
            "itemDescription",
            moda_segura
        ),

        Categoria=(
            "Categoria",
            moda_segura
        ),

        Mantenedor=(
            "Mantenedor",
            moda_segura
        ),

        Tipo=(
            "Tipo",
            moda_segura
        ),

        Torre=(
            "Torre",
            moda_segura
        ),

        TTL_Torre=(
            "TTL_Torre",
            moda_segura
        )
    )

    .reset_index()
)


# ======================================================================
# 9. MÉTRICAS DO MÓDULO 6 POR EQUIPAMENTO
# ======================================================================

metricas_m6 = (

    ocorrencias
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        Ocorrencias=(
            "itemName",
            "size"
        ),

        Ocorrencias_Fechadas=(
            "Flag_Fechada",
            "sum"
        ),

        Ocorrencias_Abertas=(
            "Flag_Aberta",
            "sum"
        ),

        Reentradas_Enquanto_Aberta=(
            "Reentradas_Enquanto_Aberta",
            "sum"
        ),

        Duracao_Media_Min=(
            "Duracao_Min",
            "mean"
        ),

        Duracao_Mediana_Min=(
            "Duracao_Min",
            "median"
        ),

        Duracao_Maxima_Min=(
            "Duracao_Min",
            "max"
        ),

        Horas_Acumuladas=(
            "Duracao_Horas",
            "sum"
        ),

        Ocorrencias_Maior_60min=(
            "Flag_Maior_60min",
            "sum"
        ),

        Ocorrencias_Maior_4h=(
            "Flag_Maior_4h",
            "sum"
        ),

        Ocorrencias_Maior_24h=(
            "Flag_Maior_24h",
            "sum"
        ),

        Tempo_Minimo_Aberto_Max_Min=(
            "Tempo_Minimo_Aberto_Ate_Fim_Base_Min",
            "max"
        )
    )

    .reset_index()
)


# ======================================================================
# 10. TAXAS DO MÓDULO 6
# ======================================================================

metricas_m6[
    "Perc_Ocorrencias_Abertas"
] = np.where(

    metricas_m6[
        "Ocorrencias"
    ] > 0,

    metricas_m6[
        "Ocorrencias_Abertas"
    ]

    /

    metricas_m6[
        "Ocorrencias"
    ]

    * 100,

    0
)


metricas_m6[
    "Perc_Ocorrencias_Maior_24h"
] = np.where(

    metricas_m6[
        "Ocorrencias_Fechadas"
    ] > 0,

    metricas_m6[
        "Ocorrencias_Maior_24h"
    ]

    /

    metricas_m6[
        "Ocorrencias_Fechadas"
    ]

    * 100,

    0
)


metricas_m6[
    "Perc_Ocorrencias_Maior_60min"
] = np.where(

    metricas_m6[
        "Ocorrencias_Fechadas"
    ] > 0,

    metricas_m6[
        "Ocorrencias_Maior_60min"
    ]

    /

    metricas_m6[
        "Ocorrencias_Fechadas"
    ]

    * 100,

    0
)


# ======================================================================
# 11. SELEÇÃO DAS COLUNAS DO MÓDULO 4
# ======================================================================

colunas_m4_desejadas = [

    "itemName",

    "Geracoes_Efetivas",

    "Geracoes_Alarme",
    "Geracoes_PreAlarme",
    "Falhas_Comunicacao",

    "Participacao_Geracoes_Perc",

    "Prioridade_Minima",
    "Prioridade_Mediana",

    "Meses_Com_Geracao",

    "Score_Frequencia",
    "Score_Prioridade",
    "Score_Severidade",
    "Score_Persistencia",

    "Indice_Criticidade_V1",
    "Classe_Criticidade",

    "Ranking_Criticidade",
    "Classe_Pareto"
]


colunas_m4 = [

    coluna

    for coluna in colunas_m4_desejadas

    if coluna in m4.columns
]


m4_final = (

    m4[
        colunas_m4
    ]
    .copy()
)


# ======================================================================
# 12. SELEÇÃO DAS COLUNAS DO MÓDULO 5
# ======================================================================

colunas_m5_desejadas = [

    "itemName",

    "Reincidencias_5min",
    "Reincidencias_10min",
    "Reincidencias_30min",
    "Reincidencias_60min",
    "Reincidencias_24h",

    "Taxa_Reincidencia_10min_Perc",
    "Taxa_Reincidencia_60min_Perc",

    "Episodios_Chattering_10min",

    "Transicoes_Chattering_10min",

    "Taxa_Chattering_10min_Perc",

    "Retornos_Rapidos_10min",
    "Reativacoes_Rapidas_10min",

    "Classe_Comportamento",

    "Indice_Prioridade_Investigacao",
    "Classe_Prioridade_Investigacao"
]


colunas_m5 = [

    coluna

    for coluna in colunas_m5_desejadas

    if coluna in m5.columns
]


m5_final = (

    m5[
        colunas_m5
    ]
    .copy()
)


# ======================================================================
# 13. CONSOLIDAÇÃO
# ======================================================================

ranking = (

    cadastro

    .merge(
        metricas_m6,
        on="itemName",
        how="outer"
    )

    .merge(
        m4_final,
        on="itemName",
        how="left"
    )

    .merge(
        m5_final,
        on="itemName",
        how="left"
    )
)


print(
    f"\nEquipamentos consolidados: "
    f"{len(ranking):,}"
)


# ======================================================================
# 14. PREENCHIMENTO DE MÉTRICAS AUSENTES
# ======================================================================

colunas_zero = [

    "Geracoes_Efetivas",

    "Geracoes_Alarme",
    "Geracoes_PreAlarme",
    "Falhas_Comunicacao",

    "Ocorrencias",
    "Ocorrencias_Fechadas",
    "Ocorrencias_Abertas",

    "Reentradas_Enquanto_Aberta",

    "Horas_Acumuladas",

    "Ocorrencias_Maior_60min",
    "Ocorrencias_Maior_4h",
    "Ocorrencias_Maior_24h",

    "Perc_Ocorrencias_Abertas",
    "Perc_Ocorrencias_Maior_24h",
    "Perc_Ocorrencias_Maior_60min",

    "Reincidencias_5min",
    "Reincidencias_10min",
    "Reincidencias_30min",
    "Reincidencias_60min",
    "Reincidencias_24h",

    "Taxa_Reincidencia_10min_Perc",
    "Taxa_Reincidencia_60min_Perc",

    "Episodios_Chattering_10min",
    "Transicoes_Chattering_10min",

    "Taxa_Chattering_10min_Perc",

    "Retornos_Rapidos_10min",
    "Reativacoes_Rapidas_10min"
]


for coluna in colunas_zero:

    if coluna in ranking.columns:

        ranking[
            coluna
        ] = (

            ranking[
                coluna
            ]
            .fillna(0)
        )


# ======================================================================
# 15. FUNÇÃO DE SCORE PERCENTIL
# ======================================================================

def score_percentil(
    serie
):

    return (

        serie
        .fillna(0)
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


# ======================================================================
# 16. SCORE FINAL DE CRITICIDADE TÉCNICA
#
# Módulo 4
# ======================================================================

if (
    "Indice_Criticidade_V1"
    in ranking.columns
):

    ranking[
        "Score_Criticidade_Final"
    ] = (

        ranking[
            "Indice_Criticidade_V1"
        ]
        .fillna(0)
        .clip(
            0,
            100
        )
    )


else:

    ranking[
        "Score_Criticidade_Final"
    ] = 0


# ======================================================================
# 17. SCORE DE CHATTERING
#
# Combina:
#
# - episódios
# - taxa de chattering
# ======================================================================

score_episodios = score_percentil(

    ranking[
        "Episodios_Chattering_10min"
    ]
)


score_taxa_chatter = (

    ranking[
        "Taxa_Chattering_10min_Perc"
    ]
    .fillna(0)
    .clip(
        0,
        100
    )
)


ranking[
    "Score_Chattering_Final"
] = (

    score_episodios
    * 0.60

    +

    score_taxa_chatter
    * 0.40
)


# ======================================================================
# 18. SCORE DE REINCIDÊNCIA
# ======================================================================

score_reincidencias = score_percentil(

    ranking[
        "Reincidencias_10min"
    ]
)


score_taxa_reincidencia = (

    ranking[
        "Taxa_Reincidencia_10min_Perc"
    ]
    .fillna(0)
    .clip(
        0,
        100
    )
)


ranking[
    "Score_Reincidencia_Final"
] = (

    score_reincidencias
    * 0.60

    +

    score_taxa_reincidencia
    * 0.40
)


# ======================================================================
# 19. SCORE DE IMPACTO TEMPORAL
#
# Robustez:
#
# Não usamos diretamente duração máxima.
#
# Usamos:
#
# - horas acumuladas
# - duração mediana
#
# Dessa maneira algumas ocorrências extremamente longas
# não dominam sozinhas o resultado.
# ======================================================================

score_horas = score_percentil(

    np.log1p(
        ranking[
            "Horas_Acumuladas"
        ]
    )
)


score_duracao_mediana = score_percentil(

    np.log1p(
        ranking[
            "Duracao_Mediana_Min"
        ]
        .fillna(0)
    )
)


ranking[
    "Score_Impacto_Temporal_Final"
] = (

    score_horas
    * 0.60

    +

    score_duracao_mediana
    * 0.40
)


# ======================================================================
# 20. SCORE DE PERSISTÊNCIA / OCORRÊNCIAS LONGAS
#
# Consideramos:
#
# 40% ocorrências abertas
# 35% percentual >24h
# 25% reentradas enquanto abertas
# ======================================================================

score_abertas = score_percentil(

    ranking[
        "Ocorrencias_Abertas"
    ]
)


score_longas = score_percentil(

    ranking[
        "Ocorrencias_Maior_24h"
    ]
)


score_reentradas = score_percentil(

    ranking[
        "Reentradas_Enquanto_Aberta"
    ]
)


ranking[
    "Score_Persistencia_Final"
] = (

    score_abertas
    * 0.40

    +

    score_longas
    * 0.35

    +

    score_reentradas
    * 0.25
)


# ======================================================================
# 21. SCORE DE FREQUÊNCIA
# ======================================================================

if (
    "Geracoes_Efetivas"
    in ranking.columns
):

    ranking[
        "Score_Frequencia_Final"
    ] = score_percentil(

        np.log1p(
            ranking[
                "Geracoes_Efetivas"
            ]
        )
    )


else:

    ranking[
        "Score_Frequencia_Final"
    ] = score_percentil(

        np.log1p(
            ranking[
                "Ocorrencias"
            ]
        )
    )


# ======================================================================
# 22. ÍNDICE CONSOLIDADO FINAL
#
# 25% Criticidade técnica
# 20% Chattering
# 15% Reincidência
# 20% Impacto temporal
# 10% Persistência / duração longa
# 10% Frequência
# ======================================================================

PESO_CRITICIDADE = 0.25
PESO_CHATTERING = 0.20
PESO_REINCIDENCIA = 0.15
PESO_TEMPORAL = 0.20
PESO_PERSISTENCIA = 0.10
PESO_FREQUENCIA = 0.10


ranking[
    "Indice_Prioridade_Final"
] = (

    ranking[
        "Score_Criticidade_Final"
    ]
    * PESO_CRITICIDADE

    +

    ranking[
        "Score_Chattering_Final"
    ]
    * PESO_CHATTERING

    +

    ranking[
        "Score_Reincidencia_Final"
    ]
    * PESO_REINCIDENCIA

    +

    ranking[
        "Score_Impacto_Temporal_Final"
    ]
    * PESO_TEMPORAL

    +

    ranking[
        "Score_Persistencia_Final"
    ]
    * PESO_PERSISTENCIA

    +

    ranking[
        "Score_Frequencia_Final"
    ]
    * PESO_FREQUENCIA
)


# ======================================================================
# 23. CLASSIFICAÇÃO FINAL
# ======================================================================

ranking[
    "Prioridade_Final"
] = np.select(

    [
        ranking[
            "Indice_Prioridade_Final"
        ] >= 80,

        ranking[
            "Indice_Prioridade_Final"
        ] >= 65,

        ranking[
            "Indice_Prioridade_Final"
        ] >= 50
    ],

    [
        "P1 - Intervenção Prioritária",
        "P2 - Alta Prioridade",
        "P3 - Programar Investigação"
    ],

    default=(
        "P4 - Monitorar"
    )
)


# ======================================================================
# 24. PERFIL DOMINANTE DO PROBLEMA
# ======================================================================

def identificar_perfil(
    row
):

    scores = {

        "Criticidade":
            row[
                "Score_Criticidade_Final"
            ],

        "Chattering":
            row[
                "Score_Chattering_Final"
            ],

        "Reincidência":
            row[
                "Score_Reincidencia_Final"
            ],

        "Duração / Impacto Temporal":
            row[
                "Score_Impacto_Temporal_Final"
            ],

        "Persistência":
            row[
                "Score_Persistencia_Final"
            ],

        "Frequência":
            row[
                "Score_Frequencia_Final"
            ]
    }


    return max(
        scores,
        key=scores.get
    )


ranking[
    "Perfil_Dominante"
] = (

    ranking
    .apply(
        identificar_perfil,
        axis=1
    )
)


# ======================================================================
# 25. FLAGS OPERACIONAIS
# ======================================================================

ranking[
    "Flag_Chattering_Relevante"
] = (

    ranking[
        "Episodios_Chattering_10min"
    ]
    >= 3
)


ranking[
    "Flag_Reincidencia_Relevante"
] = (

    ranking[
        "Reincidencias_10min"
    ]
    >= 10
)


ranking[
    "Flag_Duracao_Relevante"
] = (

    ranking[
        "Ocorrencias_Maior_24h"
    ]
    > 0
)


ranking[
    "Flag_Ocorrencia_Aberta"
] = (

    ranking[
        "Ocorrencias_Abertas"
    ]
    > 0
)


ranking[
    "Flag_Alta_Criticidade"
] = (

    ranking[
        "Score_Criticidade_Final"
    ]
    >= 80
)


# ======================================================================
# 26. RECOMENDAÇÃO AUTOMÁTICA
#
# Recomendações analíticas, não diagnóstico automático.
# ======================================================================

def gerar_recomendacao(
    row
):

    categoria = (
        str(
            row[
                "Categoria"
            ]
        )
        .lower()
    )


    mantenedor = (
        str(
            row[
                "Mantenedor"
            ]
        )
        .lower()
    )


    tipo = (
        str(
            row[
                "Tipo"
            ]
        )
        .lower()
    )


    chatter = (
        row[
            "Flag_Chattering_Relevante"
        ]
    )


    reinc = (
        row[
            "Flag_Reincidencia_Relevante"
        ]
    )


    longa = (
        row[
            "Flag_Duracao_Relevante"
        ]
    )


    aberta = (
        row[
            "Flag_Ocorrencia_Aberta"
        ]
    )


    # ------------------------------------------------------------
    # OFFLINE / AUTOMAÇÃO
    # ------------------------------------------------------------

    if (
        "off-line" in tipo
        or "offline" in tipo
    ):

        if chatter:

            return (
                "Verificar instabilidade de comunicação, "
                "rede, controlador, alimentação e qualidade "
                "do enlace. Avaliar atraso de alarme e "
                "parametrização de perda/restabelecimento."
            )


        return (
            "Investigar comunicação do dispositivo, "
            "controlador, alimentação, rede e disponibilidade."
        )


    # ------------------------------------------------------------
    # FALHA DE COMANDO / AUTOMAÇÃO
    # ------------------------------------------------------------

    if (
        "falha de comando" in tipo
        or "automação" in mantenedor
    ):

        if chatter:

            return (
                "Revisar lógica de comando, feedback, "
                "intertravamentos e temporizações. "
                "Investigar alternância rápida de estado."
            )


        if longa or aberta:

            return (
                "Verificar comando e feedback do equipamento, "
                "intertravamentos, permissivos e condição "
                "de falha persistente."
            )


    # ------------------------------------------------------------
    # HVAC - TEMPERATURA
    # ------------------------------------------------------------

    if (
        "temperatura" in tipo
    ):

        if chatter:

            return (
                "Revisar setpoint, deadband, limites de alarme, "
                "temporização, estabilidade do sensor e resposta "
                "da malha de controle."
            )


        if longa or aberta:

            return (
                "Investigar causa de desvio térmico persistente: "
                "setpoint, sensor, vazão de ar/água, capacidade, "
                "atuador e estratégia de controle."
            )


        return (
            "Verificar tendência de temperatura, setpoint, "
            "limites de alarme e condição do sensor."
        )


    # ------------------------------------------------------------
    # PRESSÃO
    # ------------------------------------------------------------

    if (
        "pressão" in tipo
        or "pressao" in tipo
    ):

        if chatter:

            return (
                "Verificar estabilidade do transmissor, "
                "deadband, pulsação de pressão e limites "
                "de alarme."
            )


        return (
            "Inspecionar transmissor, condição hidráulica/"
            "aeráulica e coerência dos limites de pressão."
        )


    # ------------------------------------------------------------
    # NÍVEL
    # ------------------------------------------------------------

    if (
        "nível" in tipo
        or "nivel" in tipo
    ):

        if chatter:

            return (
                "Verificar oscilação de nível, boia/transmissor, "
                "deadband, atraso de alarme e estabilidade "
                "do processo."
            )


        return (
            "Avaliar sensor de nível, condição do reservatório "
            "e parametrização dos limites de alarme."
        )


    # ------------------------------------------------------------
    # BOIA
    # ------------------------------------------------------------

    if (
        "boia" in tipo
    ):

        return (
            "Inspecionar boia, drenagem, bandeja, cabeamento "
            "e eventual oscilação do contato."
        )


    # ------------------------------------------------------------
    # OCORRÊNCIA LONGA / ABERTA
    # ------------------------------------------------------------

    if (
        aberta
        and longa
    ):

        return (
            "Priorizar investigação de condição persistente "
            "e confirmar se o ponto permanece efetivamente "
            "em falha ou se existe problema de normalização."
        )


    if aberta:

        return (
            "Verificar ocorrência ainda aberta e confirmar "
            "estado atual do equipamento."
        )


    if longa:

        return (
            "Investigar motivo da longa permanência em condição "
            "anormal e confirmar processo de normalização."
        )


    # ------------------------------------------------------------
    # CHATTERING
    # ------------------------------------------------------------

    if chatter:

        return (
            "Investigar chattering: revisar deadband, "
            "temporização, sensor, lógica e qualidade do sinal."
        )


    # ------------------------------------------------------------
    # REINCIDÊNCIA
    # ------------------------------------------------------------

    if reinc:

        return (
            "Investigar reincidência frequente e identificar "
            "causa raiz antes de ajustar limites de alarme."
        )


    # ------------------------------------------------------------
    # PADRÃO
    # ------------------------------------------------------------

    return (
        "Monitorar tendência e revisar histórico antes "
        "de definir ação corretiva."
    )


ranking[
    "Acao_Recomendada"
] = (

    ranking
    .apply(
        gerar_recomendacao,
        axis=1
    )
)


# ======================================================================
# 27. CLASSIFICAÇÃO DO TIPO DE TRATAMENTO
# ======================================================================

def classificar_tratamento(
    row
):

    chatter = (
        row[
            "Flag_Chattering_Relevante"
        ]
    )


    aberta = (
        row[
            "Flag_Ocorrencia_Aberta"
        ]
    )


    longa = (
        row[
            "Flag_Duracao_Relevante"
        ]
    )


    critic = (
        row[
            "Flag_Alta_Criticidade"
        ]
    )


    if (
        critic
        and
        aberta
    ):

        return (
            "Manutenção / Investigação imediata"
        )


    if (
        critic
        and
        longa
    ):

        return (
            "Análise de causa raiz"
        )


    if (
        chatter
    ):

        return (
            "Racionalização / Parametrização"
        )


    if (
        row[
            "Flag_Reincidencia_Relevante"
        ]
    ):

        return (
            "Investigação de reincidência"
        )


    return (
        "Monitoramento"
    )


ranking[
    "Tipo_Tratamento"
] = (

    ranking
    .apply(
        classificar_tratamento,
        axis=1
    )
)


# ======================================================================
# 28. SCORE DE CONFIANÇA DA ANÁLISE
#
# Não altera diretamente a prioridade.
#
# Serve para mostrar se temos volume suficiente
# de dados fechados para interpretar o ponto.
# ======================================================================

ranking[
    "Score_Confianca"
] = np.select(

    [
        (
            ranking[
                "Ocorrencias_Fechadas"
            ] >= 100
        ),

        (
            ranking[
                "Ocorrencias_Fechadas"
            ] >= 20
        ),

        (
            ranking[
                "Ocorrencias_Fechadas"
            ] >= 5
        )
    ],

    [
        100,
        80,
        60
    ],

    default=40
)


ranking[
    "Confianca_Analise"
] = np.select(

    [
        ranking[
            "Score_Confianca"
        ] >= 90,

        ranking[
            "Score_Confianca"
        ] >= 70,

        ranking[
            "Score_Confianca"
        ] >= 50
    ],

    [
        "Alta",
        "Boa",
        "Moderada"
    ],

    default="Baixa"
)


# ======================================================================
# 29. ORDENAÇÃO FINAL
# ======================================================================

ranking = (

    ranking
    .sort_values(

        [
            "Indice_Prioridade_Final",
            "Score_Criticidade_Final",
            "Horas_Acumuladas"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


ranking[
    "Ranking_Final"
] = (

    np.arange(
        1,
        len(
            ranking
        ) + 1
    )
)


# ======================================================================
# 30. PARETO DO ÍNDICE FINAL
# ======================================================================

ranking[
    "Participacao_Geracoes_Final_Perc"
] = np.where(

    ranking[
        "Geracoes_Efetivas"
    ].sum() > 0,

    ranking[
        "Geracoes_Efetivas"
    ]

    /

    ranking[
        "Geracoes_Efetivas"
    ].sum()

    * 100,

    0
)


ranking[
    "Participacao_Acumulada_Geracoes_Perc"
] = (

    ranking
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )[
        "Participacao_Geracoes_Final_Perc"
    ]
    .cumsum()
)


# ======================================================================
# 31. RESUMO DAS PRIORIDADES
# ======================================================================

resumo_prioridades = (

    ranking
    .groupby(
        "Prioridade_Final",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Ocorrencias=(
            "Ocorrencias",
            "sum"
        ),

        Ocorrencias_Abertas=(
            "Ocorrencias_Abertas",
            "sum"
        ),

        Horas_Acumuladas=(
            "Horas_Acumuladas",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Prioridade_Final",
            "mean"
        )
    )

    .reset_index()
)


# ======================================================================
# 32. RESUMO POR TIPO DE TRATAMENTO
# ======================================================================

resumo_tratamento = (

    ranking
    .groupby(
        "Tipo_Tratamento",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "size"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        Ocorrencias_Abertas=(
            "Ocorrencias_Abertas",
            "sum"
        ),

        Episodios_Chattering=(
            "Episodios_Chattering_10min",
            "sum"
        ),

        Horas_Acumuladas=(
            "Horas_Acumuladas",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Equipamentos",
        ascending=False
    )
)


# ======================================================================
# 33. FUNÇÃO DE RESUMO POR DIMENSÃO
# ======================================================================

def resumo_dimensao(
    coluna
):

    resultado = (

        ranking
        .groupby(
            coluna,
            observed=True,
            dropna=False
        )
        .agg(

            Equipamentos=(
                "itemName",
                "size"
            ),

            P1=(
                "Prioridade_Final",
                lambda x:
                (
                    x
                    ==
                    "P1 - Intervenção Prioritária"
                ).sum()
            ),

            P2=(
                "Prioridade_Final",
                lambda x:
                (
                    x
                    ==
                    "P2 - Alta Prioridade"
                ).sum()
            ),

            Geracoes_Efetivas=(
                "Geracoes_Efetivas",
                "sum"
            ),

            Ocorrencias=(
                "Ocorrencias",
                "sum"
            ),

            Ocorrencias_Abertas=(
                "Ocorrencias_Abertas",
                "sum"
            ),

            Episodios_Chattering=(
                "Episodios_Chattering_10min",
                "sum"
            ),

            Horas_Acumuladas=(
                "Horas_Acumuladas",
                "sum"
            ),

            Indice_Medio=(
                "Indice_Prioridade_Final",
                "mean"
            ),

            Indice_Maximo=(
                "Indice_Prioridade_Final",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            [
                "P1",
                "Indice_Medio"
            ],

            ascending=[
                False,
                False
            ]
        )
    )


    return resultado


# ======================================================================
# 34. RESUMOS
# ======================================================================

resumo_categoria = resumo_dimensao(
    "Categoria"
)

resumo_mantenedor = resumo_dimensao(
    "Mantenedor"
)

resumo_tipo = resumo_dimensao(
    "Tipo"
)

resumo_torre = resumo_dimensao(
    "Torre"
)


# ======================================================================
# 35. TOP 100 FINAL
# ======================================================================

top100 = (

    ranking
    .head(100)
    .copy()
)


# ======================================================================
# 36. FILAS OPERACIONAIS
# ======================================================================

fila_p1 = (

    ranking.loc[
        ranking[
            "Prioridade_Final"
        ]
        ==
        "P1 - Intervenção Prioritária"
    ]

    .copy()
)


fila_p2 = (

    ranking.loc[
        ranking[
            "Prioridade_Final"
        ]
        ==
        "P2 - Alta Prioridade"
    ]

    .copy()
)


fila_abertos = (

    ranking.loc[
        ranking[
            "Ocorrencias_Abertas"
        ]
        > 0
    ]

    .sort_values(
        [
            "Indice_Prioridade_Final",
            "Ocorrencias_Abertas"
        ],

        ascending=[
            False,
            False
        ]
    )

    .copy()
)


fila_chattering = (

    ranking.loc[
        ranking[
            "Episodios_Chattering_10min"
        ]
        > 0
    ]

    .sort_values(
        [
            "Episodios_Chattering_10min",
            "Indice_Prioridade_Final"
        ],

        ascending=[
            False,
            False
        ]
    )

    .copy()
)


fila_longas = (

    ranking.loc[
        ranking[
            "Ocorrencias_Maior_24h"
        ]
        > 0
    ]

    .sort_values(
        [
            "Ocorrencias_Maior_24h",
            "Horas_Acumuladas"
        ],

        ascending=[
            False,
            False
        ]
    )

    .copy()
)


# ======================================================================
# 37. RESUMO EXECUTIVO
# ======================================================================

TOTAL_EQUIP = len(
    ranking
)


TOTAL_P1 = (

    ranking[
        "Prioridade_Final"
    ]
    .eq(
        "P1 - Intervenção Prioritária"
    )
    .sum()
)


TOTAL_P2 = (

    ranking[
        "Prioridade_Final"
    ]
    .eq(
        "P2 - Alta Prioridade"
    )
    .sum()
)


TOTAL_P3 = (

    ranking[
        "Prioridade_Final"
    ]
    .eq(
        "P3 - Programar Investigação"
    )
    .sum()
)


TOTAL_P4 = (

    ranking[
        "Prioridade_Final"
    ]
    .eq(
        "P4 - Monitorar"
    )
    .sum()
)


lider = (
    ranking.iloc[0]
)


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Equipamentos analisados",

            "P1 - Intervenção Prioritária",

            "P2 - Alta Prioridade",

            "P3 - Programar Investigação",

            "P4 - Monitorar",

            "% equipamentos P1",

            "Equipamentos com ocorrências abertas",

            "Equipamentos com chattering",

            "Equipamentos com ocorrências >24h",

            "Equipamento líder",

            "Índice do equipamento líder",

            "Categoria do líder",

            "Mantenedor do líder",

            "Tipo do líder",

            "Perfil dominante do líder"
        ],

        "Resultado": [

            f"{TOTAL_EQUIP:,}",

            f"{TOTAL_P1:,}",

            f"{TOTAL_P2:,}",

            f"{TOTAL_P3:,}",

            f"{TOTAL_P4:,}",

            (
                f"{TOTAL_P1 / TOTAL_EQUIP * 100:.2f}%"
                if TOTAL_EQUIP > 0
                else "0.00%"
            ),

            f"{(ranking['Ocorrencias_Abertas'] > 0).sum():,}",

            f"{(ranking['Episodios_Chattering_10min'] > 0).sum():,}",

            f"{(ranking['Ocorrencias_Maior_24h'] > 0).sum():,}",

            str(
                lider[
                    "itemName"
                ]
            ),

            f"{lider['Indice_Prioridade_Final']:.2f}",

            str(
                lider[
                    "Categoria"
                ]
            ),

            str(
                lider[
                    "Mantenedor"
                ]
            ),

            str(
                lider[
                    "Tipo"
                ]
            ),

            str(
                lider[
                    "Perfil_Dominante"
                ]
            )
        ]
    }
)


# ======================================================================
# 38. EXIBIÇÃO DOS RESULTADOS
# ======================================================================

print("\n" + "=" * 115)
print("RESUMO EXECUTIVO")
print("=" * 115)

display(
    resumo_executivo
)


print("\n" + "=" * 115)
print("DISTRIBUIÇÃO DAS PRIORIDADES")
print("=" * 115)

display(
    resumo_prioridades
)


print("\n" + "=" * 115)
print("TIPOS DE TRATAMENTO")
print("=" * 115)

display(
    resumo_tratamento
)


print("\n" + "=" * 115)
print("TOP 30 - RANKING FINAL")
print("=" * 115)

colunas_exibicao = [

    "Ranking_Final",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",

    "Geracoes_Efetivas",

    "Ocorrencias",
    "Ocorrencias_Abertas",

    "Ocorrencias_Maior_24h",

    "Horas_Acumuladas",

    "Episodios_Chattering_10min",

    "Reincidencias_10min",

    "Indice_Criticidade_V1",

    "Score_Criticidade_Final",
    "Score_Chattering_Final",
    "Score_Reincidencia_Final",
    "Score_Impacto_Temporal_Final",
    "Score_Persistencia_Final",
    "Score_Frequencia_Final",

    "Indice_Prioridade_Final",

    "Prioridade_Final",

    "Perfil_Dominante",

    "Tipo_Tratamento",

    "Acao_Recomendada",

    "Confianca_Analise"
]


colunas_exibicao = [

    coluna

    for coluna in colunas_exibicao

    if coluna in ranking.columns
]


display(

    ranking[
        colunas_exibicao
    ]
    .head(30)
)


# ======================================================================
# 39. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 115)
print("VALIDAÇÕES")
print("=" * 115)


print(
    f"Equipamentos consolidados.............: "
    f"{len(ranking):,}"
)


print(
    f"itemName únicos.......................: "
    f"{ranking['itemName'].nunique():,}"
)


validacao_unicidade = (

    len(
        ranking
    )

    ==

    ranking[
        "itemName"
    ]
    .nunique()
)


print(
    "\nUma linha por itemName................:",
    validacao_unicidade
)


print(
    "Índice mínimo........................:",
    f"{ranking['Indice_Prioridade_Final'].min():.2f}"
)


print(
    "Índice máximo........................:",
    f"{ranking['Indice_Prioridade_Final'].max():.2f}"
)


validacao_indice = (

    ranking[
        "Indice_Prioridade_Final"
    ]
    .between(
        0,
        100
    )
    .all()
)


print(
    "Índice dentro de 0 a 100..............:",
    validacao_indice
)


# ======================================================================
# 40. SALVAMENTO DO RANKING FINAL
# ======================================================================

print(
    "\nSalvando ranking consolidado..."
)


ranking.to_parquet(
    ARQUIVO_RANKING_FINAL,
    index=False
)


print(
    f"Arquivo criado:\n"
    f"{ARQUIVO_RANKING_FINAL}"
)


# ======================================================================
# 41. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Componente": [

            "Criticidade Técnica",

            "Chattering",

            "Reincidência",

            "Impacto Temporal",

            "Persistência",

            "Frequência"
        ],

        "Peso_Perc": [

            25,
            20,
            15,
            20,
            10,
            10
        ],

        "Origem": [

            "Módulo 4",

            "Módulo 5",

            "Módulo 5",

            "Módulo 6",

            "Módulo 6",

            "Módulo 4 / 6"
        ],

        "Descricao": [

            (
                "Índice técnico considerando frequência, "
                "prioridade, severidade e persistência."
            ),

            (
                "Combinação de episódios e taxa de "
                "chattering em 10 minutos."
            ),

            (
                "Quantidade e taxa de reincidência "
                "em até 10 minutos."
            ),

            (
                "Horas acumuladas e duração mediana, "
                "com transformação robusta."
            ),

            (
                "Ocorrências abertas, ocorrências acima "
                "de 24 horas e reentradas."
            ),

            (
                "Volume total de gerações efetivas."
            )
        ]
    }
)


# ======================================================================
# 42. EXPORTAÇÃO PARA EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 7..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    resumo_prioridades.to_excel(
        writer,
        sheet_name="Prioridades",
        index=False
    )


    resumo_tratamento.to_excel(
        writer,
        sheet_name="Tipos Tratamento",
        index=False
    )


    ranking.to_excel(
        writer,
        sheet_name="Ranking Final",
        index=False
    )


    top100.to_excel(
        writer,
        sheet_name="Top100",
        index=False
    )


    fila_p1.to_excel(
        writer,
        sheet_name="Fila P1",
        index=False
    )


    fila_p2.to_excel(
        writer,
        sheet_name="Fila P2",
        index=False
    )


    fila_abertos.to_excel(
        writer,
        sheet_name="Ocorrencias Abertas",
        index=False
    )


    fila_chattering.to_excel(
        writer,
        sheet_name="Fila Chattering",
        index=False
    )


    fila_longas.to_excel(
        writer,
        sheet_name="Ocorrencias Longas",
        index=False
    )


    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )


    resumo_mantenedor.to_excel(
        writer,
        sheet_name="Mantenedor",
        index=False
    )


    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )


    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


# ======================================================================
# 43. FORMATAÇÃO DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (

    Font,
    Alignment
)

from openpyxl.utils import (

    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0

        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(
                        cell.value
                    )
                )


                max_length = max(
                    max_length,
                    len(
                        valor
                    )
                )


            except:
                pass


        largura = min(

            max(
                max_length + 2,
                12
            ),

            50
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 44. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 115)
print("MÓDULO 7 CONCLUÍDO COM SUCESSO")
print("=" * 115)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


print(
    f"2. {ARQUIVO_RANKING_FINAL}"
)


print(
    """
O relatório contém:

1. Resumo executivo

2. Ranking final consolidado

3. Top 100 equipamentos

4. Fila P1 - Intervenção prioritária

5. Fila P2 - Alta prioridade

6. Ocorrências abertas

7. Equipamentos com chattering

8. Ocorrências longas

9. Categoria

10. Mantenedor

11. Tipo

12. Torre

13. Tipo de tratamento

14. Recomendações de ação

15. Metodologia do índice
"""
)


# ======================================================================
# 45. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar também o ranking consolidado:"
)


print(
    f'files.download("{ARQUIVO_RANKING_FINAL}")'
)

MÓDULO 7 - CRITICIDADE CONSOLIDADA E PRIORIZAÇÃO DE AÇÕES

Carregando resultados dos módulos anteriores...
Arquivos carregados.

Módulo 4: 6,263 registros
Módulo 5: 8,691 registros
Módulo 6: 438,482 ocorrências

Equipamentos consolidados: 5,826

RESUMO EXECUTIVO


,Indicador,Resultado
0,Equipamentos analisados,"5,826"
1,P1 - Intervenção Prioritária,57
2,P2 - Alta Prioridade,668
3,P3 - Programar Investigação,"2,161"
4,P4 - Monitorar,"2,940"
5,% equipamentos P1,0.98%
6,Equipamentos com ocorrências abertas,"4,224"
7,Equipamentos com chattering,"1,909"
8,Equipamentos com ocorrências >24h,"2,454"
9,Equipamento líder,CEATOB01ST01CVAV33_ZN-TEM



DISTRIBUIÇÃO DAS PRIORIDADES


,Prioridade_Final,Equipamentos,Geracoes_Efetivas,Ocorrencias,Ocorrencias_Abertas,Horas_Acumuladas,Episodios_Chattering,Indice_Medio
0,P1 - Intervenção Prioritária,57,75450,64329,62,"163,842.21","5,955.00",82.25
1,P2 - Alta Prioridade,668,268191,200875,770,"1,602,255.63","13,099.00",72.12
2,P3 - Programar Investigação,2161,235606,154171,1704,"2,326,651.07","4,631.00",55.85
3,P4 - Monitorar,2940,45637,19107,2232,"458,711.93",692.00,36.70



TIPOS DE TRATAMENTO


...

,Ranking_Final,itemName,itemDescription,Categoria,Mantenedor,Tipo,Torre,Geracoes_Efetivas,Ocorrencias,Ocorrencias_Abertas,Ocorrencias_Maior_24h,Horas_Acumuladas,Episodios_Chattering_10min,Reincidencias_10min,Indice_Criticidade_V1,Score_Criticidade_Final,Score_Chattering_Final,Score_Reincidencia_Final,Score_Impacto_Temporal_Final,Score_Persistencia_Final,Score_Frequencia_Final,Indice_Prioridade_Final,Prioridade_Final,Perfil_Dominante,Tipo_Tratamento,Acao_Recomendada,Confianca_Analise
0,1,CEATOB01ST01CVAV33_ZN-TEM,Temperatura Ambiente,HVAC,HVAC,Temperatura,CEA,3124,1051,2,47,"7,394.91",73.00,"2,412.00",89.49,89.49,75.59,90.86,81.73,98.12,99.91,87.27,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
1,2,CEATOB07SR08CVAV71_ZN-TEM,Temp Média - SR08 e SR09 Linear,HVAC,HVAC,Temperatura,CEA,3600,3238,1,15,"2,220.72",405.00,"2,254.00",85.21,85.21,92.65,85.01,73.03,80.07,99.93,85.19,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
2,3,CEATOA03SR05CVAV19_ZN-TEM,Temperatura Ambiente,HVAC,HVAC,Temperatura,CEA,605,494,2,23,"4,186.67",63.00,320.00,84.19,84.19,85.04,80.39,78.43,95.53,97.31,85.08,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
3,4,CEITE5P3CAG1CHAG00_CH-CWC,Sensor de Vazão Agua Condensada,HVAC,HVAC,Other,TEV,1089,1083,1,10,"1,981.07",60.00,879.00,88.58,88.58,93.99,92.08,70.01,62.66,98.95,84.92,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,Priorizar investigação de condição persistente...,Alta
4,5,CEATOA04ST01CVAV11_ZN-TEM,Temperatura Ambiente,HVAC,HVAC,Temperatura,CEA,2849,2490,3,22,"3,157.66",222.00,751.00,85.24,85.24,86.38,70.23,78.42,98.19,99.85,84.61,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
5,6,CEITE5P1CM01FANC05_DA-TEM,Temperatura de insuflamento,HVAC,HVAC,Temperatura,TEV,1752,1751,1,9,"1,023.25",64.00,"1,355.00",92.04,92.04,95.38,90.80,65.89,57.08,99.52,84.55,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
6,7,CEITE5P1SL03CVAV16_ZN-MED,Temperatura Ambiente Média,HVAC,HVAC,Temperatura,TEV,2105,2102,0,25,"2,110.50",176.00,"1,848.00",86.41,86.41,96.76,95.05,70.40,47.13,99.76,83.98,P1 - Intervenção Prioritária,Frequência,Análise de causa raiz,"Revisar setpoint, deadband, limites de alarme,...",Alta
7,8,CEATOB07AA01EVAP28_ZN-TEM,Temperatura Ambiente VRF,HVAC,HVAC,Temperatura,CEA,3735,3727,1,3,342.10,227.00,"3,059.00",92.53,92.53,96.44,92.75,58.76,58.91,99.95,83.97,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Revisar setpoint, deadband, limites de alarme,...",Alta
8,9,CEATOAP1AA01EVAP13_ZN-TEM,Temperatura Ambiente,HVAC,HVAC,Temperatura,CEA,1951,1910,0,8,"1,256.12",169.00,"1,493.00",92.44,92.44,94.01,90.52,65.93,52.21,99.69,83.87,P1 - Intervenção Prioritária,Frequência,Análise de causa raiz,"Revisar setpoint, deadband, limites de alarme,...",Alta
9,10,VND016 [Nobreak Delta 02],Controller,Sistema,Automação,Off-line,CEA,474,390,1,7,"2,499.62",68.00,"1,021.00",86.69,86.69,81.65,90.98,74.92,76.02,95.80,83.81,P1 - Intervenção Prioritária,Frequência,Manutenção / Investigação imediata,"Verificar instabilidade de comunicação, rede, ...",Alta



VALIDAÇÕES
Equipamentos consolidados.............: 5,826
itemName únicos.......................: 5,826

Uma linha por itemName................: True
Índice mínimo........................: 21.36
Índice máximo........................: 87.27
Índice dentro de 0 a 100..............: True

Salvando ranking consolidado...
Arquivo criado:
Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet

Gerando relatório Excel do Módulo 7...

MÓDULO 7 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo7_Priorizacao_Final_Alarmes_Metasys.xlsx
2. Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet

O relatório contém:

1. Resumo executivo

2. Ranking final consolidado

3. Top 100 equipamentos

4. Fila P1 - Intervenção prioritária

5. Fila P2 - Alta prioridade

6. Ocorrências abertas

7. Equipamentos com chattering

8. Ocorrências longas

9. Categoria

10. Mantenedor

11. Tipo

12. Torre

13. Tipo de tratamento

14. Recomendações de ação

15. Metodologia do índice


Iniciando download do rela

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também o ranking consolidado:
files.download("Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet")


In [17]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 8
# ALARM FLOOD, CASCATAS E CONCENTRAÇÃO TEMPORAL DE EVENTOS
#
# ENTRADAS
# ----------------------------------------------------------------------
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet
#
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Analisar concentração temporal das gerações efetivas
#
# 2. Avaliar janelas de:
#       - 1 minuto
#       - 5 minutos
#       - 10 minutos
#
# 3. Identificar:
#       - picos
#       - alarm floods
#       - cascatas
#       - eventos localizados
#       - eventos multitorre
#       - eventos multicategoria
#
# 4. Criar episódios contínuos de flood
#
# 5. Medir:
#       - eventos
#       - equipamentos únicos
#       - categorias
#       - mantenedores
#       - tipos
#       - torres
#       - participação P1/P2
#
# 6. Identificar equipamentos que aparecem repetidamente
#    durante floods
#
#
# IMPORTANTE
# ----------------------------------------------------------------------
# Os thresholds deste módulo são critérios analíticos do projeto.
#
# NÃO representam parâmetros oficiais Johnson Controls / Metasys.
#
# O código também calcula percentis da própria base para permitir
# posterior calibração dos thresholds.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    300
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    300
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_BASE = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)


ARQUIVO_RANKING = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


ARQUIVO_RELATORIO = (
    "Relatorio_Modulo8_Alarm_Flood_Metasys.xlsx"
)


ARQUIVO_EPISODIOS = (
    "Episodios_Alarm_Flood_Metasys_2026.parquet"
)


print("=" * 120)

print(
    "MÓDULO 8 - ALARM FLOOD, CASCATAS "
    "E CONCENTRAÇÃO TEMPORAL"
)

print("=" * 120)


# ======================================================================
# 3. PARÂMETROS
# ======================================================================

# ------------------------------------------------------------
# Janelas de análise
# ------------------------------------------------------------

JANELAS_MINUTOS = [
    1,
    5,
    10
]


# ------------------------------------------------------------
# Threshold principal
#
# Será utilizado na janela de 10 minutos para formar
# episódios de Alarm Flood.
#
# Pode ser recalibrado depois da análise dos percentis.
# ------------------------------------------------------------

JANELA_REFERENCIA_MIN = 10


# Flood moderado
THRESHOLD_EVENTOS_MODERADO = 25

# Flood alto
THRESHOLD_EVENTOS_ALTO = 50

# Flood severo
THRESHOLD_EVENTOS_SEVERO = 100


# Exige também um mínimo de equipamentos diferentes
THRESHOLD_EQUIPAMENTOS = 10


# Distância máxima entre duas janelas flood
# para agrupá-las em um mesmo episódio
GAP_EPISODIO_MIN = 10


# ======================================================================
# 4. VERIFICAÇÃO / UPLOAD
# ======================================================================

arquivos_necessarios = [
    ARQUIVO_BASE,
    ARQUIVO_RANKING
]


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    print(
        "\nArquivos ausentes:"
    )

    for arquivo in faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos."
    )


    from google.colab import files

    uploaded = files.upload()


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            faltantes
        )
    )


# ======================================================================
# 5. CARREGAMENTO
# ======================================================================

print(
    "\nCarregando bases..."
)


df = pd.read_parquet(
    ARQUIVO_BASE
)


ranking = pd.read_parquet(
    ARQUIVO_RANKING
)


print(
    f"Base de eventos........: {len(df):,}"
)

print(
    f"Ranking final..........: {len(ranking):,}"
)


# ======================================================================
# 6. COLUNAS NECESSÁRIAS
# ======================================================================

COLUNAS_BASE = [

    "DataHoraLocal",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "priority",

    "Flag_Geracao_Efetiva",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",
    "Flag_Falha_Comunicacao"
]


faltantes_base = [

    coluna

    for coluna in COLUNAS_BASE

    if coluna not in df.columns
]


if faltantes_base:

    raise ValueError(

        "Colunas ausentes na base:\n\n"

        +

        "\n".join(
            faltantes_base
        )
    )


# ======================================================================
# 7. AJUSTE TEMPORAL
# ======================================================================

df[
    "DataHoraLocal"
] = pd.to_datetime(

    df[
        "DataHoraLocal"
    ],

    errors="coerce"
)


# Remove timezone SOMENTE para agrupamentos temporais
df[
    "DataHoraAnalise"
] = (

    df[
        "DataHoraLocal"
    ]
    .dt.tz_localize(
        None
    )
)


# ======================================================================
# 8. BASE SOMENTE DE GERAÇÕES EFETIVAS
# ======================================================================

geracoes = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()
)


geracoes = (

    geracoes
    .sort_values(
        "DataHoraAnalise"
    )
    .reset_index(
        drop=True
    )
)


TOTAL_GERACOES = len(
    geracoes
)


print("\n" + "=" * 120)

print(
    "BASE DE GERAÇÕES EFETIVAS"
)

print("=" * 120)


print(
    f"Gerações analisadas: "
    f"{TOTAL_GERACOES:,}"
)


# ======================================================================
# 9. INTEGRAÇÃO COM RANKING FINAL
# ======================================================================

colunas_ranking_desejadas = [

    "itemName",

    "Ranking_Final",

    "Indice_Prioridade_Final",

    "Prioridade_Final",

    "Perfil_Dominante",

    "Tipo_Tratamento"
]


colunas_ranking = [

    coluna

    for coluna in colunas_ranking_desejadas

    if coluna in ranking.columns
]


ranking_aux = (

    ranking[
        colunas_ranking
    ]

    .sort_values(
        "Indice_Prioridade_Final",
        ascending=False
    )

    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


geracoes = (

    geracoes
    .merge(

        ranking_aux,

        on="itemName",

        how="left"
    )
)


# ======================================================================
# 10. FLAGS P1 / P2
# ======================================================================

geracoes[
    "Flag_P1"
] = (

    geracoes[
        "Prioridade_Final"
    ]
    ==
    "P1 - Intervenção Prioritária"
)


geracoes[
    "Flag_P2"
] = (

    geracoes[
        "Prioridade_Final"
    ]
    ==
    "P2 - Alta Prioridade"
)


geracoes[
    "Flag_P1_P2"
] = (

    geracoes[
        "Flag_P1"
    ]

    |

    geracoes[
        "Flag_P2"
    ]
)


# ======================================================================
# 11. CLASSIFICAÇÃO DO TIPO DE GERAÇÃO
# ======================================================================

geracoes[
    "Classe_Geracao"
] = np.select(

    [
        geracoes[
            "Flag_Entrada_Alarme"
        ],

        geracoes[
            "Flag_Entrada_PreAlarme"
        ],

        geracoes[
            "Flag_Falha_Comunicacao"
        ]
    ],

    [
        "Alarme",
        "Pré-Alarme",
        "Comunicação"
    ],

    default="Outro"
)


# ======================================================================
# 12. FUNÇÃO DE AGREGAÇÃO TEMPORAL
# ======================================================================

def criar_janelas(
    base,
    minutos
):

    freq = (
        f"{minutos}min"
    )


    temp = (
        base
        .set_index(
            "DataHoraAnalise"
        )
        .resample(
            freq
        )
        .agg(

            Eventos=(
                "itemName",
                "size"
            ),

            Equipamentos_Unicos=(
                "itemName",
                "nunique"
            ),

            Categorias=(
                "Categoria",
                "nunique"
            ),

            Mantenedores=(
                "Mantenedor",
                "nunique"
            ),

            Tipos=(
                "Tipo",
                "nunique"
            ),

            Torres=(
                "Torre",
                "nunique"
            ),

            Alarmes=(
                "Flag_Entrada_Alarme",
                "sum"
            ),

            PreAlarmes=(
                "Flag_Entrada_PreAlarme",
                "sum"
            ),

            Falhas_Comunicacao=(
                "Flag_Falha_Comunicacao",
                "sum"
            ),

            Eventos_P1=(
                "Flag_P1",
                "sum"
            ),

            Eventos_P2=(
                "Flag_P2",
                "sum"
            ),

            Eventos_P1_P2=(
                "Flag_P1_P2",
                "sum"
            ),

            Prioridade_Minima=(
                "priority",
                "min"
            ),

            Prioridade_Mediana=(
                "priority",
                "median"
            )
        )
        .reset_index()
    )


    temp[
        "Janela_Minutos"
    ] = minutos


    temp[
        "Perc_P1_P2"
    ] = np.where(

        temp[
            "Eventos"
        ] > 0,

        temp[
            "Eventos_P1_P2"
        ]

        /

        temp[
            "Eventos"
        ]

        * 100,

        0
    )


    temp[
        "Eventos_por_Equipamento"
    ] = np.where(

        temp[
            "Equipamentos_Unicos"
        ] > 0,

        temp[
            "Eventos"
        ]

        /

        temp[
            "Equipamentos_Unicos"
        ],

        0
    )


    return temp


# ======================================================================
# 13. GERAÇÃO DAS JANELAS
# ======================================================================

janelas = {}


for janela in JANELAS_MINUTOS:

    print(
        f"Calculando janela de {janela} minuto(s)..."
    )

    janelas[
        janela
    ] = criar_janelas(
        geracoes,
        janela
    )


# ======================================================================
# 14. DISTRIBUIÇÃO ESTATÍSTICA DAS JANELAS
# ======================================================================

distribuicao = []


for janela in JANELAS_MINUTOS:

    temp = (

        janelas[
            janela
        ]
        .loc[
            janelas[
                janela
            ][
                "Eventos"
            ] > 0
        ]
    )


    serie = (
        temp[
            "Eventos"
        ]
    )


    distribuicao.append({

        "Janela_Min": janela,

        "Janelas_Com_Eventos": (
            len(
                temp
            )
        ),

        "Media_Eventos": (
            serie.mean()
        ),

        "Mediana_Eventos": (
            serie.median()
        ),

        "P90": (
            serie.quantile(
                0.90
            )
        ),

        "P95": (
            serie.quantile(
                0.95
            )
        ),

        "P99": (
            serie.quantile(
                0.99
            )
        ),

        "P995": (
            serie.quantile(
                0.995
            )
        ),

        "P999": (
            serie.quantile(
                0.999
            )
        ),

        "Maximo": (
            serie.max()
        )
    })


distribuicao_janelas = pd.DataFrame(
    distribuicao
)


# ======================================================================
# 15. CLASSIFICAÇÃO DAS JANELAS DE 10 MINUTOS
# ======================================================================

j10 = (

    janelas[
        JANELA_REFERENCIA_MIN
    ]
    .copy()
)


j10[
    "Nivel_Flood"
] = np.select(

    [
        (
            j10[
                "Eventos"
            ]
            >=
            THRESHOLD_EVENTOS_SEVERO
        ),

        (
            j10[
                "Eventos"
            ]
            >=
            THRESHOLD_EVENTOS_ALTO
        ),

        (
            j10[
                "Eventos"
            ]
            >=
            THRESHOLD_EVENTOS_MODERADO
        )
    ],

    [
        "Severo",
        "Alto",
        "Moderado"
    ],

    default="Normal"
)


# ======================================================================
# 16. FLAG DE ALARM FLOOD
#
# Exigimos:
#
# - pelo menos threshold de eventos
# - pelo menos número mínimo de equipamentos diferentes
# ======================================================================

j10[
    "Flag_Alarm_Flood"
] = (

    (
        j10[
            "Eventos"
        ]
        >=
        THRESHOLD_EVENTOS_MODERADO
    )

    &

    (
        j10[
            "Equipamentos_Unicos"
        ]
        >=
        THRESHOLD_EQUIPAMENTOS
    )
)


# ======================================================================
# 17. CLASSIFICAÇÃO DA ABRANGÊNCIA
# ======================================================================

def classificar_abrangencia(
    row
):

    torres = (
        row[
            "Torres"
        ]
    )

    categorias = (
        row[
            "Categorias"
        ]
    )


    if (
        torres >= 4
        and
        categorias >= 3
    ):

        return (
            "Cascata Ampla / Sistêmica"
        )


    if (
        torres >= 2
        and
        categorias >= 2
    ):

        return (
            "Cascata Multissistema"
        )


    if (
        torres >= 2
    ):

        return (
            "Cascata Multitorre"
        )


    if (
        categorias >= 2
    ):

        return (
            "Cascata Multicategoria"
        )


    return (
        "Flood Localizado"
    )


j10[
    "Abrangencia_Flood"
] = (

    j10
    .apply(
        classificar_abrangencia,
        axis=1
    )
)


# ======================================================================
# 18. BASE SOMENTE DOS FLOODS
# ======================================================================

floods = (

    j10.loc[
        j10[
            "Flag_Alarm_Flood"
        ]
    ]

    .copy()

    .sort_values(
        "DataHoraAnalise"
    )

    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 120)

print(
    "ALARM FLOODS IDENTIFICADOS"
)

print("=" * 120)


print(
    f"Janelas flood: "
    f"{len(floods):,}"
)


# ======================================================================
# 19. CRIAÇÃO DOS EPISÓDIOS
#
# Janelas flood próximas serão agrupadas em um episódio único.
# ======================================================================

if len(
    floods
) > 0:

    floods[
        "DataHora_Anterior"
    ] = (

        floods[
            "DataHoraAnalise"
        ]
        .shift(1)
    )


    floods[
        "Gap_Min"
    ] = (

        (
            floods[
                "DataHoraAnalise"
            ]

            -

            floods[
                "DataHora_Anterior"
            ]
        )

        .dt.total_seconds()

        /

        60
    )


    floods[
        "Flag_Novo_Episodio"
    ] = (

        floods[
            "Gap_Min"
        ]
        .isna()

        |

        (
            floods[
                "Gap_Min"
            ]
            >
            GAP_EPISODIO_MIN
        )
    )


    floods[
        "ID_Episodio"
    ] = (

        floods[
            "Flag_Novo_Episodio"
        ]
        .cumsum()
    )


else:

    floods[
        "ID_Episodio"
    ] = pd.Series(
        dtype="int64"
    )


# ======================================================================
# 20. RESUMO INICIAL DOS EPISÓDIOS
# ======================================================================

if len(
    floods
) > 0:

    episodios = (

        floods
        .groupby(
            "ID_Episodio",
            observed=True
        )
        .agg(

            Inicio_Episodio=(
                "DataHoraAnalise",
                "min"
            ),

            Fim_Episodio=(
                "DataHoraAnalise",
                "max"
            ),

            Janelas_Flood=(
                "DataHoraAnalise",
                "size"
            ),

            Eventos_Janelas=(
                "Eventos",
                "sum"
            ),

            Pico_Eventos_10min=(
                "Eventos",
                "max"
            ),

            Pico_Equipamentos_10min=(
                "Equipamentos_Unicos",
                "max"
            ),

            Max_Torres=(
                "Torres",
                "max"
            ),

            Max_Categorias=(
                "Categorias",
                "max"
            ),

            # ----------------------------------------------------------
            # IMPORTANTE:
            # Esses valores vêm da soma das janelas de 10 minutos.
            #
            # Usamos nomes diferentes para evitar colisão posteriormente
            # com Eventos_P1 / Eventos_P2 calculados sobre os eventos
            # reais de cada episódio.
            # ----------------------------------------------------------

            Eventos_P1_Janelas=(
                "Eventos_P1",
                "sum"
            ),

            Eventos_P2_Janelas=(
                "Eventos_P2",
                "sum"
            ),

            Eventos_P1_P2_Janelas=(
                "Eventos_P1_P2",
                "sum"
            )
        )

        .reset_index()
    )


    # --------------------------------------------------------------
    # Cada janela representa 10 minutos.
    # Portanto, o término real do episódio é o início da
    # última janela + duração da janela de referência.
    # --------------------------------------------------------------

    episodios[
        "Fim_Episodio"
    ] = (

        episodios[
            "Fim_Episodio"
        ]

        +

        pd.Timedelta(
            minutes=JANELA_REFERENCIA_MIN
        )
    )


    # --------------------------------------------------------------
    # Duração total do episódio
    # --------------------------------------------------------------

    episodios[
        "Duracao_Episodio_Min"
    ] = (

        (
            episodios[
                "Fim_Episodio"
            ]

            -

            episodios[
                "Inicio_Episodio"
            ]
        )

        .dt.total_seconds()

        /

        60
    )


else:

    episodios = pd.DataFrame()


# ======================================================================
# 21. DETALHAMENTO DOS EVENTOS DE CADA EPISÓDIO
# ======================================================================

detalhes_episodios = []


if len(
    episodios
) > 0:

    for episodio in episodios.itertuples(
        index=False
    ):

        inicio = (
            episodio.Inicio_Episodio
        )

        fim = (
            episodio.Fim_Episodio
        )


        temp = (

            geracoes.loc[
                (
                    geracoes[
                        "DataHoraAnalise"
                    ]
                    >=
                    inicio
                )

                &

                (
                    geracoes[
                        "DataHoraAnalise"
                    ]
                    <
                    fim
                )
            ]

            .copy()
        )


        if len(
            temp
        ) == 0:

            continue


        # --------------------------------------------------------------
        # Moda segura
        # --------------------------------------------------------------

        def moda_temp(
            serie
        ):

            serie = (
                serie
                .dropna()
            )

            if len(
                serie
            ) == 0:

                return pd.NA


            moda = (
                serie
                .mode()
            )


            if len(
                moda
            ) > 0:

                return moda.iloc[0]


            return serie.iloc[0]


        detalhes_episodios.append({

            "ID_Episodio":
                episodio.ID_Episodio,

            "Eventos_Reais":
                len(temp),

            "Equipamentos_Unicos":
                temp[
                    "itemName"
                ].nunique(),

            "Torres":
                temp[
                    "Torre"
                ].nunique(),

            "Categorias":
                temp[
                    "Categoria"
                ].nunique(),

            "Mantenedores":
                temp[
                    "Mantenedor"
                ].nunique(),

            "Tipos":
                temp[
                    "Tipo"
                ].nunique(),

            "Alarmes":
                int(
                    temp[
                        "Flag_Entrada_Alarme"
                    ].sum()
                ),

            "PreAlarmes":
                int(
                    temp[
                        "Flag_Entrada_PreAlarme"
                    ].sum()
                ),

            "Falhas_Comunicacao":
                int(
                    temp[
                        "Flag_Falha_Comunicacao"
                    ].sum()
                ),

            "Eventos_P1":
                int(
                    temp[
                        "Flag_P1"
                    ].sum()
                ),

            "Eventos_P2":
                int(
                    temp[
                        "Flag_P2"
                    ].sum()
                ),

            "Eventos_P1_P2":
    int(
        temp[
            "Flag_P1_P2"
        ].sum()
    ),

            "Equipamentos_P1":
                temp.loc[
                    temp[
                        "Flag_P1"
                    ],
                    "itemName"
                ].nunique(),

            "Equipamentos_P2":
                temp.loc[
                    temp[
                        "Flag_P2"
                    ],
                    "itemName"
                ].nunique(),

            "Torre_Dominante":
                moda_temp(
                    temp[
                        "Torre"
                    ]
                ),

            "Categoria_Dominante":
                moda_temp(
                    temp[
                        "Categoria"
                    ]
                ),

            "Tipo_Dominante":
                moda_temp(
                    temp[
                        "Tipo"
                    ]
                ),

            "Mantenedor_Dominante":
                moda_temp(
                    temp[
                        "Mantenedor"
                    ]
                )
        })


detalhes_episodios = pd.DataFrame(
    detalhes_episodios
)


# ======================================================================
# 22. MERGE COM EPISÓDIOS
# ======================================================================

if (
    len(
        episodios
    ) > 0

    and

    len(
        detalhes_episodios
    ) > 0
):

    episodios = (

        episodios
        .merge(

            detalhes_episodios,

            on="ID_Episodio",

            how="left"
        )
    )


# ======================================================================
# 23. CLASSIFICAÇÃO DE SEVERIDADE DO EPISÓDIO
# ======================================================================

if len(
    episodios
) > 0:

    episodios[
        "Perc_Eventos_P1_P2"
    ] = np.where(

        episodios[
            "Eventos_Reais"
        ] > 0,

        (
            episodios[
                "Eventos_P1"
            ]

            +

            episodios[
                "Eventos_P2"
            ]
        )

        /

        episodios[
            "Eventos_Reais"
        ]

        * 100,

        0
    )


    episodios[
        "Nivel_Severidade_Episodio"
    ] = np.select(

        [
            (
                episodios[
                    "Pico_Eventos_10min"
                ]
                >=
                THRESHOLD_EVENTOS_SEVERO
            ),

            (
                episodios[
                    "Pico_Eventos_10min"
                ]
                >=
                THRESHOLD_EVENTOS_ALTO
            )
        ],

        [
            "Severo",
            "Alto"
        ],

        default="Moderado"
    )


# ======================================================================
# 24. CLASSIFICAÇÃO DA ABRANGÊNCIA DO EPISÓDIO
# ======================================================================

def classificar_episodio(
    row
):

    if (
        row[
            "Torres"
        ] >= 4

        and

        row[
            "Categorias"
        ] >= 3
    ):

        return (
            "Sistêmico"
        )


    if (
        row[
            "Torres"
        ] >= 2

        and

        row[
            "Categorias"
        ] >= 2
    ):

        return (
            "Multissistema"
        )


    if (
        row[
            "Torres"
        ] >= 2
    ):

        return (
            "Multitorre"
        )


    return (
        "Localizado"
    )


if len(
    episodios
) > 0:

    episodios[
        "Abrangencia_Episodio"
    ] = (

        episodios
        .apply(
            classificar_episodio,
            axis=1
        )
    )


# ======================================================================
# 25. SCORE DE FLOOD
#
# Objetivo:
# Priorizar episódios considerando:
#
# 40% volume
# 25% equipamentos envolvidos
# 15% abrangência
# 20% presença de equipamentos P1/P2
# ======================================================================

if len(
    episodios
) > 0:

    episodios[
        "Score_Volume"
    ] = (

        episodios[
            "Eventos_Reais"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    episodios[
        "Score_Equipamentos"
    ] = (

        episodios[
            "Equipamentos_Unicos"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    episodios[
        "Score_Abrangencia"
    ] = (

        (
            episodios[
                "Torres"
            ]
            +
            episodios[
                "Categorias"
            ]
        )

        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    episodios[
        "Score_Prioridade"
    ] = (

        episodios[
            "Perc_Eventos_P1_P2"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    episodios[
        "Indice_Alarm_Flood"
    ] = (

        episodios[
            "Score_Volume"
        ]
        * 0.40

        +

        episodios[
            "Score_Equipamentos"
        ]
        * 0.25

        +

        episodios[
            "Score_Abrangencia"
        ]
        * 0.15

        +

        episodios[
            "Score_Prioridade"
        ]
        * 0.20
    )


    episodios[
        "Classe_Alarm_Flood"
    ] = np.select(

        [
            episodios[
                "Indice_Alarm_Flood"
            ] >= 80,

            episodios[
                "Indice_Alarm_Flood"
            ] >= 60,

            episodios[
                "Indice_Alarm_Flood"
            ] >= 40
        ],

        [
            "Crítico",
            "Alto",
            "Moderado"
        ],

        default="Baixo"
    )


# ======================================================================
# 26. RANKING DOS EPISÓDIOS
# ======================================================================

if len(
    episodios
) > 0:

    episodios = (

        episodios
        .sort_values(

            [
                "Indice_Alarm_Flood",
                "Eventos_Reais"
            ],

            ascending=[
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


    episodios[
        "Ranking_Flood"
    ] = (

        np.arange(
            1,
            len(
                episodios
            ) + 1
        )
    )


# ======================================================================
# 27. ASSOCIAÇÃO DE EVENTOS AOS EPISÓDIOS
# ======================================================================

eventos_em_flood = []


if len(
    episodios
) > 0:

    for episodio in episodios.itertuples(
        index=False
    ):

        temp = (

            geracoes.loc[
                (
                    geracoes[
                        "DataHoraAnalise"
                    ]
                    >=
                    episodio.Inicio_Episodio
                )

                &

                (
                    geracoes[
                        "DataHoraAnalise"
                    ]
                    <
                    episodio.Fim_Episodio
                )
            ]

            .copy()
        )


        temp[
            "ID_Episodio"
        ] = (
            episodio.ID_Episodio
        )


        temp[
            "Ranking_Flood"
        ] = (
            episodio.Ranking_Flood
        )


        temp[
            "Indice_Alarm_Flood"
        ] = (
            episodio.Indice_Alarm_Flood
        )


        temp[
            "Classe_Alarm_Flood"
        ] = (
            episodio.Classe_Alarm_Flood
        )


        eventos_em_flood.append(
            temp
        )


if len(
    eventos_em_flood
) > 0:

    eventos_em_flood = pd.concat(

        eventos_em_flood,

        ignore_index=True
    )


else:

    eventos_em_flood = pd.DataFrame()


# ======================================================================
# 28. EQUIPAMENTOS MAIS PRESENTES EM FLOODS
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    ranking_equip_flood = (

        eventos_em_flood
        .groupby(

            [
                "itemName",
                "itemDescription",
                "Categoria",
                "Mantenedor",
                "Tipo",
                "Torre"
            ],

            observed=True,

            dropna=False
        )
        .agg(

            Eventos_em_Flood=(
                "itemName",
                "size"
            ),

            Episodios_Flood=(
                "ID_Episodio",
                "nunique"
            ),

            Alarmes=(
                "Flag_Entrada_Alarme",
                "sum"
            ),

            PreAlarmes=(
                "Flag_Entrada_PreAlarme",
                "sum"
            ),

            Falhas_Comunicacao=(
                "Flag_Falha_Comunicacao",
                "sum"
            ),

            Indice_Max_Flood=(
                "Indice_Alarm_Flood",
                "max"
            ),

            Ranking_Final_Equipamento=(
                "Ranking_Final",
                "min"
            ),

            Indice_Prioridade_Final=(
                "Indice_Prioridade_Final",
                "max"
            )
        )

        .reset_index()

        .sort_values(

            [
                "Episodios_Flood",
                "Eventos_em_Flood"
            ],

            ascending=[
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


else:

    ranking_equip_flood = pd.DataFrame()


# ======================================================================
# 29. PERCENTUAL DAS GERAÇÕES QUE OCORREM EM FLOOD
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    eventos_unicos_flood = (

        eventos_em_flood
        .drop_duplicates(
            subset=[
                "DataHoraAnalise",
                "itemName",
                "priority",
                "Classe_Geracao"
            ]
        )
    )


    TOTAL_EVENTOS_EM_FLOOD = len(
        eventos_unicos_flood
    )


else:

    TOTAL_EVENTOS_EM_FLOOD = 0


PERC_EVENTOS_FLOOD = (

    TOTAL_EVENTOS_EM_FLOOD

    /

    TOTAL_GERACOES

    * 100

    if TOTAL_GERACOES > 0

    else 0
)


# ======================================================================
# 30. ANÁLISE MENSAL DOS FLOODS
# ======================================================================

if len(
    episodios
) > 0:

    episodios[
        "Ano"
    ] = (

        episodios[
            "Inicio_Episodio"
        ]
        .dt.year
    )


    episodios[
        "Mes_Numero"
    ] = (

        episodios[
            "Inicio_Episodio"
        ]
        .dt.month
    )


    MAPA_MESES = {

        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }


    episodios[
        "Mes"
    ] = (

        episodios[
            "Mes_Numero"
        ]
        .map(
            MAPA_MESES
        )
    )


    resumo_mensal = (

        episodios
        .groupby(

            [
                "Ano",
                "Mes_Numero",
                "Mes"
            ],

            observed=True
        )
        .agg(

            Episodios_Flood=(
                "ID_Episodio",
                "size"
            ),

            Eventos_em_Flood=(
                "Eventos_Reais",
                "sum"
            ),

            Equipamentos_Medio=(
                "Equipamentos_Unicos",
                "mean"
            ),

            Pico_Eventos_10min=(
                "Pico_Eventos_10min",
                "max"
            ),

            Floods_Criticos=(
                "Classe_Alarm_Flood",
                lambda x:
                (
                    x
                    ==
                    "Crítico"
                ).sum()
            ),

            Floods_Altos=(
                "Classe_Alarm_Flood",
                lambda x:
                (
                    x
                    ==
                    "Alto"
                ).sum()
            ),

            Indice_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            )
        )

        .reset_index()

        .sort_values(
            [
                "Ano",
                "Mes_Numero"
            ]
        )
    )


else:

    resumo_mensal = pd.DataFrame()


# ======================================================================
# 31. ANÁLISE POR HORA DO DIA
# ======================================================================

if len(
    episodios
) > 0:

    episodios[
        "Hora"
    ] = (

        episodios[
            "Inicio_Episodio"
        ]
        .dt.hour
    )


    resumo_hora = (

        episodios
        .groupby(
            "Hora",
            observed=True
        )
        .agg(

            Episodios_Flood=(
                "ID_Episodio",
                "size"
            ),

            Eventos_em_Flood=(
                "Eventos_Reais",
                "sum"
            ),

            Indice_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            ),

            Indice_Maximo=(
                "Indice_Alarm_Flood",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            "Hora"
        )
    )


else:

    resumo_hora = pd.DataFrame()


# ======================================================================
# 32. ANÁLISE POR DIA DA SEMANA
# ======================================================================

if len(
    episodios
) > 0:

    episodios[
        "Dia_Semana_Numero"
    ] = (

        episodios[
            "Inicio_Episodio"
        ]
        .dt.dayofweek
        +
        1
    )


    MAPA_DIAS = {

        1: "Segunda",
        2: "Terça",
        3: "Quarta",
        4: "Quinta",
        5: "Sexta",
        6: "Sábado",
        7: "Domingo"
    }


    episodios[
        "Dia_Semana"
    ] = (

        episodios[
            "Dia_Semana_Numero"
        ]
        .map(
            MAPA_DIAS
        )
    )


    resumo_dia_semana = (

        episodios
        .groupby(

            [
                "Dia_Semana_Numero",
                "Dia_Semana"
            ],

            observed=True
        )
        .agg(

            Episodios_Flood=(
                "ID_Episodio",
                "size"
            ),

            Eventos_em_Flood=(
                "Eventos_Reais",
                "sum"
            ),

            Indice_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            )
        )

        .reset_index()

        .sort_values(
            "Dia_Semana_Numero"
        )
    )


else:

    resumo_dia_semana = pd.DataFrame()


# ======================================================================
# 33. CATEGORIAS DURANTE FLOODS
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    resumo_categoria = (

        eventos_em_flood
        .groupby(
            "Categoria",
            observed=True,
            dropna=False
        )
        .agg(

            Eventos=(
                "itemName",
                "size"
            ),

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Episodios=(
                "ID_Episodio",
                "nunique"
            ),

            Eventos_P1=(
                "Flag_P1",
                "sum"
            ),

            Eventos_P2=(
                "Flag_P2",
                "sum"
            )
        )

        .reset_index()

        .sort_values(
            "Eventos",
            ascending=False
        )
    )


else:

    resumo_categoria = pd.DataFrame()


# ======================================================================
# 34. TORRES DURANTE FLOODS
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    resumo_torre = (

        eventos_em_flood
        .groupby(
            "Torre",
            observed=True,
            dropna=False
        )
        .agg(

            Eventos=(
                "itemName",
                "size"
            ),

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Episodios=(
                "ID_Episodio",
                "nunique"
            )
        )

        .reset_index()

        .sort_values(
            "Eventos",
            ascending=False
        )
    )


else:

    resumo_torre = pd.DataFrame()


# ======================================================================
# 35. TIPOS DURANTE FLOODS
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    resumo_tipo = (

        eventos_em_flood
        .groupby(
            "Tipo",
            observed=True,
            dropna=False
        )
        .agg(

            Eventos=(
                "itemName",
                "size"
            ),

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Episodios=(
                "ID_Episodio",
                "nunique"
            )
        )

        .reset_index()

        .sort_values(
            "Eventos",
            ascending=False
        )
    )


else:

    resumo_tipo = pd.DataFrame()


# ======================================================================
# 36. MATRIZ TORRE × CATEGORIA
#
# CORREÇÃO:
# Colunas categóricas podem provocar KeyError no pivot_table
# quando utilizadas juntamente com margins=True.
#
# Para evitar esse problema:
# - trabalhamos em uma cópia
# - convertemos Torre e Categoria temporariamente para string
# - criamos a matriz sem margins
# - calculamos Total manualmente
# ======================================================================

if len(
    eventos_em_flood
) > 0:

    # --------------------------------------------------------------
    # Cópia apenas das colunas necessárias
    # --------------------------------------------------------------

    matriz_base = (

        eventos_em_flood[
            [
                "Torre",
                "Categoria",
                "itemName"
            ]
        ]

        .copy()
    )


    # --------------------------------------------------------------
    # Remove comportamento Categorical do Pandas
    #
    # O fillna ocorre antes de astype(str) para evitar transformar
    # valores ausentes em textos inconsistentes.
    # --------------------------------------------------------------

    matriz_base[
        "Torre"
    ] = (

        matriz_base[
            "Torre"
        ]
        .astype("string")
        .fillna("Não Classificado")
    )


    matriz_base[
        "Categoria"
    ] = (

        matriz_base[
            "Categoria"
        ]
        .astype("string")
        .fillna("Não Classificado")
    )


    # --------------------------------------------------------------
    # Pivot SEM margens automáticas
    # --------------------------------------------------------------

    matriz_torre_categoria = pd.pivot_table(

        matriz_base,

        index="Torre",

        columns="Categoria",

        values="itemName",

        aggfunc="size",

        fill_value=0,

        observed=True,

        margins=False
    )


    # --------------------------------------------------------------
    # Total por linha
    # --------------------------------------------------------------

    matriz_torre_categoria[
        "Total"
    ] = (

        matriz_torre_categoria
        .sum(
            axis=1
        )
    )


    # --------------------------------------------------------------
    # Total por coluna
    # --------------------------------------------------------------

    linha_total = (

        matriz_torre_categoria
        .sum(
            axis=0
        )
    )


    linha_total.name = "Total"


    matriz_torre_categoria = pd.concat(

        [
            matriz_torre_categoria,
            linha_total.to_frame().T
        ]
    )


    # --------------------------------------------------------------
    # Ordena as torres pela quantidade total de eventos
    # mantendo a linha Total no final
    # --------------------------------------------------------------

    if len(
        matriz_torre_categoria
    ) > 1:

        dados_sem_total = (

            matriz_torre_categoria
            .drop(
                index="Total",
                errors="ignore"
            )

            .sort_values(
                "Total",
                ascending=False
            )
        )


        linha_total_df = (

            matriz_torre_categoria
            .loc[
                ["Total"]
            ]
        )


        matriz_torre_categoria = pd.concat(

            [
                dados_sem_total,
                linha_total_df
            ]
        )


else:

    matriz_torre_categoria = pd.DataFrame()


# ======================================================================
# 37. TOP 100 EPISÓDIOS
# ======================================================================

if len(
    episodios
) > 0:

    top100_episodios = (

        episodios
        .head(100)
        .copy()
    )


else:

    top100_episodios = pd.DataFrame()


# ======================================================================
# 38. RESUMO POR CLASSE
# ======================================================================

if len(
    episodios
) > 0:

    resumo_classes = (

        episodios
        .groupby(
            "Classe_Alarm_Flood",
            observed=True
        )
        .agg(

            Episodios=(
                "ID_Episodio",
                "size"
            ),

            Eventos=(
                "Eventos_Reais",
                "sum"
            ),

            Equipamentos_Medio=(
                "Equipamentos_Unicos",
                "mean"
            ),

            Duracao_Media_Min=(
                "Duracao_Episodio_Min",
                "mean"
            ),

            Indice_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            )
        )

        .reset_index()
    )


else:

    resumo_classes = pd.DataFrame()


# ======================================================================
# 39. RESUMO POR ABRANGÊNCIA
# ======================================================================

if len(
    episodios
) > 0:

    resumo_abrangencia = (

        episodios
        .groupby(
            "Abrangencia_Episodio",
            observed=True
        )
        .agg(

            Episodios=(
                "ID_Episodio",
                "size"
            ),

            Eventos=(
                "Eventos_Reais",
                "sum"
            ),

            Equipamentos_Medio=(
                "Equipamentos_Unicos",
                "mean"
            ),

            Torres_Media=(
                "Torres",
                "mean"
            ),

            Categorias_Media=(
                "Categorias",
                "mean"
            )
        )

        .reset_index()

        .sort_values(
            "Eventos",
            ascending=False
        )
    )


else:

    resumo_abrangencia = pd.DataFrame()


# ======================================================================
# 40. RESUMO EXECUTIVO
# ======================================================================

TOTAL_EPISODIOS = len(
    episodios
)


if TOTAL_EPISODIOS > 0:

    EPISODIOS_CRITICOS = (

        episodios[
            "Classe_Alarm_Flood"
        ]
        .eq(
            "Crítico"
        )
        .sum()
    )


    EPISODIOS_ALTOS = (

        episodios[
            "Classe_Alarm_Flood"
        ]
        .eq(
            "Alto"
        )
        .sum()
    )


    maior_episodio = (

        episodios
        .sort_values(
            "Eventos_Reais",
            ascending=False
        )
        .iloc[0]
    )


    maior_indice = (
        episodios.iloc[0]
    )


else:

    EPISODIOS_CRITICOS = 0
    EPISODIOS_ALTOS = 0
    maior_episodio = None
    maior_indice = None


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Gerações efetivas analisadas",

            "Janelas flood identificadas",

            "Episódios de Alarm Flood",

            "Episódios críticos",

            "Episódios altos",

            "Eventos únicos em períodos de flood",

            "% das gerações em períodos de flood",

            "Maior episódio - quantidade de eventos",

            "Maior episódio - equipamentos envolvidos",

            "Maior episódio - torres envolvidas",

            "Maior episódio - categorias envolvidas",

            "Maior índice de Alarm Flood"
        ],

        "Resultado": [

            f"{TOTAL_GERACOES:,}",

            f"{len(floods):,}",

            f"{TOTAL_EPISODIOS:,}",

            f"{EPISODIOS_CRITICOS:,}",

            f"{EPISODIOS_ALTOS:,}",

            f"{TOTAL_EVENTOS_EM_FLOOD:,}",

            f"{PERC_EVENTOS_FLOOD:.2f}%",

            (
                f"{int(maior_episodio['Eventos_Reais']):,}"
                if maior_episodio is not None
                else "0"
            ),

            (
                f"{int(maior_episodio['Equipamentos_Unicos']):,}"
                if maior_episodio is not None
                else "0"
            ),

            (
                f"{int(maior_episodio['Torres']):,}"
                if maior_episodio is not None
                else "0"
            ),

            (
                f"{int(maior_episodio['Categorias']):,}"
                if maior_episodio is not None
                else "0"
            ),

            (
                f"{maior_indice['Indice_Alarm_Flood']:.2f}"
                if maior_indice is not None
                else "0.00"
            )
        ]
    }
)


# ======================================================================
# 41. EXIBIÇÃO
# ======================================================================

print("\n" + "=" * 120)

print(
    "RESUMO EXECUTIVO"
)

print("=" * 120)


display(
    resumo_executivo
)


print("\n" + "=" * 120)

print(
    "DISTRIBUIÇÃO ESTATÍSTICA DAS JANELAS"
)

print("=" * 120)


display(
    distribuicao_janelas
)


print("\n" + "=" * 120)

print(
    "TOP 30 EPISÓDIOS DE ALARM FLOOD"
)

print("=" * 120)


if len(
    episodios
) > 0:

    display(
        episodios.head(30)
    )


print("\n" + "=" * 120)

print(
    "EQUIPAMENTOS MAIS PRESENTES EM FLOODS"
)

print("=" * 120)


if len(
    ranking_equip_flood
) > 0:

    display(
        ranking_equip_flood.head(30)
    )


print("\n" + "=" * 120)

print(
    "RESUMO MENSAL"
)

print("=" * 120)


if len(
    resumo_mensal
) > 0:

    display(
        resumo_mensal
    )


# ======================================================================
# 42. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 120)

print(
    "VALIDAÇÕES"
)

print("=" * 120)


print(
    f"Gerações efetivas.......................: "
    f"{TOTAL_GERACOES:,}"
)


print(
    f"Janelas flood...........................: "
    f"{len(floods):,}"
)


print(
    f"Episódios...............................: "
    f"{TOTAL_EPISODIOS:,}"
)


print(
    f"Eventos únicos em flood.................: "
    f"{TOTAL_EVENTOS_EM_FLOOD:,}"
)


print(
    f"% eventos em flood......................: "
    f"{PERC_EVENTOS_FLOOD:.2f}%"
)


# ----------------------------------------------------------------------
# Validação temporal
# ----------------------------------------------------------------------

if len(
    episodios
) > 0:

    episodios_invalidos = (

        episodios[
            "Fim_Episodio"
        ]

        <

        episodios[
            "Inicio_Episodio"
        ]
    ).sum()


else:

    episodios_invalidos = 0


print(
    f"Episódios com fim < início..............: "
    f"{episodios_invalidos}"
)


# ----------------------------------------------------------------------
# Meses encontrados
# ----------------------------------------------------------------------

if len(
    episodios
) > 0:

    meses_identificados = sorted(

        episodios[
            "Mes_Numero"
        ]
        .dropna()
        .unique()
        .tolist()
    )


else:

    meses_identificados = []


print(
    f"Meses encontrados.......................: "
    f"{meses_identificados}"
)


# ======================================================================
# 43. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Parametro": [

            "Evento analisado",

            "Janelas avaliadas",

            "Janela de referência",

            "Flood moderado",

            "Flood alto",

            "Flood severo",

            "Equipamentos mínimos",

            "Gap entre janelas",

            "Índice Alarm Flood",

            "Cascata sistêmica"
        ],

        "Definicao": [

            (
                "Somente gerações efetivas provenientes "
                "do Módulo 2."
            ),

            (
                "1, 5 e 10 minutos."
            ),

            (
                "10 minutos."
            ),

            (
                f">= {THRESHOLD_EVENTOS_MODERADO} eventos "
                f"em {JANELA_REFERENCIA_MIN} minutos."
            ),

            (
                f">= {THRESHOLD_EVENTOS_ALTO} eventos "
                f"em {JANELA_REFERENCIA_MIN} minutos."
            ),

            (
                f">= {THRESHOLD_EVENTOS_SEVERO} eventos "
                f"em {JANELA_REFERENCIA_MIN} minutos."
            ),

            (
                f">= {THRESHOLD_EQUIPAMENTOS} equipamentos "
                "diferentes para caracterizar flood."
            ),

            (
                f"Até {GAP_EPISODIO_MIN} minutos entre "
                "janelas consecutivas para formar um episódio."
            ),

            (
                "40% volume + 25% equipamentos + "
                "15% abrangência + 20% participação P1/P2."
            ),

            (
                "Episódio envolvendo >=4 torres "
                "e >=3 categorias."
            )
        ]
    }
)


# ======================================================================
# 44. SALVAMENTO DO PARQUET
# ======================================================================

if len(
    episodios
) > 0:

    episodios.to_parquet(

        ARQUIVO_EPISODIOS,

        index=False
    )


    print(
        f"\nArquivo criado:\n"
        f"{ARQUIVO_EPISODIOS}"
    )


# ======================================================================
# 45. PREPARAÇÃO PARA EXCEL
# ======================================================================

def remover_timezone_dataframe(
    dataframe
):

    temp = (
        dataframe
        .copy()
    )


    for coluna in temp.columns:

        if isinstance(
            temp[
                coluna
            ].dtype,
            pd.DatetimeTZDtype
        ):

            temp[
                coluna
            ] = (

                temp[
                    coluna
                ]
                .dt.tz_localize(
                    None
                )
            )


    return temp


# As datas de análise já estão timezone-naive,
# mas mantemos função de segurança.
episodios_excel = remover_timezone_dataframe(
    episodios
)

floods_excel = remover_timezone_dataframe(
    floods
)


# ======================================================================
# 46. EXPORTAÇÃO PARA EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 8..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


    distribuicao_janelas.to_excel(
        writer,
        sheet_name="Distribuicao Janelas",
        index=False
    )


    janelas[
        1
    ].sort_values(
        "Eventos",
        ascending=False
    ).head(
        5000
    ).to_excel(
        writer,
        sheet_name="Top Janelas 1min",
        index=False
    )


    janelas[
        5
    ].sort_values(
        "Eventos",
        ascending=False
    ).head(
        5000
    ).to_excel(
        writer,
        sheet_name="Top Janelas 5min",
        index=False
    )


    janelas[
        10
    ].sort_values(
        "Eventos",
        ascending=False
    ).head(
        5000
    ).to_excel(
        writer,
        sheet_name="Top Janelas 10min",
        index=False
    )


    floods_excel.to_excel(
        writer,
        sheet_name="Janelas Flood",
        index=False
    )


    episodios_excel.to_excel(
        writer,
        sheet_name="Episodios Flood",
        index=False
    )


    top100_episodios.to_excel(
        writer,
        sheet_name="Top100 Flood",
        index=False
    )


    resumo_classes.to_excel(
        writer,
        sheet_name="Classes Flood",
        index=False
    )


    resumo_abrangencia.to_excel(
        writer,
        sheet_name="Abrangencia",
        index=False
    )


    resumo_mensal.to_excel(
        writer,
        sheet_name="Mensal",
        index=False
    )


    resumo_hora.to_excel(
        writer,
        sheet_name="Hora",
        index=False
    )


    resumo_dia_semana.to_excel(
        writer,
        sheet_name="Dia Semana",
        index=False
    )


    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )


    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )


    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )


    ranking_equip_flood.to_excel(
        writer,
        sheet_name="Equipamentos Flood",
        index=False
    )


    matriz_torre_categoria.to_excel(
        writer,
        sheet_name="Matriz Torre x Cat"
    )


# ======================================================================
# 47. FORMATAÇÃO DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0


        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (

                    ""

                    if cell.value is None

                    else str(
                        cell.value
                    )
                )


                max_length = max(

                    max_length,

                    len(
                        valor
                    )
                )


            except:

                pass


        largura = min(

            max(
                max_length + 2,
                12
            ),

            50
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 48. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 120)

print(
    "MÓDULO 8 CONCLUÍDO COM SUCESSO"
)

print("=" * 120)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


if Path(
    ARQUIVO_EPISODIOS
).exists():

    print(
        f"2. {ARQUIVO_EPISODIOS}"
    )


print(
    """
O relatório contém:

1. Resumo executivo

2. Distribuição estatística de 1, 5 e 10 minutos

3. Top janelas de 1 minuto

4. Top janelas de 5 minutos

5. Top janelas de 10 minutos

6. Janelas classificadas como Flood

7. Episódios consolidados de Alarm Flood

8. Ranking de severidade dos episódios

9. Eventos P1/P2 envolvidos

10. Abrangência das cascatas

11. Análise mensal

12. Análise por hora

13. Dia da semana

14. Categoria

15. Torre

16. Tipo

17. Equipamentos mais recorrentes em floods

18. Matriz Torre x Categoria

19. Metodologia
"""
)


# ======================================================================
# 49. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


if Path(
    ARQUIVO_EPISODIOS
).exists():

    print(
        "\nPara baixar também os episódios em Parquet:"
    )

    print(
        f'files.download("{ARQUIVO_EPISODIOS}")'
    )

MÓDULO 8 - ALARM FLOOD, CASCATAS E CONCENTRAÇÃO TEMPORAL

Carregando bases...
Base de eventos........: 1,413,882
Ranking final..........: 5,826

BASE DE GERAÇÕES EFETIVAS
Gerações analisadas: 637,566
Calculando janela de 1 minuto(s)...
Calculando janela de 5 minuto(s)...
Calculando janela de 10 minuto(s)...

ALARM FLOODS IDENTIFICADOS
Janelas flood: 9,249

RESUMO EXECUTIVO


,Indicador,Resultado
0,Gerações efetivas analisadas,"637,566"
1,Janelas flood identificadas,"9,249"
2,Episódios de Alarm Flood,"1,843"
3,Episódios críticos,177
4,Episódios altos,519
5,Eventos únicos em períodos de flood,"469,534"
6,% das gerações em períodos de flood,73.64%
7,Maior episódio - quantidade de eventos,"7,326"
8,Maior episódio - equipamentos envolvidos,"1,716"
9,Maior episódio - torres envolvidas,8



DISTRIBUIÇÃO ESTATÍSTICA DAS JANELAS


,Janela_Min,Janelas_Com_Eventos,Media_Eventos,Mediana_Eventos,P90,P95,P99,P995,P999,Maximo
0,1,176659,3.61,2.00,6.00,8.00,20.00,35.00,188.00,581
1,5,53034,12.02,7.00,23.00,29.00,65.67,141.83,417.97,1436
2,10,28805,22.13,12.00,45.00,57.00,148.96,268.00,591.74,1513



TOP 30 EPISÓDIOS DE ALARM FLOOD


...

,Ano,Mes_Numero,Mes,Episodios_Flood,Eventos_em_Flood,Equipamentos_Medio,Pico_Eventos_10min,Floods_Criticos,Floods_Altos,Indice_Medio
0,2026,1,Janeiro,132,113559,384.33,1402,34,42,60.57
1,2026,2,Fevereiro,180,84563,250.27,1400,33,58,56.92
2,2026,3,Março,232,74350,179.38,428,40,62,53.00
3,2026,4,Abril,201,67981,193.33,520,36,62,58.04
4,2026,5,Maio,320,40558,101.73,603,9,92,46.39
5,2026,6,Junho,371,42565,96.51,1513,9,88,43.88
6,2026,7,Julho,407,48806,104.37,979,16,115,46.36



VALIDAÇÕES
Gerações efetivas.......................: 637,566
Janelas flood...........................: 9,249
Episódios...............................: 1,843
Eventos únicos em flood.................: 469,534
% eventos em flood......................: 73.64%
Episódios com fim < início..............: 0
Meses encontrados.......................: [1, 2, 3, 4, 5, 6, 7]

Arquivo criado:
Episodios_Alarm_Flood_Metasys_2026.parquet

Gerando relatório Excel do Módulo 8...

MÓDULO 8 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo8_Alarm_Flood_Metasys.xlsx
2. Episodios_Alarm_Flood_Metasys_2026.parquet

O relatório contém:

1. Resumo executivo

2. Distribuição estatística de 1, 5 e 10 minutos

3. Top janelas de 1 minuto

4. Top janelas de 5 minutos

5. Top janelas de 10 minutos

6. Janelas classificadas como Flood

7. Episódios consolidados de Alarm Flood

8. Ranking de severidade dos episódios

9. Eventos P1/P2 envolvidos

10. Abrangência das cascatas

11. Análise mensal

12. Análise p

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também os episódios em Parquet:
files.download("Episodios_Alarm_Flood_Metasys_2026.parquet")


In [18]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 9
# FIRST-OUT, PRECURSORES E SEQUENCIAMENTO DE CASCATAS
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Identificar os primeiros eventos de cada episódio de Alarm Flood
#
# 2. Avaliar equipamentos presentes nos primeiros:
#       - 1 minuto
#       - 2 minutos
#       - 5 minutos
#       - 10 minutos
#
# 3. Diferenciar:
#       - First-Out
#       - Precursor forte
#       - Participante recorrente
#       - Seguidor
#       - Mass Contributor
#
# 4. Medir a velocidade de expansão das cascatas:
#       - tempo até 10 equipamentos
#       - tempo até 25 equipamentos
#       - tempo até 50 equipamentos
#       - tempo até 100 equipamentos
#       - tempo até 250 equipamentos
#
# 5. Avaliar recorrência dos precursores
#
# 6. Relacionar precursor com:
#       - Categoria
#       - Tipo
#       - Torre
#       - Mantenedor
#       - prioridade
#       - ranking do Módulo 7
#
# 7. Criar relações temporais precursor -> evento posterior
#
# 8. Gerar ranking de possíveis precursores
#
# IMPORTANTE
# ----------------------------------------------------------------------
# "Precursor" significa associação temporal.
#
# O módulo NÃO afirma causalidade física automaticamente.
#
# Um equipamento aparecer antes de outros repetidamente é evidência
# para investigação, não prova de causa raiz.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    300
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    300
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_BASE = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)


ARQUIVO_RANKING = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


ARQUIVO_EPISODIOS = (
    "Episodios_Alarm_Flood_Metasys_2026.parquet"
)


ARQUIVO_RELATORIO = (
    "Relatorio_Modulo9_Precursores_FirstOut_Metasys.xlsx"
)


ARQUIVO_PRECURSORES = (
    "Ranking_Precursores_Metasys_2026.parquet"
)


ARQUIVO_CASCATAS = (
    "Cascatas_Sequenciamento_Metasys_2026.parquet"
)


print("=" * 120)

print(
    "MÓDULO 9 - FIRST-OUT, PRECURSORES "
    "E SEQUENCIAMENTO DE CASCATAS"
)

print("=" * 120)


# ======================================================================
# 3. PARÂMETROS
# ======================================================================

# Janelas utilizadas para estudar o início dos episódios
JANELAS_PRECURSOR_MIN = [
    1,
    2,
    5,
    10
]


# Marcos de expansão da cascata
MARCOS_EQUIPAMENTOS = [
    10,
    25,
    50,
    100,
    250
]


# Quantidade máxima de seguidores por precursor para análise
# de pares temporais.
#
# Essa limitação evita explosão combinatória.
MAX_SEGUIDORES_POR_EPISODIO = 100


# Janela máxima para considerar uma relação precursor -> seguidor
JANELA_RELACAO_MIN = 30


# Número mínimo de episódios para considerar um padrão
# recorrente na classificação final
MIN_EPISODIOS_RECORRENCIA = 5


# ======================================================================
# 4. VERIFICAÇÃO / UPLOAD DOS ARQUIVOS
# ======================================================================

arquivos_necessarios = [

    ARQUIVO_BASE,
    ARQUIVO_RANKING,
    ARQUIVO_EPISODIOS
]


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    print(
        "\nArquivos ausentes:"
    )

    for arquivo in faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos solicitados."
    )


    from google.colab import files

    uploaded = files.upload()


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            faltantes
        )
    )


# ======================================================================
# 5. CARREGAMENTO
# ======================================================================

print(
    "\nCarregando bases..."
)


df = pd.read_parquet(
    ARQUIVO_BASE
)


ranking = pd.read_parquet(
    ARQUIVO_RANKING
)


episodios = pd.read_parquet(
    ARQUIVO_EPISODIOS
)


print(
    f"Base de eventos........: {len(df):,}"
)

print(
    f"Ranking final..........: {len(ranking):,}"
)

print(
    f"Episódios de flood.....: {len(episodios):,}"
)


# ======================================================================
# 6. VALIDAÇÃO DAS COLUNAS
# ======================================================================

COLUNAS_BASE = [

    "DataHoraLocal",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",

    "priority",

    "Flag_Geracao_Efetiva",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",
    "Flag_Falha_Comunicacao"
]


faltantes_base = [

    coluna

    for coluna in COLUNAS_BASE

    if coluna not in df.columns
]


if faltantes_base:

    raise ValueError(

        "Colunas ausentes na base de eventos:\n\n"

        +

        "\n".join(
            faltantes_base
        )
    )


COLUNAS_EPISODIOS = [

    "ID_Episodio",

    "Inicio_Episodio",
    "Fim_Episodio",

    "Ranking_Flood",
    "Indice_Alarm_Flood",
    "Classe_Alarm_Flood"
]


faltantes_episodios = [

    coluna

    for coluna in COLUNAS_EPISODIOS

    if coluna not in episodios.columns
]


if faltantes_episodios:

    raise ValueError(

        "Colunas ausentes na base de episódios:\n\n"

        +

        "\n".join(
            faltantes_episodios
        )
    )


# ======================================================================
# 7. TRATAMENTO TEMPORAL
# ======================================================================

df[
    "DataHoraLocal"
] = pd.to_datetime(

    df[
        "DataHoraLocal"
    ],

    errors="coerce"
)


df[
    "DataHoraAnalise"
] = (

    df[
        "DataHoraLocal"
    ]
    .dt.tz_localize(
        None
    )
)


episodios[
    "Inicio_Episodio"
] = pd.to_datetime(

    episodios[
        "Inicio_Episodio"
    ],

    errors="coerce"
)


episodios[
    "Fim_Episodio"
] = pd.to_datetime(

    episodios[
        "Fim_Episodio"
    ],

    errors="coerce"
)


# Segurança caso algum parquet mantenha timezone
for coluna in [
    "Inicio_Episodio",
    "Fim_Episodio"
]:

    try:

        episodios[
            coluna
        ] = (

            episodios[
                coluna
            ]
            .dt.tz_localize(
                None
            )
        )

    except:

        pass


# ======================================================================
# 8. SOMENTE GERAÇÕES EFETIVAS
# ======================================================================

geracoes = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()

    .sort_values(
        "DataHoraAnalise"
    )

    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 120)

print(
    "BASE ANALÍTICA"
)

print("=" * 120)


print(
    f"Gerações efetivas.......: {len(geracoes):,}"
)

print(
    f"Episódios................: {len(episodios):,}"
)


# ======================================================================
# 9. INTEGRAÇÃO COM RANKING DO MÓDULO 7
# ======================================================================

colunas_ranking_desejadas = [

    "itemName",

    "Ranking_Final",

    "Indice_Prioridade_Final",

    "Prioridade_Final",

    "Perfil_Dominante",

    "Tipo_Tratamento"
]


colunas_ranking = [

    coluna

    for coluna in colunas_ranking_desejadas

    if coluna in ranking.columns
]


ranking_aux = (

    ranking[
        colunas_ranking
    ]

    .copy()
)


if (
    "Indice_Prioridade_Final"
    in ranking_aux.columns
):

    ranking_aux = (

        ranking_aux
        .sort_values(
            "Indice_Prioridade_Final",
            ascending=False
        )
    )


ranking_aux = (

    ranking_aux
    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


geracoes = (

    geracoes
    .merge(

        ranking_aux,

        on="itemName",

        how="left"
    )
)


# ======================================================================
# 10. FLAGS P1 / P2
# ======================================================================

geracoes[
    "Flag_P1"
] = (

    geracoes[
        "Prioridade_Final"
    ]
    ==
    "P1 - Intervenção Prioritária"
)


geracoes[
    "Flag_P2"
] = (

    geracoes[
        "Prioridade_Final"
    ]
    ==
    "P2 - Alta Prioridade"
)


geracoes[
    "Flag_P1_P2"
] = (

    geracoes[
        "Flag_P1"
    ]

    |

    geracoes[
        "Flag_P2"
    ]
)


# ======================================================================
# 11. CLASSE DA GERAÇÃO
# ======================================================================

geracoes[
    "Classe_Geracao"
] = np.select(

    [
        geracoes[
            "Flag_Entrada_Alarme"
        ],

        geracoes[
            "Flag_Entrada_PreAlarme"
        ],

        geracoes[
            "Flag_Falha_Comunicacao"
        ]
    ],

    [
        "Alarme",
        "Pré-Alarme",
        "Comunicação"
    ],

    default="Outro"
)


# ======================================================================
# 12. PREPARAÇÃO DAS ESTRUTURAS
# ======================================================================

eventos_episodios_lista = []

first_out_lista = []

expansao_lista = []

pares_lista = []


# ======================================================================
# 13. FUNÇÕES AUXILIARES
# ======================================================================

def moda_segura(
    serie
):

    serie = (

        serie
        .dropna()
    )


    if len(
        serie
    ) == 0:

        return pd.NA


    moda = (
        serie
        .mode()
    )


    if len(
        moda
    ) > 0:

        return moda.iloc[0]


    return serie.iloc[0]


# ======================================================================
# 14. PROCESSAMENTO DOS EPISÓDIOS
# ======================================================================

print("\n" + "=" * 120)

print(
    "PROCESSANDO EPISÓDIOS"
)

print("=" * 120)


TOTAL_EPISODIOS = len(
    episodios
)


for contador, episodio in enumerate(

    episodios.itertuples(
        index=False
    ),

    start=1
):

    if (
        contador == 1

        or

        contador % 100 == 0

        or

        contador == TOTAL_EPISODIOS
    ):

        print(
            f"Processando episódio "
            f"{contador:,}/{TOTAL_EPISODIOS:,}"
        )


    id_episodio = (
        episodio.ID_Episodio
    )


    inicio = (
        episodio.Inicio_Episodio
    )


    fim = (
        episodio.Fim_Episodio
    )


    # --------------------------------------------------------------
    # Eventos efetivos dentro do episódio
    # --------------------------------------------------------------

    temp = (

        geracoes.loc[
            (
                geracoes[
                    "DataHoraAnalise"
                ]
                >=
                inicio
            )

            &

            (
                geracoes[
                    "DataHoraAnalise"
                ]
                <
                fim
            )
        ]

        .copy()

        .sort_values(
            [
                "DataHoraAnalise",
                "itemName"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    if len(
        temp
    ) == 0:

        continue


    # --------------------------------------------------------------
    # Tempo desde o início do episódio
    # --------------------------------------------------------------

    temp[
        "Minutos_Desde_Inicio"
    ] = (

        (
            temp[
                "DataHoraAnalise"
            ]

            -

            inicio
        )

        .dt.total_seconds()

        /

        60
    )


    # --------------------------------------------------------------
    # Informações do episódio
    # --------------------------------------------------------------

    temp[
        "ID_Episodio"
    ] = id_episodio


    temp[
        "Ranking_Flood"
    ] = getattr(
        episodio,
        "Ranking_Flood",
        np.nan
    )


    temp[
        "Indice_Alarm_Flood"
    ] = getattr(
        episodio,
        "Indice_Alarm_Flood",
        np.nan
    )


    temp[
        "Classe_Alarm_Flood"
    ] = getattr(
        episodio,
        "Classe_Alarm_Flood",
        pd.NA
    )


    # --------------------------------------------------------------
    # Primeiro timestamp do episódio
    # --------------------------------------------------------------

    primeiro_timestamp = (

        temp[
            "DataHoraAnalise"
        ]
        .min()
    )


    temp[
        "Segundos_Desde_FirstOut"
    ] = (

        (
            temp[
                "DataHoraAnalise"
            ]

            -

            primeiro_timestamp
        )

        .dt.total_seconds()
    )


    temp[
        "Minutos_Desde_FirstOut"
    ] = (

        temp[
            "Segundos_Desde_FirstOut"
        ]

        /

        60
    )


    # --------------------------------------------------------------
    # Flags das janelas iniciais
    # --------------------------------------------------------------

    temp[
        "Flag_FirstOut"
    ] = (

        temp[
            "DataHoraAnalise"
        ]
        ==
        primeiro_timestamp
    )


    for janela in JANELAS_PRECURSOR_MIN:

        temp[
            f"Flag_Inicio_{janela}min"
        ] = (

            temp[
                "Minutos_Desde_FirstOut"
            ]

            <=
            janela
        )


    eventos_episodios_lista.append(
        temp
    )


    # ==============================================================
    # 14.1 FIRST-OUT
    # ==============================================================

    first = (

        temp.loc[
            temp[
                "Flag_FirstOut"
            ]
        ]

        .copy()
    )


    for linha in first.itertuples(
        index=False
    ):

        first_out_lista.append({

            "ID_Episodio":
                id_episodio,

            "Ranking_Flood":
                getattr(
                    episodio,
                    "Ranking_Flood",
                    np.nan
                ),

            "Indice_Alarm_Flood":
                getattr(
                    episodio,
                    "Indice_Alarm_Flood",
                    np.nan
                ),

            "Classe_Alarm_Flood":
                getattr(
                    episodio,
                    "Classe_Alarm_Flood",
                    pd.NA
                ),

            "Inicio_Episodio":
                inicio,

            "DataHora_FirstOut":
                primeiro_timestamp,

            "itemName":
                linha.itemName,

            "itemDescription":
                linha.itemDescription,

            "Categoria":
                linha.Categoria,

            "Mantenedor":
                linha.Mantenedor,

            "Tipo":
                linha.Tipo,

            "Torre":
                linha.Torre,

            "priority":
                linha.priority,

            "Classe_Geracao":
                linha.Classe_Geracao,

            "Ranking_Final":
                getattr(
                    linha,
                    "Ranking_Final",
                    np.nan
                ),

            "Indice_Prioridade_Final":
                getattr(
                    linha,
                    "Indice_Prioridade_Final",
                    np.nan
                ),

            "Prioridade_Final":
                getattr(
                    linha,
                    "Prioridade_Final",
                    pd.NA
                )
        })


    # ==============================================================
    # 14.2 EXPANSÃO DA CASCATA
    # ==============================================================

    primeira_ocorrencia_equip = (

        temp
        .groupby(
            "itemName",
            observed=True
        )[
            "DataHoraAnalise"
        ]
        .min()

        .sort_values()
    )


    expansao = {

        "ID_Episodio":
            id_episodio,

        "Inicio_Episodio":
            inicio,

        "Fim_Episodio":
            fim,

        "Equipamentos_Total":
            len(
                primeira_ocorrencia_equip
            ),

        "Eventos_Total":
            len(
                temp
            ),

        "Indice_Alarm_Flood":
            getattr(
                episodio,
                "Indice_Alarm_Flood",
                np.nan
            ),

        "Classe_Alarm_Flood":
            getattr(
                episodio,
                "Classe_Alarm_Flood",
                pd.NA
            )
    }


    for marco in MARCOS_EQUIPAMENTOS:

        nome_coluna = (
            f"Tempo_ate_{marco}_Equip_Min"
        )


        if len(
            primeira_ocorrencia_equip
        ) >= marco:

            timestamp_marco = (

                primeira_ocorrencia_equip
                .iloc[
                    marco - 1
                ]
            )


            expansao[
                nome_coluna
            ] = (

                (
                    timestamp_marco

                    -

                    primeiro_timestamp
                )

                .total_seconds()

                /

                60
            )


        else:

            expansao[
                nome_coluna
            ] = np.nan


    expansao_lista.append(
        expansao
    )


    # ==============================================================
    # 14.3 RELAÇÕES TEMPORAIS PRECURSOR -> SEGUIDOR
    #
    # Para evitar explosão combinatória:
    #
    # - usa-se o primeiro timestamp de cada equipamento
    # - precursor = equipamento nos primeiros 5 min
    # - seguidor = equipamento que surge depois
    # - máximo 100 seguidores por episódio
    # ==============================================================

    primeira_equip_df = (

        temp
        .sort_values(
            "DataHoraAnalise"
        )

        .drop_duplicates(
            subset="itemName",
            keep="first"
        )

        [
            [
                "itemName",
                "DataHoraAnalise",
                "Categoria",
                "Tipo",
                "Torre"
            ]
        ]

        .copy()
    )


    primeira_equip_df[
        "Minutos_FirstOut"
    ] = (

        (
            primeira_equip_df[
                "DataHoraAnalise"
            ]

            -

            primeiro_timestamp
        )

        .dt.total_seconds()

        /

        60
    )


    precursores_ep = (

        primeira_equip_df.loc[
            primeira_equip_df[
                "Minutos_FirstOut"
            ]
            <=
            5
        ]

        .copy()
    )


    seguidores_ep = (

        primeira_equip_df.loc[
            (
                primeira_equip_df[
                    "Minutos_FirstOut"
                ]
                > 0
            )

            &

            (
                primeira_equip_df[
                    "Minutos_FirstOut"
                ]
                <=
                JANELA_RELACAO_MIN
            )
        ]

        .sort_values(
            "DataHoraAnalise"
        )

        .head(
            MAX_SEGUIDORES_POR_EPISODIO
        )

        .copy()
    )


    # --------------------------------------------------------------
    # Relações entre os primeiros equipamentos e seguidores
    # --------------------------------------------------------------

    for precursor in precursores_ep.itertuples(
        index=False
    ):

        for seguidor in seguidores_ep.itertuples(
            index=False
        ):

            if (
                precursor.itemName
                ==
                seguidor.itemName
            ):

                continue


            if (
                seguidor.DataHoraAnalise
                <=
                precursor.DataHoraAnalise
            ):

                continue


            delta_min = (

                (
                    seguidor.DataHoraAnalise

                    -

                    precursor.DataHoraAnalise
                )

                .total_seconds()

                /

                60
            )


            if (
                delta_min
                >
                JANELA_RELACAO_MIN
            ):

                continue


            pares_lista.append({

                "ID_Episodio":
                    id_episodio,

                "Precursor":
                    precursor.itemName,

                "Categoria_Precursor":
                    precursor.Categoria,

                "Tipo_Precursor":
                    precursor.Tipo,

                "Torre_Precursor":
                    precursor.Torre,

                "Seguidor":
                    seguidor.itemName,

                "Categoria_Seguidor":
                    seguidor.Categoria,

                "Tipo_Seguidor":
                    seguidor.Tipo,

                "Torre_Seguidor":
                    seguidor.Torre,

                "Delta_Min":
                    delta_min,

                "Indice_Alarm_Flood":
                    getattr(
                        episodio,
                        "Indice_Alarm_Flood",
                        np.nan
                    )
            })


# ======================================================================
# 15. CONSOLIDAÇÃO
# ======================================================================

if len(
    eventos_episodios_lista
) > 0:

    eventos_episodios = pd.concat(

        eventos_episodios_lista,

        ignore_index=True
    )


else:

    eventos_episodios = pd.DataFrame()


first_out = pd.DataFrame(
    first_out_lista
)


expansao_cascatas = pd.DataFrame(
    expansao_lista
)


pares_temporais = pd.DataFrame(
    pares_lista
)


print("\n" + "=" * 120)

print(
    "CONSOLIDAÇÃO CONCLUÍDA"
)

print("=" * 120)


print(
    f"Eventos associados a episódios....: "
    f"{len(eventos_episodios):,}"
)

print(
    f"Registros First-Out...............: "
    f"{len(first_out):,}"
)

print(
    f"Episódios com expansão analisada..: "
    f"{len(expansao_cascatas):,}"
)

print(
    f"Relações temporais................: "
    f"{len(pares_temporais):,}"
)


# ======================================================================
# 16. RANKING FIRST-OUT
# ======================================================================

if len(
    first_out
) > 0:

    ranking_first_out = (

        first_out
        .groupby(

            [
                "itemName",
                "itemDescription",
                "Categoria",
                "Mantenedor",
                "Tipo",
                "Torre"
            ],

            observed=True,

            dropna=False
        )
        .agg(

            FirstOut_Total=(
                "ID_Episodio",
                "size"
            ),

            Episodios_FirstOut=(
                "ID_Episodio",
                "nunique"
            ),

            Floods_Criticos=(
                "Classe_Alarm_Flood",

                lambda x:
                (
                    x
                    ==
                    "Crítico"
                ).sum()
            ),

            Floods_Altos=(
                "Classe_Alarm_Flood",

                lambda x:
                (
                    x
                    ==
                    "Alto"
                ).sum()
            ),

            Indice_Flood_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            ),

            Indice_Flood_Max=(
                "Indice_Alarm_Flood",
                "max"
            ),

            Ranking_Final_Equipamento=(
                "Ranking_Final",
                "min"
            ),

            Indice_Prioridade_Final=(
                "Indice_Prioridade_Final",
                "max"
            )
        )

        .reset_index()

        .sort_values(

            [
                "Episodios_FirstOut",
                "Indice_Flood_Medio"
            ],

            ascending=[
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


else:

    ranking_first_out = pd.DataFrame()


# ======================================================================
# 17. PARTICIPAÇÃO NOS PRIMEIROS 1/2/5/10 MINUTOS
# ======================================================================

ranking_inicio_lista = []


if len(
    eventos_episodios
) > 0:

    # Primeira aparição do equipamento em cada episódio
    primeira_aparicao = (

        eventos_episodios
        .sort_values(
            "DataHoraAnalise"
        )

        .drop_duplicates(

            subset=[
                "ID_Episodio",
                "itemName"
            ],

            keep="first"
        )

        .copy()
    )


    agrupamento_base = [

        "itemName",
        "itemDescription",
        "Categoria",
        "Mantenedor",
        "Tipo",
        "Torre"
    ]


    for janela in JANELAS_PRECURSOR_MIN:

        temp = (

            primeira_aparicao.loc[
                primeira_aparicao[
                    "Minutos_Desde_FirstOut"
                ]
                <=
                janela
            ]

            .copy()
        )


        resumo = (

            temp
            .groupby(

                agrupamento_base,

                observed=True,

                dropna=False
            )
            .agg(

                Episodios=(
                    "ID_Episodio",
                    "nunique"
                ),

                Minutos_Medio=(
                    "Minutos_Desde_FirstOut",
                    "mean"
                ),

                Indice_Flood_Medio=(
                    "Indice_Alarm_Flood",
                    "mean"
                ),

                Ranking_Final_Equipamento=(
                    "Ranking_Final",
                    "min"
                ),

                Indice_Prioridade_Final=(
                    "Indice_Prioridade_Final",
                    "max"
                )
            )

            .reset_index()
        )


        resumo[
            "Janela_Min"
        ] = janela


        ranking_inicio_lista.append(
            resumo
        )


if len(
    ranking_inicio_lista
) > 0:

    ranking_inicio = pd.concat(

        ranking_inicio_lista,

        ignore_index=True
    )


else:

    ranking_inicio = pd.DataFrame()


# ======================================================================
# 18. PIVOT DOS PRECURSORES
# ======================================================================

if len(
    ranking_inicio
) > 0:

    base_pivot = (

        ranking_inicio[
            [
                "itemName",
                "Janela_Min",
                "Episodios"
            ]
        ]

        .copy()
    )


    # Evita problemas com categorical
    base_pivot[
        "itemName"
    ] = (

        base_pivot[
            "itemName"
        ]
        .astype("string")
    )


    pivot_precursores = (

        base_pivot
        .pivot_table(

            index="itemName",

            columns="Janela_Min",

            values="Episodios",

            aggfunc="max",

            fill_value=0,

            observed=True
        )

        .reset_index()
    )


    # --------------------------------------------------------------
    # Garante todas as colunas esperadas
    # --------------------------------------------------------------

    for janela in JANELAS_PRECURSOR_MIN:

        if (
            janela
            not in pivot_precursores.columns
        ):

            pivot_precursores[
                janela
            ] = 0


    pivot_precursores = (

        pivot_precursores
        .rename(

            columns={

                1:
                    "Episodios_1min",

                2:
                    "Episodios_2min",

                5:
                    "Episodios_5min",

                10:
                    "Episodios_10min"
            }
        )
    )


else:

    pivot_precursores = pd.DataFrame()


# ======================================================================
# 19. PARTICIPAÇÃO TOTAL EM FLOODS POR EQUIPAMENTO
# ======================================================================

if len(
    eventos_episodios
) > 0:

    participacao_total = (

        eventos_episodios
        .groupby(
            "itemName",
            observed=True
        )
        .agg(

            Episodios_Total=(
                "ID_Episodio",
                "nunique"
            ),

            Eventos_Total_Flood=(
                "itemName",
                "size"
            )
        )

        .reset_index()
    )


else:

    participacao_total = pd.DataFrame()


# ======================================================================
# 20. BASE DESCRITIVA DOS EQUIPAMENTOS
# ======================================================================

if len(
    eventos_episodios
) > 0:

    cadastro_equipamentos = (

        eventos_episodios
        .sort_values(
            "DataHoraAnalise"
        )

        .drop_duplicates(
            subset="itemName",
            keep="first"
        )

        [
            [
                "itemName",
                "itemDescription",
                "Categoria",
                "Mantenedor",
                "Tipo",
                "Torre",
                "Ranking_Final",
                "Indice_Prioridade_Final",
                "Prioridade_Final"
            ]
        ]

        .copy()
    )


else:

    cadastro_equipamentos = pd.DataFrame()


# ======================================================================
# 21. RANKING CONSOLIDADO DE PRECURSORES
# ======================================================================

if (
    len(
        cadastro_equipamentos
    ) > 0
):

    ranking_precursores = (

        cadastro_equipamentos
        .merge(

            participacao_total,

            on="itemName",

            how="left"
        )
    )


    if len(
        pivot_precursores
    ) > 0:

        ranking_precursores = (

            ranking_precursores
            .merge(

                pivot_precursores,

                on="itemName",

                how="left"
            )
        )


    if len(
        ranking_first_out
    ) > 0:

        first_aux = (

            ranking_first_out[
                [
                    "itemName",
                    "Episodios_FirstOut",
                    "Floods_Criticos",
                    "Floods_Altos",
                    "Indice_Flood_Medio"
                ]
            ]

            .copy()
        )


        ranking_precursores = (

            ranking_precursores
            .merge(

                first_aux,

                on="itemName",

                how="left"
            )
        )


else:

    ranking_precursores = pd.DataFrame()


# ======================================================================
# 22. TRATAMENTO DOS NULOS
# ======================================================================

colunas_contagem = [

    "Episodios_Total",

    "Eventos_Total_Flood",

    "Episodios_1min",

    "Episodios_2min",

    "Episodios_5min",

    "Episodios_10min",

    "Episodios_FirstOut",

    "Floods_Criticos",

    "Floods_Altos"
]


if len(
    ranking_precursores
) > 0:

    for coluna in colunas_contagem:

        if (
            coluna
            not in ranking_precursores.columns
        ):

            ranking_precursores[
                coluna
            ] = 0


        ranking_precursores[
            coluna
        ] = (

            pd.to_numeric(

                ranking_precursores[
                    coluna
                ],

                errors="coerce"
            )

            .fillna(0)
        )


# ======================================================================
# 23. TAXAS DE PRECURSOR
# ======================================================================

if len(
    ranking_precursores
) > 0:

    ranking_precursores[
        "Taxa_FirstOut"
    ] = np.where(

        ranking_precursores[
            "Episodios_Total"
        ] > 0,

        ranking_precursores[
            "Episodios_FirstOut"
        ]

        /

        ranking_precursores[
            "Episodios_Total"
        ]

        * 100,

        0
    )


    ranking_precursores[
        "Taxa_1min"
    ] = np.where(

        ranking_precursores[
            "Episodios_Total"
        ] > 0,

        ranking_precursores[
            "Episodios_1min"
        ]

        /

        ranking_precursores[
            "Episodios_Total"
        ]

        * 100,

        0
    )


    ranking_precursores[
        "Taxa_5min"
    ] = np.where(

        ranking_precursores[
            "Episodios_Total"
        ] > 0,

        ranking_precursores[
            "Episodios_5min"
        ]

        /

        ranking_precursores[
            "Episodios_Total"
        ]

        * 100,

        0
    )


    ranking_precursores[
        "Taxa_10min"
    ] = np.where(

        ranking_precursores[
            "Episodios_Total"
        ] > 0,

        ranking_precursores[
            "Episodios_10min"
        ]

        /

        ranking_precursores[
            "Episodios_Total"
        ]

        * 100,

        0
    )


# ======================================================================
# 24. SCORE DE PRECURSOR
#
# Combina:
#
# 30% frequência nos primeiros 1 min
# 25% frequência nos primeiros 5 min
# 20% First-Out
# 15% quantidade absoluta de episódios iniciais
# 10% criticidade do equipamento no Módulo 7
# ======================================================================

if len(
    ranking_precursores
) > 0:

    ranking_precursores[
        "Score_1min"
    ] = (

        ranking_precursores[
            "Taxa_1min"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    ranking_precursores[
        "Score_5min"
    ] = (

        ranking_precursores[
            "Taxa_5min"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    ranking_precursores[
        "Score_FirstOut"
    ] = (

        ranking_precursores[
            "Taxa_FirstOut"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    ranking_precursores[
        "Score_Recorrencia"
    ] = (

        ranking_precursores[
            "Episodios_5min"
        ]
        .rank(
            pct=True,
            method="average"
        )

        * 100
    )


    if (
        "Indice_Prioridade_Final"
        in ranking_precursores.columns
    ):

        ranking_precursores[
            "Score_Prioridade"
        ] = (

            pd.to_numeric(

                ranking_precursores[
                    "Indice_Prioridade_Final"
                ],

                errors="coerce"
            )

            .rank(
                pct=True,
                method="average"
            )

            * 100
        )


    else:

        ranking_precursores[
            "Score_Prioridade"
        ] = 0


    ranking_precursores[
        "Indice_Precursor"
    ] = (

        ranking_precursores[
            "Score_1min"
        ]
        * 0.30

        +

        ranking_precursores[
            "Score_5min"
        ]
        * 0.25

        +

        ranking_precursores[
            "Score_FirstOut"
        ]
        * 0.20

        +

        ranking_precursores[
            "Score_Recorrencia"
        ]
        * 0.15

        +

        ranking_precursores[
            "Score_Prioridade"
        ]
        * 0.10
    )


# ======================================================================
# 25. CLASSIFICAÇÃO COMPORTAMENTAL
# ======================================================================

def classificar_comportamento(
    row
):

    episodios_total = (
        row[
            "Episodios_Total"
        ]
    )


    eventos_total = (
        row[
            "Eventos_Total_Flood"
        ]
    )


    episodios_5 = (
        row[
            "Episodios_5min"
        ]
    )


    taxa_first = (
        row[
            "Taxa_FirstOut"
        ]
    )


    taxa_1 = (
        row[
            "Taxa_1min"
        ]
    )


    taxa_5 = (
        row[
            "Taxa_5min"
        ]
    )


    # --------------------------------------------------------------
    # Precursor forte
    # --------------------------------------------------------------

    if (
        episodios_5
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        taxa_1
        >=
        40

        and

        taxa_5
        >=
        60
    ):

        return (
            "Precursor Forte"
        )


    # --------------------------------------------------------------
    # First-Out recorrente
    # --------------------------------------------------------------

    if (
        episodios_total
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        taxa_first
        >=
        25
    ):

        return (
            "First-Out Recorrente"
        )


    # --------------------------------------------------------------
    # Mass contributor
    # --------------------------------------------------------------

    eventos_por_episodio = (

        eventos_total
        /
        episodios_total

        if episodios_total > 0

        else 0
    )


    if (
        episodios_total
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        eventos_por_episodio
        >=
        5

        and

        taxa_5
        <
        40
    ):

        return (
            "Mass Contributor"
        )


    # --------------------------------------------------------------
    # Participante recorrente
    # --------------------------------------------------------------

    if (
        episodios_total
        >=
        20
    ):

        return (
            "Participante Recorrente"
        )


    # --------------------------------------------------------------
    # Seguidor
    # --------------------------------------------------------------

    if (
        episodios_total
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        taxa_10
        if False
        else False
    ):

        return (
            "Seguidor"
        )


    return (
        "Participação Ocasional"
    )


# ======================================================================
# 26. CLASSIFICAÇÃO COMPORTAMENTAL - VERSÃO FINAL
#
# A função anterior permanece apenas como referência lógica.
# Utilizamos abaixo a versão efetivamente aplicada.
# ======================================================================

def classificar_comportamento_final(
    row
):

    episodios_total = (
        row[
            "Episodios_Total"
        ]
    )


    eventos_total = (
        row[
            "Eventos_Total_Flood"
        ]
    )


    episodios_5 = (
        row[
            "Episodios_5min"
        ]
    )


    taxa_first = (
        row[
            "Taxa_FirstOut"
        ]
    )


    taxa_1 = (
        row[
            "Taxa_1min"
        ]
    )


    taxa_5 = (
        row[
            "Taxa_5min"
        ]
    )


    taxa_10 = (
        row[
            "Taxa_10min"
        ]
    )


    eventos_por_episodio = (

        eventos_total
        /
        episodios_total

        if episodios_total > 0

        else 0
    )


    if (
        episodios_5
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        taxa_1
        >=
        40

        and

        taxa_5
        >=
        60
    ):

        return (
            "Precursor Forte"
        )


    if (
        episodios_total
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        taxa_first
        >=
        25
    ):

        return (
            "First-Out Recorrente"
        )


    if (
        episodios_total
        >=
        MIN_EPISODIOS_RECORRENCIA

        and

        eventos_por_episodio
        >=
        5

        and

        taxa_5
        <
        40
    ):

        return (
            "Mass Contributor"
        )


    if (
        episodios_total
        >=
        20

        and

        taxa_10
        <
        20
    ):

        return (
            "Seguidor Recorrente"
        )


    if (
        episodios_total
        >=
        20
    ):

        return (
            "Participante Recorrente"
        )


    return (
        "Participação Ocasional"
    )


if len(
    ranking_precursores
) > 0:

    ranking_precursores[
        "Classificacao_Comportamental"
    ] = (

        ranking_precursores
        .apply(

            classificar_comportamento_final,

            axis=1
        )
    )


# ======================================================================
# 27. RANKING FINAL DOS PRECURSORES
# ======================================================================

if len(
    ranking_precursores
) > 0:

    ranking_precursores = (

        ranking_precursores
        .sort_values(

            [
                "Indice_Precursor",
                "Episodios_5min",
                "Episodios_FirstOut"
            ],

            ascending=[
                False,
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


    ranking_precursores[
        "Ranking_Precursor"
    ] = (

        np.arange(
            1,
            len(
                ranking_precursores
            ) + 1
        )
    )


# ======================================================================
# 28. ANÁLISE DOS PARES PRECURSOR -> SEGUIDOR
# ======================================================================

if len(
    pares_temporais
) > 0:

    ranking_pares = (

        pares_temporais
        .groupby(

            [
                "Precursor",
                "Categoria_Precursor",
                "Tipo_Precursor",
                "Torre_Precursor",

                "Seguidor",
                "Categoria_Seguidor",
                "Tipo_Seguidor",
                "Torre_Seguidor"
            ],

            observed=True,

            dropna=False
        )
        .agg(

            Episodios_Com_Relacao=(
                "ID_Episodio",
                "nunique"
            ),

            Ocorrencias=(
                "ID_Episodio",
                "size"
            ),

            Delta_Medio_Min=(
                "Delta_Min",
                "mean"
            ),

            Delta_Mediano_Min=(
                "Delta_Min",
                "median"
            ),

            Delta_Minimo_Min=(
                "Delta_Min",
                "min"
            ),

            Indice_Flood_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            )
        )

        .reset_index()

        .sort_values(

            [
                "Episodios_Com_Relacao",
                "Ocorrencias"
            ],

            ascending=[
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


else:

    ranking_pares = pd.DataFrame()


# ======================================================================
# 29. PRECURSORES POR CATEGORIA
# ======================================================================

if len(
    ranking_precursores
) > 0:

    resumo_categoria_precursor = (

        ranking_precursores
        .groupby(
            "Categoria",
            observed=True,
            dropna=False
        )
        .agg(

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Precursores_Fortes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "Precursor Forte"
                ).sum()
            ),

            FirstOut_Recorrentes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "First-Out Recorrente"
                ).sum()
            ),

            Indice_Precursor_Medio=(
                "Indice_Precursor",
                "mean"
            ),

            Indice_Precursor_Max=(
                "Indice_Precursor",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            "Indice_Precursor_Max",
            ascending=False
        )
    )


else:

    resumo_categoria_precursor = pd.DataFrame()


# ======================================================================
# 30. PRECURSORES POR TORRE
# ======================================================================

if len(
    ranking_precursores
) > 0:

    resumo_torre_precursor = (

        ranking_precursores
        .groupby(
            "Torre",
            observed=True,
            dropna=False
        )
        .agg(

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Precursores_Fortes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "Precursor Forte"
                ).sum()
            ),

            FirstOut_Recorrentes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "First-Out Recorrente"
                ).sum()
            ),

            Indice_Precursor_Medio=(
                "Indice_Precursor",
                "mean"
            ),

            Indice_Precursor_Max=(
                "Indice_Precursor",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            "Indice_Precursor_Max",
            ascending=False
        )
    )


else:

    resumo_torre_precursor = pd.DataFrame()


# ======================================================================
# 31. PRECURSORES POR TIPO
# ======================================================================

if len(
    ranking_precursores
) > 0:

    resumo_tipo_precursor = (

        ranking_precursores
        .groupby(
            "Tipo",
            observed=True,
            dropna=False
        )
        .agg(

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Precursores_Fortes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "Precursor Forte"
                ).sum()
            ),

            FirstOut_Recorrentes=(
                "Classificacao_Comportamental",

                lambda x:
                (
                    x
                    ==
                    "First-Out Recorrente"
                ).sum()
            ),

            Indice_Precursor_Medio=(
                "Indice_Precursor",
                "mean"
            ),

            Indice_Precursor_Max=(
                "Indice_Precursor",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            "Indice_Precursor_Max",
            ascending=False
        )
    )


else:

    resumo_tipo_precursor = pd.DataFrame()


# ======================================================================
# 32. DISTRIBUIÇÃO DA VELOCIDADE DE EXPANSÃO
# ======================================================================

resumo_expansao_lista = []


if len(
    expansao_cascatas
) > 0:

    for marco in MARCOS_EQUIPAMENTOS:

        coluna = (
            f"Tempo_ate_{marco}_Equip_Min"
        )


        serie = (

            pd.to_numeric(

                expansao_cascatas[
                    coluna
                ],

                errors="coerce"
            )

            .dropna()
        )


        if len(
            serie
        ) == 0:

            continue


        resumo_expansao_lista.append({

            "Marco_Equipamentos":
                marco,

            "Episodios_que_Atingiram":
                len(
                    serie
                ),

            "Media_Min":
                serie.mean(),

            "Mediana_Min":
                serie.median(),

            "P25_Min":
                serie.quantile(
                    0.25
                ),

            "P75_Min":
                serie.quantile(
                    0.75
                ),

            "P90_Min":
                serie.quantile(
                    0.90
                ),

            "P95_Min":
                serie.quantile(
                    0.95
                ),

            "Minimo_Min":
                serie.min(),

            "Maximo_Min":
                serie.max()
        })


resumo_expansao = pd.DataFrame(
    resumo_expansao_lista
)


# ======================================================================
# 33. CLASSIFICAÇÃO DA VELOCIDADE DA CASCATA
# ======================================================================

if len(
    expansao_cascatas
) > 0:

    def classificar_velocidade(
        row
    ):

        t50 = (
            row.get(
                "Tempo_ate_50_Equip_Min",
                np.nan
            )
        )


        t100 = (
            row.get(
                "Tempo_ate_100_Equip_Min",
                np.nan
            )
        )


        if (
            pd.notna(
                t100
            )

            and

            t100 <= 5
        ):

            return (
                "Explosiva"
            )


        if (
            pd.notna(
                t50
            )

            and

            t50 <= 5
        ):

            return (
                "Muito Rápida"
            )


        if (
            pd.notna(
                t50
            )

            and

            t50 <= 15
        ):

            return (
                "Rápida"
            )


        if (
            pd.notna(
                t50
            )

            and

            t50 <= 30
        ):

            return (
                "Moderada"
            )


        if pd.notna(
            t50
        ):

            return (
                "Progressiva"
            )


        return (
            "Baixa Expansão"
        )


    expansao_cascatas[
        "Classe_Velocidade"
    ] = (

        expansao_cascatas
        .apply(

            classificar_velocidade,

            axis=1
        )
    )


# ======================================================================
# 34. RESUMO DA VELOCIDADE
# ======================================================================

if len(
    expansao_cascatas
) > 0:

    resumo_velocidade = (

        expansao_cascatas
        .groupby(
            "Classe_Velocidade",
            observed=True
        )
        .agg(

            Episodios=(
                "ID_Episodio",
                "size"
            ),

            Equipamentos_Medio=(
                "Equipamentos_Total",
                "mean"
            ),

            Eventos_Medio=(
                "Eventos_Total",
                "mean"
            ),

            Indice_Flood_Medio=(
                "Indice_Alarm_Flood",
                "mean"
            )
        )

        .reset_index()

        .sort_values(
            "Episodios",
            ascending=False
        )
    )


else:

    resumo_velocidade = pd.DataFrame()


# ======================================================================
# 35. DISTRIBUIÇÃO DAS CLASSIFICAÇÕES COMPORTAMENTAIS
# ======================================================================

if len(
    ranking_precursores
) > 0:

    resumo_comportamento = (

        ranking_precursores
        .groupby(
            "Classificacao_Comportamental",
            observed=True
        )
        .agg(

            Equipamentos=(
                "itemName",
                "nunique"
            ),

            Episodios_Total=(
                "Episodios_Total",
                "sum"
            ),

            Eventos_Total=(
                "Eventos_Total_Flood",
                "sum"
            ),

            Indice_Precursor_Medio=(
                "Indice_Precursor",
                "mean"
            ),

            Indice_Precursor_Max=(
                "Indice_Precursor",
                "max"
            )
        )

        .reset_index()

        .sort_values(
            "Indice_Precursor_Max",
            ascending=False
        )
    )


else:

    resumo_comportamento = pd.DataFrame()


# ======================================================================
# 36. TOP 100 PRECURSORES
# ======================================================================

if len(
    ranking_precursores
) > 0:

    top100_precursores = (

        ranking_precursores
        .head(100)
        .copy()
    )


else:

    top100_precursores = pd.DataFrame()


# ======================================================================
# 37. TOP FIRST-OUT
# ======================================================================

if len(
    ranking_first_out
) > 0:

    top100_first_out = (

        ranking_first_out
        .head(100)
        .copy()
    )


else:

    top100_first_out = pd.DataFrame()


# ======================================================================
# 38. TOP RELAÇÕES TEMPORAIS
# ======================================================================

if len(
    ranking_pares
) > 0:

    top500_relacoes = (

        ranking_pares
        .head(500)
        .copy()
    )


else:

    top500_relacoes = pd.DataFrame()


# ======================================================================
# 39. RESUMO EXECUTIVO
# ======================================================================

TOTAL_EQUIP_ANALISADOS = (

    ranking_precursores[
        "itemName"
    ].nunique()

    if len(
        ranking_precursores
    ) > 0

    else 0
)


PRECURSORES_FORTES = (

    ranking_precursores[
        "Classificacao_Comportamental"
    ]
    .eq(
        "Precursor Forte"
    )
    .sum()

    if len(
        ranking_precursores
    ) > 0

    else 0
)


FIRST_OUT_RECORRENTES = (

    ranking_precursores[
        "Classificacao_Comportamental"
    ]
    .eq(
        "First-Out Recorrente"
    )
    .sum()

    if len(
        ranking_precursores
    ) > 0

    else 0
)


MASS_CONTRIBUTORS = (

    ranking_precursores[
        "Classificacao_Comportamental"
    ]
    .eq(
        "Mass Contributor"
    )
    .sum()

    if len(
        ranking_precursores
    ) > 0

    else 0
)


SEGUIDORES_RECORRENTES = (

    ranking_precursores[
        "Classificacao_Comportamental"
    ]
    .eq(
        "Seguidor Recorrente"
    )
    .sum()

    if len(
        ranking_precursores
    ) > 0

    else 0
)


if len(
    ranking_precursores
) > 0:

    melhor_precursor = (
        ranking_precursores.iloc[0]
    )


else:

    melhor_precursor = None


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Episódios analisados",

            "Equipamentos analisados",

            "Precursores fortes",

            "First-Out recorrentes",

            "Mass Contributors",

            "Seguidores recorrentes",

            "Relações temporais únicas",

            "Principal precursor candidato",

            "Índice do principal precursor",

            "Episódios do principal precursor em 1 min",

            "Episódios do principal precursor em 5 min"
        ],

        "Resultado": [

            f"{len(expansao_cascatas):,}",

            f"{TOTAL_EQUIP_ANALISADOS:,}",

            f"{PRECURSORES_FORTES:,}",

            f"{FIRST_OUT_RECORRENTES:,}",

            f"{MASS_CONTRIBUTORS:,}",

            f"{SEGUIDORES_RECORRENTES:,}",

            f"{len(ranking_pares):,}",

            (
                str(
                    melhor_precursor[
                        "itemName"
                    ]
                )

                if melhor_precursor is not None

                else ""
            ),

            (
                f"{melhor_precursor['Indice_Precursor']:.2f}"

                if melhor_precursor is not None

                else "0.00"
            ),

            (
                f"{int(melhor_precursor['Episodios_1min']):,}"

                if melhor_precursor is not None

                else "0"
            ),

            (
                f"{int(melhor_precursor['Episodios_5min']):,}"

                if melhor_precursor is not None

                else "0"
            )
        ]
    }
)


# ======================================================================
# 40. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Parametro": [

            "First-Out",

            "Janela inicial 1 min",

            "Janela inicial 2 min",

            "Janela inicial 5 min",

            "Janela inicial 10 min",

            "Relação temporal",

            "Precursor Forte",

            "First-Out Recorrente",

            "Mass Contributor",

            "Seguidor Recorrente",

            "Índice Precursor",

            "Interpretação causal"
        ],

        "Definicao": [

            (
                "Equipamento com geração efetiva no primeiro "
                "timestamp observado do episódio."
            ),

            (
                "Primeira aparição do equipamento até 1 minuto "
                "após o First-Out."
            ),

            (
                "Primeira aparição do equipamento até 2 minutos "
                "após o First-Out."
            ),

            (
                "Primeira aparição do equipamento até 5 minutos "
                "após o First-Out."
            ),

            (
                "Primeira aparição do equipamento até 10 minutos "
                "após o First-Out."
            ),

            (
                f"Precursor seguido por outro equipamento em até "
                f"{JANELA_RELACAO_MIN} minutos."
            ),

            (
                ">=5 episódios, >=40% das participações dentro "
                "de 1 min e >=60% dentro de 5 min."
            ),

            (
                ">=5 episódios e First-Out em >=25% das "
                "participações."
            ),

            (
                ">=5 episódios, >=5 eventos por episódio e "
                "taxa inicial de 5 min <40%."
            ),

            (
                ">=20 episódios e menos de 20% das participações "
                "nos primeiros 10 minutos."
            ),

            (
                "30% taxa 1min + 25% taxa 5min + "
                "20% First-Out + 15% recorrência + "
                "10% prioridade do Módulo 7."
            ),

            (
                "Associação temporal não deve ser interpretada "
                "automaticamente como causalidade física."
            )
        ]
    }
)


# ======================================================================
# 41. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 120)

print(
    "VALIDAÇÕES"
)

print("=" * 120)


print(
    f"Episódios esperados.....................: "
    f"{len(episodios):,}"
)


print(
    f"Episódios processados...................: "
    f"{len(expansao_cascatas):,}"
)


print(
    f"Equipamentos analisados.................: "
    f"{TOTAL_EQUIP_ANALISADOS:,}"
)


print(
    f"Precursores fortes......................: "
    f"{PRECURSORES_FORTES:,}"
)


print(
    f"First-Out recorrentes...................: "
    f"{FIRST_OUT_RECORRENTES:,}"
)


print(
    f"Mass Contributors.......................: "
    f"{MASS_CONTRIBUTORS:,}"
)


print(
    f"Seguidores recorrentes..................: "
    f"{SEGUIDORES_RECORRENTES:,}"
)


# ======================================================================
# 42. EXIBIÇÃO DOS RESULTADOS
# ======================================================================

print("\n" + "=" * 120)

print(
    "RESUMO EXECUTIVO"
)

print("=" * 120)


display(
    resumo_executivo
)


print("\n" + "=" * 120)

print(
    "TOP 30 PRECURSORES"
)

print("=" * 120)


if len(
    ranking_precursores
) > 0:

    display(
        ranking_precursores.head(30)
    )


print("\n" + "=" * 120)

print(
    "TOP 30 FIRST-OUT"
)

print("=" * 120)


if len(
    ranking_first_out
) > 0:

    display(
        ranking_first_out.head(30)
    )


print("\n" + "=" * 120)

print(
    "VELOCIDADE DE EXPANSÃO"
)

print("=" * 120)


if len(
    resumo_expansao
) > 0:

    display(
        resumo_expansao
    )


print("\n" + "=" * 120)

print(
    "CLASSIFICAÇÃO COMPORTAMENTAL"
)

print("=" * 120)


if len(
    resumo_comportamento
) > 0:

    display(
        resumo_comportamento
    )


# ======================================================================
# 43. SALVAMENTO DOS PARQUETS
# ======================================================================

if len(
    ranking_precursores
) > 0:

    ranking_precursores.to_parquet(

        ARQUIVO_PRECURSORES,

        index=False
    )


if len(
    expansao_cascatas
) > 0:

    expansao_cascatas.to_parquet(

        ARQUIVO_CASCATAS,

        index=False
    )


# ======================================================================
# 44. FUNÇÃO PARA REMOVER TIMEZONE DO EXCEL
# ======================================================================

def remover_timezone_dataframe(
    dataframe
):

    temp = (
        dataframe
        .copy()
    )


    for coluna in temp.columns:

        if isinstance(
            temp[
                coluna
            ].dtype,
            pd.DatetimeTZDtype
        ):

            temp[
                coluna
            ] = (

                temp[
                    coluna
                ]
                .dt.tz_localize(
                    None
                )
            )


    return temp


# ======================================================================
# 45. PREPARAÇÃO DOS DATAFRAMES PARA EXCEL
# ======================================================================

resumo_executivo_excel = remover_timezone_dataframe(
    resumo_executivo
)

ranking_precursores_excel = remover_timezone_dataframe(
    ranking_precursores
)

ranking_first_out_excel = remover_timezone_dataframe(
    ranking_first_out
)

first_out_excel = remover_timezone_dataframe(
    first_out
)

expansao_excel = remover_timezone_dataframe(
    expansao_cascatas
)

resumo_expansao_excel = remover_timezone_dataframe(
    resumo_expansao
)

resumo_velocidade_excel = remover_timezone_dataframe(
    resumo_velocidade
)

resumo_comportamento_excel = remover_timezone_dataframe(
    resumo_comportamento
)

ranking_inicio_excel = remover_timezone_dataframe(
    ranking_inicio
)

ranking_pares_excel = remover_timezone_dataframe(
    ranking_pares
)

categoria_excel = remover_timezone_dataframe(
    resumo_categoria_precursor
)

torre_excel = remover_timezone_dataframe(
    resumo_torre_precursor
)

tipo_excel = remover_timezone_dataframe(
    resumo_tipo_precursor
)

metodologia_excel = remover_timezone_dataframe(
    metodologia
)


# ======================================================================
# 46. EXPORTAÇÃO PARA EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 9..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo_excel.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    metodologia_excel.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


    top100_precursores.to_excel(
        writer,
        sheet_name="Top100 Precursores",
        index=False
    )


    ranking_precursores_excel.to_excel(
        writer,
        sheet_name="Ranking Precursores",
        index=False
    )


    top100_first_out.to_excel(
        writer,
        sheet_name="Top100 FirstOut",
        index=False
    )


    ranking_first_out_excel.to_excel(
        writer,
        sheet_name="Ranking FirstOut",
        index=False
    )


    first_out_excel.to_excel(
        writer,
        sheet_name="Eventos FirstOut",
        index=False
    )


    ranking_inicio_excel.to_excel(
        writer,
        sheet_name="Janelas Iniciais",
        index=False
    )


    expansao_excel.to_excel(
        writer,
        sheet_name="Expansao Cascatas",
        index=False
    )


    resumo_expansao_excel.to_excel(
        writer,
        sheet_name="Resumo Expansao",
        index=False
    )


    resumo_velocidade_excel.to_excel(
        writer,
        sheet_name="Velocidade",
        index=False
    )


    resumo_comportamento_excel.to_excel(
        writer,
        sheet_name="Comportamento",
        index=False
    )


    top500_relacoes.to_excel(
        writer,
        sheet_name="Top500 Relacoes",
        index=False
    )


    categoria_excel.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )


    torre_excel.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )


    tipo_excel.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )


# ======================================================================
# 47. FORMATAÇÃO DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1

        and

        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(

            horizontal="center",

            vertical="center",

            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0


        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (

                    ""

                    if cell.value is None

                    else str(
                        cell.value
                    )
                )


                max_length = max(

                    max_length,

                    len(
                        valor
                    )
                )


            except:

                pass


        largura = min(

            max(
                max_length + 2,
                12
            ),

            50
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 48. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 120)

print(
    "MÓDULO 9 CONCLUÍDO COM SUCESSO"
)

print("=" * 120)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


if Path(
    ARQUIVO_PRECURSORES
).exists():

    print(
        f"2. {ARQUIVO_PRECURSORES}"
    )


if Path(
    ARQUIVO_CASCATAS
).exists():

    print(
        f"3. {ARQUIVO_CASCATAS}"
    )


print(
    """
O relatório contém:

1. Resumo executivo

2. Metodologia

3. Top 100 precursores

4. Ranking completo de precursores

5. Top 100 First-Out

6. Ranking First-Out

7. Eventos individuais First-Out

8. Participação nos primeiros 1/2/5/10 minutos

9. Expansão das cascatas

10. Distribuição da velocidade de expansão

11. Classificação de velocidade

12. Classificação comportamental

13. Top relações precursor -> seguidor

14. Precursores por categoria

15. Precursores por torre

16. Precursores por tipo
"""
)


# ======================================================================
# 49. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nArquivos Parquet disponíveis para os próximos módulos:"
)


if Path(
    ARQUIVO_PRECURSORES
).exists():

    print(
        f'files.download("{ARQUIVO_PRECURSORES}")'
    )


if Path(
    ARQUIVO_CASCATAS
).exists():

    print(
        f'files.download("{ARQUIVO_CASCATAS}")'
    )

MÓDULO 9 - FIRST-OUT, PRECURSORES E SEQUENCIAMENTO DE CASCATAS

Carregando bases...
Base de eventos........: 1,413,882
Ranking final..........: 5,826
Episódios de flood.....: 1,843

BASE ANALÍTICA
Gerações efetivas.......: 637,566
Episódios................: 1,843

PROCESSANDO EPISÓDIOS
Processando episódio 1/1,843
Processando episódio 100/1,843
Processando episódio 200/1,843
Processando episódio 300/1,843
Processando episódio 400/1,843
Processando episódio 500/1,843
Processando episódio 600/1,843
Processando episódio 700/1,843
Processando episódio 800/1,843
Processando episódio 900/1,843
Processando episódio 1,000/1,843
Processando episódio 1,100/1,843
Processando episódio 1,200/1,843
Processando episódio 1,300/1,843
Processando episódio 1,400/1,843
Processando episódio 1,500/1,843
Processando episódio 1,600/1,843
Processando episódio 1,700/1,843
Processando episódio 1,800/1,843
Processando episódio 1,843/1,843

CONSOLIDAÇÃO CONCLUÍDA
Eventos associados a episódios....: 472,382
Registr

,Indicador,Resultado
0,Episódios analisados,"1,843"
1,Equipamentos analisados,"5,618"
2,Precursores fortes,61
3,First-Out recorrentes,5
4,Mass Contributors,59
5,Seguidores recorrentes,"1,166"
6,Relações temporais únicas,"1,000,895"
7,Principal precursor candidato,TE2-PG-NAE-55114-MC [...
8,Índice do principal precursor,98.72
9,Episódios do principal precursor em 1 min,147



TOP 30 PRECURSORES


,itemName,itemDescription,Categoria,Mantenedor,Tipo,Torre,Ranking_Final,Indice_Prioridade_Final,Prioridade_Final,Episodios_Total,Eventos_Total_Flood,Episodios_1min,Episodios_2min,Episodios_5min,Episodios_10min,Episodios_FirstOut,Floods_Criticos,Floods_Altos,Indice_Flood_Medio,Taxa_FirstOut,Taxa_1min,Taxa_5min,Taxa_10min,Score_1min,Score_5min,Score_FirstOut,Score_Recorrencia,Score_Prioridade,Indice_Precursor,Classificacao_Comportamental,Ranking_Precursor
0,TE2-PG-NAE-55114-MC [...,<NA>,Sistema,Automação,Other,TOS,42,80.67,P1 - Intervenção Prioritária,179,1817,147.00,147.00,147.00,149.00,120.00,12.00,433.00,71.01,67.04,82.12,82.12,83.24,99.30,96.34,99.88,99.65,99.21,98.72,Precursor Forte,1
1,CEATOB09CM01VENT02_SA-STS,Estado VAE 02 - 09º Andar CEA,Geral,Others,Falha de Comando,CEA,148,77.31,P2 - Alta Prioridade,303,319,209.00,209.00,210.00,257.00,206.00,66.00,44.00,63.91,67.99,68.98,69.31,84.82,98.75,95.32,99.91,99.96,97.23,98.15,Precursor Forte,2
2,TE2-PG-SNE-11113-MC [ENERGY VALVE SUP],<NA>,Sistema,Automação,Falha de Comando,TOS,98,78.74,P2 - Alta Prioridade,242,363,150.00,157.00,169.00,199.00,93.00,2.00,86.00,69.95,38.43,61.98,69.83,82.23,98.55,95.38,99.66,99.81,98.16,98.13,Precursor Forte,3
3,CEATOB09CM01VENT01_SA-STS,Estado VAE 01 - 09º Andar CEA,Geral,Others,Falha de Comando,CEA,168,76.88,P2 - Alta Prioridade,303,319,209.00,209.00,210.00,257.00,204.00,65.00,44.00,63.94,67.33,68.98,69.31,84.82,98.75,95.32,99.89,99.96,96.79,98.11,Precursor Forte,4
4,TE2-PG-NAE-55114-MC [...,<NA>,Sistema,Automação,Other,TOS,42,80.67,P1 - Intervenção Prioritária,179,1817,147.00,147.00,147.00,149.00,4.00,0.00,0.00,42.60,2.23,82.12,82.12,83.24,99.30,96.34,92.94,99.65,99.21,97.33,Precursor Forte,5
5,CEATOB09CM01VENT02_SA-STS,Estado VAE 02 - 09º Andar CEA,Geral,Others,Falha de Comando,CEA,148,77.31,P2 - Alta Prioridade,303,319,209.00,209.00,210.00,257.00,9.00,3.00,1.00,61.35,2.97,68.98,69.31,84.82,98.75,95.32,94.54,99.96,97.23,97.08,Precursor Forte,6
6,CEATOB09CM01VENT01_SA-STS,Estado VAE 01 - 09º Andar CEA,Geral,Others,Falha de Comando,CEA,168,76.88,P2 - Alta Prioridade,303,319,209.00,209.00,210.00,257.00,9.00,3.00,1.00,61.35,2.97,68.98,69.31,84.82,98.75,95.32,94.54,99.96,96.79,97.03,Precursor Forte,7
7,CEATOB04AA01QAUT02_DA-TE2,Temperatura Insuflamento Duto Principal CJ403B,Geral,Others,Other,CEA,24,82.50,P1 - Intervenção Prioritária,257,621,89.00,98.00,129.00,174.00,76.00,52.00,17.00,82.44,29.57,34.63,50.19,67.70,96.67,92.92,99.59,99.22,99.56,96.99,First-Out Recorrente,8
8,CEATOB09CM01EXAU01_EF-STS,Estado EXA 01 - 09º Andar CEA,Geral,Others,Falha de Comando,CEA,807,63.31,P3 - Programar Investigação,300,307,209.00,209.00,210.00,257.00,201.00,65.00,43.00,64.09,67.00,69.67,70.00,85.67,98.80,95.42,99.86,99.96,85.22,96.98,Precursor Forte,9
9,CEATOB04SH01QDEL01_QDL-S2,Estado da Iluminacao C2,Iluminação,Elétrica,Falha de Comando,CEA,677,66.10,P2 - Alta Prioridade,150,157,118.00,118.00,118.00,136.00,5.00,3.00,2.00,83.77,3.33,78.67,78.67,90.67,98.96,95.98,95.09,99.06,87.55,96.31,Precursor Forte,10



TOP 30 FIRST-OUT


...

,Classificacao_Comportamental,Equipamentos,Episodios_Total,Eventos_Total,Indice_Precursor_Medio,Indice_Precursor_Max
4,Precursor Forte,57,7726,13358,87.26,98.72
0,First-Out Recorrente,5,1154,1617,93.03,96.99
2,Participante Recorrente,1868,161469,273518,65.92,95.06
1,Mass Contributor,55,6118,48966,70.69,92.45
3,Participação Ocasional,2474,14653,19243,38.44,87.13
5,Seguidor Recorrente,1162,100826,128827,45.68,78.66



Gerando relatório Excel do Módulo 9...

MÓDULO 9 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo9_Precursores_FirstOut_Metasys.xlsx
2. Ranking_Precursores_Metasys_2026.parquet
3. Cascatas_Sequenciamento_Metasys_2026.parquet

O relatório contém:

1. Resumo executivo

2. Metodologia

3. Top 100 precursores

4. Ranking completo de precursores

5. Top 100 First-Out

6. Ranking First-Out

7. Eventos individuais First-Out

8. Participação nos primeiros 1/2/5/10 minutos

9. Expansão das cascatas

10. Distribuição da velocidade de expansão

11. Classificação de velocidade

12. Classificação comportamental

13. Top relações precursor -> seguidor

14. Precursores por categoria

15. Precursores por torre

16. Precursores por tipo


Iniciando download do relatório Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Arquivos Parquet disponíveis para os próximos módulos:
files.download("Ranking_Precursores_Metasys_2026.parquet")
files.download("Cascatas_Sequenciamento_Metasys_2026.parquet")


In [19]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 10
# ASSINATURAS DE CASCATA E HIPÓTESES DE CAUSA RAIZ
#
# ENTRADAS
# ----------------------------------------------------------------------
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet
#
# Episodios_Alarm_Flood_Metasys_2026.parquet
#
# Ranking_Precursores_Metasys_2026.parquet
#
# Cascatas_Sequenciamento_Metasys_2026.parquet
#
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Reconstruir a composição de cada episódio de Alarm Flood
#
# 2. Criar uma "assinatura" para cada cascata:
#
#       Categoria dominante
#       Tipo dominante
#       Torre dominante
#       Mantenedor dominante
#       Classe de geração dominante
#       Abrangência
#       Velocidade de propagação
#
# 3. Avaliar os primeiros eventos / First-Out
#
# 4. Cruzar os episódios com o ranking de precursores do Módulo 9
#
# 5. Criar hipóteses analíticas de causa:
#
#       - Comunicação / Automação
#       - HVAC / Temperatura
#       - Falha de Comando
#       - Hidráulica
#       - Pressão
#       - Nível
#       - Cascata Multissistema
#       - Evento Localizado
#       - Padrão Misto
#
# 6. Calcular confiança da hipótese
#
# 7. Identificar assinaturas recorrentes
#
# 8. Gerar ações recomendadas para investigação
#
#
# IMPORTANTE
# ----------------------------------------------------------------------
# Hipótese de causa raiz ≠ diagnóstico comprovado.
#
# A associação temporal e estatística deverá posteriormente ser
# confrontada com:
#
# - arquitetura de automação
# - diagramas
# - rede
# - controladores
# - lógica de controle
# - alimentação elétrica
# - operação
# - ordens de serviço
# - histórico de manutenção
#
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    350
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    320
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_BASE = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)


ARQUIVO_RANKING_FINAL = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


ARQUIVO_EPISODIOS = (
    "Episodios_Alarm_Flood_Metasys_2026.parquet"
)


ARQUIVO_PRECURSORES = (
    "Ranking_Precursores_Metasys_2026.parquet"
)


ARQUIVO_CASCATAS = (
    "Cascatas_Sequenciamento_Metasys_2026.parquet"
)


ARQUIVO_RESULTADO = (
    "Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet"
)


ARQUIVO_RELATORIO = (
    "Relatorio_Modulo10_Assinaturas_Causa_Raiz_Metasys.xlsx"
)


print("=" * 125)

print(
    "MÓDULO 10 - ASSINATURAS DE CASCATA "
    "E HIPÓTESES DE CAUSA RAIZ"
)

print("=" * 125)


# ======================================================================
# 3. VERIFICAÇÃO DOS ARQUIVOS
# ======================================================================

arquivos_necessarios = [

    ARQUIVO_BASE,
    ARQUIVO_RANKING_FINAL,
    ARQUIVO_EPISODIOS,
    ARQUIVO_PRECURSORES,
    ARQUIVO_CASCATAS
]


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    print(
        "\nArquivos não encontrados:"
    )

    for arquivo in faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos necessários."
    )


    from google.colab import files

    uploaded = files.upload()


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            faltantes
        )
    )


# ======================================================================
# 4. CARREGAMENTO
# ======================================================================

print(
    "\nCarregando bases..."
)


df = pd.read_parquet(
    ARQUIVO_BASE
)


ranking_final = pd.read_parquet(
    ARQUIVO_RANKING_FINAL
)


episodios = pd.read_parquet(
    ARQUIVO_EPISODIOS
)


precursores = pd.read_parquet(
    ARQUIVO_PRECURSORES
)


cascatas = pd.read_parquet(
    ARQUIVO_CASCATAS
)


print(
    f"Base de eventos............: {len(df):,}"
)

print(
    f"Ranking final..............: {len(ranking_final):,}"
)

print(
    f"Episódios..................: {len(episodios):,}"
)

print(
    f"Precursores................: {len(precursores):,}"
)

print(
    f"Cascatas sequenciadas......: {len(cascatas):,}"
)


# ======================================================================
# 5. FUNÇÕES AUXILIARES
# ======================================================================

def moda_segura(serie):

    serie = (
        serie
        .dropna()
    )

    if len(
        serie
    ) == 0:

        return pd.NA


    moda = (
        serie
        .mode()
    )


    if len(
        moda
    ) > 0:

        return moda.iloc[0]


    return serie.iloc[0]


def percentual(
    numerador,
    denominador
):

    if (
        denominador is None
        or
        pd.isna(
            denominador
        )
        or
        denominador == 0
    ):

        return 0.0


    return (
        numerador
        /
        denominador
        *
        100
    )


# ======================================================================
# 6. PADRONIZAÇÃO TEMPORAL
# ======================================================================

df[
    "DataHoraLocal"
] = pd.to_datetime(

    df[
        "DataHoraLocal"
    ],

    errors="coerce"
)


try:

    df[
        "DataHoraAnalise"
    ] = (

        df[
            "DataHoraLocal"
        ]
        .dt.tz_localize(
            None
        )
    )


except:

    df[
        "DataHoraAnalise"
    ] = (

        df[
            "DataHoraLocal"
        ]
    )


for coluna in [

    "Inicio_Episodio",
    "Fim_Episodio"

]:

    episodios[
        coluna
    ] = pd.to_datetime(

        episodios[
            coluna
        ],

        errors="coerce"
    )


    try:

        episodios[
            coluna
        ] = (

            episodios[
                coluna
            ]
            .dt.tz_localize(
                None
            )
        )

    except:

        pass


# ======================================================================
# 7. BASE SOMENTE DE GERAÇÕES EFETIVAS
# ======================================================================

geracoes = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()

    .sort_values(
        "DataHoraAnalise"
    )

    .reset_index(
        drop=True
    )
)


# ======================================================================
# 8. CLASSE DA GERAÇÃO
# ======================================================================

geracoes[
    "Classe_Geracao"
] = np.select(

    [
        geracoes[
            "Flag_Entrada_Alarme"
        ],

        geracoes[
            "Flag_Entrada_PreAlarme"
        ],

        geracoes[
            "Flag_Falha_Comunicacao"
        ]
    ],

    [
        "Alarme",
        "Pré-Alarme",
        "Comunicação"
    ],

    default="Outro"
)


# ======================================================================
# 9. RANKING FINAL POR EQUIPAMENTO
# ======================================================================

colunas_ranking = [

    "itemName",

    "Ranking_Final",

    "Indice_Prioridade_Final",

    "Prioridade_Final",

    "Perfil_Dominante",

    "Tipo_Tratamento"
]


colunas_ranking = [

    coluna

    for coluna in colunas_ranking

    if coluna in ranking_final.columns
]


ranking_aux = (

    ranking_final[
        colunas_ranking
    ]

    .copy()
)


if (
    "Indice_Prioridade_Final"
    in ranking_aux.columns
):

    ranking_aux = (

        ranking_aux
        .sort_values(
            "Indice_Prioridade_Final",
            ascending=False
        )
    )


ranking_aux = (

    ranking_aux
    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


geracoes = (

    geracoes
    .merge(

        ranking_aux,

        on="itemName",

        how="left"
    )
)


# ======================================================================
# 10. PRECURSORES DO MÓDULO 9
# ======================================================================

colunas_precursor = [

    "itemName",

    "Ranking_Precursor",

    "Indice_Precursor",

    "Classificacao_Comportamental",

    "Episodios_FirstOut",

    "Episodios_1min",

    "Episodios_5min",

    "Taxa_FirstOut",

    "Taxa_1min",

    "Taxa_5min"
]


colunas_precursor = [

    coluna

    for coluna in colunas_precursor

    if coluna in precursores.columns
]


precursor_aux = (

    precursores[
        colunas_precursor
    ]
    .copy()
)


if (
    "Indice_Precursor"
    in precursor_aux.columns
):

    precursor_aux = (

        precursor_aux
        .sort_values(
            "Indice_Precursor",
            ascending=False
        )
    )


precursor_aux = (

    precursor_aux
    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


geracoes = (

    geracoes
    .merge(

        precursor_aux,

        on="itemName",

        how="left"
    )
)


# ======================================================================
# 11. CASCATAS / VELOCIDADE DO MÓDULO 9
# ======================================================================

colunas_cascata = [

    "ID_Episodio",

    "Equipamentos_Total",

    "Eventos_Total",

    "Tempo_ate_10_Equip_Min",

    "Tempo_ate_25_Equip_Min",

    "Tempo_ate_50_Equip_Min",

    "Tempo_ate_100_Equip_Min",

    "Tempo_ate_250_Equip_Min",

    "Classe_Velocidade"
]


colunas_cascata = [

    coluna

    for coluna in colunas_cascata

    if coluna in cascatas.columns
]


cascata_aux = (

    cascatas[
        colunas_cascata
    ]

    .drop_duplicates(
        subset="ID_Episodio",
        keep="first"
    )
)


# ======================================================================
# 12. PROCESSAMENTO DOS EPISÓDIOS
# ======================================================================

resultados = []

eventos_assinaturas = []


TOTAL_EPISODIOS = len(
    episodios
)


print("\n" + "=" * 125)

print(
    "ANALISANDO ASSINATURAS DOS EPISÓDIOS"
)

print("=" * 125)


for contador, episodio in enumerate(

    episodios.itertuples(
        index=False
    ),

    start=1

):

    if (

        contador == 1

        or

        contador % 100 == 0

        or

        contador == TOTAL_EPISODIOS

    ):

        print(

            f"Processando episódio "
            f"{contador:,}/{TOTAL_EPISODIOS:,}"
        )


    id_ep = (
        episodio.ID_Episodio
    )


    inicio = (
        episodio.Inicio_Episodio
    )


    fim = (
        episodio.Fim_Episodio
    )


    temp = (

        geracoes.loc[
            (
                geracoes[
                    "DataHoraAnalise"
                ]
                >=
                inicio
            )

            &

            (
                geracoes[
                    "DataHoraAnalise"
                ]
                <
                fim
            )
        ]

        .copy()

        .sort_values(
            [
                "DataHoraAnalise",
                "itemName"
            ]
        )
    )


    if len(
        temp
    ) == 0:

        continue


    total = len(
        temp
    )


    # ==================================================================
    # 12.1 COMPOSIÇÃO POR CATEGORIA
    # ==================================================================

    cont_categoria = (

        temp[
            "Categoria"
        ]
        .astype("string")
        .fillna(
            "Não Classificado"
        )
        .value_counts()
    )


    categoria_dominante = (
        cont_categoria.index[0]
    )


    qtd_categoria_dominante = (
        cont_categoria.iloc[0]
    )


    perc_categoria_dominante = percentual(

        qtd_categoria_dominante,
        total
    )


    # ==================================================================
    # 12.2 TIPO
    # ==================================================================

    cont_tipo = (

        temp[
            "Tipo"
        ]
        .astype("string")
        .fillna(
            "Não Classificado"
        )
        .value_counts()
    )


    tipo_dominante = (
        cont_tipo.index[0]
    )


    qtd_tipo_dominante = (
        cont_tipo.iloc[0]
    )


    perc_tipo_dominante = percentual(

        qtd_tipo_dominante,
        total
    )


    # ==================================================================
    # 12.3 TORRE
    # ==================================================================

    cont_torre = (

        temp[
            "Torre"
        ]
        .astype("string")
        .fillna(
            "Não Classificado"
        )
        .value_counts()
    )


    torre_dominante = (
        cont_torre.index[0]
    )


    qtd_torre_dominante = (
        cont_torre.iloc[0]
    )


    perc_torre_dominante = percentual(

        qtd_torre_dominante,
        total
    )


    # ==================================================================
    # 12.4 MANTENEDOR
    # ==================================================================

    cont_mantenedor = (

        temp[
            "Mantenedor"
        ]
        .astype("string")
        .fillna(
            "Não Classificado"
        )
        .value_counts()
    )


    mantenedor_dominante = (
        cont_mantenedor.index[0]
    )


    perc_mantenedor_dominante = percentual(

        cont_mantenedor.iloc[0],
        total
    )


    # ==================================================================
    # 12.5 CLASSE DA GERAÇÃO
    # ==================================================================

    cont_classe = (

        temp[
            "Classe_Geracao"
        ]
        .value_counts()
    )


    classe_dominante = (
        cont_classe.index[0]
    )


    perc_classe_dominante = percentual(

        cont_classe.iloc[0],
        total
    )


    perc_alarme = percentual(

        (
            temp[
                "Classe_Geracao"
            ]
            ==
            "Alarme"
        ).sum(),

        total
    )


    perc_prealarme = percentual(

        (
            temp[
                "Classe_Geracao"
            ]
            ==
            "Pré-Alarme"
        ).sum(),

        total
    )


    perc_comunicacao = percentual(

        (
            temp[
                "Classe_Geracao"
            ]
            ==
            "Comunicação"
        ).sum(),

        total
    )


    # ==================================================================
    # 12.6 FIRST-OUT REAL DO EPISÓDIO
    # ==================================================================

    primeiro_timestamp = (

        temp[
            "DataHoraAnalise"
        ]
        .min()
    )


    first = (

        temp.loc[
            temp[
                "DataHoraAnalise"
            ]
            ==
            primeiro_timestamp
        ]

        .copy()
    )


    first_item = (
        moda_segura(
            first[
                "itemName"
            ]
        )
    )


    first_categoria = (
        moda_segura(
            first[
                "Categoria"
            ]
        )
    )


    first_tipo = (
        moda_segura(
            first[
                "Tipo"
            ]
        )
    )


    first_torre = (
        moda_segura(
            first[
                "Torre"
            ]
        )
    )


    first_mantenedor = (
        moda_segura(
            first[
                "Mantenedor"
            ]
        )
    )


    first_classe = (
        moda_segura(
            first[
                "Classe_Geracao"
            ]
        )
    )


    # ==================================================================
    # 12.7 PRIMEIROS 2 MINUTOS
    # ==================================================================

    limite_2min = (

        primeiro_timestamp

        +

        pd.Timedelta(
            minutes=2
        )
    )


    inicio2 = (

        temp.loc[
            temp[
                "DataHoraAnalise"
            ]
            <=
            limite_2min
        ]

        .copy()
    )


    categoria_inicio = (
        moda_segura(
            inicio2[
                "Categoria"
            ]
        )
    )


    tipo_inicio = (
        moda_segura(
            inicio2[
                "Tipo"
            ]
        )
    )


    torre_inicio = (
        moda_segura(
            inicio2[
                "Torre"
            ]
        )
    )


    # ==================================================================
    # 12.8 MELHOR PRECURSOR PRESENTE NO INÍCIO
    # ==================================================================

    inicio2_validos = (

        inicio2.loc[
            inicio2[
                "Indice_Precursor"
            ].notna()
        ]

        .sort_values(
            "Indice_Precursor",
            ascending=False
        )
    )


    if len(
        inicio2_validos
    ) > 0:

        precursor_linha = (
            inicio2_validos.iloc[0]
        )


        precursor_item = (
            precursor_linha[
                "itemName"
            ]
        )


        indice_precursor = (
            precursor_linha[
                "Indice_Precursor"
            ]
        )


        classificacao_precursor = (
            precursor_linha.get(
                "Classificacao_Comportamental",
                pd.NA
            )
        )


    else:

        precursor_item = pd.NA

        indice_precursor = np.nan

        classificacao_precursor = pd.NA


    # ==================================================================
    # 12.9 ABRANGÊNCIA
    # ==================================================================

    qtd_torres = (

        temp[
            "Torre"
        ]
        .nunique(
            dropna=True
        )
    )


    qtd_categorias = (

        temp[
            "Categoria"
        ]
        .nunique(
            dropna=True
        )
    )


    qtd_tipos = (

        temp[
            "Tipo"
        ]
        .nunique(
            dropna=True
        )
    )


    qtd_equip = (

        temp[
            "itemName"
        ]
        .nunique()
    )


    # ==================================================================
    # 12.10 VELOCIDADE
    # ==================================================================

    linha_cascata = (

        cascata_aux.loc[
            cascata_aux[
                "ID_Episodio"
            ]
            ==
            id_ep
        ]
    )


    if len(
        linha_cascata
    ) > 0:

        linha_cascata = (
            linha_cascata.iloc[0]
        )


        classe_velocidade = (

            linha_cascata.get(
                "Classe_Velocidade",
                pd.NA
            )
        )


        tempo_50 = (

            linha_cascata.get(
                "Tempo_ate_50_Equip_Min",
                np.nan
            )
        )


        tempo_100 = (

            linha_cascata.get(
                "Tempo_ate_100_Equip_Min",
                np.nan
            )
        )


    else:

        classe_velocidade = pd.NA

        tempo_50 = np.nan

        tempo_100 = np.nan


    # ==================================================================
    # 12.11 INDICADORES ESPECÍFICOS
    # ==================================================================

    tipo_normalizado = (

        temp[
            "Tipo"
        ]
        .astype("string")
        .str.lower()
        .fillna("")
    )


    categoria_normalizada = (

        temp[
            "Categoria"
        ]
        .astype("string")
        .str.lower()
        .fillna("")
    )


    mantenedor_normalizado = (

        temp[
            "Mantenedor"
        ]
        .astype("string")
        .str.lower()
        .fillna("")
    )


    perc_offline = percentual(

        tipo_normalizado
        .str.contains(
            "off-line|offline",
            regex=True
        )
        .sum(),

        total
    )


    perc_temperatura = percentual(

        tipo_normalizado
        .str.contains(
            "temperatura",
            regex=False
        )
        .sum(),

        total
    )


    perc_falha_comando = percentual(

        tipo_normalizado
        .str.contains(
            "falha de comando",
            regex=False
        )
        .sum(),

        total
    )


    perc_pressao = percentual(

        tipo_normalizado
        .str.contains(
            "pressão|pressao",
            regex=True
        )
        .sum(),

        total
    )


    perc_nivel = percentual(

        tipo_normalizado
        .str.contains(
            "nível|nivel",
            regex=True
        )
        .sum(),

        total
    )


    perc_hvac = percentual(

        categoria_normalizada
        .str.contains(
            "hvac",
            regex=False
        )
        .sum(),

        total
    )


    perc_hidraulica = percentual(

        categoria_normalizada
        .str.contains(
            "hidráulica|hidraulica",
            regex=True
        )
        .sum(),

        total
    )


    perc_sistema = percentual(

        categoria_normalizada
        .str.contains(
            "sistema",
            regex=False
        )
        .sum(),

        total
    )


    perc_automacao = percentual(

        mantenedor_normalizado
        .str.contains(
            "automação|automacao",
            regex=True
        )
        .sum(),

        total
    )


    # ==================================================================
    # 12.12 HIPÓTESE DE CAUSA RAIZ
    # ==================================================================

    if (

        perc_comunicacao >= 50

        or

        perc_offline >= 50

    ):

        hipotese = (
            "Comunicação / Automação"
        )


    elif (

        perc_falha_comando >= 40

    ):

        hipotese = (
            "Falha de Comando / Lógica de Controle"
        )


    elif (

        perc_hvac >= 60

        and

        perc_temperatura >= 40

    ):

        hipotese = (
            "HVAC / Desvio de Temperatura"
        )


    elif (

        perc_hidraulica >= 50

        and

        perc_nivel >= 30

    ):

        hipotese = (
            "Hidráulica / Nível"
        )


    elif (

        perc_pressao >= 40

    ):

        hipotese = (
            "Pressão / Processo"
        )


    elif (

        qtd_torres >= 4

        and

        qtd_categorias >= 3

        and

        perc_categoria_dominante < 50

    ):

        hipotese = (
            "Cascata Multissistema / Causa Comum"
        )


    elif (

        perc_torre_dominante >= 70

        and

        qtd_torres <= 2

    ):

        hipotese = (
            "Evento Localizado na Torre"
        )


    else:

        hipotese = (
            "Padrão Misto / Causa Não Determinada"
        )


    # ==================================================================
    # 12.13 SCORE DE CONFIANÇA
    #
    # COMPONENTES:
    #
    # 35% dominância da assinatura
    # 25% coerência First-Out
    # 20% precursor conhecido
    # 20% concentração em causa técnica
    # ==================================================================

    dominancia_assinatura = np.mean(

        [
            perc_categoria_dominante,
            perc_tipo_dominante,
            perc_mantenedor_dominante
        ]
    )


    coerencia_first = 0


    if (
        str(
            first_categoria
        )
        ==
        str(
            categoria_dominante
        )
    ):

        coerencia_first += 35


    if (
        str(
            first_tipo
        )
        ==
        str(
            tipo_dominante
        )
    ):

        coerencia_first += 35


    if (
        str(
            first_torre
        )
        ==
        str(
            torre_dominante
        )
    ):

        coerencia_first += 30


    score_precursor = (

        float(
            indice_precursor
        )

        if pd.notna(
            indice_precursor
        )

        else 0
    )


    score_tecnico = max(

        perc_comunicacao,
        perc_offline,
        perc_temperatura,
        perc_falha_comando,
        perc_pressao,
        perc_nivel,
        perc_hvac,
        perc_hidraulica
    )


    confianca = (

        dominancia_assinatura
        * 0.35

        +

        coerencia_first
        * 0.25

        +

        score_precursor
        * 0.20

        +

        score_tecnico
        * 0.20
    )


    confianca = min(
        max(
            confianca,
            0
        ),
        100
    )


    if confianca >= 80:

        classe_confianca = (
            "Alta"
        )


    elif confianca >= 60:

        classe_confianca = (
            "Boa"
        )


    elif confianca >= 40:

        classe_confianca = (
            "Moderada"
        )


    else:

        classe_confianca = (
            "Baixa"
        )


    # ==================================================================
    # 12.14 ASSINATURA
    # ==================================================================

    assinatura = (

        f"{categoria_dominante}"

        +

        " | "

        +

        f"{tipo_dominante}"

        +

        " | "

        +

        f"{torre_dominante}"

        +

        " | "

        +

        f"{classe_dominante}"

        +

        " | "

        +

        f"{classe_velocidade}"
    )


    # ==================================================================
    # 12.15 RESULTADO
    # ==================================================================

    resultados.append({

        "ID_Episodio":
            id_ep,

        "Inicio_Episodio":
            inicio,

        "Fim_Episodio":
            fim,

        "Eventos":
            total,

        "Equipamentos_Unicos":
            qtd_equip,

        "Torres":
            qtd_torres,

        "Categorias":
            qtd_categorias,

        "Tipos":
            qtd_tipos,

        "Categoria_Dominante":
            categoria_dominante,

        "Perc_Categoria_Dominante":
            perc_categoria_dominante,

        "Tipo_Dominante":
            tipo_dominante,

        "Perc_Tipo_Dominante":
            perc_tipo_dominante,

        "Torre_Dominante":
            torre_dominante,

        "Perc_Torre_Dominante":
            perc_torre_dominante,

        "Mantenedor_Dominante":
            mantenedor_dominante,

        "Perc_Mantenedor_Dominante":
            perc_mantenedor_dominante,

        "Classe_Geracao_Dominante":
            classe_dominante,

        "Perc_Classe_Dominante":
            perc_classe_dominante,

        "Perc_Alarmes":
            perc_alarme,

        "Perc_PreAlarmes":
            perc_prealarme,

        "Perc_Comunicacao":
            perc_comunicacao,

        "Perc_Offline":
            perc_offline,

        "Perc_Temperatura":
            perc_temperatura,

        "Perc_Falha_Comando":
            perc_falha_comando,

        "Perc_Pressao":
            perc_pressao,

        "Perc_Nivel":
            perc_nivel,

        "Perc_HVAC":
            perc_hvac,

        "Perc_Hidraulica":
            perc_hidraulica,

        "Perc_Sistema":
            perc_sistema,

        "Perc_Automacao":
            perc_automacao,

        "FirstOut_Item":
            first_item,

        "FirstOut_Categoria":
            first_categoria,

        "FirstOut_Tipo":
            first_tipo,

        "FirstOut_Torre":
            first_torre,

        "FirstOut_Mantenedor":
            first_mantenedor,

        "FirstOut_Classe_Geracao":
            first_classe,

        "Categoria_Inicio_2min":
            categoria_inicio,

        "Tipo_Inicio_2min":
            tipo_inicio,

        "Torre_Inicio_2min":
            torre_inicio,

        "Melhor_Precursor_Inicio":
            precursor_item,

        "Indice_Precursor":
            indice_precursor,

        "Classificacao_Precursor":
            classificacao_precursor,

        "Classe_Velocidade":
            classe_velocidade,

        "Tempo_50_Equip_Min":
            tempo_50,

        "Tempo_100_Equip_Min":
            tempo_100,

        "Hipotese_Causa_Raiz":
            hipotese,

        "Confianca_Hipotese":
            confianca,

        "Classe_Confianca":
            classe_confianca,

        "Assinatura_Cascata":
            assinatura,

        "Indice_Alarm_Flood":
            getattr(
                episodio,
                "Indice_Alarm_Flood",
                np.nan
            ),

        "Classe_Alarm_Flood":
            getattr(
                episodio,
                "Classe_Alarm_Flood",
                pd.NA
            )
    })


# ======================================================================
# 13. DATAFRAME DE RESULTADOS
# ======================================================================

assinaturas = pd.DataFrame(
    resultados
)


print("\n" + "=" * 125)

print(
    "ASSINATURAS CONSTRUÍDAS"
)

print("=" * 125)


print(
    f"Episódios classificados: "
    f"{len(assinaturas):,}"
)


# ======================================================================
# 14. RECORRÊNCIA DAS ASSINATURAS
# ======================================================================

recorrencia_assinaturas = (

    assinaturas
    .groupby(
        "Assinatura_Cascata",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Equipamentos_Medio=(
            "Equipamentos_Unicos",
            "mean"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Flood_Medio=(
            "Indice_Alarm_Flood",
            "mean"
        ),

        Primeira_Ocorrencia=(
            "Inicio_Episodio",
            "min"
        ),

        Ultima_Ocorrencia=(
            "Inicio_Episodio",
            "max"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


recorrencia_assinaturas[
    "Ranking_Assinatura"
] = (

    np.arange(
        1,
        len(
            recorrencia_assinaturas
        ) + 1
    )
)


# ======================================================================
# 15. MERGE DA RECORRÊNCIA NO EPISÓDIO
# ======================================================================

assinaturas = (

    assinaturas
    .merge(

        recorrencia_assinaturas[
            [
                "Assinatura_Cascata",
                "Episodios",
                "Ranking_Assinatura"
            ]
        ]

        .rename(
            columns={
                "Episodios":
                    "Recorrencia_Assinatura"
            }
        ),

        on="Assinatura_Cascata",

        how="left"
    )
)


# ======================================================================
# 16. SCORE DE RECORRÊNCIA DA ASSINATURA
# ======================================================================

assinaturas[
    "Score_Recorrencia_Assinatura"
] = (

    assinaturas[
        "Recorrencia_Assinatura"
    ]
    .rank(
        pct=True,
        method="average"
    )

    * 100
)


# ======================================================================
# 17. ÍNDICE DE PRIORIDADE DE INVESTIGAÇÃO DA CASCATA
#
# 35% Alarm Flood
# 25% confiança da hipótese
# 20% recorrência da assinatura
# 20% precursor
# ======================================================================

assinaturas[
    "Indice_Prioridade_Causa"
] = (

    assinaturas[
        "Indice_Alarm_Flood"
    ]
    .fillna(0)
    * 0.35

    +

    assinaturas[
        "Confianca_Hipotese"
    ]
    .fillna(0)
    * 0.25

    +

    assinaturas[
        "Score_Recorrencia_Assinatura"
    ]
    .fillna(0)
    * 0.20

    +

    assinaturas[
        "Indice_Precursor"
    ]
    .fillna(0)
    * 0.20
)


# ======================================================================
# 18. PRIORIDADE DA INVESTIGAÇÃO
# ======================================================================

assinaturas[
    "Prioridade_Investigacao_Causa"
] = np.select(

    [
        assinaturas[
            "Indice_Prioridade_Causa"
        ] >= 80,

        assinaturas[
            "Indice_Prioridade_Causa"
        ] >= 65,

        assinaturas[
            "Indice_Prioridade_Causa"
        ] >= 50
    ],

    [
        "P1 - Investigar Imediatamente",
        "P2 - Alta Prioridade",
        "P3 - Investigação Programada"
    ],

    default=(
        "P4 - Monitorar"
    )
)


# ======================================================================
# 19. RECOMENDAÇÕES DE INVESTIGAÇÃO
# ======================================================================

def recomendar_acao(
    row
):

    hipotese = (
        row[
            "Hipotese_Causa_Raiz"
        ]
    )


    if hipotese == (
        "Comunicação / Automação"
    ):

        return (
            "Correlacionar primeiro evento com controlador/NAE, "
            "rede BACnet/IP ou MS/TP, switches, alimentação, "
            "reinicialização de dispositivos e perda de comunicação. "
            "Verificar se múltiplos pontos pertencem ao mesmo controlador."
        )


    if hipotese == (
        "Falha de Comando / Lógica de Controle"
    ):

        return (
            "Revisar lógica de comando e feedback, permissivos, "
            "intertravamentos, temporizações e sequenciamento. "
            "Verificar qual comando ocorreu imediatamente antes da cascata."
        )


    if hipotese == (
        "HVAC / Desvio de Temperatura"
    ):

        return (
            "Correlacionar temperaturas com setpoints, horário de partida, "
            "status de FAC/AHU/VAV, válvulas, dampers e disponibilidade "
            "de água gelada. Verificar causa comum antes de ajustar limites."
        )


    if hipotese == (
        "Hidráulica / Nível"
    ):

        return (
            "Verificar nível, bombas, boias/transmissores, enchimento, "
            "drenagem e estados de comando. Identificar se um único "
            "evento hidráulico antecede os demais."
        )


    if hipotese == (
        "Pressão / Processo"
    ):

        return (
            "Correlacionar transmissores de pressão, bombas/ventiladores, "
            "válvulas/dampers e mudanças de comando. Avaliar instabilidade "
            "do processo e do sensor."
        )


    if hipotese == (
        "Cascata Multissistema / Causa Comum"
    ):

        return (
            "Priorizar infraestrutura comum: alimentação elétrica, "
            "automação, rede, controladores e eventos de partida/parada. "
            "Comparar First-Out entre torres e sistemas."
        )


    if hipotese == (
        "Evento Localizado na Torre"
    ):

        return (
            "Investigar infraestrutura compartilhada da torre: "
            "controlador, alimentação, equipamento central, rede local "
            "e sequência operacional."
        )


    return (
        "Revisar eventos First-Out e primeiros 2 minutos, "
        "agrupar equipamentos por controlador/sistema físico e "
        "confrontar com histórico operacional e manutenção."
    )


assinaturas[
    "Acao_Recomendada"
] = (

    assinaturas
    .apply(
        recomendar_acao,
        axis=1
    )
)


# ======================================================================
# 20. ORDENAÇÃO FINAL
# ======================================================================

assinaturas = (

    assinaturas
    .sort_values(

        [
            "Indice_Prioridade_Causa",
            "Indice_Alarm_Flood",
            "Confianca_Hipotese"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


assinaturas[
    "Ranking_Investigacao"
] = (

    np.arange(
        1,
        len(
            assinaturas
        ) + 1
    )
)


# ======================================================================
# 21. RESUMO POR HIPÓTESE
# ======================================================================

resumo_hipoteses = (

    assinaturas
    .groupby(
        "Hipotese_Causa_Raiz",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Equipamentos_Medio=(
            "Equipamentos_Unicos",
            "mean"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Flood_Medio=(
            "Indice_Alarm_Flood",
            "mean"
        ),

        Prioridade_Causa_Media=(
            "Indice_Prioridade_Causa",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


resumo_hipoteses[
    "Percentual_Episodios"
] = (

    resumo_hipoteses[
        "Episodios"
    ]

    /

    len(
        assinaturas
    )

    * 100
)


# ======================================================================
# 22. RESUMO POR CONFIANÇA
# ======================================================================

resumo_confianca = (

    assinaturas[
        "Classe_Confianca"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)


resumo_confianca.columns = [

    "Classe_Confianca",
    "Episodios"
]


resumo_confianca[
    "Percentual"
] = (

    resumo_confianca[
        "Episodios"
    ]

    /

    len(
        assinaturas
    )

    * 100
)


# ======================================================================
# 23. RESUMO POR PRIORIDADE
# ======================================================================

resumo_prioridade = (

    assinaturas
    .groupby(
        "Prioridade_Investigacao_Causa",
        observed=True
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Indice_Medio=(
            "Indice_Prioridade_Causa",
            "mean"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        )
    )

    .reset_index()
)


# ======================================================================
# 24. RESUMO MENSAL
# ======================================================================

assinaturas[
    "Ano"
] = (

    assinaturas[
        "Inicio_Episodio"
    ]
    .dt.year
)


assinaturas[
    "Mes_Numero"
] = (

    assinaturas[
        "Inicio_Episodio"
    ]
    .dt.month
)


MAPA_MESES = {

    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}


assinaturas[
    "Mes"
] = (

    assinaturas[
        "Mes_Numero"
    ]
    .map(
        MAPA_MESES
    )
)


resumo_mensal = (

    assinaturas
    .groupby(

        [
            "Ano",
            "Mes_Numero",
            "Mes"
        ],

        observed=True
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        P1=(
            "Prioridade_Investigacao_Causa",

            lambda x:
            (
                x
                ==
                "P1 - Investigar Imediatamente"
            ).sum()
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Causa_Medio=(
            "Indice_Prioridade_Causa",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
)


# ======================================================================
# 25. MATRIZ HIPÓTESE × TORRE
#
# Evitamos categorical + margins=True.
# ======================================================================

matriz_base = (

    assinaturas[
        [
            "Hipotese_Causa_Raiz",
            "Torre_Dominante"
        ]
    ]

    .copy()
)


matriz_base[
    "Hipotese_Causa_Raiz"
] = (

    matriz_base[
        "Hipotese_Causa_Raiz"
    ]
    .astype("string")
    .fillna(
        "Não Classificado"
    )
)


matriz_base[
    "Torre_Dominante"
] = (

    matriz_base[
        "Torre_Dominante"
    ]
    .astype("string")
    .fillna(
        "Não Classificado"
    )
)


matriz_hipotese_torre = pd.crosstab(

    matriz_base[
        "Hipotese_Causa_Raiz"
    ],

    matriz_base[
        "Torre_Dominante"
    ]
)


matriz_hipotese_torre[
    "Total"
] = (

    matriz_hipotese_torre
    .sum(
        axis=1
    )
)


linha_total = (

    matriz_hipotese_torre
    .sum(
        axis=0
    )
)


linha_total.name = "Total"


matriz_hipotese_torre = pd.concat(

    [
        matriz_hipotese_torre,
        linha_total.to_frame().T
    ]
)


# ======================================================================
# 26. MATRIZ HIPÓTESE × TIPO
# ======================================================================

matriz_tipo_base = (

    assinaturas[
        [
            "Hipotese_Causa_Raiz",
            "Tipo_Dominante"
        ]
    ]

    .copy()
)


matriz_tipo_base[
    "Hipotese_Causa_Raiz"
] = (

    matriz_tipo_base[
        "Hipotese_Causa_Raiz"
    ]
    .astype("string")
    .fillna(
        "Não Classificado"
    )
)


matriz_tipo_base[
    "Tipo_Dominante"
] = (

    matriz_tipo_base[
        "Tipo_Dominante"
    ]
    .astype("string")
    .fillna(
        "Não Classificado"
    )
)


matriz_hipotese_tipo = pd.crosstab(

    matriz_tipo_base[
        "Hipotese_Causa_Raiz"
    ],

    matriz_tipo_base[
        "Tipo_Dominante"
    ]
)


matriz_hipotese_tipo[
    "Total"
] = (

    matriz_hipotese_tipo
    .sum(
        axis=1
    )
)


# ======================================================================
# 27. TOP EPISÓDIOS
# ======================================================================

top100_causa = (

    assinaturas
    .head(100)
    .copy()
)


# ======================================================================
# 28. ASSINATURAS RECORRENTES
# ======================================================================

top_assinaturas = (

    recorrencia_assinaturas
    .head(100)
    .copy()
)


# ======================================================================
# 29. FIRST-OUT MAIS FREQUENTE POR HIPÓTESE
# ======================================================================

firstout_por_hipotese = (

    assinaturas
    .groupby(

        [
            "Hipotese_Causa_Raiz",
            "FirstOut_Item"
        ],

        observed=True,

        dropna=False
    )
    .size()

    .reset_index(
        name="Episodios"
    )
)


firstout_por_hipotese = (

    firstout_por_hipotese
    .sort_values(

        [
            "Hipotese_Causa_Raiz",
            "Episodios"
        ],

        ascending=[
            True,
            False
        ]
    )
)


firstout_top = (

    firstout_por_hipotese
    .groupby(
        "Hipotese_Causa_Raiz",
        observed=True
    )
    .head(20)
    .reset_index(
        drop=True
    )
)


# ======================================================================
# 30. PRECURSOR MAIS FREQUENTE POR HIPÓTESE
# ======================================================================

precursor_por_hipotese = (

    assinaturas
    .loc[
        assinaturas[
            "Melhor_Precursor_Inicio"
        ].notna()
    ]

    .groupby(

        [
            "Hipotese_Causa_Raiz",
            "Melhor_Precursor_Inicio"
        ],

        observed=True
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Indice_Precursor_Medio=(
            "Indice_Precursor",
            "mean"
        )
    )

    .reset_index()

    .sort_values(

        [
            "Hipotese_Causa_Raiz",
            "Episodios"
        ],

        ascending=[
            True,
            False
        ]
    )
)


# ======================================================================
# 31. RESUMO EXECUTIVO
# ======================================================================

TOTAL_EPISODIOS = len(
    assinaturas
)


TOTAL_P1 = (

    assinaturas[
        "Prioridade_Investigacao_Causa"
    ]
    .eq(
        "P1 - Investigar Imediatamente"
    )
    .sum()
)


TOTAL_ALTA_CONFIANCA = (

    assinaturas[
        "Classe_Confianca"
    ]
    .eq(
        "Alta"
    )
    .sum()
)


if TOTAL_EPISODIOS > 0:

    principal = (
        assinaturas.iloc[0]
    )


    hipotese_lider = (

        resumo_hipoteses
        .iloc[0]
    )


else:

    principal = None
    hipotese_lider = None


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Episódios analisados",

            "Hipóteses P1",

            "% episódios P1",

            "Episódios com confiança alta",

            "% com confiança alta",

            "Assinaturas distintas",

            "Hipótese mais frequente",

            "Episódios da hipótese mais frequente",

            "Episódio líder de investigação",

            "Hipótese do episódio líder",

            "Confiança do episódio líder",

            "Assinatura do episódio líder",

            "First-Out do episódio líder",

            "Precursor candidato do episódio líder"
        ],

        "Resultado": [

            f"{TOTAL_EPISODIOS:,}",

            f"{TOTAL_P1:,}",

            (
                f"{TOTAL_P1 / TOTAL_EPISODIOS * 100:.2f}%"
                if TOTAL_EPISODIOS > 0
                else "0.00%"
            ),

            f"{TOTAL_ALTA_CONFIANCA:,}",

            (
                f"{TOTAL_ALTA_CONFIANCA / TOTAL_EPISODIOS * 100:.2f}%"
                if TOTAL_EPISODIOS > 0
                else "0.00%"
            ),

            f"{assinaturas['Assinatura_Cascata'].nunique():,}",

            (
                str(
                    hipotese_lider[
                        "Hipotese_Causa_Raiz"
                    ]
                )
                if hipotese_lider is not None
                else ""
            ),

            (
                f"{int(hipotese_lider['Episodios']):,}"
                if hipotese_lider is not None
                else "0"
            ),

            (
                str(
                    principal[
                        "ID_Episodio"
                    ]
                )
                if principal is not None
                else ""
            ),

            (
                str(
                    principal[
                        "Hipotese_Causa_Raiz"
                    ]
                )
                if principal is not None
                else ""
            ),

            (
                f"{principal['Confianca_Hipotese']:.2f}%"
                if principal is not None
                else "0.00%"
            ),

            (
                str(
                    principal[
                        "Assinatura_Cascata"
                    ]
                )
                if principal is not None
                else ""
            ),

            (
                str(
                    principal[
                        "FirstOut_Item"
                    ]
                )
                if principal is not None
                else ""
            ),

            (
                str(
                    principal[
                        "Melhor_Precursor_Inicio"
                    ]
                )
                if principal is not None
                else ""
            )
        ]
    }
)


# ======================================================================
# 32. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Elemento": [

            "Assinatura",

            "First-Out",

            "Precursor",

            "Comunicação / Automação",

            "Falha de Comando",

            "HVAC / Temperatura",

            "Hidráulica / Nível",

            "Pressão",

            "Multissistema",

            "Evento Localizado",

            "Confiança",

            "Prioridade de investigação",

            "Interpretação"
        ],

        "Definicao": [

            (
                "Combinação de categoria, tipo, torre, "
                "classe de geração e velocidade."
            ),

            (
                "Primeiro evento observado temporalmente "
                "no episódio."
            ),

            (
                "Equipamento com maior índice precursor "
                "presente nos primeiros 2 minutos."
            ),

            (
                "Predominância de eventos de comunicação "
                "ou Off-line."
            ),

            (
                "Predominância de eventos classificados "
                "como Falha de Comando."
            ),

            (
                "Predominância HVAC combinada com "
                "eventos de temperatura."
            ),

            (
                "Predominância hidráulica combinada "
                "com eventos de nível."
            ),

            (
                "Predominância de eventos de pressão."
            ),

            (
                "Múltiplas torres/categorias sem uma "
                "categoria claramente dominante."
            ),

            (
                "Alta concentração em uma única torre."
            ),

            (
                "35% dominância + 25% coerência First-Out + "
                "20% precursor + 20% padrão técnico."
            ),

            (
                "35% Flood + 25% confiança + "
                "20% recorrência + 20% precursor."
            ),

            (
                "Hipóteses estatísticas e temporais devem "
                "ser validadas operacionalmente."
            )
        ]
    }
)


# ======================================================================
# 33. EXIBIÇÃO
# ======================================================================

print("\n" + "=" * 125)

print(
    "RESUMO EXECUTIVO"
)

print("=" * 125)


display(
    resumo_executivo
)


print("\n" + "=" * 125)

print(
    "HIPÓTESES DE CAUSA RAIZ"
)

print("=" * 125)


display(
    resumo_hipoteses
)


print("\n" + "=" * 125)

print(
    "CONFIANÇA DAS HIPÓTESES"
)

print("=" * 125)


display(
    resumo_confianca
)


print("\n" + "=" * 125)

print(
    "TOP 30 EPISÓDIOS PARA INVESTIGAÇÃO"
)

print("=" * 125)


display(
    assinaturas.head(30)
)


print("\n" + "=" * 125)

print(
    "TOP ASSINATURAS RECORRENTES"
)

print("=" * 125)


display(
    recorrencia_assinaturas.head(30)
)


# ======================================================================
# 34. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 125)

print(
    "VALIDAÇÕES"
)

print("=" * 125)


print(
    f"Episódios origem.......................: "
    f"{len(episodios):,}"
)


print(
    f"Episódios classificados...............: "
    f"{len(assinaturas):,}"
)


print(
    f"Assinaturas distintas.................: "
    f"{assinaturas['Assinatura_Cascata'].nunique():,}"
)


print(
    f"Hipóteses distintas....................: "
    f"{assinaturas['Hipotese_Causa_Raiz'].nunique():,}"
)


validacao_episodios = (

    len(
        assinaturas
    )

    <=

    len(
        episodios
    )
)


print(
    "\nQuantidade de episódios consistente....:",
    validacao_episodios
)


validacao_confianca = (

    assinaturas[
        "Confianca_Hipotese"
    ]
    .between(
        0,
        100
    )
    .all()
)


print(
    "Confiança dentro de 0 a 100.............:",
    validacao_confianca
)


validacao_indice = (

    assinaturas[
        "Indice_Prioridade_Causa"
    ]
    .between(
        0,
        100
    )
    .all()
)


print(
    "Índice de causa dentro de 0 a 100.......:",
    validacao_indice
)


# ======================================================================
# 35. SALVAMENTO PARQUET
# ======================================================================

assinaturas.to_parquet(

    ARQUIVO_RESULTADO,

    index=False
)


print(
    f"\nBase criada:\n"
    f"{ARQUIVO_RESULTADO}"
)


# ======================================================================
# 36. FUNÇÃO PARA REMOVER TIMEZONE
# ======================================================================

def remover_timezone_dataframe(
    dataframe
):

    temp = (
        dataframe
        .copy()
    )


    for coluna in temp.columns:

        if isinstance(
            temp[
                coluna
            ].dtype,
            pd.DatetimeTZDtype
        ):

            temp[
                coluna
            ] = (

                temp[
                    coluna
                ]
                .dt.tz_localize(
                    None
                )
            )


    return temp


# ======================================================================
# 37. PREPARAÇÃO EXCEL
# ======================================================================

resumo_executivo_excel = remover_timezone_dataframe(
    resumo_executivo
)

assinaturas_excel = remover_timezone_dataframe(
    assinaturas
)

resumo_hipoteses_excel = remover_timezone_dataframe(
    resumo_hipoteses
)

resumo_confianca_excel = remover_timezone_dataframe(
    resumo_confianca
)

resumo_prioridade_excel = remover_timezone_dataframe(
    resumo_prioridade
)

recorrencia_excel = remover_timezone_dataframe(
    recorrencia_assinaturas
)

top100_excel = remover_timezone_dataframe(
    top100_causa
)

top_assinaturas_excel = remover_timezone_dataframe(
    top_assinaturas
)

firstout_excel = remover_timezone_dataframe(
    firstout_top
)

precursor_hip_excel = remover_timezone_dataframe(
    precursor_por_hipotese
)

mensal_excel = remover_timezone_dataframe(
    resumo_mensal
)


# ======================================================================
# 38. EXPORTAÇÃO PARA EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 10..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo_excel.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


    resumo_hipoteses_excel.to_excel(
        writer,
        sheet_name="Hipoteses Causa",
        index=False
    )


    resumo_confianca_excel.to_excel(
        writer,
        sheet_name="Confianca",
        index=False
    )


    resumo_prioridade_excel.to_excel(
        writer,
        sheet_name="Prioridades",
        index=False
    )


    top100_excel.to_excel(
        writer,
        sheet_name="Top100 Investigacao",
        index=False
    )


    assinaturas_excel.to_excel(
        writer,
        sheet_name="Episodios",
        index=False
    )


    recorrencia_excel.to_excel(
        writer,
        sheet_name="Assinaturas",
        index=False
    )


    top_assinaturas_excel.to_excel(
        writer,
        sheet_name="Top Assinaturas",
        index=False
    )


    firstout_excel.to_excel(
        writer,
        sheet_name="FirstOut por Hipotese",
        index=False
    )


    precursor_hip_excel.to_excel(
        writer,
        sheet_name="Precursor por Hipotese",
        index=False
    )


    mensal_excel.to_excel(
        writer,
        sheet_name="Mensal",
        index=False
    )


    matriz_hipotese_torre.to_excel(
        writer,
        sheet_name="Hipotese x Torre"
    )


    matriz_hipotese_tipo.to_excel(
        writer,
        sheet_name="Hipotese x Tipo"
    )


# ======================================================================
# 39. FORMATAÇÃO DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1

        and

        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0


        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (

                    ""

                    if cell.value is None

                    else str(
                        cell.value
                    )
                )


                max_length = max(

                    max_length,

                    len(
                        valor
                    )
                )


            except:

                pass


        largura = min(

            max(
                max_length + 2,
                12
            ),

            55
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 40. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 125)

print(
    "MÓDULO 10 CONCLUÍDO COM SUCESSO"
)

print("=" * 125)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


print(
    f"2. {ARQUIVO_RESULTADO}"
)


print(
    """
O relatório contém:

1. Resumo executivo

2. Metodologia

3. Distribuição das hipóteses de causa

4. Confiança das hipóteses

5. Prioridade de investigação

6. Top 100 episódios

7. Todos os episódios classificados

8. Assinaturas recorrentes

9. Top assinaturas

10. First-Out por hipótese

11. Precursores por hipótese

12. Evolução mensal

13. Matriz Hipótese x Torre

14. Matriz Hipótese x Tipo

15. Ações recomendadas de investigação
"""
)


# ======================================================================
# 41. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar a base Parquet:"
)


print(
    f'files.download("{ARQUIVO_RESULTADO}")'
)

MÓDULO 10 - ASSINATURAS DE CASCATA E HIPÓTESES DE CAUSA RAIZ

Carregando bases...
Base de eventos............: 1,413,882
Ranking final..............: 5,826
Episódios..................: 1,843
Precursores................: 5,648
Cascatas sequenciadas......: 1,843

ANALISANDO ASSINATURAS DOS EPISÓDIOS
Processando episódio 1/1,843
Processando episódio 100/1,843
Processando episódio 200/1,843
Processando episódio 300/1,843
Processando episódio 400/1,843
Processando episódio 500/1,843
Processando episódio 600/1,843
Processando episódio 700/1,843
Processando episódio 800/1,843
Processando episódio 900/1,843
Processando episódio 1,000/1,843
Processando episódio 1,100/1,843
Processando episódio 1,200/1,843
Processando episódio 1,300/1,843
Processando episódio 1,400/1,843
Processando episódio 1,500/1,843
Processando episódio 1,600/1,843
Processando episódio 1,700/1,843
Processando episódio 1,800/1,843
Processando episódio 1,843/1,843

ASSINATURAS CONSTRUÍDAS
Episódios classificados: 1,843

RESUMO

,Indicador,Resultado
0,Episódios analisados,"1,843"
1,Hipóteses P1,63
2,% episódios P1,3.42%
3,Episódios com confiança alta,868
4,% com confiança alta,47.10%
5,Assinaturas distintas,110
6,Hipótese mais frequente,HVAC / Desvio de Temperatura
7,Episódios da hipótese mais frequente,"1,451"
8,Episódio líder de investigação,536
9,Hipótese do episódio líder,HVAC / Desvio de Temperatura



HIPÓTESES DE CAUSA RAIZ


,Hipotese_Causa_Raiz,Episodios,Eventos,Equipamentos_Medio,Confianca_Media,Indice_Flood_Medio,Prioridade_Causa_Media,Percentual_Episodios
4,HVAC / Desvio de Temperatura,1451,414422,168.97,80.76,52.24,67.87,78.73
1,Comunicação / Automação,196,44958,161.93,70.84,42.37,52.56,10.63
3,Falha de Comando / Lógica de Controle,126,4470,33.33,66.94,40.00,57.12,6.84
5,Padrão Misto / Causa Não Determinada,57,7544,92.02,61.47,44.27,55.54,3.09
0,Cascata Multissistema / Causa Comum,11,938,62.18,50.47,45.87,47.73,0.60
2,Evento Localizado na Torre,2,50,24.00,60.65,10.64,38.88,0.11



CONFIANÇA DAS HIPÓTESES


...

,Assinatura_Cascata,Episodios,Eventos,Equipamentos_Medio,Confianca_Media,Indice_Flood_Medio,Primeira_Ocorrencia,Ultima_Ocorrencia,Ranking_Assinatura
32,HVAC | Temperatura | CEA | Pré-Alarme | Baixa ...,422,13220,28.99,82.33,28.72,2026-01-16 18:40:00,2026-07-31 17:40:00,1
34,HVAC | Temperatura | CEA | Pré-Alarme | Moderada,322,83559,153.28,82.27,63.41,2026-01-22 09:10:00,2026-07-31 18:20:00,2
36,HVAC | Temperatura | CEA | Pré-Alarme | Rápida,201,91473,296.76,76.67,71.20,2026-01-06 05:50:00,2026-07-31 06:00:00,3
23,HVAC | Temperatura | CEA | Alarme | Baixa Expa...,94,3370,30.21,77.37,34.32,2026-01-01 09:20:00,2026-07-30 08:30:00,4
12,HVAC | Falha de Comando | CEA | Alarme | Baixa...,94,3183,32.52,66.51,41.07,2026-01-06 20:00:00,2026-07-31 20:00:00,5
35,HVAC | Temperatura | CEA | Pré-Alarme | Muito ...,83,44870,390.57,72.38,76.81,2026-01-28 06:00:00,2026-07-24 06:00:00,6
25,HVAC | Temperatura | CEA | Alarme | Moderada,65,25587,179.37,83.59,69.07,2026-01-01 09:40:00,2026-07-27 06:30:00,7
28,HVAC | Temperatura | CEA | Alarme | Rápida,55,34783,293.62,78.89,71.58,2026-01-01 06:30:00,2026-07-13 19:50:00,8
101,Sistema | Off-line | TOS | Comunicação | Baixa...,46,1893,30.22,74.94,25.71,2026-02-04 22:50:00,2026-07-18 23:00:00,9
60,HVAC | Temperatura | TOS | Pré-Alarme | Baixa ...,28,859,28.68,83.23,20.58,2026-02-17 03:10:00,2026-07-28 09:00:00,10



VALIDAÇÕES
Episódios origem.......................: 1,843
Episódios classificados...............: 1,843
Assinaturas distintas.................: 110
Hipóteses distintas....................: 6

Quantidade de episódios consistente....: True
Confiança dentro de 0 a 100.............: True
Índice de causa dentro de 0 a 100.......: True

Base criada:
Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet

Gerando relatório Excel do Módulo 10...

MÓDULO 10 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo10_Assinaturas_Causa_Raiz_Metasys.xlsx
2. Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet

O relatório contém:

1. Resumo executivo

2. Metodologia

3. Distribuição das hipóteses de causa

4. Confiança das hipóteses

5. Prioridade de investigação

6. Top 100 episódios

7. Todos os episódios classificados

8. Assinaturas recorrentes

9. Top assinaturas

10. First-Out por hipótese

11. Precursores por hipótese

12. Evolução mensal

13. Matriz Hipótese x Torre

14. Matriz Hipótese

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar a base Parquet:
files.download("Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet")


In [20]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 11
# KPIs DE ALARM MANAGEMENT,
# OPORTUNIDADES DE REDUÇÃO E PLANO DE AÇÃO
#
# ENTRADAS
# ----------------------------------------------------------------------
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Ranking_Modulo5_Reincidencia_Chattering.parquet
#
# Base_Ocorrencias_Reconstruidas_Modulo6.parquet
#
# Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet
#
# Episodios_Alarm_Flood_Metasys_2026.parquet
#
# Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet
#
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Consolidar os principais KPIs dos módulos anteriores
#
# 2. Criar visão executiva de Alarm Management
#
# 3. Quantificar:
#       - volume total
#       - gerações efetivas
#       - alarmes
#       - pré-alarmes
#       - comunicação
#       - criticidade
#       - chattering
#       - reincidência
#       - alarm floods
#       - ocorrências longas
#       - ocorrências abertas
#       - hipóteses de causa
#
# 4. Classificar equipamentos em alavancas de melhoria
#
# 5. Criar grupos MUTUAMENTE EXCLUSIVOS para evitar
#    dupla contagem das oportunidades
#
# 6. Calcular três cenários analíticos:
#
#       Conservador
#       Base
#       Potencial
#
# 7. Criar carteira de ações
#
# 8. Preparar estrutura para dashboard / relatório executivo
#
#
# IMPORTANTE
# ----------------------------------------------------------------------
# "Volume Exposto" não significa automaticamente volume eliminável.
#
# As estimativas de redução são cenários analíticos baseados em
# percentuais configuráveis neste código.
#
# Os resultados devem ser validados com manutenção, operação,
# engenharia e gestão antes de qualquer alteração no Metasys.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    350
)

pd.set_option(
    "display.max_rows",
    250
)

pd.set_option(
    "display.width",
    320
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS
# ======================================================================

ARQUIVO_BASE = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)


ARQUIVO_MODULO5 = (
    "Ranking_Modulo5_Reincidencia_Chattering.parquet"
)


ARQUIVO_MODULO6 = (
    "Base_Ocorrencias_Reconstruidas_Modulo6.parquet"
)


ARQUIVO_RANKING_FINAL = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


ARQUIVO_FLOODS = (
    "Episodios_Alarm_Flood_Metasys_2026.parquet"
)


ARQUIVO_CAUSA = (
    "Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet"
)


ARQUIVO_RELATORIO = (
    "Relatorio_Modulo11_Alarm_Management_Metasys.xlsx"
)


ARQUIVO_CARTEIRA = (
    "Carteira_Acoes_Alarm_Management_Metasys_2026.parquet"
)


print("=" * 125)

print(
    "MÓDULO 11 - KPIs DE ALARM MANAGEMENT, "
    "OPORTUNIDADES DE REDUÇÃO E PLANO DE AÇÃO"
)

print("=" * 125)


# ======================================================================
# 3. PARÂMETROS DOS CENÁRIOS
#
# Percentual estimado de redução dentro do VOLUME EXPOSTO
# de cada alavanca.
#
# São premissas analíticas editáveis.
# ======================================================================

TAXAS_CENARIOS = {

    "Racionalização - Chattering": {

        "Conservador": 0.10,
        "Base": 0.25,
        "Potencial": 0.40
    },

    "Racionalização - Reincidência": {

        "Conservador": 0.05,
        "Base": 0.15,
        "Potencial": 0.25
    },

    "Comunicação / Automação": {

        "Conservador": 0.05,
        "Base": 0.15,
        "Potencial": 0.30
    },

    "Persistência / Normalização": {

        "Conservador": 0.03,
        "Base": 0.10,
        "Potencial": 0.20
    },

    "Criticidade / Causa Raiz": {

        "Conservador": 0.02,
        "Base": 0.05,
        "Potencial": 0.10
    },

    "Monitoramento": {

        "Conservador": 0.00,
        "Base": 0.00,
        "Potencial": 0.00
    },

    "Sem Classificação": {

        "Conservador": 0.00,
        "Base": 0.00,
        "Potencial": 0.00
    }
}


# ======================================================================
# 4. VERIFICAÇÃO / UPLOAD
# ======================================================================

arquivos_necessarios = [

    ARQUIVO_BASE,
    ARQUIVO_MODULO5,
    ARQUIVO_MODULO6,
    ARQUIVO_RANKING_FINAL,
    ARQUIVO_FLOODS,
    ARQUIVO_CAUSA
]


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    print(
        "\nArquivos ausentes:"
    )

    for arquivo in faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos necessários."
    )


    from google.colab import files

    uploaded = files.upload()


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            faltantes
        )
    )


# ======================================================================
# 5. CARREGAMENTO
# ======================================================================

print(
    "\nCarregando resultados dos módulos anteriores..."
)


df = pd.read_parquet(
    ARQUIVO_BASE
)


m5 = pd.read_parquet(
    ARQUIVO_MODULO5
)


ocorrencias = pd.read_parquet(
    ARQUIVO_MODULO6
)


ranking = pd.read_parquet(
    ARQUIVO_RANKING_FINAL
)


floods = pd.read_parquet(
    ARQUIVO_FLOODS
)


causas = pd.read_parquet(
    ARQUIVO_CAUSA
)


print(
    f"Base de eventos..............: {len(df):,}"
)

print(
    f"Ranking Módulo 5............: {len(m5):,}"
)

print(
    f"Ocorrências Módulo 6........: {len(ocorrencias):,}"
)

print(
    f"Ranking Final Módulo 7......: {len(ranking):,}"
)

print(
    f"Episódios Flood..............: {len(floods):,}"
)

print(
    f"Assinaturas / Causa..........: {len(causas):,}"
)


# ======================================================================
# 6. FUNÇÕES AUXILIARES
# ======================================================================

def moda_segura(
    serie
):

    serie = serie.dropna()

    if len(
        serie
    ) == 0:

        return pd.NA


    moda = serie.mode()


    if len(
        moda
    ) > 0:

        return moda.iloc[0]


    return serie.iloc[0]


def percentual(
    numerador,
    denominador
):

    if (
        denominador is None
        or
        pd.isna(
            denominador
        )
        or
        denominador == 0
    ):

        return 0.0


    return (

        numerador

        /

        denominador

        * 100
    )


# ======================================================================
# 7. PADRONIZAÇÃO DE DATAS
# ======================================================================

df[
    "DataHoraLocal"
] = pd.to_datetime(

    df[
        "DataHoraLocal"
    ],

    errors="coerce"
)


try:

    df[
        "DataHoraAnalise"
    ] = (

        df[
            "DataHoraLocal"
        ]
        .dt.tz_localize(
            None
        )
    )


except:

    df[
        "DataHoraAnalise"
    ] = (

        df[
            "DataHoraLocal"
        ]
    )


# ======================================================================
# 8. PERÍODO ANALISADO
# ======================================================================

DATA_INICIAL = (

    df[
        "DataHoraAnalise"
    ]
    .min()
)


DATA_FINAL = (

    df[
        "DataHoraAnalise"
    ]
    .max()
)


DIAS_ANALISADOS = (

    df[
        "DataHoraAnalise"
    ]
    .dt.normalize()
    .nunique()
)


# ======================================================================
# 9. BASE SOMENTE DE GERAÇÕES EFETIVAS
# ======================================================================

geracoes = (

    df.loc[
        df[
            "Flag_Geracao_Efetiva"
        ] == True
    ]

    .copy()
)


TOTAL_EVENTOS = len(
    df
)


TOTAL_GERACOES = len(
    geracoes
)


TOTAL_OPERACIONAIS = int(

    df[
        "Flag_Evento_Operacional"
    ]
    .sum()
)


TOTAL_ALARMES = int(

    geracoes[
        "Flag_Entrada_Alarme"
    ]
    .sum()
)


TOTAL_PREALARMES = int(

    geracoes[
        "Flag_Entrada_PreAlarme"
    ]
    .sum()
)


TOTAL_COMUNICACAO = int(

    geracoes[
        "Flag_Falha_Comunicacao"
    ]
    .sum()
)


# ======================================================================
# 10. KPIs BÁSICOS DE VOLUME
# ======================================================================

MEDIA_EVENTOS_DIA = (

    TOTAL_EVENTOS

    /

    DIAS_ANALISADOS
)


MEDIA_GERACOES_DIA = (

    TOTAL_GERACOES

    /

    DIAS_ANALISADOS
)


PERC_GERACOES = percentual(

    TOTAL_GERACOES,
    TOTAL_EVENTOS
)


PERC_ALARMES = percentual(

    TOTAL_ALARMES,
    TOTAL_GERACOES
)


PERC_PREALARMES = percentual(

    TOTAL_PREALARMES,
    TOTAL_GERACOES
)


PERC_COMUNICACAO = percentual(

    TOTAL_COMUNICACAO,
    TOTAL_GERACOES
)


# ======================================================================
# 11. DEDUPLICAÇÃO DO MÓDULO 5
# ======================================================================

if (
    "Indice_Prioridade_Investigacao"
    in m5.columns
):

    m5 = (

        m5
        .sort_values(
            "Indice_Prioridade_Investigacao",
            ascending=False
        )
    )


m5 = (

    m5
    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


# ======================================================================
# 12. DEDUPLICAÇÃO DO RANKING FINAL
# ======================================================================

if (
    "Indice_Prioridade_Final"
    in ranking.columns
):

    ranking = (

        ranking
        .sort_values(
            "Indice_Prioridade_Final",
            ascending=False
        )
    )


ranking = (

    ranking
    .drop_duplicates(
        subset="itemName",
        keep="first"
    )
)


# ======================================================================
# 13. MÉTRICAS DO MÓDULO 6 POR EQUIPAMENTO
# ======================================================================

metricas_m6 = (

    ocorrencias
    .groupby(
        "itemName",
        observed=True,
        dropna=False
    )
    .agg(

        Ocorrencias=(
            "itemName",
            "size"
        ),

        Fechadas=(
            "Flag_Fechada",
            "sum"
        ),

        Abertas=(
            "Flag_Aberta",
            "sum"
        ),

        Reentradas=(
            "Reentradas_Enquanto_Aberta",
            "sum"
        ),

        Duracao_Mediana_Min=(
            "Duracao_Min",
            "median"
        ),

        Duracao_Maxima_Min=(
            "Duracao_Min",
            "max"
        ),

        Horas_Acumuladas=(
            "Duracao_Horas",
            "sum"
        ),

        Maior_60min=(
            "Flag_Maior_60min",
            "sum"
        ),

        Maior_4h=(
            "Flag_Maior_4h",
            "sum"
        ),

        Maior_24h=(
            "Flag_Maior_24h",
            "sum"
        )
    )

    .reset_index()
)


# ======================================================================
# 14. CADASTRO CONSOLIDADO
# ======================================================================

cadastro = (

    ranking[
        [
            coluna

            for coluna in [

                "itemName",
                "itemDescription",

                "Categoria",
                "Mantenedor",
                "Tipo",
                "Torre",
                "TTL_Torre",

                "Geracoes_Efetivas",

                "Indice_Criticidade_V1",
                "Classe_Criticidade",

                "Indice_Prioridade_Final",
                "Prioridade_Final",

                "Perfil_Dominante",
                "Tipo_Tratamento",

                "Ocorrencias",
                "Ocorrencias_Abertas",

                "Ocorrencias_Maior_24h",
                "Horas_Acumuladas"
            ]

            if coluna in ranking.columns
        ]
    ]

    .copy()
)


# ======================================================================
# 15. MÉTRICAS M5 PARA MERGE
# ======================================================================

colunas_m5 = [

    coluna

    for coluna in [

        "itemName",

        "Reincidencias_10min",
        "Reincidencias_60min",

        "Taxa_Reincidencia_10min_Perc",

        "Episodios_Chattering_10min",

        "Transicoes_Chattering_10min",

        "Taxa_Chattering_10min_Perc",

        "Classe_Comportamento"
    ]

    if coluna in m5.columns
]


m5_aux = (

    m5[
        colunas_m5
    ]

    .copy()
)


# ======================================================================
# 16. CONSOLIDAÇÃO POR EQUIPAMENTO
# ======================================================================

equipamentos = (

    cadastro

    .merge(
        m5_aux,
        on="itemName",
        how="left"
    )

    .merge(
        metricas_m6,
        on="itemName",
        how="left",
        suffixes=(
            "",
            "_M6"
        )
    )
)


# ======================================================================
# 17. PREENCHIMENTO DE NULOS
# ======================================================================

colunas_zero = [

    "Geracoes_Efetivas",

    "Reincidencias_10min",
    "Reincidencias_60min",

    "Taxa_Reincidencia_10min_Perc",

    "Episodios_Chattering_10min",

    "Transicoes_Chattering_10min",

    "Taxa_Chattering_10min_Perc",

    "Ocorrencias",

    "Fechadas",

    "Abertas",

    "Reentradas",

    "Horas_Acumuladas",

    "Maior_60min",
    "Maior_4h",
    "Maior_24h"
]


for coluna in colunas_zero:

    if coluna in equipamentos.columns:

        equipamentos[
            coluna
        ] = (

            pd.to_numeric(

                equipamentos[
                    coluna
                ],

                errors="coerce"
            )

            .fillna(0)
        )


# ======================================================================
# 18. CLASSIFICAÇÃO DAS ALAVANCAS
#
# HIERARQUIA MUTUAMENTE EXCLUSIVA
#
# 1. Chattering
# 2. Reincidência
# 3. Comunicação
# 4. Persistência
# 5. Criticidade
# 6. Monitoramento
#
# Cada equipamento pertence apenas a UMA alavanca principal.
# ======================================================================

def classificar_alavanca(
    row
):

    episodios_chattering = (
        row.get(
            "Episodios_Chattering_10min",
            0
        )
    )


    reincidencias = (
        row.get(
            "Reincidencias_10min",
            0
        )
    )


    tipo = (

        str(
            row.get(
                "Tipo",
                ""
            )
        )

        .lower()
    )


    abertas = (
        row.get(
            "Abertas",
            0
        )
    )


    longas = (
        row.get(
            "Maior_24h",
            0
        )
    )


    prioridade = (

        str(
            row.get(
                "Prioridade_Final",
                ""
            )
        )
    )


    # --------------------------------------------------------------
    # 1. Chattering
    # --------------------------------------------------------------

    if (
        episodios_chattering
        >=
        3
    ):

        return (
            "Racionalização - Chattering"
        )


    # --------------------------------------------------------------
    # 2. Reincidência
    # --------------------------------------------------------------

    if (
        reincidencias
        >=
        10
    ):

        return (
            "Racionalização - Reincidência"
        )


    # --------------------------------------------------------------
    # 3. Comunicação
    # --------------------------------------------------------------

    if (
        "off-line" in tipo
        or
        "offline" in tipo
    ):

        return (
            "Comunicação / Automação"
        )


    # --------------------------------------------------------------
    # 4. Persistência
    # --------------------------------------------------------------

    if (
        abertas > 0

        or

        longas > 0
    ):

        return (
            "Persistência / Normalização"
        )


    # --------------------------------------------------------------
    # 5. Criticidade
    # --------------------------------------------------------------

    if (
        prioridade.startswith(
            "P1"
        )

        or

        prioridade.startswith(
            "P2"
        )
    ):

        return (
            "Criticidade / Causa Raiz"
        )


    return (
        "Monitoramento"
    )


equipamentos[
    "Alavanca_Principal"
] = (

    equipamentos
    .apply(
        classificar_alavanca,
        axis=1
    )
)


# ======================================================================
# 19. RANKING DAS ALAVANCAS
# ======================================================================

ORDEM_ALAVANCAS = {

    "Racionalização - Chattering":
        1,

    "Racionalização - Reincidência":
        2,

    "Comunicação / Automação":
        3,

    "Persistência / Normalização":
        4,

    "Criticidade / Causa Raiz":
        5,

    "Monitoramento":
        6,

    "Sem Classificação":
        7
}


equipamentos[
    "Ordem_Alavanca"
] = (

    equipamentos[
        "Alavanca_Principal"
    ]
    .map(
        ORDEM_ALAVANCAS
    )
)


# ======================================================================
# 20. MERGE DA ALAVANCA NA BASE DE GERAÇÕES
#
# Isso permite quantificar o volume EXPOSTO por alavanca.
#
# Como cada itemName recebe uma única alavanca,
# não existe dupla contagem entre grupos.
# ======================================================================

alavanca_item = (

    equipamentos[
        [
            "itemName",
            "Alavanca_Principal"
        ]
    ]

    .drop_duplicates(
        subset="itemName"
    )
)


geracoes_analise = (

    geracoes
    .merge(

        alavanca_item,

        on="itemName",

        how="left"
    )
)


geracoes_analise[
    "Alavanca_Principal"
] = (

    geracoes_analise[
        "Alavanca_Principal"
    ]
    .fillna(
        "Sem Classificação"
    )
)


# ======================================================================
# 21. VOLUME EXPOSTO POR ALAVANCA
# ======================================================================

volume_alavancas = (

    geracoes_analise
    .groupby(
        "Alavanca_Principal",
        observed=True,
        dropna=False
    )
    .agg(

        Volume_Exposto=(
            "itemName",
            "size"
        ),

        Equipamentos=(
            "itemName",
            "nunique"
        ),

        Alarmes=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        PreAlarmes=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()
)


volume_alavancas[
    "Percentual_Geracoes"
] = (

    volume_alavancas[
        "Volume_Exposto"
    ]

    /

    TOTAL_GERACOES

    * 100
)


volume_alavancas[
    "Ordem"
] = (

    volume_alavancas[
        "Alavanca_Principal"
    ]
    .map(
        ORDEM_ALAVANCAS
    )
)


volume_alavancas = (

    volume_alavancas
    .sort_values(
        "Ordem"
    )
    .drop(
        columns="Ordem"
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================
# 22. CÁLCULO DOS CENÁRIOS
# ======================================================================

cenarios_lista = []


for linha in volume_alavancas.itertuples(
    index=False
):

    alavanca = (
        linha.Alavanca_Principal
    )


    volume = (
        linha.Volume_Exposto
    )


    taxas = (

        TAXAS_CENARIOS.get(

            alavanca,

            {
                "Conservador": 0,
                "Base": 0,
                "Potencial": 0
            }
        )
    )


    for nome_cenario in [

        "Conservador",
        "Base",
        "Potencial"

    ]:

        taxa = (
            taxas[
                nome_cenario
            ]
        )


        reducao = (

            volume

            *

            taxa
        )


        cenarios_lista.append({

            "Alavanca":
                alavanca,

            "Cenario":
                nome_cenario,

            "Volume_Exposto":
                int(
                    volume
                ),

            "Taxa_Reducao_Assumida_Perc":
                taxa * 100,

            "Reducao_Estimada":
                int(
                    round(
                        reducao
                    )
                ),

            "Volume_Remanescente":
                int(
                    round(
                        volume
                        -
                        reducao
                    )
                )
        })


cenarios_detalhados = pd.DataFrame(
    cenarios_lista
)


# ======================================================================
# 23. RESUMO DOS CENÁRIOS
# ======================================================================

resumo_cenarios = (

    cenarios_detalhados
    .groupby(
        "Cenario",
        observed=True
    )
    .agg(

        Reducao_Estimada=(
            "Reducao_Estimada",
            "sum"
        )
    )

    .reset_index()
)


resumo_cenarios[
    "Geracoes_Atuais"
] = TOTAL_GERACOES


resumo_cenarios[
    "Geracoes_Estimadas_Pos_Acao"
] = (

    resumo_cenarios[
        "Geracoes_Atuais"
    ]

    -

    resumo_cenarios[
        "Reducao_Estimada"
    ]
)


resumo_cenarios[
    "Reducao_Total_Perc"
] = (

    resumo_cenarios[
        "Reducao_Estimada"
    ]

    /

    TOTAL_GERACOES

    * 100
)


resumo_cenarios[
    "Media_Diaria_Atual"
] = MEDIA_GERACOES_DIA


resumo_cenarios[
    "Media_Diaria_Pos_Acao"
] = (

    resumo_cenarios[
        "Geracoes_Estimadas_Pos_Acao"
    ]

    /

    DIAS_ANALISADOS
)


ordem_cenario = {

    "Conservador": 1,
    "Base": 2,
    "Potencial": 3
}


resumo_cenarios[
    "Ordem"
] = (

    resumo_cenarios[
        "Cenario"
    ]
    .map(
        ordem_cenario
    )
)


resumo_cenarios = (

    resumo_cenarios
    .sort_values(
        "Ordem"
    )
    .drop(
        columns="Ordem"
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================
# 24. KPIs DE CHATTERING / REINCIDÊNCIA
# ======================================================================

TOTAL_EPISODIOS_CHATTERING = int(

    pd.to_numeric(

        m5[
            "Episodios_Chattering_10min"
        ],

        errors="coerce"
    )

    .fillna(0)

    .sum()
)


EQUIP_CHATTERING = (

    m5.loc[

        pd.to_numeric(

            m5[
                "Episodios_Chattering_10min"
            ],

            errors="coerce"
        )

        .fillna(0)

        > 0,

        "itemName"
    ]

    .nunique()
)


TOTAL_REINCIDENCIAS_10MIN = int(

    pd.to_numeric(

        m5[
            "Reincidencias_10min"
        ],

        errors="coerce"
    )

    .fillna(0)

    .sum()
)


# ======================================================================
# 25. KPIs DE DURAÇÃO
# ======================================================================

TOTAL_OCORRENCIAS = len(
    ocorrencias
)


TOTAL_ABERTAS = int(

    ocorrencias[
        "Flag_Aberta"
    ]
    .sum()
)


TOTAL_FECHADAS = int(

    ocorrencias[
        "Flag_Fechada"
    ]
    .sum()
)


TOTAL_LONGAS_24H = int(

    ocorrencias[
        "Flag_Maior_24h"
    ]
    .sum()
)


DURACAO_MEDIANA = (

    ocorrencias.loc[
        ocorrencias[
            "Flag_Fechada"
        ] == True,
        "Duracao_Min"
    ]

    .median()
)


HORAS_ACUMULADAS = (

    ocorrencias[
        "Duracao_Horas"
    ]
    .sum()
)


# ======================================================================
# 26. KPIs DE PRIORIDADE FINAL
# ======================================================================

TOTAL_EQUIPAMENTOS = (

    ranking[
        "itemName"
    ]
    .nunique()
)


TOTAL_P1 = (

    ranking[
        "Prioridade_Final"
    ]
    .astype("string")
    .str.startswith(
        "P1"
    )
    .sum()
)


TOTAL_P2 = (

    ranking[
        "Prioridade_Final"
    ]
    .astype("string")
    .str.startswith(
        "P2"
    )
    .sum()
)


EQUIP_P1_P2 = (

    TOTAL_P1

    +

    TOTAL_P2
)


# ======================================================================
# 27. EVENTOS P1 / P2
# ======================================================================

prioridade_item = (

    ranking[
        [
            "itemName",
            "Prioridade_Final"
        ]
    ]

    .drop_duplicates(
        subset="itemName"
    )
)


geracoes_prioridade = (

    geracoes
    .merge(

        prioridade_item,

        on="itemName",

        how="left"
    )
)


flag_p1_p2 = (

    geracoes_prioridade[
        "Prioridade_Final"
    ]
    .astype("string")
    .str.startswith(
        ("P1", "P2")
    )
)


EVENTOS_P1_P2 = int(
    flag_p1_p2.sum()
)


PERC_EVENTOS_P1_P2 = percentual(

    EVENTOS_P1_P2,
    TOTAL_GERACOES
)


# ======================================================================
# 28. KPIs DE ALARM FLOOD
# ======================================================================

TOTAL_FLOODS = len(
    floods
)


if (
    "Eventos_Reais"
    in floods.columns
):

    EVENTOS_FLOOD = int(

        floods[
            "Eventos_Reais"
        ]
        .sum()
    )


elif (
    "Eventos"
    in floods.columns
):

    EVENTOS_FLOOD = int(

        floods[
            "Eventos"
        ]
        .sum()
    )


else:

    EVENTOS_FLOOD = 0


EVENTOS_FLOOD = min(

    EVENTOS_FLOOD,
    TOTAL_GERACOES
)


PERC_EVENTOS_FLOOD = percentual(

    EVENTOS_FLOOD,
    TOTAL_GERACOES
)


MEDIA_FLOODS_DIA = (

    TOTAL_FLOODS

    /

    DIAS_ANALISADOS
)


# ======================================================================
# 29. KPIs DE CAUSA RAIZ
# ======================================================================

TOTAL_CAUSAS = len(
    causas
)


TOTAL_CAUSA_P1 = (

    causas[
        "Prioridade_Investigacao_Causa"
    ]
    .astype("string")
    .str.startswith(
        "P1"
    )
    .sum()
)


TOTAL_CONFIANCA_ALTA = (

    causas[
        "Classe_Confianca"
    ]
    .astype("string")
    .eq(
        "Alta"
    )
    .sum()
)


TOTAL_CONFIANCA_BOA = (

    causas[
        "Classe_Confianca"
    ]
    .astype("string")
    .eq(
        "Boa"
    )
    .sum()
)


HIPOTESE_LIDER = (

    causas[
        "Hipotese_Causa_Raiz"
    ]
    .value_counts()
    .index[0]
)


EPISODIOS_HIPOTESE_LIDER = (

    causas[
        "Hipotese_Causa_Raiz"
    ]
    .value_counts()
    .iloc[0]
)


PERC_HIPOTESE_LIDER = percentual(

    EPISODIOS_HIPOTESE_LIDER,
    TOTAL_CAUSAS
)


# ======================================================================
# 30. SCORECARD EXECUTIVO
# ======================================================================

scorecard = pd.DataFrame(
    {
        "Pilar": [

            "Volume",
            "Volume",
            "Volume",

            "Composição",
            "Composição",
            "Composição",

            "Qualidade",
            "Qualidade",

            "Persistência",
            "Persistência",
            "Persistência",

            "Flood",
            "Flood",
            "Flood",

            "Criticidade",
            "Criticidade",

            "Diagnóstico",
            "Diagnóstico",
            "Diagnóstico"
        ],

        "Indicador": [

            "Eventos totais",
            "Gerações efetivas",
            "Média de gerações por dia",

            "% Alarmes",
            "% Pré-Alarmes",
            "% Comunicação",

            "Episódios de chattering",
            "Reincidências até 10 min",

            "Ocorrências abertas",
            "Ocorrências >24h",
            "Duração mediana das fechadas - min",

            "Episódios de Alarm Flood",
            "% gerações em períodos de Flood",
            "Média de Floods por dia",

            "Equipamentos P1 + P2",
            "% gerações oriundas de P1 + P2",

            "Hipóteses de causa analisadas",
            "% hipóteses Boa ou Alta",
            "% hipótese dominante"
        ],

        "Valor": [

            TOTAL_EVENTOS,
            TOTAL_GERACOES,
            MEDIA_GERACOES_DIA,

            PERC_ALARMES,
            PERC_PREALARMES,
            PERC_COMUNICACAO,

            TOTAL_EPISODIOS_CHATTERING,
            TOTAL_REINCIDENCIAS_10MIN,

            TOTAL_ABERTAS,
            TOTAL_LONGAS_24H,
            DURACAO_MEDIANA,

            TOTAL_FLOODS,
            PERC_EVENTOS_FLOOD,
            MEDIA_FLOODS_DIA,

            EQUIP_P1_P2,
            PERC_EVENTOS_P1_P2,

            TOTAL_CAUSAS,

            percentual(
                TOTAL_CONFIANCA_ALTA
                +
                TOTAL_CONFIANCA_BOA,
                TOTAL_CAUSAS
            ),

            PERC_HIPOTESE_LIDER
        ],

        "Unidade": [

            "registros",
            "eventos",
            "eventos/dia",

            "%",
            "%",
            "%",

            "episódios",
            "eventos",

            "ocorrências",
            "ocorrências",
            "min",

            "episódios",
            "%",
            "episódios/dia",

            "equipamentos",
            "%",

            "episódios",
            "%",
            "%"
        ]
    }
)


# ======================================================================
# 31. RESUMO DAS HIPÓTESES
# ======================================================================

resumo_hipoteses = (

    causas
    .groupby(
        "Hipotese_Causa_Raiz",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Causa_Medio=(
            "Indice_Prioridade_Causa",
            "mean"
        ),

        P1=(
            "Prioridade_Investigacao_Causa",

            lambda x:
            x.astype("string")
            .str.startswith(
                "P1"
            )
            .sum()
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


resumo_hipoteses[
    "Percentual_Episodios"
] = (

    resumo_hipoteses[
        "Episodios"
    ]

    /

    TOTAL_CAUSAS

    * 100
)


# ======================================================================
# 32. ASSINATURAS RECORRENTES
# ======================================================================

resumo_assinaturas = (

    causas
    .groupby(
        "Assinatura_Cascata",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Flood_Medio=(
            "Indice_Alarm_Flood",
            "mean"
        ),

        Primeira_Ocorrencia=(
            "Inicio_Episodio",
            "min"
        ),

        Ultima_Ocorrencia=(
            "Inicio_Episodio",
            "max"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


resumo_assinaturas[
    "Participacao_Episodios_Perc"
] = (

    resumo_assinaturas[
        "Episodios"
    ]

    /

    TOTAL_CAUSAS

    * 100
)


# ======================================================================
# 33. PARETO DAS ASSINATURAS
# ======================================================================

resumo_assinaturas[
    "Participacao_Acumulada_Perc"
] = (

    resumo_assinaturas[
        "Participacao_Episodios_Perc"
    ]
    .cumsum()
)


resumo_assinaturas[
    "Classe_Pareto"
] = np.select(

    [
        resumo_assinaturas[
            "Participacao_Acumulada_Perc"
        ]
        <= 80,

        resumo_assinaturas[
            "Participacao_Acumulada_Perc"
        ]
        <= 95
    ],

    [
        "A",
        "B"
    ],

    default="C"
)


# ======================================================================
# 34. CARTEIRA OPERACIONAL DE AÇÕES
# ======================================================================

carteira = (

    equipamentos
    .copy()
)


# ======================================================================
# 35. SCORE DA ALAVANCA
# ======================================================================

carteira[
    "Score_Alavanca"
] = np.select(

    [
        carteira[
            "Alavanca_Principal"
        ]
        ==
        "Racionalização - Chattering",

        carteira[
            "Alavanca_Principal"
        ]
        ==
        "Racionalização - Reincidência",

        carteira[
            "Alavanca_Principal"
        ]
        ==
        "Comunicação / Automação",

        carteira[
            "Alavanca_Principal"
        ]
        ==
        "Persistência / Normalização",

        carteira[
            "Alavanca_Principal"
        ]
        ==
        "Criticidade / Causa Raiz"
    ],

    [
        100,
        90,
        80,
        70,
        60
    ],

    default=30
)


# ======================================================================
# 36. SCORE DE VOLUME
# ======================================================================

carteira[
    "Score_Volume"
] = (

    pd.to_numeric(

        carteira[
            "Geracoes_Efetivas"
        ],

        errors="coerce"
    )

    .fillna(0)

    .rank(
        pct=True,
        method="average"
    )

    * 100
)


# ======================================================================
# 37. SCORE DE PRIORIDADE
# ======================================================================

carteira[
    "Score_Prioridade_Carteira"
] = (

    pd.to_numeric(

        carteira[
            "Indice_Prioridade_Final"
        ],

        errors="coerce"
    )

    .fillna(0)
)


# ======================================================================
# 38. ÍNDICE DE AÇÃO
#
# 40% alavanca
# 35% prioridade
# 25% volume
# ======================================================================

carteira[
    "Indice_Acao"
] = (

    carteira[
        "Score_Alavanca"
    ]
    * 0.40

    +

    carteira[
        "Score_Prioridade_Carteira"
    ]
    * 0.35

    +

    carteira[
        "Score_Volume"
    ]
    * 0.25
)


# ======================================================================
# 39. PRIORIDADE DA CARTEIRA
# ======================================================================

carteira[
    "Prioridade_Acao"
] = np.select(

    [
        carteira[
            "Indice_Acao"
        ] >= 80,

        carteira[
            "Indice_Acao"
        ] >= 65,

        carteira[
            "Indice_Acao"
        ] >= 50
    ],

    [
        "A1 - Imediata",
        "A2 - Alta",
        "A3 - Programada"
    ],

    default="A4 - Monitorar"
)


# ======================================================================
# 40. AÇÃO RECOMENDADA POR ALAVANCA
# ======================================================================

def recomendar_acao(
    row
):

    alavanca = (
        row[
            "Alavanca_Principal"
        ]
    )


    tipo = (

        str(
            row.get(
                "Tipo",
                ""
            )
        )
        .lower()
    )


    if (
        alavanca
        ==
        "Racionalização - Chattering"
    ):

        return (
            "Analisar tendência do ponto e revisar deadband, "
            "limites, temporização, atraso de alarme, sensor "
            "e estabilidade da malha antes de qualquer ajuste."
        )


    if (
        alavanca
        ==
        "Racionalização - Reincidência"
    ):

        return (
            "Identificar causa da repetição em curto intervalo. "
            "Revisar processo, histerese, lógica e retorno à "
            "condição normal."
        )


    if (
        alavanca
        ==
        "Comunicação / Automação"
    ):

        return (
            "Investigar controlador, NAE, BACnet/IP ou MS/TP, "
            "switches, alimentação, rede e reinicializações. "
            "Mapear equipamentos que compartilham infraestrutura."
        )


    if (
        alavanca
        ==
        "Persistência / Normalização"
    ):

        return (
            "Investigar ocorrências longas ou ainda abertas. "
            "Confirmar causa física, retorno à condição normal "
            "e eventual falha na lógica de normalização."
        )


    if (
        alavanca
        ==
        "Criticidade / Causa Raiz"
    ):

        return (
            "Executar análise de causa raiz priorizando "
            "criticidade, First-Out, precursor, histórico "
            "operacional e condição física do equipamento."
        )


    return (
        "Manter monitoramento e reavaliar após atuação "
        "sobre os grupos prioritários."
    )


carteira[
    "Acao_Recomendada"
] = (

    carteira
    .apply(
        recomendar_acao,
        axis=1
    )
)


# ======================================================================
# 41. ORDENAÇÃO DA CARTEIRA
# ======================================================================

carteira = (

    carteira
    .sort_values(

        [
            "Indice_Acao",
            "Geracoes_Efetivas"
        ],

        ascending=[
            False,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


carteira[
    "Ranking_Acao"
] = (

    np.arange(
        1,
        len(
            carteira
        ) + 1
    )
)


# ======================================================================
# 42. RESUMO DA CARTEIRA
# ======================================================================

resumo_carteira = (

    carteira
    .groupby(
        "Alavanca_Principal",
        observed=True,
        dropna=False
    )
    .agg(

        Equipamentos=(
            "itemName",
            "nunique"
        ),

        Geracoes_Efetivas=(
            "Geracoes_Efetivas",
            "sum"
        ),

        A1=(
            "Prioridade_Acao",

            lambda x:
            (
                x
                ==
                "A1 - Imediata"
            ).sum()
        ),

        A2=(
            "Prioridade_Acao",

            lambda x:
            (
                x
                ==
                "A2 - Alta"
            ).sum()
        ),

        Indice_Acao_Medio=(
            "Indice_Acao",
            "mean"
        )
    )

    .reset_index()
)


resumo_carteira[
    "Participacao_Geracoes_Perc"
] = (

    resumo_carteira[
        "Geracoes_Efetivas"
    ]

    /

    TOTAL_GERACOES

    * 100
)


# ======================================================================
# 43. PLANO DE AÇÃO MACRO
# ======================================================================

plano_acao = pd.DataFrame(
    {
        "Ordem": [

            1,
            2,
            3,
            4,
            5,
            6
        ],

        "Frente": [

            "Chattering",

            "Reincidência",

            "HVAC / Temperatura",

            "Comunicação / Automação",

            "Persistência",

            "Alarm Flood / Cascatas"
        ],

        "Objetivo": [

            (
                "Reduzir alternâncias repetitivas "
                "e alarmes de baixo valor operacional."
            ),

            (
                "Eliminar causas que geram repetição "
                "em curtos intervalos."
            ),

            (
                "Atacar assinaturas recorrentes de "
                "temperatura e pré-alarme."
            ),

            (
                "Reduzir cascatas relacionadas a "
                "indisponibilidade de comunicação."
            ),

            (
                "Eliminar condições anormais de "
                "longa duração ou sem normalização."
            ),

            (
                "Reduzir o impacto operacional de "
                "grandes concentrações temporais."
            )
        ],

        "Acao_Principal": [

            (
                "Revisar deadband, temporização, "
                "limites e estabilidade dos sensores."
            ),

            (
                "Investigar causa raiz antes de "
                "alterar parametrização."
            ),

            (
                "Correlacionar setpoints, horários, "
                "FAC/AHU/VAV, válvulas e água gelada."
            ),

            (
                "Mapear controladores, rede, switches "
                "e alimentação compartilhada."
            ),

            (
                "Revisar ocorrências abertas e eventos "
                "acima de 24 horas."
            ),

            (
                "Investigar First-Out, precursor e "
                "assinaturas recorrentes."
            )
        ],

        "Indicador_de_Sucesso": [

            (
                "Redução de episódios de chattering"
            ),

            (
                "Redução de reincidências <=10 min"
            ),

            (
                "Redução de eventos HVAC / Temperatura"
            ),

            (
                "Redução de eventos Off-line"
            ),

            (
                "Redução de ocorrências abertas e >24h"
            ),

            (
                "Redução de episódios e eventos em Flood"
            )
        ]
    }
)


# ======================================================================
# 44. TABELA DE PREMISSAS DOS CENÁRIOS
# ======================================================================

premissas_lista = []


for alavanca, taxas in TAXAS_CENARIOS.items():

    premissas_lista.append({

        "Alavanca":
            alavanca,

        "Conservador_Perc":
            taxas[
                "Conservador"
            ]
            * 100,

        "Base_Perc":
            taxas[
                "Base"
            ]
            * 100,

        "Potencial_Perc":
            taxas[
                "Potencial"
            ]
            * 100
    })


premissas_cenarios = pd.DataFrame(
    premissas_lista
)


# ======================================================================
# 45. EVOLUÇÃO MENSAL DAS GERAÇÕES
# ======================================================================

geracoes_analise[
    "Ano"
] = (

    geracoes_analise[
        "DataHoraAnalise"
    ]
    .dt.year
)


geracoes_analise[
    "Mes_Numero"
] = (

    geracoes_analise[
        "DataHoraAnalise"
    ]
    .dt.month
)


MAPA_MESES = {

    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}


geracoes_analise[
    "Mes"
] = (

    geracoes_analise[
        "Mes_Numero"
    ]
    .map(
        MAPA_MESES
    )
)


mensal = (

    geracoes_analise
    .groupby(

        [
            "Ano",
            "Mes_Numero",
            "Mes"
        ],

        observed=True
    )
    .agg(

        Geracoes_Efetivas=(
            "itemName",
            "size"
        ),

        Alarmes=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        PreAlarmes=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Equipamentos=(
            "itemName",
            "nunique"
        )
    )

    .reset_index()

    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
)


# ======================================================================
# 46. MATRIZ MÊS × ALAVANCA
# ======================================================================

matriz_mes_alavanca = pd.pivot_table(

    geracoes_analise.assign(

        Alavanca_Principal=(

            geracoes_analise[
                "Alavanca_Principal"
            ]
            .astype("string")
        )
    ),

    index=[
        "Ano",
        "Mes_Numero",
        "Mes"
    ],

    columns="Alavanca_Principal",

    values="itemName",

    aggfunc="size",

    fill_value=0,

    observed=True
)


matriz_mes_alavanca[
    "Total"
] = (

    matriz_mes_alavanca
    .sum(
        axis=1
    )
)


# ======================================================================
# 47. RESUMO EXECUTIVO
# ======================================================================

cenario_base = (

    resumo_cenarios.loc[
        resumo_cenarios[
            "Cenario"
        ]
        ==
        "Base"
    ]
    .iloc[0]
)


cenario_conservador = (

    resumo_cenarios.loc[
        resumo_cenarios[
            "Cenario"
        ]
        ==
        "Conservador"
    ]
    .iloc[0]
)


cenario_potencial = (

    resumo_cenarios.loc[
        resumo_cenarios[
            "Cenario"
        ]
        ==
        "Potencial"
    ]
    .iloc[0]
)


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [

            "Período inicial",

            "Período final",

            "Dias analisados",

            "Eventos totais",

            "Gerações efetivas",

            "Média de gerações por dia",

            "% Pré-Alarmes",

            "% Alarmes",

            "% Comunicação",

            "Equipamentos analisados",

            "Equipamentos P1 + P2",

            "% gerações de equipamentos P1 + P2",

            "Episódios de chattering",

            "Equipamentos com chattering",

            "Reincidências <=10 min",

            "Ocorrências abertas",

            "Ocorrências >24h",

            "Alarm Floods",

            "% gerações em períodos de Flood",

            "Hipótese de causa dominante",

            "% episódios da hipótese dominante",

            "Hipóteses com confiança Boa + Alta",

            "Redução cenário Conservador",

            "Redução cenário Base",

            "Redução cenário Potencial"
        ],

        "Resultado": [

            str(
                DATA_INICIAL
            ),

            str(
                DATA_FINAL
            ),

            f"{DIAS_ANALISADOS:,}",

            f"{TOTAL_EVENTOS:,}",

            f"{TOTAL_GERACOES:,}",

            f"{MEDIA_GERACOES_DIA:,.2f}",

            f"{PERC_PREALARMES:.2f}%",

            f"{PERC_ALARMES:.2f}%",

            f"{PERC_COMUNICACAO:.2f}%",

            f"{TOTAL_EQUIPAMENTOS:,}",

            f"{EQUIP_P1_P2:,}",

            f"{PERC_EVENTOS_P1_P2:.2f}%",

            f"{TOTAL_EPISODIOS_CHATTERING:,}",

            f"{EQUIP_CHATTERING:,}",

            f"{TOTAL_REINCIDENCIAS_10MIN:,}",

            f"{TOTAL_ABERTAS:,}",

            f"{TOTAL_LONGAS_24H:,}",

            f"{TOTAL_FLOODS:,}",

            f"{PERC_EVENTOS_FLOOD:.2f}%",

            str(
                HIPOTESE_LIDER
            ),

            f"{PERC_HIPOTESE_LIDER:.2f}%",

            (
                f"{percentual(
                    TOTAL_CONFIANCA_ALTA
                    +
                    TOTAL_CONFIANCA_BOA,
                    TOTAL_CAUSAS
                ):.2f}%"
            ),

            f"{cenario_conservador['Reducao_Total_Perc']:.2f}%",

            f"{cenario_base['Reducao_Total_Perc']:.2f}%",

            f"{cenario_potencial['Reducao_Total_Perc']:.2f}%"
        ]
    }
)


# ======================================================================
# 48. EXIBIÇÃO
# ======================================================================

print("\n" + "=" * 125)

print(
    "RESUMO EXECUTIVO"
)

print("=" * 125)


display(
    resumo_executivo
)


print("\n" + "=" * 125)

print(
    "SCORECARD DE ALARM MANAGEMENT"
)

print("=" * 125)


display(
    scorecard
)


print("\n" + "=" * 125)

print(
    "VOLUME EXPOSTO POR ALAVANCA"
)

print("=" * 125)


display(
    volume_alavancas
)


print("\n" + "=" * 125)

print(
    "CENÁRIOS DE REDUÇÃO"
)

print("=" * 125)


display(
    resumo_cenarios
)


print("\n" + "=" * 125)

print(
    "CARTEIRA DE AÇÕES - TOP 30"
)

print("=" * 125)


display(
    carteira.head(30)
)


print("\n" + "=" * 125)

print(
    "HIPÓTESES DE CAUSA"
)

print("=" * 125)


display(
    resumo_hipoteses
)


print("\n" + "=" * 125)

print(
    "ASSINATURAS RECORRENTES - TOP 30"
)

print("=" * 125)


display(
    resumo_assinaturas.head(30)
)


# ======================================================================
# 49. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 125)

print(
    "VALIDAÇÕES"
)

print("=" * 125)


soma_volume_alavancas = int(

    volume_alavancas[
        "Volume_Exposto"
    ]
    .sum()
)


print(
    f"Gerações efetivas.........................: "
    f"{TOTAL_GERACOES:,}"
)


print(
    f"Soma volume das alavancas................: "
    f"{soma_volume_alavancas:,}"
)


print(
    "\nVolume mutuamente exclusivo consistente..:",
    soma_volume_alavancas
    ==
    TOTAL_GERACOES
)


print(
    "Equipamentos únicos carteira.............:",
    carteira[
        "itemName"
    ]
    .nunique()
    ==
    len(
        carteira
    )
)


print(
    "Cenários com redução <= volume atual.....:",
    (
        resumo_cenarios[
            "Reducao_Estimada"
        ]
        <=
        TOTAL_GERACOES
    )
    .all()
)


# ======================================================================
# 50. SALVAMENTO DA CARTEIRA
# ======================================================================

carteira.to_parquet(

    ARQUIVO_CARTEIRA,

    index=False
)


print(
    f"\nCarteira criada:\n"
    f"{ARQUIVO_CARTEIRA}"
)


# ======================================================================
# 51. METODOLOGIA
# ======================================================================

metodologia = pd.DataFrame(
    {
        "Elemento": [

            "Volume Exposto",

            "Alavancas",

            "Hierarquia",

            "Chattering",

            "Reincidência",

            "Comunicação",

            "Persistência",

            "Criticidade",

            "Cenário Conservador",

            "Cenário Base",

            "Cenário Potencial",

            "Redução Estimada",

            "Importante"
        ],

        "Definicao": [

            (
                "Quantidade de gerações efetivas associadas "
                "a equipamentos pertencentes à alavanca."
            ),

            (
                "Grupos analíticos de oportunidade "
                "para atuação."
            ),

            (
                "Cada equipamento pertence a apenas uma "
                "alavanca principal para evitar dupla contagem."
            ),

            (
                "Equipamentos com >=3 episódios de "
                "chattering em 10 minutos."
            ),

            (
                "Equipamentos sem chattering prioritário "
                "e com >=10 reincidências em 10 minutos."
            ),

            (
                "Equipamentos classificados como Off-line."
            ),

            (
                "Equipamentos com ocorrências abertas "
                "ou ocorrências acima de 24 horas."
            ),

            (
                "Equipamentos P1/P2 não classificados "
                "nas alavancas anteriores."
            ),

            (
                "Hipótese de redução com percentuais "
                "mais prudentes."
            ),

            (
                "Hipótese intermediária para planejamento."
            ),

            (
                "Hipótese de maior captura das oportunidades."
            ),

            (
                "Volume Exposto × taxa assumida para "
                "cada alavanca e cenário."
            ),

            (
                "As estimativas são cenários analíticos e "
                "não representam economia garantida."
            )
        ]
    }
)


# ======================================================================
# 52. PREPARAÇÃO PARA EXCEL
# ======================================================================

def remover_timezone_dataframe(
    dataframe
):

    temp = (
        dataframe
        .copy()
    )


    for coluna in temp.columns:

        try:

            if (
                hasattr(
                    temp[
                        coluna
                    ].dtype,
                    "tz"
                )

                and

                temp[
                    coluna
                ].dtype.tz
                is not None
            ):

                temp[
                    coluna
                ] = (

                    temp[
                        coluna
                    ]
                    .dt.tz_localize(
                        None
                    )
                )

        except:

            pass


    return temp


# ======================================================================
# 53. DATAFRAMES EXCEL
# ======================================================================

resumo_executivo_excel = remover_timezone_dataframe(
    resumo_executivo
)

scorecard_excel = remover_timezone_dataframe(
    scorecard
)

volume_alavancas_excel = remover_timezone_dataframe(
    volume_alavancas
)

cenarios_detalhados_excel = remover_timezone_dataframe(
    cenarios_detalhados
)

resumo_cenarios_excel = remover_timezone_dataframe(
    resumo_cenarios
)

carteira_excel = remover_timezone_dataframe(
    carteira
)

resumo_carteira_excel = remover_timezone_dataframe(
    resumo_carteira
)

resumo_hipoteses_excel = remover_timezone_dataframe(
    resumo_hipoteses
)

resumo_assinaturas_excel = remover_timezone_dataframe(
    resumo_assinaturas
)

mensal_excel = remover_timezone_dataframe(
    mensal
)


# ======================================================================
# 54. EXPORTAÇÃO PARA EXCEL
# ======================================================================

print(
    "\nGerando relatório Excel do Módulo 11..."
)


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo_excel.to_excel(
        writer,
        sheet_name="Resumo Executivo",
        index=False
    )


    scorecard_excel.to_excel(
        writer,
        sheet_name="Scorecard",
        index=False
    )


    metodologia.to_excel(
        writer,
        sheet_name="Metodologia",
        index=False
    )


    premissas_cenarios.to_excel(
        writer,
        sheet_name="Premissas Cenarios",
        index=False
    )


    volume_alavancas_excel.to_excel(
        writer,
        sheet_name="Volume por Alavanca",
        index=False
    )


    resumo_cenarios_excel.to_excel(
        writer,
        sheet_name="Cenarios",
        index=False
    )


    cenarios_detalhados_excel.to_excel(
        writer,
        sheet_name="Cenarios Detalhados",
        index=False
    )


    resumo_carteira_excel.to_excel(
        writer,
        sheet_name="Resumo Carteira",
        index=False
    )


    carteira_excel.to_excel(
        writer,
        sheet_name="Carteira Acoes",
        index=False
    )


    carteira_excel.head(
        100
    ).to_excel(
        writer,
        sheet_name="Top100 Acoes",
        index=False
    )


    plano_acao.to_excel(
        writer,
        sheet_name="Plano Macro",
        index=False
    )


    resumo_hipoteses_excel.to_excel(
        writer,
        sheet_name="Hipoteses Causa",
        index=False
    )


    resumo_assinaturas_excel.to_excel(
        writer,
        sheet_name="Assinaturas",
        index=False
    )


    mensal_excel.to_excel(
        writer,
        sheet_name="Mensal",
        index=False
    )


    matriz_mes_alavanca.to_excel(
        writer,
        sheet_name="Mes x Alavanca"
    )


# ======================================================================
# 55. FORMATAÇÃO DO EXCEL
# ======================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    Alignment
)

from openpyxl.utils import (
    get_column_letter
)


wb = load_workbook(
    ARQUIVO_RELATORIO
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1

        and

        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(

            horizontal="center",

            vertical="center",

            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0


        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                valor = (

                    ""

                    if cell.value is None

                    else str(
                        cell.value
                    )
                )


                max_length = max(

                    max_length,

                    len(
                        valor
                    )
                )


            except:

                pass


        largura = min(

            max(
                max_length + 2,
                12
            ),

            55
        )


        ws.column_dimensions[
            letra
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ======================================================================
# 56. FINALIZAÇÃO
# ======================================================================

print("\n" + "=" * 125)

print(
    "MÓDULO 11 CONCLUÍDO COM SUCESSO"
)

print("=" * 125)


print(
    "\nArquivos gerados:"
)


print(
    f"\n1. {ARQUIVO_RELATORIO}"
)


print(
    f"2. {ARQUIVO_CARTEIRA}"
)


print(
    """
O relatório contém:

1. Resumo Executivo

2. Scorecard de Alarm Management

3. Metodologia

4. Premissas dos cenários

5. Volume exposto por alavanca

6. Cenário Conservador

7. Cenário Base

8. Cenário Potencial

9. Cenários detalhados

10. Resumo da carteira de ações

11. Carteira completa

12. Top 100 ações

13. Plano Macro

14. Hipóteses de causa

15. Assinaturas recorrentes

16. Evolução mensal

17. Matriz Mês x Alavanca
"""
)


# ======================================================================
# 57. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar também a carteira em Parquet:"
)


print(
    f'files.download("{ARQUIVO_CARTEIRA}")'
)

MÓDULO 11 - KPIs DE ALARM MANAGEMENT, OPORTUNIDADES DE REDUÇÃO E PLANO DE AÇÃO

Carregando resultados dos módulos anteriores...
Base de eventos..............: 1,413,882
Ranking Módulo 5............: 8,691
Ocorrências Módulo 6........: 438,482
Ranking Final Módulo 7......: 5,826
Episódios Flood..............: 1,843
Assinaturas / Causa..........: 1,843

RESUMO EXECUTIVO


,Indicador,Resultado
0,Período inicial,2026-01-01 00:00:29
1,Período final,2026-07-31 23:58:32
2,Dias analisados,211
3,Eventos totais,"1,413,882"
4,Gerações efetivas,"637,566"
5,Média de gerações por dia,"3,021.64"
6,% Pré-Alarmes,46.54%
7,% Alarmes,37.69%
8,% Comunicação,15.77%
9,Equipamentos analisados,"5,826"



SCORECARD DE ALARM MANAGEMENT


,Pilar,Indicador,Valor,Unidade
0,Volume,Eventos totais,"1,413,882.00",registros
1,Volume,Gerações efetivas,"637,566.00",eventos
2,Volume,Média de gerações por dia,"3,021.64",eventos/dia
3,Composição,% Alarmes,37.69,%
4,Composição,% Pré-Alarmes,46.54,%
5,Composição,% Comunicação,15.77,%
6,Qualidade,Episódios de chattering,"24,377.00",episódios
7,Qualidade,Reincidências até 10 min,"114,111.00",eventos
8,Persistência,Ocorrências abertas,"4,768.00",ocorrências
9,Persistência,Ocorrências >24h,"24,168.00",ocorrências



VOLUME EXPOSTO POR ALAVANCA


...

,Assinatura_Cascata,Episodios,Eventos,Confianca_Media,Indice_Flood_Medio,Primeira_Ocorrencia,Ultima_Ocorrencia,Participacao_Episodios_Perc,Participacao_Acumulada_Perc,Classe_Pareto
32,HVAC | Temperatura | CEA | Pré-Alarme | Baixa ...,422,13220,82.33,28.72,2026-01-16 18:40:00,2026-07-31 17:40:00,22.90,22.90,A
34,HVAC | Temperatura | CEA | Pré-Alarme | Moderada,322,83559,82.27,63.41,2026-01-22 09:10:00,2026-07-31 18:20:00,17.47,40.37,A
36,HVAC | Temperatura | CEA | Pré-Alarme | Rápida,201,91473,76.67,71.20,2026-01-06 05:50:00,2026-07-31 06:00:00,10.91,51.28,A
23,HVAC | Temperatura | CEA | Alarme | Baixa Expa...,94,3370,77.37,34.32,2026-01-01 09:20:00,2026-07-30 08:30:00,5.10,56.38,A
12,HVAC | Falha de Comando | CEA | Alarme | Baixa...,94,3183,66.51,41.07,2026-01-06 20:00:00,2026-07-31 20:00:00,5.10,61.48,A
35,HVAC | Temperatura | CEA | Pré-Alarme | Muito ...,83,44870,72.38,76.81,2026-01-28 06:00:00,2026-07-24 06:00:00,4.50,65.98,A
25,HVAC | Temperatura | CEA | Alarme | Moderada,65,25587,83.59,69.07,2026-01-01 09:40:00,2026-07-27 06:30:00,3.53,69.51,A
28,HVAC | Temperatura | CEA | Alarme | Rápida,55,34783,78.89,71.58,2026-01-01 06:30:00,2026-07-13 19:50:00,2.98,72.49,A
101,Sistema | Off-line | TOS | Comunicação | Baixa...,46,1893,74.94,25.71,2026-02-04 22:50:00,2026-07-18 23:00:00,2.50,74.99,A
60,HVAC | Temperatura | TOS | Pré-Alarme | Baixa ...,28,859,83.23,20.58,2026-02-17 03:10:00,2026-07-28 09:00:00,1.52,76.51,A



VALIDAÇÕES
Gerações efetivas.........................: 637,566
Soma volume das alavancas................: 637,566

Volume mutuamente exclusivo consistente..: True
Equipamentos únicos carteira.............: True
Cenários com redução <= volume atual.....: True

Carteira criada:
Carteira_Acoes_Alarm_Management_Metasys_2026.parquet

Gerando relatório Excel do Módulo 11...

MÓDULO 11 CONCLUÍDO COM SUCESSO

Arquivos gerados:

1. Relatorio_Modulo11_Alarm_Management_Metasys.xlsx
2. Carteira_Acoes_Alarm_Management_Metasys_2026.parquet

O relatório contém:

1. Resumo Executivo

2. Scorecard de Alarm Management

3. Metodologia

4. Premissas dos cenários

5. Volume exposto por alavanca

6. Cenário Conservador

7. Cenário Base

8. Cenário Potencial

9. Cenários detalhados

10. Resumo da carteira de ações

11. Carteira completa

12. Top 100 ações

13. Plano Macro

14. Hipóteses de causa

15. Assinaturas recorrentes

16. Evolução mensal

17. Matriz Mês x Alavanca


Iniciando download do relatório Ex

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também a carteira em Parquet:
files.download("Carteira_Acoes_Alarm_Management_Metasys_2026.parquet")


In [22]:
# ======================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 12
# DASHBOARD EXECUTIVO, HEALTH SCORE E RELATÓRIO GERENCIAL
#
# OBJETIVOS
# ----------------------------------------------------------------------
# 1. Consolidar os resultados dos Módulos 1 a 11
#
# 2. Criar Dashboard Executivo profissional
#
# 3. Criar:
#       - Alarm Management Health Score - AMHS V1
#       - Scorecard Executivo
#       - Pareto das oportunidades
#       - Cenários de redução
#       - Top equipamentos
#       - Hipóteses de causa
#       - Assinaturas recorrentes
#       - Alarm Flood
#       - Tendência mensal
#       - Plano de ação
#
# 4. Criar diagnóstico executivo automático
#
# 5. Gerar estrutura pronta para apresentação à equipe
#
#
# IMPORTANTE
# ----------------------------------------------------------------------
# AMHS V1 é um indicador ANALÍTICO INTERNO do projeto.
#
# Não representa índice oficial Johnson Controls / Metasys
# nem classificação normativa.
#
# Os pesos e critérios podem ser recalibrados futuramente.
# ======================================================================


# ======================================================================
# 1. BIBLIOTECAS
# ======================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Border,
    Side,
    Alignment
)

from openpyxl.chart import (
    BarChart,
    LineChart,
    DoughnutChart,
    Reference
)

from openpyxl.chart.label import DataLabelList

from openpyxl.utils import get_column_letter


pd.set_option(
    "display.max_columns",
    350
)

pd.set_option(
    "display.max_rows",
    250
)

pd.set_option(
    "display.width",
    320
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ======================================================================
# 2. ARQUIVOS DE ENTRADA
# ======================================================================

ARQUIVO_MODULO11 = (
    "Relatorio_Modulo11_Alarm_Management_Metasys.xlsx"
)

ARQUIVO_CARTEIRA = (
    "Carteira_Acoes_Alarm_Management_Metasys_2026.parquet"
)

ARQUIVO_CAUSAS = (
    "Assinaturas_Cascatas_Causa_Raiz_Metasys_2026.parquet"
)

ARQUIVO_FLOODS = (
    "Episodios_Alarm_Flood_Metasys_2026.parquet"
)

ARQUIVO_RANKING_FINAL = (
    "Ranking_Final_Priorizacao_Alarmes_Metasys_2026.parquet"
)


# ======================================================================
# 3. ARQUIVO DE SAÍDA
# ======================================================================

ARQUIVO_DASHBOARD = (
    "Dashboard_Executivo_Alarm_Management_Metasys_2026.xlsx"
)


print("=" * 125)

print(
    "MÓDULO 12 - DASHBOARD EXECUTIVO, "
    "HEALTH SCORE E RELATÓRIO GERENCIAL"
)

print("=" * 125)


# ======================================================================
# 4. PARÂMETROS DO HEALTH SCORE
#
# Os pesos somam 100%.
#
# O score representa a SAÚDE relativa do sistema:
#
# 100 = melhor condição
#   0 = pior condição
#
# ======================================================================

PESO_QUALIDADE = 0.25
PESO_PERSISTENCIA = 0.20
PESO_FLOOD = 0.20
PESO_CRITICIDADE = 0.20
PESO_DIAGNOSTICO = 0.15


# ======================================================================
# 5. FAIXAS DO HEALTH SCORE
# ======================================================================

FAIXAS_AMHS = {

    "Muito Saudável": 80,
    "Saudável": 65,
    "Atenção": 50,
    "Crítico": 35,
    "Muito Crítico": 0
}


# ======================================================================
# 6. VERIFICAÇÃO / UPLOAD
# ======================================================================

arquivos_necessarios = [

    ARQUIVO_MODULO11,
    ARQUIVO_CARTEIRA,
    ARQUIVO_CAUSAS,
    ARQUIVO_FLOODS,
    ARQUIVO_RANKING_FINAL
]


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    print(
        "\nArquivos não encontrados:"
    )

    for arquivo in faltantes:

        print(
            f" - {arquivo}"
        )


    print(
        "\nFaça o upload dos arquivos solicitados."
    )


    from google.colab import files

    uploaded = files.upload()


faltantes = [

    arquivo

    for arquivo in arquivos_necessarios

    if not Path(
        arquivo
    ).exists()
]


if faltantes:

    raise FileNotFoundError(

        "Os seguintes arquivos continuam ausentes:\n\n"

        +

        "\n".join(
            faltantes
        )
    )


# ======================================================================
# 7. CARREGAMENTO DOS DADOS
# ======================================================================

print(
    "\nCarregando dados consolidados..."
)


# ----------------------------------------------------------------------
# Excel Módulo 11
# ----------------------------------------------------------------------

resumo_exec = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Resumo Executivo"
)


scorecard = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Scorecard"
)


volume_alavancas = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Volume por Alavanca"
)


cenarios = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Cenarios"
)


mensal = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Mensal"
)


plano_macro = pd.read_excel(

    ARQUIVO_MODULO11,

    sheet_name="Plano Macro"
)


# ----------------------------------------------------------------------
# Parquets
# ----------------------------------------------------------------------

carteira = pd.read_parquet(
    ARQUIVO_CARTEIRA
)


causas = pd.read_parquet(
    ARQUIVO_CAUSAS
)


floods = pd.read_parquet(
    ARQUIVO_FLOODS
)


ranking_final = pd.read_parquet(
    ARQUIVO_RANKING_FINAL
)


print(
    f"Carteira de ações...............: {len(carteira):,}"
)

print(
    f"Assinaturas de causa............: {len(causas):,}"
)

print(
    f"Episódios Flood.................: {len(floods):,}"
)

print(
    f"Equipamentos ranking final......: {len(ranking_final):,}"
)


# ======================================================================
# 8. FUNÇÕES AUXILIARES - VERSÃO CORRIGIDA
# ======================================================================

def converter_percentual(valor):
    """
    Converte percentuais de forma robusta.

    Exemplos aceitos:
    '74,09%'  -> 74.09
    '74.09%'  -> 74.09
    '92,78'   -> 92.78
    '92.78'   -> 92.78
    92.78     -> 92.78
    0.9278    -> 92.78  (quando recebido como percentual Excel)
    """

    if pd.isna(valor):
        return 0.0

    # --------------------------------------------------------------
    # Valor numérico
    # --------------------------------------------------------------

    if isinstance(
        valor,
        (
            int,
            float,
            np.integer,
            np.floating
        )
    ):

        numero = float(
            valor
        )

        # Excel pode armazenar 92,78% como 0.9278
        if (
            0 <= numero <= 1
        ):

            return (
                numero
                * 100
            )

        return numero


    # --------------------------------------------------------------
    # Valor textual
    # --------------------------------------------------------------

    texto = (
        str(
            valor
        )
        .strip()
        .replace(
            "%",
            ""
        )
        .replace(
            " ",
            ""
        )
    )


    if (
        texto == ""
        or
        texto.lower()
        in [
            "nan",
            "none"
        ]
    ):

        return 0.0


    # --------------------------------------------------------------
    # Casos contendo vírgula E ponto
    # --------------------------------------------------------------

    if (
        ","
        in texto

        and

        "."
        in texto
    ):

        # Exemplo brasileiro:
        # 1.234,56
        if (
            texto.rfind(
                ","
            )
            >
            texto.rfind(
                "."
            )
        ):

            texto = (
                texto
                .replace(
                    ".",
                    ""
                )
                .replace(
                    ",",
                    "."
                )
            )


        # Exemplo internacional:
        # 1,234.56
        else:

            texto = (
                texto
                .replace(
                    ",",
                    ""
                )
            )


    # --------------------------------------------------------------
    # Somente vírgula
    # --------------------------------------------------------------

    elif (
        ","
        in texto
    ):

        texto = (
            texto
            .replace(
                ",",
                "."
            )
        )


    try:

        numero = float(
            texto
        )


        # Segurança para percentual Excel textual
        if (
            0 <= numero <= 1
        ):

            numero *= 100


        return numero


    except Exception:

        return 0.0


def obter_indicador(
    dataframe,
    indicador
):

    linha = (

        dataframe.loc[
            dataframe[
                "Indicador"
            ]
            ==
            indicador
        ]
    )


    if len(
        linha
    ) == 0:

        return None


    return linha.iloc[0][
        "Resultado"
    ]


def percentual(
    numerador,
    denominador
):

    if (
        denominador is None
        or
        pd.isna(
            denominador
        )
        or
        denominador == 0
    ):

        return 0.0


    return (

        numerador
        /
        denominador
        *
        100
    )


# ======================================================================
# 9. EXTRAÇÃO DOS PRINCIPAIS INDICADORES
# ======================================================================

TOTAL_EVENTOS = int(

    str(
        obter_indicador(
            resumo_exec,
            "Eventos totais"
        )
    )
    .replace(
        ",",
        ""
    )
)


TOTAL_GERACOES = int(

    str(
        obter_indicador(
            resumo_exec,
            "Gerações efetivas"
        )
    )
    .replace(
        ",",
        ""
    )
)


MEDIA_GERACOES_DIA = float(

    str(
        obter_indicador(
            resumo_exec,
            "Média de gerações por dia"
        )
    )
    .replace(
        ",",
        ""
    )
)


PERC_PREALARMES = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% Pré-Alarmes"
    )
)


PERC_ALARMES = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% Alarmes"
    )
)


PERC_COMUNICACAO = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% Comunicação"
    )
)


PERC_P1_P2 = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% gerações de equipamentos P1 + P2"
    )
)


PERC_FLOOD = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% gerações em períodos de Flood"
    )
)


PERC_CONFIANCA = converter_percentual(

    obter_indicador(
        resumo_exec,
        "Hipóteses com confiança Boa + Alta"
    )
)


HIPOTESE_DOMINANTE = str(

    obter_indicador(
        resumo_exec,
        "Hipótese de causa dominante"
    )
)


PERC_HIPOTESE_DOMINANTE = converter_percentual(

    obter_indicador(
        resumo_exec,
        "% episódios da hipótese dominante"
    )
)


# ======================================================================
# 10. VOLUME DAS ALAVANCAS
# ======================================================================

def obter_percentual_alavanca(
    nome
):

    linha = (

        volume_alavancas.loc[
            volume_alavancas[
                "Alavanca_Principal"
            ]
            ==
            nome
        ]
    )


    if len(
        linha
    ) == 0:

        return 0.0


    valor = (

        linha.iloc[0][
            "Percentual_Geracoes"
        ]
    )


    return converter_percentual(
        valor
    )


PERC_CHATTERING = obter_percentual_alavanca(
    "Racionalização - Chattering"
)


PERC_REINCIDENCIA = obter_percentual_alavanca(
    "Racionalização - Reincidência"
)


PERC_PERSISTENCIA = obter_percentual_alavanca(
    "Persistência / Normalização"
)


PERC_AUTOMACAO = obter_percentual_alavanca(
    "Comunicação / Automação"
)


# ======================================================================
# 11. VALIDAÇÃO DOS PERCENTUAIS DE ENTRADA
# ======================================================================

print("\n" + "=" * 125)

print(
    "VALIDAÇÃO DOS PERCENTUAIS DO DASHBOARD"
)

print("=" * 125)


percentuais_validacao = pd.DataFrame(
    {
        "Indicador": [

            "Pré-Alarmes",

            "Alarmes",

            "Comunicação",

            "P1 + P2",

            "Alarm Flood",

            "Confiança Diagnóstica",

            "Chattering",

            "Reincidência",

            "Persistência",

            "Comunicação / Automação"
        ],

        "Percentual": [

            PERC_PREALARMES,

            PERC_ALARMES,

            PERC_COMUNICACAO,

            PERC_P1_P2,

            PERC_FLOOD,

            PERC_CONFIANCA,

            PERC_CHATTERING,

            PERC_REINCIDENCIA,

            PERC_PERSISTENCIA,

            PERC_AUTOMACAO
        ]
    }
)


display(
    percentuais_validacao
)


# Todos devem estar entre 0 e 100
validacao_percentuais = (

    percentuais_validacao[
        "Percentual"
    ]
    .between(
        0,
        100
    )
    .all()
)


print(
    "\nPercentuais entre 0 e 100...............:",
    validacao_percentuais
)


if not validacao_percentuais:

    raise ValueError(
        "Foram encontrados percentuais fora da faixa 0–100."
    )


# ======================================================================
# 12. COMPONENTE 1 - QUALIDADE / REPETITIVIDADE
#
# Chattering + Reincidência
#
# Como as alavancas do Módulo 11 são mutuamente exclusivas,
# os percentuais podem ser somados sem dupla contagem.
# ======================================================================

EXPOSICAO_QUALIDADE = min(

    PERC_CHATTERING
    +
    PERC_REINCIDENCIA,

    100
)


SCORE_QUALIDADE = max(

    100
    -
    EXPOSICAO_QUALIDADE,

    0
)


# ======================================================================
# 13. COMPONENTE 2 - PERSISTÊNCIA
# ======================================================================

SCORE_PERSISTENCIA = max(

    100
    -
    PERC_PERSISTENCIA,

    0
)


# ======================================================================
# 14. COMPONENTE 3 - ALARM FLOOD
# ======================================================================

SCORE_FLOOD = max(

    100
    -
    PERC_FLOOD,

    0
)


# ======================================================================
# 15. COMPONENTE 4 - CRITICIDADE
# ======================================================================

SCORE_CRITICIDADE = max(

    100
    -
    PERC_P1_P2,

    0
)


# ======================================================================
# 16. COMPONENTE 5 - CAPACIDADE DIAGNÓSTICA
#
# Neste componente:
#
# quanto maior a confiança, MELHOR o score.
# ======================================================================

SCORE_DIAGNOSTICO = min(

    max(
        PERC_CONFIANCA,
        0
    ),

    100
)


# ======================================================================
# 17. HEALTH SCORE - AMHS V1
# ======================================================================

AMHS = (

    SCORE_QUALIDADE
    *
    PESO_QUALIDADE

    +

    SCORE_PERSISTENCIA
    *
    PESO_PERSISTENCIA

    +

    SCORE_FLOOD
    *
    PESO_FLOOD

    +

    SCORE_CRITICIDADE
    *
    PESO_CRITICIDADE

    +

    SCORE_DIAGNOSTICO
    *
    PESO_DIAGNOSTICO
)


# Segurança matemática
AMHS = min(
    max(
        AMHS,
        0
    ),
    100
)


# ======================================================================
# 18. CLASSIFICAÇÃO DO AMHS
# ======================================================================

if AMHS >= 80:

    CLASSE_AMHS = (
        "Muito Saudável"
    )


elif AMHS >= 65:

    CLASSE_AMHS = (
        "Saudável"
    )


elif AMHS >= 50:

    CLASSE_AMHS = (
        "Atenção"
    )


elif AMHS >= 35:

    CLASSE_AMHS = (
        "Crítico"
    )


else:

    CLASSE_AMHS = (
        "Muito Crítico"
    )


# ======================================================================
# 19. TABELA DE COMPOSIÇÃO DO HEALTH SCORE
# ======================================================================

health_score = pd.DataFrame(
    {
        "Dimensao": [

            "Qualidade / Repetitividade",

            "Persistência",

            "Alarm Flood",

            "Criticidade",

            "Capacidade Diagnóstica"
        ],

        "Indicador_Base": [

            "% volume Chattering + Reincidência",

            "% volume Persistência / Normalização",

            "% gerações em períodos de Flood",

            "% gerações de equipamentos P1 + P2",

            "% hipóteses com confiança Boa + Alta"
        ],

        "Valor_Indicador_Perc": [

            EXPOSICAO_QUALIDADE,

            PERC_PERSISTENCIA,

            PERC_FLOOD,

            PERC_P1_P2,

            PERC_CONFIANCA
        ],

        "Score_0_100": [

            SCORE_QUALIDADE,

            SCORE_PERSISTENCIA,

            SCORE_FLOOD,

            SCORE_CRITICIDADE,

            SCORE_DIAGNOSTICO
        ],

        "Peso_Perc": [

            PESO_QUALIDADE * 100,

            PESO_PERSISTENCIA * 100,

            PESO_FLOOD * 100,

            PESO_CRITICIDADE * 100,

            PESO_DIAGNOSTICO * 100
        ]
    }
)


health_score[
    "Contribuicao_AMHS"
] = (

    health_score[
        "Score_0_100"
    ]

    *

    health_score[
        "Peso_Perc"
    ]

    /

    100
)


# ======================================================================
# 20. VALIDAÇÃO FINAL DO AMHS
# ======================================================================

print("\n" + "=" * 125)

print(
    "COMPOSIÇÃO DO AMHS CORRIGIDO"
)

print("=" * 125)


display(
    health_score
)


print(
    f"\nExposição Qualidade.....................: "
    f"{EXPOSICAO_QUALIDADE:.2f}%"
)


print(
    f"Score Qualidade.........................: "
    f"{SCORE_QUALIDADE:.2f}"
)


print(
    f"Score Persistência......................: "
    f"{SCORE_PERSISTENCIA:.2f}"
)


print(
    f"Score Alarm Flood.......................: "
    f"{SCORE_FLOOD:.2f}"
)


print(
    f"Score Criticidade.......................: "
    f"{SCORE_CRITICIDADE:.2f}"
)


print(
    f"Score Capacidade Diagnóstica............: "
    f"{SCORE_DIAGNOSTICO:.2f}"
)


print(
    f"\nAMHS V1.................................: "
    f"{AMHS:.2f}/100"
)


print(
    f"Classificação...........................: "
    f"{CLASSE_AMHS}"
)


# ======================================================================
# 21. VALIDAÇÕES DE COERÊNCIA
# ======================================================================

assert (
    0
    <=
    PERC_FLOOD
    <=
    100
), "Alarm Flood fora da faixa 0–100."


assert (
    0
    <=
    PERC_P1_P2
    <=
    100
), "P1/P2 fora da faixa 0–100."


assert (
    0
    <=
    PERC_CONFIANCA
    <=
    100
), "Confiança fora da faixa 0–100."


assert (
    0
    <=
    AMHS
    <=
    100
), "AMHS fora da faixa 0–100."


print(
    "\nValidações críticas concluídas com sucesso."
)


# ======================================================================
# 19. CENÁRIOS
# ======================================================================

cenario_conservador = (

    cenarios.loc[
        cenarios[
            "Cenario"
        ]
        ==
        "Conservador"
    ]
    .iloc[0]
)


cenario_base = (

    cenarios.loc[
        cenarios[
            "Cenario"
        ]
        ==
        "Base"
    ]
    .iloc[0]
)


cenario_potencial = (

    cenarios.loc[
        cenarios[
            "Cenario"
        ]
        ==
        "Potencial"
    ]
    .iloc[0]
)


# ======================================================================
# 20. TOP AÇÕES
# ======================================================================

top_acoes = (

    carteira
    .sort_values(
        "Ranking_Acao"
    )
    .head(50)
    .copy()
)


# ======================================================================
# 21. TOP CAUSAS
# ======================================================================

resumo_causas = (

    causas
    .groupby(
        "Hipotese_Causa_Raiz",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Causa_Medio=(
            "Indice_Prioridade_Causa",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


resumo_causas[
    "Participacao_Episodios_Perc"
] = (

    resumo_causas[
        "Episodios"
    ]

    /

    resumo_causas[
        "Episodios"
    ].sum()

    * 100
)


# ======================================================================
# 22. ASSINATURAS
# ======================================================================

resumo_assinaturas = (

    causas
    .groupby(
        "Assinatura_Cascata",
        observed=True,
        dropna=False
    )
    .agg(

        Episodios=(
            "ID_Episodio",
            "size"
        ),

        Eventos=(
            "Eventos",
            "sum"
        ),

        Confianca_Media=(
            "Confianca_Hipotese",
            "mean"
        ),

        Indice_Flood_Medio=(
            "Indice_Alarm_Flood",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        "Episodios",
        ascending=False
    )
)


resumo_assinaturas[
    "Participacao_Perc"
] = (

    resumo_assinaturas[
        "Episodios"
    ]

    /

    resumo_assinaturas[
        "Episodios"
    ].sum()

    * 100
)


resumo_assinaturas[
    "Participacao_Acumulada_Perc"
] = (

    resumo_assinaturas[
        "Participacao_Perc"
    ]
    .cumsum()
)


# ======================================================================
# 23. RESUMO DE FLOODS
# ======================================================================

TOTAL_FLOODS = len(
    floods
)


if (
    "Classe_Alarm_Flood"
    in floods.columns
):

    resumo_floods = (

        floods[
            "Classe_Alarm_Flood"
        ]
        .value_counts()
        .reset_index()
    )


    resumo_floods.columns = [

        "Classe_Flood",
        "Episodios"
    ]


else:

    resumo_floods = pd.DataFrame(
        {
            "Classe_Flood": [],
            "Episodios": []
        }
    )


# ======================================================================
# 24. DIAGNÓSTICO AUTOMÁTICO
# ======================================================================

alavanca_lider = (

    volume_alavancas
    .sort_values(
        "Percentual_Geracoes",
        ascending=False
    )
    .iloc[0]
)


assinatura_lider = (

    resumo_assinaturas
    .iloc[0]
)


equipamento_lider = (

    top_acoes
    .iloc[0]
)


diagnosticos = []


diagnosticos.append(

    (
        f"1. A principal alavanca de atuação é "
        f"'{alavanca_lider['Alavanca_Principal']}', "
        f"responsável por "
        f"{alavanca_lider['Percentual_Geracoes']:.2f}% "
        f"das gerações efetivas."
    )
)


diagnosticos.append(

    (
        f"2. Alarm Flood apresenta elevada participação: "
        f"{PERC_FLOOD:.2f}% das gerações ocorreram "
        f"durante períodos classificados como flood."
    )
)


diagnosticos.append(

    (
        f"3. A hipótese de causa dominante é "
        f"'{HIPOTESE_DOMINANTE}', presente em "
        f"{PERC_HIPOTESE_DOMINANTE:.2f}% dos episódios analisados."
    )
)


diagnosticos.append(

    (
        f"4. A assinatura recorrente mais frequente é "
        f"'{assinatura_lider['Assinatura_Cascata']}', "
        f"observada em {int(assinatura_lider['Episodios']):,} episódios."
    )
)


diagnosticos.append(

    (
        f"5. O equipamento com maior prioridade na carteira é "
        f"'{equipamento_lider['itemName']}', "
        f"classificado como "
        f"'{equipamento_lider['Prioridade_Acao']}'."
    )
)


diagnosticos.append(

    (
        f"6. O cenário Base estima potencial analítico de redução "
        f"de {cenario_base['Reducao_Total_Perc']:.2f}% das gerações, "
        f"reduzindo a média diária para aproximadamente "
        f"{cenario_base['Media_Diaria_Pos_Acao']:,.0f} eventos/dia."
    )
)


diagnosticos.append(

    (
        f"7. O Alarm Management Health Score atual é "
        f"{AMHS:.2f}/100, classificado como '{CLASSE_AMHS}'."
    )
)


diagnostico_df = pd.DataFrame(
    {
        "Diagnostico": diagnosticos
    }
)


# ======================================================================
# 25. RECOMENDAÇÕES EXECUTIVAS
# ======================================================================

recomendacoes = pd.DataFrame(
    {
        "Prioridade": [

            1,
            2,
            3,
            4,
            5,
            6
        ],

        "Frente": [

            "Chattering",

            "HVAC / Temperatura",

            "Persistência",

            "Alarm Flood",

            "Comunicação / Automação",

            "Governança de Alarmes"
        ],

        "Recomendacao": [

            (
                "Atuar inicialmente nos equipamentos A1/A2 com "
                "maior quantidade de episódios de chattering, "
                "revisando deadband, temporização, histerese, "
                "sensores e estabilidade da malha."
            ),

            (
                "Executar campanha específica para alarmes e "
                "pré-alarmes de temperatura, priorizando CEA e "
                "equipamentos presentes nas assinaturas recorrentes."
            ),

            (
                "Revisar ocorrências ainda abertas e acima de 24h, "
                "diferenciando falha física persistente de falha "
                "de normalização ou configuração."
            ),

            (
                "Analisar os maiores floods usando First-Out, "
                "precursor e assinatura para identificar causas comuns."
            ),

            (
                "Mapear controladores, NAE, redes BACnet, switches, "
                "alimentações e equipamentos que compartilham "
                "infraestrutura de comunicação."
            ),

            (
                "Criar rotina mensal de acompanhamento com os mesmos "
                "KPIs e recalcular o AMHS após cada ciclo de melhorias."
            )
        ],

        "Indicador_Acompanhamento": [

            "Episódios de chattering",

            "Eventos HVAC / Temperatura",

            "Ocorrências abertas e >24h",

            "% eventos em Alarm Flood",

            "Eventos Off-line",

            "AMHS e média diária"
        ]
    }
)


# ======================================================================
# 26. RESUMO EXECUTIVO PARA O DASHBOARD
# ======================================================================

dashboard_kpis = pd.DataFrame(
    {
        "Indicador": [

            "Eventos Totais",

            "Gerações Efetivas",

            "Média / Dia",

            "Alarm Flood",

            "P1 + P2",

            "AMHS",

            "Cenário Base",

            "Confiança Diagnóstica"
        ],

        "Valor": [

            TOTAL_EVENTOS,

            TOTAL_GERACOES,

            MEDIA_GERACOES_DIA,

            PERC_FLOOD,

            PERC_P1_P2,

            AMHS,

            float(
                cenario_base[
                    "Reducao_Total_Perc"
                ]
            ),

            PERC_CONFIANCA
        ],

        "Unidade": [

            "registros",

            "eventos",

            "eventos/dia",

            "%",

            "%",

            "/100",

            "% redução",

            "%"
        ]
    }
)


# ======================================================================
# 27. CRIAÇÃO DO EXCEL
# ======================================================================

print(
    "\nConstruindo Dashboard Executivo..."
)


with pd.ExcelWriter(

    ARQUIVO_DASHBOARD,

    engine="openpyxl"

) as writer:


    # ------------------------------------------------------------------
    # Dashboard será formatado posteriormente
    # ------------------------------------------------------------------

    pd.DataFrame(
        {
            "Dashboard":
                [
                    "Alarm Management Dashboard"
                ]
        }
    ).to_excel(

        writer,

        sheet_name="Dashboard Executivo",

        index=False
    )


    health_score.to_excel(

        writer,

        sheet_name="Health Score",

        index=False
    )


    dashboard_kpis.to_excel(

        writer,

        sheet_name="KPIs",

        index=False
    )


    volume_alavancas.to_excel(

        writer,

        sheet_name="Oportunidades",

        index=False
    )


    cenarios.to_excel(

        writer,

        sheet_name="Cenarios",

        index=False
    )


    top_acoes.to_excel(

        writer,

        sheet_name="Top Acoes",

        index=False
    )


    resumo_causas.to_excel(

        writer,

        sheet_name="Causas",

        index=False
    )


    resumo_assinaturas.head(
        100
    ).to_excel(

        writer,

        sheet_name="Assinaturas",

        index=False
    )


    resumo_floods.to_excel(

        writer,

        sheet_name="Floods",

        index=False
    )


    mensal.to_excel(

        writer,

        sheet_name="Tendencia Mensal",

        index=False
    )


    diagnostico_df.to_excel(

        writer,

        sheet_name="Diagnostico",

        index=False
    )


    recomendacoes.to_excel(

        writer,

        sheet_name="Recomendacoes",

        index=False
    )


    plano_macro.to_excel(

        writer,

        sheet_name="Plano de Acao",

        index=False
    )


# ======================================================================
# 28. CARREGAMENTO PARA FORMATAÇÃO
# ======================================================================

wb = load_workbook(
    ARQUIVO_DASHBOARD
)


# ======================================================================
# 29. PALETA VISUAL
# ======================================================================

COR_AZUL_ESCURO = "17365D"

COR_AZUL = "2F75B5"

COR_AZUL_CLARO = "D9EAF7"

COR_VERDE = "70AD47"

COR_VERDE_CLARO = "E2F0D9"

COR_AMARELO = "FFC000"

COR_AMARELO_CLARO = "FFF2CC"

COR_LARANJA = "ED7D31"

COR_VERMELHO = "C00000"

COR_VERMELHO_CLARO = "F4CCCC"

COR_CINZA = "D9E1F2"

COR_CINZA_CLARO = "F3F6F8"

COR_BRANCO = "FFFFFF"

COR_PRETO = "1F1F1F"


borda_fina = Border(

    left=Side(
        style="thin",
        color="D9D9D9"
    ),

    right=Side(
        style="thin",
        color="D9D9D9"
    ),

    top=Side(
        style="thin",
        color="D9D9D9"
    ),

    bottom=Side(
        style="thin",
        color="D9D9D9"
    )
)


# ======================================================================
# 30. FUNÇÃO DE ESTILO DAS ABAS DE DADOS
# ======================================================================

def formatar_aba_dados(
    ws
):

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.fill = PatternFill(

            "solid",

            fgColor=COR_AZUL_ESCURO
        )

        cell.font = Font(

            bold=True,

            color=COR_BRANCO
        )

        cell.alignment = Alignment(

            horizontal="center",

            vertical="center",

            wrap_text=True
        )


    for coluna_cells in ws.columns:

        max_length = 0


        letra = get_column_letter(
            coluna_cells[0].column
        )


        for cell in coluna_cells:

            try:

                texto = (

                    ""

                    if cell.value is None

                    else str(
                        cell.value
                    )
                )


                max_length = max(

                    max_length,

                    len(
                        texto
                    )
                )


            except:

                pass


        largura = min(

            max(
                max_length + 2,
                11
            ),

            48
        )


        ws.column_dimensions[
            letra
        ].width = largura


    for linha in ws.iter_rows(
        min_row=2
    ):

        for cell in linha:

            cell.alignment = Alignment(

                vertical="top",

                wrap_text=True
            )


# ======================================================================
# 31. FORMATAÇÃO DAS ABAS DE DADOS
# ======================================================================

for nome in wb.sheetnames:

    if nome != "Dashboard Executivo":

        formatar_aba_dados(
            wb[
                nome
            ]
        )


# ======================================================================
# 32. DASHBOARD EXECUTIVO
# ======================================================================

ws = wb[
    "Dashboard Executivo"
]


# Limpa conteúdo inicial
for row in ws.iter_rows():

    for cell in row:

        cell.value = None


# Remove gridlines
ws.sheet_view.showGridLines = False


# ======================================================================
# 33. TÍTULO
# ======================================================================

ws.merge_cells(
    "A1:N2"
)


ws[
    "A1"
] = (
    "ALARM MANAGEMENT DASHBOARD – METASYS 2026"
)


ws[
    "A1"
].fill = PatternFill(

    "solid",

    fgColor=COR_AZUL_ESCURO
)


ws[
    "A1"
].font = Font(

    bold=True,

    color=COR_BRANCO,

    size=20
)


ws[
    "A1"
].alignment = Alignment(

    horizontal="center",

    vertical="center"
)


# ======================================================================
# 34. SUBTÍTULO
# ======================================================================

ws.merge_cells(
    "A3:N3"
)


ws[
    "A3"
] = (
    "Período: Janeiro a Julho de 2026 | "
    "Dashboard Executivo de Gestão e Racionalização de Alarmes"
)


ws[
    "A3"
].font = Font(

    italic=True,

    color=COR_AZUL_ESCURO,

    size=11
)


ws[
    "A3"
].alignment = Alignment(

    horizontal="center"
)


# ======================================================================
# 35. FUNÇÃO DE CARD
# ======================================================================

def criar_card(
    ws,
    intervalo,
    titulo,
    valor,
    subtitulo="",
    cor=COR_AZUL
):

    ws.merge_cells(
        intervalo
    )


    celula = (

        intervalo
        .split(
            ":"
        )[0]
    )


    ws[
        celula
    ] = (

        f"{titulo}\n"
        f"{valor}\n"
        f"{subtitulo}"
    )


    ws[
        celula
    ].fill = PatternFill(

        "solid",

        fgColor=cor
    )


    ws[
        celula
    ].font = Font(

        bold=True,

        color=COR_BRANCO,

        size=12
    )


    ws[
        celula
    ].alignment = Alignment(

        horizontal="center",

        vertical="center",

        wrap_text=True
    )


# ======================================================================
# 36. CARDS PRINCIPAIS
# ======================================================================

criar_card(

    ws,

    "A5:C8",

    "GERAÇÕES EFETIVAS",

    f"{TOTAL_GERACOES:,.0f}",

    f"{MEDIA_GERACOES_DIA:,.0f}/dia",

    COR_AZUL
)


criar_card(

    ws,

    "D5:F8",

    "ALARM FLOOD",

    f"{PERC_FLOOD:.2f}%",

    f"{TOTAL_FLOODS:,} episódios",

    COR_VERMELHO
)


criar_card(

    ws,

    "G5:I8",

    "EQUIPAMENTOS P1 + P2",

    f"{PERC_P1_P2:.2f}%",

    "das gerações",

    COR_LARANJA
)


cor_amhs = (

    COR_VERDE

    if AMHS >= 65

    else

    COR_AMARELO

    if AMHS >= 50

    else

    COR_VERMELHO
)


criar_card(

    ws,

    "J5:L8",

    "AMHS V1",

    f"{AMHS:.1f}/100",

    CLASSE_AMHS,

    cor_amhs
)


criar_card(

    ws,

    "M5:N8",

    "CENÁRIO BASE",

    f"-{cenario_base['Reducao_Total_Perc']:.2f}%",

    "potencial analítico",

    COR_VERDE
)


# ======================================================================
# 37. ÁREA DE DIAGNÓSTICO EXECUTIVO
# ======================================================================

ws.merge_cells(
    "A10:N10"
)


ws[
    "A10"
] = (
    "DIAGNÓSTICO EXECUTIVO"
)


ws[
    "A10"
].fill = PatternFill(

    "solid",

    fgColor=COR_AZUL_ESCURO
)


ws[
    "A10"
].font = Font(

    bold=True,

    color=COR_BRANCO,

    size=12
)


for i, texto in enumerate(
    diagnosticos,
    start=11
):

    ws.merge_cells(
        start_row=i,
        start_column=1,
        end_row=i,
        end_column=14
    )


    ws.cell(
        i,
        1
    ).value = texto


    ws.cell(
        i,
        1
    ).alignment = Alignment(

        wrap_text=True,

        vertical="top"
    )


    ws.cell(
        i,
        1
    ).fill = PatternFill(

        "solid",

        fgColor=(

            COR_CINZA_CLARO

            if i % 2 == 1

            else

            COR_BRANCO
        )
    )


# ======================================================================
# 38. SEÇÃO DE GRÁFICOS
# ======================================================================

ws.merge_cells(
    "A20:N20"
)


ws[
    "A20"
] = (
    "VISÃO EXECUTIVA"
)


ws[
    "A20"
].fill = PatternFill(

    "solid",

    fgColor=COR_AZUL_ESCURO
)


ws[
    "A20"
].font = Font(

    bold=True,

    color=COR_BRANCO,

    size=12
)


# ======================================================================
# 39. GRÁFICO 1 - ALAVANCAS
# ======================================================================

ws_op = wb[
    "Oportunidades"
]


chart1 = BarChart()

chart1.type = "bar"

chart1.style = 10

chart1.title = (
    "Distribuição das Gerações por Alavanca"
)

chart1.y_axis.title = (
    "Alavanca"
)

chart1.x_axis.title = (
    "% das Gerações"
)


# Descobre coluna Percentual_Geracoes
headers_op = {

    cell.value:
        cell.column

    for cell in ws_op[1]
}


col_pct = headers_op[
    "Percentual_Geracoes"
]


col_nome = headers_op[
    "Alavanca_Principal"
]


data = Reference(

    ws_op,

    min_col=col_pct,

    min_row=1,

    max_row=ws_op.max_row
)


cats = Reference(

    ws_op,

    min_col=col_nome,

    min_row=2,

    max_row=ws_op.max_row
)


chart1.add_data(

    data,

    titles_from_data=True
)


chart1.set_categories(
    cats
)


chart1.height = 8

chart1.width = 14

chart1.legend = None


ws.add_chart(

    chart1,

    "A21"
)


# ======================================================================
# 40. GRÁFICO 2 - CENÁRIOS
# ======================================================================

ws_cen = wb[
    "Cenarios"
]


headers_cen = {

    cell.value:
        cell.column

    for cell in ws_cen[1]
}


col_red = headers_cen[
    "Reducao_Total_Perc"
]


col_cen = headers_cen[
    "Cenario"
]


chart2 = BarChart()

chart2.type = "col"

chart2.style = 10

chart2.title = (
    "Potencial Analítico de Redução"
)

chart2.y_axis.title = (
    "% de Redução"
)


data = Reference(

    ws_cen,

    min_col=col_red,

    min_row=1,

    max_row=ws_cen.max_row
)


cats = Reference(

    ws_cen,

    min_col=col_cen,

    min_row=2,

    max_row=ws_cen.max_row
)


chart2.add_data(

    data,

    titles_from_data=True
)


chart2.set_categories(
    cats
)


chart2.height = 8

chart2.width = 12

chart2.legend = None


ws.add_chart(

    chart2,

    "H21"
)


# ======================================================================
# 41. GRÁFICO 3 - CAUSAS
# ======================================================================

ws_causas = wb[
    "Causas"
]


headers_causas = {

    cell.value:
        cell.column

    for cell in ws_causas[1]
}


col_epi = headers_causas[
    "Episodios"
]


col_hip = headers_causas[
    "Hipotese_Causa_Raiz"
]


max_causas = min(

    ws_causas.max_row,

    7
)


chart3 = DoughnutChart()

chart3.title = (
    "Hipóteses de Causa"
)


labels = Reference(

    ws_causas,

    min_col=col_hip,

    min_row=2,

    max_row=max_causas
)


data = Reference(

    ws_causas,

    min_col=col_epi,

    min_row=1,

    max_row=max_causas
)


chart3.add_data(

    data,

    titles_from_data=True
)


chart3.set_categories(
    labels
)


chart3.height = 8

chart3.width = 12


chart3.dataLabels = DataLabelList()

chart3.dataLabels.showPercent = True


ws.add_chart(

    chart3,

    "A37"
)


# ======================================================================
# 42. GRÁFICO 4 - TENDÊNCIA MENSAL
# ======================================================================

ws_mensal = wb[
    "Tendencia Mensal"
]


headers_m = {

    cell.value:
        cell.column

    for cell in ws_mensal[1]
}


if (
    "Geracoes_Efetivas"
    in headers_m

    and

    "Mes"
    in headers_m
):

    chart4 = LineChart()

    chart4.style = 10

    chart4.title = (
        "Evolução Mensal das Gerações Efetivas"
    )

    chart4.y_axis.title = (
        "Gerações"
    )

    chart4.x_axis.title = (
        "Mês"
    )


    data = Reference(

        ws_mensal,

        min_col=headers_m[
            "Geracoes_Efetivas"
        ],

        min_row=1,

        max_row=ws_mensal.max_row
    )


    cats = Reference(

        ws_mensal,

        min_col=headers_m[
            "Mes"
        ],

        min_row=2,

        max_row=ws_mensal.max_row
    )


    chart4.add_data(

        data,

        titles_from_data=True
    )


    chart4.set_categories(
        cats
    )


    chart4.height = 8

    chart4.width = 12


    ws.add_chart(

        chart4,

        "H37"
    )


# ======================================================================
# 43. SEÇÃO DE PRIORIDADES
# ======================================================================

ws.merge_cells(
    "A53:N53"
)


ws[
    "A53"
] = (
    "PRIORIDADES DE ATUAÇÃO"
)


ws[
    "A53"
].fill = PatternFill(

    "solid",

    fgColor=COR_AZUL_ESCURO
)


ws[
    "A53"
].font = Font(

    bold=True,

    color=COR_BRANCO,

    size=12
)


# ======================================================================
# 44. TOP 10 AÇÕES NO DASHBOARD
# ======================================================================

colunas_dashboard = [

    "Ranking_Acao",

    "itemName",

    "Categoria",

    "Mantenedor",

    "Tipo",

    "Torre",

    "Geracoes_Efetivas",

    "Indice_Acao",

    "Prioridade_Acao"
]


colunas_dashboard = [

    coluna

    for coluna in colunas_dashboard

    if coluna in top_acoes.columns
]


top10_dashboard = (

    top_acoes[
        colunas_dashboard
    ]
    .head(10)
)


linha_inicio = 55


for j, coluna in enumerate(

    top10_dashboard.columns,

    start=1
):

    cell = ws.cell(

        linha_inicio,
        j
    )


    cell.value = coluna


    cell.fill = PatternFill(

        "solid",

        fgColor=COR_AZUL
    )


    cell.font = Font(

        bold=True,

        color=COR_BRANCO
    )


    cell.alignment = Alignment(

        horizontal="center",

        vertical="center",

        wrap_text=True
    )


for i, row in enumerate(

    top10_dashboard.itertuples(
        index=False
    ),

    start=linha_inicio + 1
):

    for j, valor in enumerate(
        row,
        start=1
    ):

        cell = ws.cell(
            i,
            j
        )

        cell.value = valor

        cell.border = borda_fina

        cell.alignment = Alignment(

            vertical="top",

            wrap_text=True
        )


# ======================================================================
# 45. COLUNAS DO DASHBOARD
# ======================================================================

larguras_dashboard = {

    "A": 13,
    "B": 25,
    "C": 18,
    "D": 18,
    "E": 18,
    "F": 13,
    "G": 15,
    "H": 15,
    "I": 20,
    "J": 15,
    "K": 15,
    "L": 15,
    "M": 15,
    "N": 15
}


for coluna, largura in larguras_dashboard.items():

    ws.column_dimensions[
        coluna
    ].width = largura


# ======================================================================
# 46. ALTURA DAS LINHAS
# ======================================================================

ws.row_dimensions[1].height = 28
ws.row_dimensions[2].height = 28
ws.row_dimensions[3].height = 22


for linha in range(
    5,
    9
):

    ws.row_dimensions[
        linha
    ].height = 25


for linha in range(
    11,
    18
):

    ws.row_dimensions[
        linha
    ].height = 28


# ======================================================================
# 47. HEALTH SCORE - FORMATAÇÃO
# ======================================================================

ws_h = wb[
    "Health Score"
]


# Título adicional
ws_h.insert_rows(
    1,
    4
)


ws_h.merge_cells(
    "A1:F2"
)


ws_h[
    "A1"
] = (
    f"ALARM MANAGEMENT HEALTH SCORE – "
    f"{AMHS:.2f}/100 | {CLASSE_AMHS}"
)


ws_h[
    "A1"
].fill = PatternFill(

    "solid",

    fgColor=cor_amhs
)


ws_h[
    "A1"
].font = Font(

    bold=True,

    color=COR_BRANCO,

    size=16
)


ws_h[
    "A1"
].alignment = Alignment(

    horizontal="center",

    vertical="center"
)


ws_h.merge_cells(
    "A3:F3"
)


ws_h[
    "A3"
] = (
    "AMHS V1 – indicador analítico interno. "
    "Não representa índice oficial Johnson Controls / Metasys."
)


ws_h[
    "A3"
].font = Font(

    italic=True,

    color=COR_AZUL_ESCURO
)


ws_h[
    "A3"
].alignment = Alignment(

    horizontal="center"
)


# ======================================================================
# 48. TABELA DE FAIXAS DO AMHS
# ======================================================================

linha_faixas = (
    ws_h.max_row
    +
    3
)


ws_h.cell(
    linha_faixas,
    1
).value = (
    "Faixa"
)


ws_h.cell(
    linha_faixas,
    2
).value = (
    "Interpretação"
)


faixas_tabela = [

    (
        "80–100",
        "Muito Saudável"
    ),

    (
        "65–79,99",
        "Saudável"
    ),

    (
        "50–64,99",
        "Atenção"
    ),

    (
        "35–49,99",
        "Crítico"
    ),

    (
        "0–34,99",
        "Muito Crítico"
    )
]


for j in range(
    1,
    3
):

    ws_h.cell(
        linha_faixas,
        j
    ).fill = PatternFill(

        "solid",

        fgColor=COR_AZUL_ESCURO
    )

    ws_h.cell(
        linha_faixas,
        j
    ).font = Font(

        bold=True,

        color=COR_BRANCO
    )


for i, (
    faixa,
    descricao
) in enumerate(

    faixas_tabela,

    start=linha_faixas + 1
):

    ws_h.cell(
        i,
        1
    ).value = faixa


    ws_h.cell(
        i,
        2
    ).value = descricao


# ======================================================================
# 49. PLANO DE AÇÃO - STATUS
#
# Acrescentamos estrutura para uso posterior pela equipe
# ======================================================================

ws_plano = wb[
    "Plano de Acao"
]


headers_plano = {

    cell.value:
        cell.column

    for cell in ws_plano[1]
}


coluna_nova = (
    ws_plano.max_column
    +
    1
)


novas_colunas = [

    "Responsavel",

    "Prazo",

    "Status",

    "Resultado_Obtido",

    "Observacoes"
]


for i, coluna in enumerate(

    novas_colunas,

    start=coluna_nova
):

    ws_plano.cell(
        1,
        i
    ).value = coluna


    ws_plano.cell(
        1,
        i
    ).fill = PatternFill(

        "solid",

        fgColor=COR_AZUL_ESCURO
    )


    ws_plano.cell(
        1,
        i
    ).font = Font(

        bold=True,

        color=COR_BRANCO
    )


# ======================================================================
# 50. RECOMENDAÇÕES - DESTAQUE
# ======================================================================

ws_rec = wb[
    "Recomendacoes"
]


for row in ws_rec.iter_rows(
    min_row=2
):

    prioridade = (
        row[0].value
    )


    if prioridade in [
        1,
        2
    ]:

        cor = (
            COR_VERMELHO_CLARO
        )


    elif prioridade in [
        3,
        4
    ]:

        cor = (
            COR_AMARELO_CLARO
        )


    else:

        cor = (
            COR_VERDE_CLARO
        )


    for cell in row:

        cell.fill = PatternFill(

            "solid",

            fgColor=cor
        )


# ======================================================================
# 51. TOP AÇÕES - DESTAQUE
# ======================================================================

ws_top = wb[
    "Top Acoes"
]


headers_top = {

    cell.value:
        cell.column

    for cell in ws_top[1]
}


if (
    "Prioridade_Acao"
    in headers_top
):

    col_prioridade = (

        headers_top[
            "Prioridade_Acao"
        ]
    )


    for linha in range(
        2,
        ws_top.max_row + 1
    ):

        prioridade = str(

            ws_top.cell(
                linha,
                col_prioridade
            ).value
        )


        if prioridade.startswith(
            "A1"
        ):

            cor = (
                COR_VERMELHO_CLARO
            )


        elif prioridade.startswith(
            "A2"
        ):

            cor = (
                COR_AMARELO_CLARO
            )


        elif prioridade.startswith(
            "A3"
        ):

            cor = (
                COR_AZUL_CLARO
            )


        else:

            cor = (
                COR_VERDE_CLARO
            )


        for cell in ws_top[
            linha
        ]:

            cell.fill = PatternFill(

                "solid",

                fgColor=cor
            )


# ======================================================================
# 52. METODOLOGIA / DICIONÁRIO
# ======================================================================

ws_dic = wb.create_sheet(
    "Dicionario Executivo"
)


dicionario = [

    (
        "AMHS V1",
        (
            "Alarm Management Health Score analítico interno "
            "do projeto, variando de 0 a 100."
        )
    ),

    (
        "Chattering",
        (
            "Alternância rápida entre entrada e retorno "
            "de condição."
        )
    ),

    (
        "Reincidência",
        (
            "Nova geração do mesmo equipamento em curto intervalo."
        )
    ),

    (
        "Persistência",
        (
            "Ocorrências abertas, longas ou com dificuldade "
            "de normalização."
        )
    ),

    (
        "Alarm Flood",
        (
            "Concentração elevada de eventos em pequena "
            "janela temporal."
        )
    ),

    (
        "First-Out",
        (
            "Primeiro evento observado no início de uma cascata."
        )
    ),

    (
        "Precursor",
        (
            "Evento ou equipamento que aparece sistematicamente "
            "no início de cascatas."
        )
    ),

    (
        "Assinatura",
        (
            "Padrão recorrente formado por categoria, tipo, torre, "
            "classe do evento e velocidade de expansão."
        )
    ),

    (
        "Hipótese de Causa",
        (
            "Hipótese estatística e temporal para investigação; "
            "não representa diagnóstico físico comprovado."
        )
    ),

    (
        "Volume Exposto",
        (
            "Volume associado a uma alavanca de oportunidade. "
            "Não significa volume automaticamente eliminável."
        )
    ),

    (
        "Cenário Base",
        (
            "Estimativa analítica intermediária de potencial "
            "de redução após ações."
        )
    ),

    (
        "A1 / A2",
        (
            "Prioridades operacionais da carteira de ações."
        )
    )
]


ws_dic.append(
    [
        "Termo",
        "Definicao"
    ]
)


for item in dicionario:

    ws_dic.append(
        list(
            item
        )
    )


formatar_aba_dados(
    ws_dic
)


# ======================================================================
# 53. ABA DE METODOLOGIA DO AMHS
# ======================================================================

ws_met = wb.create_sheet(
    "Metodologia AMHS"
)


metodologia_amhs = [

    [
        "Componente",
        "Indicador",
        "Regra de Score",
        "Peso"
    ],

    [
        "Qualidade / Repetitividade",
        "% Chattering + Reincidência",
        "100 - exposição (%)",
        "25%"
    ],

    [
        "Persistência",
        "% volume Persistência / Normalização",
        "100 - exposição (%)",
        "20%"
    ],

    [
        "Alarm Flood",
        "% gerações em períodos de Flood",
        "100 - exposição (%)",
        "20%"
    ],

    [
        "Criticidade",
        "% gerações de P1 + P2",
        "100 - concentração (%)",
        "20%"
    ],

    [
        "Capacidade Diagnóstica",
        "% hipóteses Boa + Alta",
        "Percentual observado",
        "15%"
    ]
]


for linha in metodologia_amhs:

    ws_met.append(
        linha
    )


formatar_aba_dados(
    ws_met
)


# ======================================================================
# 54. ORDEM DAS ABAS
# ======================================================================

ordem_abas = [

    "Dashboard Executivo",

    "Health Score",

    "KPIs",

    "Oportunidades",

    "Cenarios",

    "Top Acoes",

    "Causas",

    "Assinaturas",

    "Floods",

    "Tendencia Mensal",

    "Diagnostico",

    "Recomendacoes",

    "Plano de Acao",

    "Metodologia AMHS",

    "Dicionario Executivo"
]


wb._sheets = [

    wb[
        nome
    ]

    for nome in ordem_abas

    if nome in wb.sheetnames
]


# ======================================================================
# 55. VALIDAÇÕES
# ======================================================================

print("\n" + "=" * 125)

print(
    "VALIDAÇÕES"
)

print("=" * 125)


print(
    f"AMHS....................................: "
    f"{AMHS:.2f}/100"
)


print(
    f"Classificação...........................: "
    f"{CLASSE_AMHS}"
)


print(
    f"Soma dos pesos.........................: "
    f"{(
        PESO_QUALIDADE
        +
        PESO_PERSISTENCIA
        +
        PESO_FLOOD
        +
        PESO_CRITICIDADE
        +
        PESO_DIAGNOSTICO
    ) * 100:.2f}%"
)


validacao_pesos = np.isclose(

    (
        PESO_QUALIDADE
        +
        PESO_PERSISTENCIA
        +
        PESO_FLOOD
        +
        PESO_CRITICIDADE
        +
        PESO_DIAGNOSTICO
    ),

    1.0
)


print(
    "Pesos do AMHS consistentes............:",
    validacao_pesos
)


validacao_amhs = (

    0
    <=
    AMHS
    <=
    100
)


print(
    "AMHS entre 0 e 100.....................:",
    validacao_amhs
)


print(
    f"Abas criadas............................: "
    f"{len(wb.sheetnames)}"
)


# ======================================================================
# 56. SALVAMENTO
# ======================================================================

wb.save(
    ARQUIVO_DASHBOARD
)


print("\n" + "=" * 125)

print(
    "MÓDULO 12 CONCLUÍDO COM SUCESSO"
)

print("=" * 125)


print(
    f"\nArquivo gerado:\n"
    f"{ARQUIVO_DASHBOARD}"
)


print(
    """
ESTRUTURA DO DASHBOARD:

1. Dashboard Executivo
   - Cards principais
   - Health Score
   - Diagnóstico automático
   - Gráficos de oportunidade
   - Cenários
   - Causas
   - Tendência mensal
   - Top 10 ações

2. Health Score
   - Componentes
   - Indicadores
   - Scores
   - Pesos
   - Contribuições

3. KPIs
   - Scorecard executivo

4. Oportunidades
   - Volume por alavanca

5. Cenários
   - Conservador
   - Base
   - Potencial

6. Top Ações
   - Carteira priorizada

7. Causas
   - Hipóteses de causa raiz

8. Assinaturas
   - Padrões recorrentes

9. Floods
   - Classificação dos episódios

10. Tendência Mensal

11. Diagnóstico

12. Recomendações

13. Plano de Ação
    - Responsável
    - Prazo
    - Status
    - Resultado
    - Observações

14. Metodologia AMHS

15. Dicionário Executivo
"""
)


# ======================================================================
# 57. DOWNLOAD
# ======================================================================

from google.colab import files


print(
    "\nIniciando download do Dashboard Executivo..."
)


files.download(
    ARQUIVO_DASHBOARD
)

MÓDULO 12 - DASHBOARD EXECUTIVO, HEALTH SCORE E RELATÓRIO GERENCIAL

Carregando dados consolidados...
Carteira de ações...............: 5,826
Assinaturas de causa............: 1,843
Episódios Flood.................: 1,843
Equipamentos ranking final......: 5,826

VALIDAÇÃO DOS PERCENTUAIS DO DASHBOARD


,Indicador,Percentual
0,Pré-Alarmes,46.54
1,Alarmes,37.69
2,Comunicação,15.77
3,P1 + P2,55.44
4,Alarm Flood,74.09
5,Confiança Diagnóstica,92.78
6,Chattering,44.39
7,Reincidência,9.77
8,Persistência,40.26
9,Comunicação / Automação,3.74



Percentuais entre 0 e 100...............: True

COMPOSIÇÃO DO AMHS CORRIGIDO


,Dimensao,Indicador_Base,Valor_Indicador_Perc,Score_0_100,Peso_Perc,Contribuicao_AMHS
0,Qualidade / Repetitividade,% volume Chattering + Reincidência,54.15,45.85,25.00,11.46
1,Persistência,% volume Persistência / Normalização,40.26,59.74,20.00,11.95
2,Alarm Flood,% gerações em períodos de Flood,74.09,25.91,20.00,5.18
3,Criticidade,% gerações de equipamentos P1 + P2,55.44,44.56,20.00,8.91
4,Capacidade Diagnóstica,% hipóteses com confiança Boa + Alta,92.78,92.78,15.00,13.92



Exposição Qualidade.....................: 54.15%
Score Qualidade.........................: 45.85
Score Persistência......................: 59.74
Score Alarm Flood.......................: 25.91
Score Criticidade.......................: 44.56
Score Capacidade Diagnóstica............: 92.78

AMHS V1.................................: 51.42/100
Classificação...........................: Atenção

Validações críticas concluídas com sucesso.

Construindo Dashboard Executivo...

VALIDAÇÕES
AMHS....................................: 51.42/100
Classificação...........................: Atenção
Soma dos pesos.........................: 100.00%
Pesos do AMHS consistentes............: True
AMHS entre 0 e 100.....................: True
Abas criadas............................: 15

MÓDULO 12 CONCLUÍDO COM SUCESSO

Arquivo gerado:
Dashboard_Executivo_Alarm_Management_Metasys_2026.xlsx

ESTRUTURA DO DASHBOARD:

1. Dashboard Executivo
   - Cards principais
   - Health Score
   - Diagnóstico automático
   - 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>